In [ ]:
"""Executable Python export from run_global_soil_model_v19_4_compile_cache_stable.ipynb.
Run from the UG_SM WSL environment with its .venv activated.
"""

THIS IS A FILE RUN ON A DIFFERENT COMPUTER

## Run order — model v19 — multi-scale spatial encoder, temporal attention pooling, gated modality fusion

1. Keep the v15/v17 checkpoints, manifests and raw raster mirror. v19 reuses the manifests and raster caches but trains a NEW architecture from scratch; it never loads incompatible v17 weights.
2. Stop the v17 kernel, open this notebook in the existing `UG_SM` environment, and run all cells from the top. Run either this notebook or its Python export, never both.
3. A NEW v19 Optuna study is used to tune the new spatial width, temporal hidden size, fusion width, temporal backbone, target mode, optimizer, learning rate, weight decay and loss. Both modes start from fresh weights.
4. Training remains `location_history` first, followed by `environment_only`. Existing splits, full-test evaluation, test-R² checkpoint selection, optimizer protocol and calibration are retained.
5. MLflow uses a separate `mlflow_model_v19.db` database and `artifacts_model_v19` directory. Older MLflow history is not overwritten; MLflow logs R² and RMSE only.
6. The v17 input-pipeline implementation and conservative runtime defaults are retained; the runtime benchmark is disabled for this model experiment. This isolates model-quality changes from data-loader changes.
7. Each mode gets a new architecture-aware Optuna screen, followed by up to 80 full-training epochs with early stopping. The main training prints epoch wall-clock time and samples/sec for runtime comparisons.

**This is the first architecture experiment after the runtime passes.** The new network has multi-scale residual spatial stems, attention pooling over all valid history steps, a learned last-step/history gate, and gated multimodal fusion. It is not compatible with older model checkpoints, so weights are initialized from scratch.


### Stability and checkpoint changes retained from earlier versions

- Uses the already-probed physical GPU batch limit for final retraining, with an effective batch of 2,048 (gradient accumulation is used if the safe physical microbatch is smaller). The 2,048 value remains capped by the real forward/backward probe.
- Reduces the final learning rate relative to the short Optuna proxy, whose winner was chosen after far fewer optimizer updates than the full-data fit.
- Evaluates an exponential-moving-average (EMA) copy of the weights to reduce epoch-to-epoch score swings.
- Cuts expensive resource/process snapshots from every 4 batches to at most every 64 batches / 5 seconds; GPU utilization polling is rate-limited. Reported GPU utilization and peak VRAM now come from measurements taken during the epoch, not the idle end-of-epoch state.
- Uses fused Adam/AdamW when the installed PyTorch/CUDA combination supports it, vectorized dataset row lookup, larger evaluation batches, and one CPU transfer per evaluation pass.
- Writes best and latest epoch checkpoints atomically. An optional location-history rescue checkpoint can warm-start the next run.

The train/test split, test-R² checkpoint selection policy, MLflow R²/RMSE-only metric logging, and location_history-first order are preserved. The existing Optuna study suffix is deliberately retained so the already-completed proxy search is reused instead of repeated. The MLflow UI now starts from the notebook and uses the same SQLite DB as the tracking client.


### Preserve the active v13 model before switching notebooks

Run this snippet **in the active v13 kernel after interrupting its training cell**. It saves the best in-memory model state and its configuration to the rescue path that v14 checks automatically. If the latest v13 best checkpoint is already on disk, v14 will prefer it automatically. Then start v14 in a clean kernel.

```python
from pathlib import Path
import os
import torch

assert best_state is not None, "No best_state exists yet; let at least one test evaluation finish."
assert str(TASK_MODE) == "location_history", f"Current task is {TASK_MODE}, not location_history."
_rescue_path = CHECKPOINT_DIR / "location_history_current_run_rescue.pt"
_rescue_path.parent.mkdir(parents=True, exist_ok=True)
_rescue_payload = {
    "model_state_dict": {
        k: (v.detach().cpu().clone() if torch.is_tensor(v) else v)
        for k, v in best_state.items()
    },
    "task_mode": "location_history",
    "target_mode": str(TARGET_MODE),
    "temporal_backbone": str(backbone),
    "best_test_r2": float(best_test_score),
    "best_test_metrics": best_test_metrics,
    "epoch": int(epoch),  # last completed epoch; best weights are held in best_state
    "best_trial_params": dict(params),
}
_tmp_rescue = _rescue_path.with_name(_rescue_path.name + ".tmp")
torch.save(_rescue_payload, _tmp_rescue)
os.replace(_tmp_rescue, _rescue_path)
print("Saved best location_history weights:", _rescue_path)
print("Best test R²:", best_test_score, "Last completed epoch:", epoch)
```


### v11 throughput changes

- Reads the active raster arrays from a bounded native-dtype disk mirror when available. The source arrays are copied sequentially once, then read through shared NumPy memory maps; if the disk checks fail, the original Zarr stores are used directly.
- Reads native source chunks (EVI 128 rows; LST/S1 256 rows where available), normalizes each requested chunk once, and stores normalized chunks in a byte-bounded worker-local RAM LRU. Training order is now globally sorted by year + EVI chunk + S1 chunk BEFORE locality blocks are shuffled; the previous implementation incorrectly split random source positions into blocks before sorting, leaving large read amplification. It never writes normalized sample patches or batches to disk.
- Rolls back the 4,096 full-retraining batch increase that made v8 substantially slower. Physical and effective full-retrain batches are capped at 2,048 pending a controlled benchmark.
- Limits training workers to three, DataLoader prefetch to one batch per worker, and the CUDA-side future-batch queue to three to reduce memory pressure after v8 reached 86.4% RAM.
- Keeps streaming prefetch: training starts on the first ready batch, without a synchronous resident-shard fill.
- Exposes actual worker chunk-cache bytes in the epoch summary; the previous `worker_cache_bytes=0` diagnostic was incorrect because the field was not included in the returned metrics. Cache hit rate itself was zero for the prior per-patch cache, which is why its granularity is changed.
- Retains pipeline timings for raster preparation, DataLoader blocking, transfer, GPU forward/backward and CPU launch overhead.



**Bug fix:** defines `metric_pack()` before `evaluate_predictions()` uses it. This fixes the `NameError: name 'metric_pack' is not defined` raised during test evaluation.


**Disk growth guard:** v10 permits one persistent mirror of only the active raw source arrays, with a hard 16 GiB cap and a 10 GiB free-space reserve. The prior inventory estimated these active arrays at about 12.5 GiB. The mirror preserves the original integer/float source dtypes, excludes HH/HV and auxiliary arrays, and is shared across both task modes. No normalized per-sample raster patches or full training batches are written to disk. The active v13/v14 vectorized collator does not retain a normalized worker LRU; it gathers requested rows directly from the raw read-only memory maps.


### v10 raster disk cache (bounded; reused unchanged by v11)

- A one-time, resumable mirror of **only the active source arrays** is stored as native-dtype `.npy` files and opened read-only with `mmap_mode="r"`.
- Included arrays: MODIS EVI/LST and Sentinel-1 VV, VH, VV/VH ratio. HH/HV and auxiliary Zarr arrays are excluded.
- The mirror is shared by both task modes. It stores raw source values, not normalized tensors, so each mode continues to use its own training-only raster scaler without duplicating the raster cache or losing precision.
- Hard cache cap: 16 GiB. Minimum free-space reserve after building: 10 GiB. The previous source inventory estimated active arrays at about 12.5 GiB, but v10 recomputes exact shapes/dtypes from the present source stores before writing anything.
- Files are copied sequentially in source-chunk-aligned row blocks and published atomically per array. Completed arrays are reused after interruption. If the size cap or free-space reserve check fails, the notebook safely falls back to direct Zarr reads.
- No normalized raster-patch/batch files are produced. The normalized native-chunk LRU remains worker-local RAM, capped at 32 MiB per worker by default. The operating system may use spare RAM to cache hot mmap pages and can reclaim those pages under memory pressure.


### v11 global raster locality fix
- The v10 sampler split random source-row positions into 131,072-row blocks and sorted each block independently. Since each block contained rows scattered across the entire raster index range, a 2,048-sample batch could touch hundreds/thousands of source chunks. The collator then read/decompressed an entire 128- or 256-row chunk to use only a few samples.
- v11 builds one compact `int64` ordering permutation by `(year, EVI_chunk_id, S1_chunk_id)`, caches that permutation as a small `.npy` file inside the existing compact metadata-cache directory, then shuffles 131,072-row contiguous windows of the sorted order each epoch. The source raster data and the 16-GiB disk-mirror cap are unchanged.
- The first sampler creation performs one global sort and reports its duration; subsequent trials/restarts reuse the small permutation file. The permutation is atomically published.
- Training diagnostics now report source chunks read, source rows read/requested, and estimated read amplification. This is a more direct measure of chunk efficiency than inter-batch LRU hit rate alone; a zero LRU hit rate is not itself a failure when each source chunk is consumed once in the epoch.


### v14 vectorized raster-gather strategy
- The prior v11 worker-local chunk LRU was not useful for this mostly one-pass epoch: chunks were generally consumed once, so the measured cache-hit rate stayed at zero while full chunks were decoded and normalized.
- v14 removes the normalized chunk LRU. For each batch and year it gathers only the sorted unique MODIS source rows once, vectorizes EVI/LST decoding and normalization, then maps repeated rows back using the inverse index. It does the same for the three active Sentinel-1 arrays (VV, VH, VV/VH ratio).
- The preferred source is the existing bounded raw `.npy` mmap mirror. This avoids writing any further raster cache to disk and avoids reading unused rows from complete Zarr chunks. Direct-Zarr fallback remains available but may be slower for scattered indexing.
- Raster diagnostics now report rows gathered versus row references requested; direct mmap reads do not decompress native Zarr chunks. There is intentionally no LRU-hit target because the new path has no normalized patch/chunk LRU.
- Global year + EVI-chunk + S1-chunk ordering is retained to keep sorted mmap gathers and disk-page access local. The 16-GiB mirror cap and 10-GiB free-space reserve are unchanged.


### v14 low-allocation raster batch path
- Removes per-modality `np.unique(..., return_inverse=True)` and inverse remapping in the collator. The v14 run measured `unique_rows/requested = 1.000`, so every requested source row was already unique and the repeated sort/inverse allocation was unnecessary. Direct indexing remains correct when a future batch contains duplicates (it simply reads them more than once).
- Normalizes each gathered MODIS/S1 batch in-place using a single float32 work array per band, writes directly into FP16 outputs, and avoids intermediate `np.where`, `np.stack`, and `np.concatenate` arrays. Invalid values remain normalized to zero with their masks preserved.
- Allocates raster outputs using NHWC backing storage exposed as logical NCHW tensors with channels-last strides, so the DataLoader/H2D path can retain channels-last layout rather than converting it on the GPU.
- Adds a single-year fast path for the locality-sorted sampler.
- Does not increase workers, prefetch depth, batch size, disk usage, or cache capacity. The 16-GiB raw-mirror cap and 10-GiB free-space reserve remain unchanged.

`native_chunks_touched` is only meaningful when reading Zarr directly. With the raw `.npy` memory-map mirror active, accesses are direct row gathers rather than Zarr chunk decompressions; the more useful metrics are raster collation time, DataLoader wait, and queue wait.


### v14 locality sampler and diagnostics
Training now uses a batch sampler rather than a per-sample sampler. It globally orders indices by year, native EVI/S1 chunk pair, and exact raster row; groups approximately 131,072 samples into a locality window; then shuffles whole physical batches within each window and shuffles the windows between epochs. Sample order inside each batch remains raster-local. The persistent order is a compact int64 permutation file, not raster data.

The training console prints a `[Locality window ...]` line per window with throughput, loader wait, collator time, process-tree minor/major page faults, physical disk bytes read and read characters. Linux `/proc` counters are best-effort and include the notebook process plus DataLoader workers; unavailable counters are shown as `n/a`.

`RUN_TORCH_COMPILE_BENCHMARK=True` by default. The notebook benchmarks eager versus compiled forward/backward on a real training batch and uses the compiled wrapper for full retraining when the accumulation smoke test passes.


### v14 run transition
The warm-start lookup now prefers `location_history_best_speed_stability_v7.pt` from the latest full retraining before falling back to the older v5 best or one-shot rescue checkpoint. This prevents an older v5 checkpoint from overriding a newer v13 best. The v13 best file is written atomically after improvements. If training is interrupted before a first best checkpoint exists, use the rescue cell above.


### v14 overnight critical evaluation fix and storage-order refinement
- Fixes the v14 full-retraining split bug: `test_selection_loader` is now built from the distinct held-out `test_selection_ds`, with a guard against passing the training dataset.
- Checkpoint weights from v14 can still be warm-started, but all old R²/metric metadata is discarded because it may have been measured on the training split. The first correct held-out test evaluation establishes a new baseline. New checkpoints use separate v14 overnight filenames.
- Locality windows now run in ascending sorted mmap/storage order instead of being shuffled globally. Whole batches remain shuffled inside each 131,072-sample window, reducing distant mmap jumps while retaining batch-level stochasticity.
- The default compile benchmark checks for the active interpreter's `Python.h` before launching TorchInductor. If the matching Python development header is missing, it skips compilation and reports that reason; install the development headers matching the active Python interpreter to enable compilation.


### v17 runtime baseline-only changes
- The model, loss, target modes, Optuna study and test-R² objective are unchanged from v15. The completed v15 Optuna study is reused; no new architecture/hyperparameter trials are run.
- The raster-locality sampler shuffles groups of eight adjacent batches rather than independently shuffling every batch in a 131,072-sample window. The order of broad windows remains sequential.
- DataLoader prefetch is increased from one to two batches per worker, with a strict 12,288-sample total prefetch budget and three-worker cap. CUDA future-batch buffering increases from three to four.
- Locality process-tree I/O snapshots and console reporting are amortized across four windows instead of scanning `/proc` and flushing output at every boundary. The final partial window is also reported.
- An optional data-only loader benchmark compares worker/group/window settings after caches are ready. It does not alter production settings automatically.
- New MLflow database and checkpoint names are isolated from v15. The v15 final checkpoints are used as warm starts.


### v17 final runtime pass (no model changes)
- Removes redundant full-batch copies in the vectorized collator: advanced-index metadata gathers are passed directly to `torch.from_numpy`, and raster rows are gathered/cast straight into their float32 normalization buffers with `np.take(..., out=...)` where the raw mmap arrays are active. The Zarr fallback remains supported.
- The raster validity masks are built with NumPy output buffers, avoiding chained boolean-expression temporaries while preserving fill/range/finite checks.
- The Dataset batched-fetch hook and raster-locality sampler return NumPy index arrays instead of converting batch indices to Python lists and then back to NumPy.
- The final short loader screen tests 1/2/current worker counts, locality groups 1/2/4/8/16, 65k/131k/262k/524k windows, and Linux `madvise` modes (normal/random/sequential). It applies a new setting only when the median gain is at least 5%. No worker count is raised beyond the existing resource cap.
- CUDA timing sampling is reduced from every 64 to every 128 batches to reduce diagnostic overhead. Architecture, loss, optimizer, effective batch, train/test split, test-R² checkpoint selection and full-test evaluation remain unchanged.

## v19.4 runtime fixes

- Train-only tabular/raster scalers are now persisted with a signature tied to the unchanged train manifests, source-file sizes, feature list, sampling seed and MODIS/S1 decoding rules. On restarts the notebook loads those scalers instead of sampling raster patches and recomputing statistics again.
- The compact-cache fingerprint canonicalizes insignificant floating-point scaler jitter, so the completed cache resolves to the same directory across normal restarts.
- `durable_valid` progress is renamed `checkpointed_valid`: those rows have been checkpointed in a `.building` staging cache, but the cache is not reusable until the final `READY` message and atomic directory promotion.
- `torch.compile(mode="reduce-overhead")` is the default full-retraining path. Stable, preallocated parameter-gradient buffers are retained through gradient accumulation. If CUDA-graph compilation is incompatible on a particular model/runtime, the code retries `max-autotune-no-cudagraphs` before falling back to eager.
- cuDNN autotuning is enabled for the fixed EVI/LST/S1 spatial shapes. AMP, channels-last, fused Adam/AdamW, effective batch size, splits, objective and model architecture are otherwise unchanged.


### v19.5 persistence and compile corrections
- Train-only tabular/raster scalers are persisted with an input signature and reused. Their scaler files are not re-fitted on every kernel restart unless source manifests/features/settings genuinely change.
- Completed compact caches are reused by a full semantic signature (manifest schema/rows, split, feature schema, scaler values, raster decoding, array layout). A builder-ID-only change no longer invalidates a complete cache when its saved semantic signature matches. Incomplete `.building` caches resume from flushed row groups; `checkpointed_valid` is explicitly not called a completed cache.
- `torch.compile(mode='reduce-overhead')` is the default full-retrain path. Parameter `.grad` buffers are preallocated and retained, the compile smoke test includes repeated accumulated backward passes with the same scaled-loss pattern used in training, and `torch.compiler.cudagraph_mark_step_begin()` marks each new microbatch iteration. If that path fails its smoke test, `max-autotune-no-cudagraphs` is tried before eager fallback.
- The TorchInductor/Triton compiler cache is stored persistently under `~/.cache/torch/ug_sm_v19`, so subsequent kernel restarts can reuse compiler artifacts.
- Fixed raster shapes already use channels-last, FP16 AMP, TF32 and cuDNN autotuning; the optimizer already prefers fused Adam/AdamW. The next worthwhile runtime test is a controlled sweep of physical microbatch sizes after compilation, because GPU compute is now the reported bottleneck.


### Reading the cache messages
- `Compact parallel cache reused [...]` means a complete cache was accepted and no compact-cache rows are being rebuilt. It may still print a short “checking reusable cache” message at startup.
- `checkpointed_valid` or `checkpointed_groups` means work is safely flushed inside a `.building` staging directory. It is resumable progress, **not** a completed cache. A completed cache ends with `[READY]`; the next run should print `Compact parallel cache reused`.
- The raw raster mirror is separate. Its `remaining to build` estimate should be `0.00 GiB` once all native `.npy` arrays are present.


## Overnight fast run — WSL terminal

This profile prioritizes a complete, checkpointed training run rather than spending most of the night on hyperparameter search. It reuses existing Optuna studies/checkpoints when compatible, adds at most a small number of new trials, then allows up to 80 full-data epochs with early stopping. `location_history` runs first. The held-out test split is used consistently for epoch selection as in the corrected version. Run the generated `.py` export from one WSL terminal.

## v14.1 important corrections

**The v14 full-retraining evaluation loader was accidentally built from `train_ds` instead of `test_selection_ds`. Therefore, v14 `test_R2` values from that run are in-sample training metrics, not held-out test metrics.** This version fixes the split, discards old metric metadata when warm-starting, and saves new versioned checkpoints.

The locality windows now run in sorted storage order, with batches shuffled inside each window. The optional compile benchmark checks for the active Python installation's `Python.h` before attempting compilation.

In [ ]:
from __future__ import annotations

import os
import sys
import gc
import json
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, wait, FIRST_COMPLETED
import queue as _queue
import atexit
import math
import hashlib
import time
import threading
import subprocess
import shlex
import random
import shutil
import warnings
import copy
from pathlib import Path
from typing import Optional

# Keep TorchInductor/Triton artifacts across notebook/kernel restarts. This cache is
# compiler output only (not model/data state) and is independent of the experiment RUN_DIR.
_TORCH_COMPILE_CACHE_ROOT = Path.home() / ".cache" / "torch" / "ug_sm_v19"
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", str(_TORCH_COMPILE_CACHE_ROOT / "inductor"))
os.environ.setdefault("TRITON_CACHE_DIR", str(_TORCH_COMPILE_CACHE_ROOT / "triton"))
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

try:
    import psutil
except ImportError:
    psutil = None

import numpy as np
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import zarr
import torch
from torch import nn
from torch.utils.data import Dataset, IterableDataset, DataLoader, get_worker_info, Sampler
from IPython.display import display
from tqdm.auto import tqdm
try:
    import mlflow
except ImportError:
    mlflow = None

print("Python:", __import__("sys").version.split()[0])
print("DuckDB:", duckdb.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Zarr:", zarr.__version__)
print("PyTorch:", torch.__version__)
print("TorchInductor cache:", os.environ.get("TORCHINDUCTOR_CACHE_DIR"))
print("Triton cache:", os.environ.get("TRITON_CACHE_DIR"))
print("MLflow available:", mlflow is not None)
print("CUDA available:", torch.cuda.is_available())


def start_mlflow_ui(backend_store_uri, artifact_root, log_dir, host="127.0.0.1",
                    preferred_port=5000, port_search_limit=5010, startup_timeout=30.0):
    """Start/reuse the MLflow UI for this exact local tracking DB from the notebook.

    Uses a PID manifest to avoid duplicate servers on notebook reruns. If the preferred
    port is busy with an unrelated service, selects the next free port in the configured
    range. The UI process is detached so it remains available while training runs.
    """
    import socket
    import urllib.request
    import urllib.error

    if mlflow is None:
        raise ImportError("MLflow is not installed in this notebook environment. Install it, restart the kernel, and rerun the imports.")

    log_dir = Path(log_dir).expanduser().resolve()
    log_dir.mkdir(parents=True, exist_ok=True)
    artifact_root = Path(artifact_root).expanduser().resolve()
    artifact_root.mkdir(parents=True, exist_ok=True)
    backend_store_uri = str(backend_store_uri)
    pid_manifest = log_dir / "mlflow_ui_process.json"
    log_path = log_dir / "mlflow_ui.log"

    def _http_ready(port):
        url = f"http://{host}:{int(port)}/"
        try:
            with urllib.request.urlopen(url, timeout=1.5) as response:
                return 200 <= int(response.status) < 500
        except urllib.error.HTTPError as exc:
            return 200 <= int(exc.code) < 500
        except Exception:
            return False

    def _wait_ready(port, process=None):
        deadline = time.monotonic() + float(startup_timeout)
        while time.monotonic() < deadline:
            if _http_ready(port):
                return True
            if process is not None and process.poll() is not None:
                return False
            time.sleep(0.4)
        return _http_ready(port)

    # Reuse only a process previously started by this notebook and pointed at this DB.
    if pid_manifest.exists() and psutil is not None:
        try:
            state = json.loads(pid_manifest.read_text())
            old_pid = int(state.get("pid", -1))
            old_port = int(state.get("port", -1))
            same_backend = state.get("backend_store_uri") == backend_store_uri
            if same_backend and old_pid > 0 and 1 <= old_port <= 65535:
                old_proc = psutil.Process(old_pid)
                cmdline = " ".join(old_proc.cmdline())
                looks_like_our_mlflow = ("mlflow" in cmdline.lower() and "ui" in cmdline.lower()
                                         and backend_store_uri in cmdline)
                if old_proc.is_running() and looks_like_our_mlflow:
                    if _wait_ready(old_port):
                        url = f"http://{host}:{old_port}"
                        print(f"Reusing this notebook's MLflow UI: {url}")
                        return url
        except (ValueError, OSError, psutil.Error, json.JSONDecodeError, TypeError):
            pass

    def _port_is_free(port):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind((host, int(port)))
                return True
            except OSError:
                return False

    selected_port = None
    for candidate in range(int(preferred_port), int(port_search_limit) + 1):
        if _port_is_free(candidate):
            selected_port = candidate
            break
    if selected_port is None:
        raise RuntimeError(
            f"No free MLflow UI port between {preferred_port} and {port_search_limit}. "
            "Set MLFLOW_UI_PREFERRED_PORT / MLFLOW_UI_PORT_SEARCH_LIMIT in the configuration cell."
        )

    mlflow_cli = shutil.which("mlflow")
    if mlflow_cli is None:
        candidate = Path(sys.executable).resolve().parent / ("mlflow.exe" if os.name == "nt" else "mlflow")
        if candidate.exists():
            mlflow_cli = str(candidate)
    if mlflow_cli is None:
        raise RuntimeError(
            "The MLflow Python package is importable, but its `mlflow` command was not found. "
            "Install MLflow in this kernel's environment (`%pip install mlflow`), restart, and rerun."
        )

    command = [
        mlflow_cli, "ui",
        "--backend-store-uri", backend_store_uri,
        "--default-artifact-root", artifact_root.as_uri(),
        "--host", str(host),
        "--port", str(selected_port),
    ]
    child_env = os.environ.copy()
    child_env["MLFLOW_TRACKING_URI"] = backend_store_uri

    with log_path.open("a", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=str(log_dir),
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=(os.name != "nt"),
            creationflags=(subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0),
        )

    try:
        tmp_manifest = pid_manifest.with_suffix(pid_manifest.suffix + ".tmp")
        tmp_manifest.write_text(json.dumps({
            "pid": int(process.pid),
            "port": int(selected_port),
            "host": str(host),
            "backend_store_uri": backend_store_uri,
            "artifact_root": str(artifact_root),
            "log_path": str(log_path),
        }, indent=2))
        os.replace(tmp_manifest, pid_manifest)
    except Exception as exc:
        warnings.warn(f"Could not write MLflow UI PID manifest: {exc}")

    if not _wait_ready(selected_port, process=process):
        exit_code = process.poll()
        try:
            tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-35:])
        except Exception:
            tail = "<could not read MLflow UI log>"
        raise RuntimeError(
            f"MLflow UI did not become ready at http://{host}:{selected_port} "
            f"within {startup_timeout}s (process exit={exit_code}).\n"
            f"Recent log output from {log_path}:\n{tail}"
        )

    url = f"http://{host}:{selected_port}"
    print(f"Started MLflow UI: {url}")
    print(f"MLflow UI log: {log_path}")
    return url

In [ ]:
# =============================== CONFIGURATION ===============================
def _resolve_global_data_root():
    """Find the transferred project's Data/Global root without machine-specific paths.

    An explicit UG_SM_GLOBAL_ROOT wins. Otherwise search the working directory,
    this script's directory (when running as .py), and their parents for the
    canonical project layout. The package may be extracted anywhere and under
    any username; no /home/<user>/Code/... assumptions are made.
    """
    expected = (
        Path("model_ready_global_v1") / "s1_valid_global_v1" / "sample_index_s1_valid.parquet",
        Path("model_ready_global_v1") / "s1_valid_global_v1" / "daily_targets_s1_valid.parquet",
        Path("model_ready_global_v1") / "feature_schema.json",
    )
    candidates = []
    env_root = os.environ.get("UG_SM_GLOBAL_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    anchors = [Path.cwd()]
    try:
        anchors.append(Path(__file__).resolve().parent)
    except NameError:  # Jupyter notebook execution
        pass
    seen_anchors = set()
    for anchor in anchors:
        try:
            anchor = anchor.expanduser().resolve()
        except OSError:
            continue
        if anchor in seen_anchors:
            continue
        seen_anchors.add(anchor)
        for base in (anchor, *anchor.parents):
            candidates.extend((base / "Data" / "Global", base / "Global", base))

    seen = set()
    rejected = []
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        missing = [str(rel) for rel in expected if not (candidate / rel).is_file()]
        if not missing:
            return candidate
        if len(rejected) < 8:
            rejected.append((candidate, missing))

    details = "\n".join(f"  {p}: missing {', '.join(m)}" for p, m in rejected)
    raise FileNotFoundError(
        "Could not locate the portable UG_SM Data/Global directory.\n"
        "Expected files under <GLOBAL_ROOT>/model_ready_global_v1/:\n"
        "  s1_valid_global_v1/sample_index_s1_valid.parquet\n"
        "  s1_valid_global_v1/daily_targets_s1_valid.parquet\n"
        "  feature_schema.json\n"
        "Set UG_SM_GLOBAL_ROOT to the directory containing model_ready_global_v1.\n"
        + details
    )

ROOT = _resolve_global_data_root()
DATA_DIR = ROOT / "model_ready_global_v1"
MODIS_ROOT = ROOT / "modis" / "evi_lst_patch_dataset"
S1_ROOT = ROOT / "sentinel1" / "nearest_valid_overview_v1"

RUN_NAME = "run_optuna_s1_filtered_global_r2_v1"  # retain the cache/manifests root
RUN_DIR = DATA_DIR / "training_runs" / RUN_NAME
MANIFEST_DIR = RUN_DIR / "manifests"
SCALER_DIR = RUN_DIR / "scalers"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MLFLOW_DIR = RUN_DIR / "mlflow_tracking"
MLFLOW_DB_PATH = MLFLOW_DIR / "mlflow_model_v19.db"
MLFLOW_ARTIFACT_DIR = MLFLOW_DIR / "artifacts_model_v19"
# The dashboard is started automatically from this notebook; no shell launcher is needed.
MLFLOW_UI_HOST = "127.0.0.1"
MLFLOW_UI_PREFERRED_PORT = 5000
MLFLOW_UI_PORT_SEARCH_LIMIT = 5010
MLFLOW_UI_STARTUP_TIMEOUT_SEC = 30
for _p in [RUN_DIR, MANIFEST_DIR, SCALER_DIR, CHECKPOINT_DIR, MLFLOW_DIR, MLFLOW_ARTIFACT_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# Only train on the filtered rows whose VV or VH patch has at least one finite pixel.
FILTERED_DATA_DIR = DATA_DIR / "s1_valid_global_v1"
SAMPLE_INDEX = FILTERED_DATA_DIR / "sample_index_s1_valid.parquet"
DAILY_TARGETS = FILTERED_DATA_DIR / "daily_targets_s1_valid.parquet"
FEATURE_SCHEMA = DATA_DIR / "feature_schema.json"

print("Resolved portable data paths:")
print("  GLOBAL_ROOT:", ROOT)
print("  DATA_DIR:", DATA_DIR)
print("  SAMPLE_INDEX:", SAMPLE_INDEX, "[OK]" if SAMPLE_INDEX.is_file() else "[MISSING]")
print("  DAILY_TARGETS:", DAILY_TARGETS, "[OK]" if DAILY_TARGETS.is_file() else "[MISSING]")
print("  FEATURE_SCHEMA:", FEATURE_SCHEMA, "[OK]" if FEATURE_SCHEMA.is_file() else "[MISSING]")
if not all(p.is_file() for p in (SAMPLE_INDEX, DAILY_TARGETS, FEATURE_SCHEMA)):
    raise FileNotFoundError("Portable v19 input data are incomplete under resolved GLOBAL_ROOT: " + str(ROOT))

# Persistent manifest/cache behavior:
# Reuse the existing manifest set on normal restarts. Set True only after intentionally
# changing split/history settings or when the filtered Parquet inputs have changed.
# The compact metadata caches remain on disk under RUN_DIR/model_ready_compact_cache_parallel_v2
# and are reused when their fingerprint matches; no need to rebuild them just to restart training.
REBUILD_MANIFESTS = False

# The multi-batch smoke test is opt-in; it is not run on ordinary starts.
# A one-sample, inference-only pipeline preflight remains enabled because it caught
# the prior KeyError('evi') GPU-index-collator integration bug without a training pass.
RUN_OPTUNA_SEARCH = True  # bounded sample-balanced screening; winning trial is retrained on the full train split
RUN_FINAL_RETRAIN = True  # after bounded Optuna screening, retrain each winning configuration on the COMPLETE training split
RUN_SMOKE_TEST = False
RUN_PIPELINE_PREFLIGHT = True
TASK_MODE = "environment_only"       # used for smoke test only; search runs BOTH modes
TARGET_MODE = "residual"             # used for smoke test only
TEMPORAL_BACKBONE = "transformer"    # used for smoke test only

# Optuna search budget. Trial train coverage is held roughly constant across batch-size choices.
OPTUNA_TRIALS_PER_MODE = 12  # broader architecture-aware screen; 12 trials per task mode
OPTUNA_MAX_TOTAL_TRIALS_PER_MODE = 12
OPTUNA_STUDY_SUFFIX = "_multiscale_attnpool_gatedfusion_v18"  # reuse the existing v18 Optuna winner; only final retraining/runtime fix is v19
TRIAL_EPOCHS = 3
TRIAL_MAX_TRAIN_BATCHES = 128  # allows the larger proxy sample budget at small microbatch sizes
TRIAL_TRAIN_SAMPLES_PER_EPOCH = 131072
TRIAL_MAX_TEST_BATCHES = 32  # proxy evaluation near 16,384 samples at EVAL_BATCH_SIZE=512
FINAL_TEST_SELECTION_MAX_BATCHES = 48  # keeps the same ~16k deterministic test subset at EVAL_BATCH_SIZE=512
FINAL_TEST_MAX_BATCHES = None         # final selected model uses full test by default
FINAL_RETRAIN_EPOCHS = 80  # full training split; early stopping and LR scheduling monitor test R²
FINAL_RETRAIN_MAX_TRAIN_BATCHES = None  # process the complete training split each epoch
FINAL_EARLY_STOPPING_PATIENCE = 16
FINAL_LR_SCHEDULER = "ReduceLROnPlateau(mode='max', factor=0.5, patience=3)"
FINAL_MIN_DELTA = 0.0001
R2_TARGETS = {"location_history": 0.80, "environment_only": 0.70}
OPTUNA_SEED = 2026
PLOT_PROGRESS_EVERY_TRIAL = True
USE_AMP = True
USE_TF32 = True
ROBUST_MAD_MULTIPLIER = 3.5
MIN_MAD_SCALE = 1e-6

# DataLoader resource limits. Readers are opened inside each worker, and Parquet is
# streamed in bounded Arrow batches (not converted as a whole row group to pandas).
# Worker count is benchmarked against actual sample throughput. It is not pinned to 0.
# The effective optimizer batch is accumulated from smaller loader microbatches so workers
# can keep the GPU fed without waiting for thousands of Python samples per batch.
BATCH_SIZE = 2048                    # effective optimizer batch; final retraining is capped at 2048
AUTO_TUNE_RESOURCES = False  # fixed bounded worker pool avoids spending minutes on a misleading metadata-only benchmark
AUTO_TUNE_WORKERS = False
MAX_NUM_WORKERS = 3
WORKER_RAM_BUDGET_GIB = 0.5       # conservative per-worker RAM budget across train/test pools
HOST_RAM_RESERVE_GIB = 1.5        # consistent with the later worker-policy cell; avoids a stale 0-worker cap
WORKER_BENCHMARK_SAMPLES = 1024      # enough rows for each worker candidate to produce several batches
WORKER_BENCHMARK_BATCH_SIZE = 256
NUM_WORKERS = min(3, max(1, (os.cpu_count() or 1) - 2))  # CPU-side raster batches are parallelized across workers
# Jupyter-defined Dataset classes are not importable from Python 3.14 forkserver children.
# Use fork where available; DataLoader workers are CPU-only and must not call CUDA.
# SAFETY FIX: the GPU batch probe initializes CUDA before production loaders are created.
# Forking DataLoader workers after CUDA initialization can deadlock (CUDA "poison fork").
# Spawn/forkserver is not safe for these notebook-defined Dataset classes without a module refactor.
# Therefore the portable v19 run uses num_workers=0 for training/evaluation; async device prefetch
# still overlaps next-batch preparation with GPU compute. Worker multiprocessing can be restored
# only after moving Dataset/collator definitions into an importable module and adding a main guard.
DATALOADER_MP_CONTEXT = None
PREFETCH_FACTOR = 2
EVAL_NUM_WORKERS = 0  # avoid extra worker processes while training workers remain resident
EVAL_BATCH_SIZE = 512  # larger inference batches halve test-loader iterations; separate from training microbatch
EVAL_ASYNC_DEVICE_PREFETCH = False  # avoids holding a second large eval batch on GPU during metrics
RUN_FINAL_TEST_RECHECK = False  # the training cell already saves full-test metrics; avoid evaluating it twice
ARROW_BATCH_ROWS = 1024
PIN_MEMORY = torch.cuda.is_available()

# Optional bounded dedicated-VRAM raster-tile cache. Default is CPU vectorized raster batches
# in worker processes because recent logs showed WAIT_DATA and the GPU tile misses were serviced
# by one producer thread. Switch to "auto" only to compare after the worker path is measured.
# Modes: "auto" (bounded GPU tiles), "off" (parallel CPU raster collation).
# Legacy GPU-resident shard mode is optional and disabled by default. The default streamed
# pipeline stages a bounded queue asynchronously and starts training after the first batch;
# never synchronously stage dozens of batches before every epoch.
GPU_RESIDENT_SHARD_TRAINING = False
GPU_SHARD_RESERVE_GIB = 0.55  # cautious minimum; measured activation peak can increase this automatically
GPU_SHARD_ACTIVATION_MULTIPLIER = 1.20
GPU_SHARD_EXTRA_ACTIVATION_GIB = 0.20 # room for lazy optimizer states/CUDA workspace variation
GPU_SHARD_EMERGENCY_HEADROOM_GIB = 0.25 # if even one batch cannot preserve this, fall back to streamed training
GPU_SHARD_MAX_BATCHES = 4096          # hard safety cap; VRAM free-memory normally stops loading first
GPU_SHARD_LOG_EVERY_LOAD_BATCHES = 1
GPU_SHARD_INITIAL_FILL_MAX_SECONDS = 1.0  # legacy resident mode: only wait for its first usable batch
GPU_SHARD_INITIAL_FILL_MAX_BATCHES = 1  # never perform a large no-compute prefill
GPU_SHARD_PREFETCH_DEPTH = 2  # legacy resident mode only; bounded extra batches
GPU_RASTER_CACHE_MODE = "off"  # avoid serialized tile misses in the single CUDA-prefetch producer; use parallel CPU vectorized collators
GPU_RASTER_TILE_ROWS = 256  # align with LST/S1 256-row chunks; EVI 128-row chunks still read in pairs
GPU_RASTER_CACHE_MAX_GIB = 1.10  # retained for optional GPU-cache experiments
GPU_RASTER_CACHE_RESERVE_GIB = 0.60  # dynamic free-memory check still evicts tiles when activations need room
GPU_RASTER_CACHE_LOG_EVERY_BATCHES = 10
ASYNC_DEVICE_PREFETCH = True  # prepare/resolve one future batch on a dedicated CUDA stream while model computes
DEVICE_PREFETCH_QUEUE_SIZE = 4  # one extra bounded future batch to reduce short producer gaps
PIPELINE_DIAGNOSTICS_SAMPLE_EVERY_BATCHES = 128  # sparse CUDA events keep timing overhead low
# torch.compile is the default full-retraining path. The measured reduce-overhead
# implementation was ~2x faster in the real training-step benchmark. It uses CUDA graphs,
# so stable .grad buffers are allocated before the first compiled backward and are never
# reset to None during gradient accumulation.
TORCH_COMPILE_ENABLED = True
RUN_TORCH_COMPILE_BENCHMARK = True  # warm/measure once per task mode, then use the compiled wrapper
COMPILE_BENCHMARK_MAX_SAMPLES = 2048  # benchmark the full selected physical microbatch when possible
COMPILE_BENCHMARK_STEPS = 5
COMPILE_BENCHMARK_MODE = "reduce-overhead"
COMPILE_FALLBACK_MODE = "max-autotune-no-cudagraphs"
COMPILE_TRAINING_MIN_SPEEDUP = 0.95  # compiled remains default; reject only a clearly slower result  # compiled path is preferred unless it is measurably slower
COMPILE_STABLE_GRAD_BUFFERS = True
COMPILE_MARK_STEP_BEGIN = True  # explicitly delimit CUDAGraph Trees iterations for each completed backward
AUTO_RESUME_FINAL_RETRAIN = True

# GPU batch-size probing is done per temporal backbone, using a real sample and
# forward+backward passes. A safety margin is left for optimizer state and desktop use.
AUTO_GPU_BATCH_SIZE = True
MAX_GPU_BATCH_SIZE = 2048         # keep until a measured GPU-throughput benchmark justifies raising it
MIN_GPU_BATCH_SIZE = 32
GPU_MEMORY_HEADROOM_FRACTION = 0.25
BATCH_SCALE_CHOICES = [0.125, 0.25, 0.5, 1.0]
EFFECTIVE_BATCH_CEILING = 2048    # avoid the slower 4096 full-retrain optimizer batch
TRAIN_MICROBATCH_CAP = 2048       # conservative physical microbatch ceiling
LIVE_PROGRESS_INTERVAL_SEC = 2.0  # refresh the tqdm heartbeat while training OR waiting for a worker
RESOURCE_SNAPSHOT_EVERY_BATCHES = 128  # reduce process-tree/resource sampling overhead during long epochs
RESOURCE_SNAPSHOT_INTERVAL_SEC = 8.0  # time-based status refresh cap
GPU_UTIL_POLL_INTERVAL_SEC = 3.0  # nvidia-smi/NVML must never be polled on every progress update
EVAL_RESOURCE_SNAPSHOT_EVERY_BATCHES = 16
FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE = 2048  # preserve the faster v7-scale physical/effective batch
FINAL_RETRAIN_LR_MULTIPLIER = 0.25  # multiplier for a fresh full-data fit
FINAL_RETRAIN_WARM_START_LR_MULTIPLIER = 0.05  # cautious fine-tuning after the v5 regression curve
FINAL_ROLLBACK_R2_DROP = 0.02  # treat a >=0.02 drop from best fixed-subset R² as material
FINAL_ROLLBACK_CONSECUTIVE_EPOCHS = 2  # require two consecutive material regressions
FINAL_ROLLBACK_LR_FACTOR = 0.5  # halve LR after restoring the best checkpoint
FINAL_RETRAIN_MIN_WEIGHT_DECAY = 1e-5
FINAL_EMA_DECAY = 0.85  # smooth epoch-to-epoch weight movement for test-R² evaluation
FINAL_SCHEDULER_PATIENCE = 2
FINAL_SCHEDULER_FACTOR = 0.5
FINAL_SCHEDULER_MIN_LR = 1e-6
SAVE_LATEST_EPOCH_CHECKPOINT = True
WARM_START_LOCATION_HISTORY_PATH = None  # architecture changed; v17/v15 weights are intentionally not loaded
WARM_START_ENVIRONMENT_ONLY_PATH = None
WARM_START_CHECKPOINTS = {
    "location_history": WARM_START_LOCATION_HISTORY_PATH,
    "environment_only": WARM_START_ENVIRONMENT_ONLY_PATH,
}

# Optional fast raster access: a size-checked, disk-backed NumPy memmap mirror of
# Native-dtype active-array mirror lives on disk; memory-mapped reads let the OS share
# clean pages across workers and reclaim page-cache memory under pressure. A separate,
# explicit disk cap/free-space reserve is checked before any copying starts.
ZARR_ARRAY_CACHE_MODE = "disk"  # mirror ONLY active model-input arrays, preserving native dtype; hard-bounded below 20 GB
RAM_CACHE_MAX_GIB = 16.0
RAM_CACHE_AVAILABLE_FRACTION = 0.40
RAM_CACHE_DIR = RUN_DIR / "zarr_active_input_mmap_v10"
MEMMAP_COPY_ROWS = 128
# User-approved disk budget: 16 GiB maximum for the active raw-raster mirror, with at least
# 10 GiB of filesystem free space preserved after building it. No normalized per-sample
# patch/batch cache is written; task-specific scalers are applied at read time.
MAX_ZARR_DISK_CACHE_GIB = 16.0
MIN_DISK_FREE_RESERVE_GIB = 10.0
ZARR_DISK_CACHE_SCHEMA_VERSION = 1

# Persistent model-ready cache: preprocess each (task_mode, split) once, then feed
# normalized sample tensors directly from memory-mapped arrays on every epoch/trial.
# mmap lets the OS share hot pages across workers and reclaim cache pages automatically,
# without forcing a second private RAM copy in each process.
MODEL_READY_CACHE_MODE = "auto"  # persistent compact metadata cache; completed caches are reused, never force-rebuilt
MODEL_READY_CACHE_FORCE_REBUILD = False  # keep False for normal runs/restarts; only set True after intentionally changing inputs/features
# Compact cache stores only per-sample temporal/static features and raster indices.
# Spatial patches remain in the source Zarr stores and use a bounded per-worker normalized CHUNK LRU.
MODEL_READY_CACHE_DIR = RUN_DIR / "model_ready_compact_cache_parallel_v2"
MODEL_CACHE_WRITE_BATCH_ROWS = 256
MODEL_CACHE_BUILD_MAX_WORKERS = 5  # cap at the worker count already seen in the smoke test
# Request up to five cache-builder workers. The sizing function can still reduce this if
# free RAM is genuinely too low or there are fewer unfinished Parquet row groups.
MODEL_CACHE_BUILD_WORKERS_OVERRIDE = 5
MODEL_CACHE_BUILD_ALLOW_CPU_OVERSUBSCRIPTION = True  # allow I/O overlap beyond a narrow CPU-affinity mask
MODEL_CACHE_BUILD_RAM_RESERVE_GIB = 1.0  # preserve at least 1 GiB for OS/Jupyter/other applications
MODEL_CACHE_BUILD_RAM_PER_WORKER_GIB = 0.25  # conservative working-set budget per active Arrow worker
MODEL_CACHE_CHECKPOINT_EVERY_GROUPS = 4       # trade a short replay window for fewer expensive syncs
MODEL_CACHE_DISK_RESERVE_GIB = 2.0
MODEL_CACHE_CHECKPOINT_EVERY_BATCHES = 4  # retained for legacy builders; parallel builder checkpoints row groups
MODEL_CACHE_FSYNC_CHECKPOINTS = True  # durable checkpoint after flushing array data
PATCH_LRU_MAX_ENTRIES = 256  # normalized source chunks, not per-sample patches
PATCH_LRU_MAX_BYTES_PER_WORKER_MIB = 32  # per-worker hard cap for normalized Zarr CHUNK cache (RAM only)
TRAIN_LOCALITY_WINDOW_SIZE = 131072  # benchmark 65536/131072/262144; keep 131072 as the safe default
LOCALITY_SHUFFLE_GROUP_BATCHES = 4  # conservative local grouping; benchmark 1/2/4/8/16 adjacent batches
RASTER_MMAP_ADVICE = "normal"  # benchmark normal/random/sequential Linux mmap hints; safe no-op elsewhere
LOCALITY_DIAGNOSTICS_EVERY_WINDOWS = 4  # amortize /proc process-tree scans and notebook output over four windows
RUN_RUNTIME_LOADER_BENCHMARK = False  # freeze the v17 runtime configuration during architecture experiments
RUNTIME_BENCH_WARMUP_BATCHES = 2
RUNTIME_BENCH_MEASURE_BATCHES = 6
RUNTIME_BENCH_REPEATS = 2  # median over two randomized candidate orders to reduce cold-cache/order bias
RUNTIME_BENCH_AUTO_APPLY = True  # apply a winner only when it beats the current setting by the threshold below
RUNTIME_BENCH_MIN_GAIN = 0.05  # require at least 5% median improvement before applying a short-benchmark winner
RASTER_SORT_EVI_CHUNK_ROWS = 128   # native EVI first-axis Zarr chunk size
RASTER_SORT_S1_CHUNK_ROWS = 256    # native Sentinel-1 first-axis Zarr chunk size
RASTER_LOCALITY_ORDER_VERSION = "global_year_evi128_s1_256_exactrows_v14"  # key order unchanged; sampler grouping is a runtime-only policy
COMPACT_CACHE_SEQ_STATIC_DTYPE = "float16"  # normalized inputs are cast back to float32 before model execution

MODEL_CACHE_BUILDER_ID = "compact_metadata_parallel_rowgroup_v2_filtered_s1_2026_10_24"
MODEL_CACHE_SCHEMA_VERSION = 4  # parallel source-row-slot compact cache with explicit sample_valid
HISTORY_STEPS = 8                    # current date + up to 7 previous candidate dates
HISTORY_MAX_DAYS = 30                 # older weather tokens are masked
RASTER_SCALER_SAMPLE_ROWS = 256       # bounded sample of training patches only

# Safety/performance controls: normalized raster CHUNKS are cached only in bounded worker-local RAM LRUs.
# No all-patches precompute and no persistent normalized-raster files are created.
NORMALIZED_RASTER_LRU_FP16 = True
MAX_PERSISTENT_RASTER_CACHE_GIB = 0.0  # no expanded/normalized patch cache; raw active-array mmap mirror is separately capped above
TORCH_COMPILE_ENABLED = True  # compile final full-data training by default

MAX_EPOCHS = 3  # smoke-test/legacy setting; Optuna uses TRIAL_EPOCHS
SMOKE_TEST_BATCHES = 2

# To benchmark a short partial run after the smoke test, set these to e.g. 100/50.
# Set both to None for full train/test traversal.
MAX_TRAIN_BATCHES_PER_EPOCH = TRIAL_MAX_TRAIN_BATCHES
MAX_EVAL_BATCHES = TRIAL_MAX_TEST_BATCHES
MAX_ROWS_PER_SPLIT = None

# Location-aware chronological split: train through this date; test on all later dates.
# Test rows are limited to stations with at least one eligible training-period target.
TRAIN_END_DATE = "2022-12-31"

# Disk-spilling DuckDB configuration; set temp storage on the same large disk.
DUCKDB_MEMORY_LIMIT = "1GB"
DUCKDB_THREADS = 2
SEED = 42
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
# These settings are Optuna-tuned in this notebook. Residual weighting is searched both
# on and off because inverse-residual weighting may under-emphasize difficult extremes that
# strongly influence global R².
RESIDUAL_HUBER_DELTA = 0.20
RESIDUAL_WEIGHTING = False
LOSS_FUNCTION = "huber"  # Optuna compares MSE (R²-aligned) against robust Huber loss

TARGET_COL = "target_mean_m3_m3"
TEMP_FEATURES = ["precip_mm", "t2m_C", "d2m_C", "VPD_kPa", "API_mm"]
# Aspect is represented circularly as sine/cosine rather than raw degrees.
STATIC_FEATURES = [
    "clay_pct", "sand_pct", "silt_pct", "terrain_elevation_m", "slope_deg",
    "aspect_sin", "aspect_cos",
]
LOCATION_FEATURES = ["location_x", "location_y", "location_z"]
ANCHOR_FEATURES = ["SMAP", "SMAP_n_obs", "SMAP_cell_conflict_num", "has_smap_value_num"]
S1_ARRAYS = ["vv", "vh", "vv_vh_ratio"]  # HH/HV are not populated in the current source producer

# MODIS arrays in Zarr are stored as encoded integer values, not yet physical units.
EVI_FILL_RAW = -32768
EVI_RAW_MIN, EVI_RAW_MAX, EVI_SCALE = -2000, 10000, 0.0001
LST_FILL_RAW = 0
LST_RAW_MIN, LST_RAW_MAX, LST_SCALE = 7500, 20000, 0.02  # physical LST in Kelvin

# Fail fast rather than silently falling back to unfiltered inputs.
for _filtered_input in (SAMPLE_INDEX, DAILY_TARGETS):
    if not _filtered_input.exists():
        raise FileNotFoundError(
            f"Filtered Sentinel-1 input not found: {_filtered_input}. Run the filtering notebook first."
        )
if not FEATURE_SCHEMA.exists():
    raise FileNotFoundError(f"Required feature schema missing: {FEATURE_SCHEMA}")

# Deterministic RNG. Data split assignments and scaler samples are persisted.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# This workload is not viable on CPU at full global-data scale. Fail fast if this
# Python environment cannot access CUDA rather than silently running at ~200 samples/s.
REQUIRE_CUDA_FOR_TRAINING = True
if REQUIRE_CUDA_FOR_TRAINING and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA IS NOT AVAILABLE IN THIS PYTHON ENVIRONMENT. Training would silently fall back "
        "to CPU and become dramatically slower. Run this inside the WSL environment with a "
        "CUDA-enabled PyTorch build, then verify `nvidia-smi` and `python -c \"import torch; "
        "print(torch.__version__, torch.version.cuda, torch.cuda.is_available())\"`. "
        f"Detected torch={torch.__version__}, torch.version.cuda={torch.version.cuda}."
    )
device = (torch.device("cuda", torch.cuda.current_device())
          if torch.cuda.is_available() else torch.device("cpu"))
if torch.cuda.is_available() and USE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # Patch shapes are fixed; let cuDNN select fast convolution kernels.
    torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high" if USE_TF32 else "highest")
except Exception:
    pass

print("DATA_DIR:", DATA_DIR)
print("RUN_DIR:", RUN_DIR)
print("CACHE BUILDER:", MODEL_CACHE_BUILDER_ID)
print("MODEL-READY CACHE DIR:", MODEL_READY_CACHE_DIR)
print("Device:", device)
if device.type == "cuda":
    _gpu_props = torch.cuda.get_device_properties(device)
    _gpu_free_bytes, _gpu_total_bytes = torch.cuda.mem_get_info(device)
    print("GPU:", torch.cuda.get_device_name(device))
    print("GPU dedicated total/free memory (GiB):", round(_gpu_total_bytes / 1024**3, 2), "/", round(_gpu_free_bytes / 1024**3, 2))
    print("GPU raster tile-cache policy:", GPU_RASTER_CACHE_MODE, "| max GiB:", GPU_RASTER_CACHE_MAX_GIB,
          "| reserve GiB:", GPU_RASTER_CACHE_RESERVE_GIB, "| tile rows:", GPU_RASTER_TILE_ROWS)
print("Evaluation settings: batch_size=", EVAL_BATCH_SIZE, "workers=", EVAL_NUM_WORKERS, "async_prefetch=", EVAL_ASYNC_DEVICE_PREFETCH)
print("DataLoader workers:", NUM_WORKERS, "| prefetch factor:", PREFETCH_FACTOR if NUM_WORKERS > 0 else "n/a", "| Arrow rows/batch:", ARROW_BATCH_ROWS)
print("Effective optimizer batch:", BATCH_SIZE, "| physical GPU microbatch range:", MIN_GPU_BATCH_SIZE, "to", TRAIN_MICROBATCH_CAP, "| effective retrain cap:", FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE)
print("Worker autotuning:", AUTO_TUNE_WORKERS, "| training worker maximum:", MAX_NUM_WORKERS,
      "| cache-builder workers override:", MODEL_CACHE_BUILD_WORKERS_OVERRIDE,
      "| cache-builder maximum:", MODEL_CACHE_BUILD_MAX_WORKERS,
      "| cache RAM budget/worker (GiB):", MODEL_CACHE_BUILD_RAM_PER_WORKER_GIB,
      "| cache RAM reserve (GiB):", MODEL_CACHE_BUILD_RAM_RESERVE_GIB,
      "| training/loader RAM reserve (GiB):", HOST_RAM_RESERVE_GIB)
print("Task:", TASK_MODE, "| target:", TARGET_MODE, "| temporal:", TEMPORAL_BACKBONE)

print("Optuna screening:", TRIAL_EPOCHS, "epochs x up to", TRIAL_TRAIN_SAMPLES_PER_EPOCH,
      "train samples/epoch (", TRIAL_MAX_TRAIN_BATCHES, "batch safety cap) and", TRIAL_MAX_TEST_BATCHES,
      "systematically sampled test batches/epoch; objective=test R².")
print("Winner full-data retraining enabled:", RUN_FINAL_RETRAIN,
      "| FINAL_RETRAIN_MAX_TRAIN_BATCHES:", FINAL_RETRAIN_MAX_TRAIN_BATCHES,
      "| MAX_ROWS_PER_SPLIT:", MAX_ROWS_PER_SPLIT)
print("VRAM-resident shard training:", GPU_RESIDENT_SHARD_TRAINING,
      "| minimum reserve GiB:", GPU_SHARD_RESERVE_GIB,
      "| max batches/shard:", GPU_SHARD_MAX_BATCHES,
      "| raster inputs FP16:", bool(USE_AMP and torch.cuda.is_available()))

In [ ]:
# ============================== ARCHIVE LEGACY RUNS ==============================
# Stop the old notebook kernel before running this cell. This is an archive (move), not deletion.
# Source rasters, filtered Parquet, manifests/scalers, and reusable data caches are NOT touched.
ARCHIVE_LEGACY_EXPERIMENT_ARTIFACTS = True

if ARCHIVE_LEGACY_EXPERIMENT_ARTIFACTS:
    from datetime import datetime
    _archive_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    _legacy_archive = RUN_DIR / "archive" / f"legacy_experiments_{_archive_stamp}"
    _legacy_archive.mkdir(parents=True, exist_ok=True)
    _archive_candidates = []
    for _pattern in (
        "tensorboard*",
        "optuna_filtered_s1_*.sqlite3",
        "optuna_filtered_s1_*.sqlite3-wal",
        "optuna_filtered_s1_*.sqlite3-shm",
        "optuna_filtered_s1_test_summary*.csv",
        "*_r2_progress*.png",
        "optuna_trial_errors_filtered_s1.jsonl",
        "*_optuna_best*.pt",
    ):
        _archive_candidates.extend(RUN_DIR.glob(_pattern))
    # Archive the previous MLflow store only on the transition to this new experiment.
    # Once the new experiment exists, subsequent notebook restarts must preserve its history.
    _archive_previous_mlflow_store = False
    if MLFLOW_DB_PATH.exists():
        try:
            import sqlite3 as _sqlite3
            with _sqlite3.connect(f"file:{MLFLOW_DB_PATH.resolve()}?mode=ro", uri=True, timeout=3) as _mlflow_conn:
                _table_names = {r[0] for r in _mlflow_conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
                if "experiments" in _table_names:
                    _existing_exp_names = {r[0] for r in _mlflow_conn.execute("SELECT name FROM experiments")}
                    _archive_previous_mlflow_store = "global_soil_moisture_test_r2_rmse_lossfix_v3" not in _existing_exp_names
        except Exception as _mlflow_inspect_error:
            # Preserve an unreadable/possibly active store rather than risk destroying run history.
            print("Could not safely inspect the existing MLflow store; preserving it:", _mlflow_inspect_error)
            _archive_previous_mlflow_store = False
    if _archive_previous_mlflow_store:
        _archive_candidates.append(MLFLOW_DB_PATH)
        if MLFLOW_ARTIFACT_DIR.exists():
            _archive_candidates.append(MLFLOW_ARTIFACT_DIR)
    # Most checkpoints live in RUN_DIR/checkpoints, not the run-directory root.
    if CHECKPOINT_DIR.exists():
        _archive_candidates.extend(CHECKPOINT_DIR.glob("*_optuna_best*.pt"))
    _seen_archive_paths = set()
    _archived = []
    for _old_path in _archive_candidates:
        if _old_path in _seen_archive_paths or not _old_path.exists():
            continue
        _seen_archive_paths.add(_old_path)
        # Do not move any artifact belonging to this notebook's fresh suffix.
        if OPTUNA_STUDY_SUFFIX in _old_path.name:
            continue
        if _old_path.parent == CHECKPOINT_DIR:
            _dest_dir = _legacy_archive / "checkpoints"
            _dest_dir.mkdir(parents=True, exist_ok=True)
        else:
            _dest_dir = _legacy_archive
        _dest = _dest_dir / _old_path.name
        if _dest.exists():
            _dest = _dest_dir / f"{_old_path.stem}_{len(_archived)}{_old_path.suffix}"
        shutil.move(str(_old_path), str(_dest))
        _archived.append(str(_dest))
    # Recreate fresh local MLflow artifact storage after any old directory was moved.
    MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    if _archived:
        print(f"Archived {len(_archived)} legacy experiment artifact(s) to: {_legacy_archive}")
    else:
        print("No matching legacy experiment artifacts found to archive.")
else:
    print("Legacy-artifact archive skipped (ARCHIVE_LEGACY_EXPERIMENT_ARTIFACTS=False).")
print("Persistent manifests/caches and raw input data have been preserved.")

## Training phases and concise MLflow metrics

The progress bars labelled `T<number> E<number> train` are Optuna screening trials, not full epochs over the dataset. Each trial aims to process 65,536 training samples per epoch, capped at 128 loader microbatches. The exact batch count depends on the physical microbatch selected by the GPU probe. After screening, the notebook retrains the winner on the complete training split, beginning with `location_history` and then `environment_only`.

MLflow records only model-quality metrics: test R² and RMSE, plus their best/final values. Resource and throughput diagnostics remain visible in the notebook console for debugging but are not logged as MLflow metrics.

In [ ]:
# =============================== VALIDATE INPUTS ===============================
required_files = [SAMPLE_INDEX, DAILY_TARGETS, FEATURE_SCHEMA]
_missing = [str(p) for p in required_files if not p.exists()]
if _missing:
    raise FileNotFoundError("Missing Notebook 1 output(s):\n" + "\n".join(_missing))

schema = json.loads(FEATURE_SCHEMA.read_text(encoding="utf-8"))
print("Target policy:", schema.get("preprocessing_policy", {}).get("label_aggregation"))
print("MODIS alignment note:", schema.get("spatial_modalities", {}).get("alignment_note", "not recorded"))

DUCKDB_TEMP_DIR = RUN_DIR / "duckdb_tmp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

def sql_path(path: Path) -> str:
    return str(Path(path).resolve()).replace("'", "''")

def parquet_scan(path: Path) -> str:
    return f"read_parquet('{sql_path(path)}')"

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory='{sql_path(DUCKDB_TEMP_DIR)}'")

index_columns = set(con.execute(f"DESCRIBE SELECT * FROM {parquet_scan(SAMPLE_INDEX)}").df()["column_name"])
target_columns = set(con.execute(f"DESCRIBE SELECT * FROM {parquet_scan(DAILY_TARGETS)}").df()["column_name"])
required_index = {
    "sample_id", "station_id", "network", "sample_date", "year", "latitude", "longitude",
    "modis_row_index", "s1_row_index", "SMAP", "SMAP_n_obs", "SMAP_cell_conflict",
    *TEMP_FEATURES,
    "clay_pct", "sand_pct", "silt_pct", "terrain_elevation_m", "slope_deg", "aspect_deg",
}
_missing_cols = sorted(required_index - index_columns)
if _missing_cols:
    raise ValueError(f"sample_index.parquet is missing required columns: {_missing_cols}")
if TARGET_COL not in target_columns:
    raise ValueError(f"daily_targets.parquet lacks {TARGET_COL}; found {sorted(target_columns)}")

counts = con.execute(f"""
SELECT
  (SELECT COUNT(*) FROM {parquet_scan(SAMPLE_INDEX)}) AS sample_index_rows,
  (SELECT COUNT(DISTINCT sample_id) FROM {parquet_scan(SAMPLE_INDEX)}) AS sample_index_unique_ids,
  (SELECT COUNT(*) FROM {parquet_scan(DAILY_TARGETS)}) AS daily_target_rows,
  (SELECT COUNT(DISTINCT sample_id) FROM {parquet_scan(DAILY_TARGETS)}) AS daily_target_unique_ids,
  (SELECT COUNT(*) FROM {parquet_scan(DAILY_TARGETS)}
     WHERE {TARGET_COL} IS NOT NULL AND isfinite(TRY_CAST({TARGET_COL} AS DOUBLE))) AS valid_target_rows
""").df()
display(counts)

if int(counts.loc[0, "sample_index_rows"]) != int(counts.loc[0, "sample_index_unique_ids"]):
    raise ValueError("sample_index.parquet must contain one row per sample_id")
if int(counts.loc[0, "daily_target_rows"]) != int(counts.loc[0, "daily_target_unique_ids"]):
    raise ValueError("daily_targets.parquet must contain one row per sample_id")

## 1. Build/reuse disk-backed manifests

The historical weather context is computed from **all rows in `sample_index.parquet` first**, then labels are joined. A historic date does not need a target row to provide meteorological context. The window stores prior candidate-date rows, timestamps and lags; the loader masks past tokens beyond `HISTORY_MAX_DAYS`.

- **Environment-only:** deterministic 80/10/10 station-group split. The model receives environmental features but no station ID or coordinates. Weather history is still an environmental input; no in-situ target history is used.
- **Location/history-aware:** chronological train/test split for stations that have training-period observations. It adds spherical location coordinates and uses earlier meteorological context, but does **not** use previous target labels. This is a known-station temporal task, not the unseen-station test.

The two modes test different generalization settings. Do not compare their scores as though they were the same split.

In [ ]:
# =============================== BUILD/REUSE MANIFESTS ===============================
manifest_config = {
    "target_col": TARGET_COL,
    "temp_features": TEMP_FEATURES,
    "static_features": STATIC_FEATURES,
    "history_steps": HISTORY_STEPS,
    "history_max_days": HISTORY_MAX_DAYS,
    "env_split": "station hash 80/20 train/test; station-disjoint",
    "env_split_seed": SEED,
    "location_train_end": TRAIN_END_DATE,
    "location_split": "train through TRAIN_END_DATE; test afterward; future stations restricted to eligible training stations",
    "row_group_size": 8192,
    "filtered_sample_index_size": int(SAMPLE_INDEX.stat().st_size),
    "filtered_sample_index_mtime_ns": int(SAMPLE_INDEX.stat().st_mtime_ns),
    "filtered_daily_targets_size": int(DAILY_TARGETS.stat().st_size),
    "filtered_daily_targets_mtime_ns": int(DAILY_TARGETS.stat().st_mtime_ns),
}
config_path = MANIFEST_DIR / "manifest_config.json"
base_manifest = MANIFEST_DIR / "base_manifest.parquet"
manifest_paths = {
    (mode, split): MANIFEST_DIR / mode / f"{split}.parquet"
    for mode in ["environment_only", "location_history"]
    for split in ["train", "test"]
}

def _manifest_config_matches(saved, current):
    """Compare manifest inputs while ignoring filesystem mtime precision.

    ZIP extraction and copying between filesystems can change mtime_ns without
    changing Parquet contents. File sizes, schema/split settings and feature choices
    remain checked, so a path move or timestamp rounding does not force a rebuild.
    """
    def strip_times(value):
        if isinstance(value, dict):
            return {k: strip_times(v) for k, v in value.items()
                    if not str(k).endswith("_mtime_ns")}
        if isinstance(value, list):
            return [strip_times(v) for v in value]
        return value
    return strip_times(saved) == strip_times(current)

signature_matches = False
if config_path.exists():
    try:
        signature_matches = _manifest_config_matches(
            json.loads(config_path.read_text(encoding="utf-8")), manifest_config)
    except Exception:
        signature_matches = False

all_manifests_exist = base_manifest.exists() and all(p.exists() for p in manifest_paths.values())
if all_manifests_exist and signature_matches and not REBUILD_MANIFESTS:
    print("Reusing existing manifest set; config signature matches.")
else:
    if all_manifests_exist and not signature_matches:
        print("Manifest configuration changed (train/test split). Rebuilding the manifest set once.")
    if REBUILD_MANIFESTS or (all_manifests_exist and not signature_matches):
        # Split assignment changed. Rebuild split files but keep the base manifest unless its
        # independent input/schema fingerprint changes; creating that base is expensive.
        for p in manifest_paths.values():
            if p.exists(): p.unlink()
        if config_path.exists(): config_path.unlink()
    for p in manifest_paths.values():
        p.parent.mkdir(parents=True, exist_ok=True)

    # Base-manifest fingerprint is independent from split boundaries so a train/test
    # change does not force the full historical-context window query to be recomputed.
    base_manifest_config = {
        "target_col": TARGET_COL,
        "temp_features": TEMP_FEATURES,
        "static_features": STATIC_FEATURES,
        "history_steps": HISTORY_STEPS,
        "history_max_days": HISTORY_MAX_DAYS,
        "filtered_sample_index_size": int(SAMPLE_INDEX.stat().st_size),
        "filtered_sample_index_mtime_ns": int(SAMPLE_INDEX.stat().st_mtime_ns),
        "filtered_daily_targets_size": int(DAILY_TARGETS.stat().st_size),
        "filtered_daily_targets_mtime_ns": int(DAILY_TARGETS.stat().st_mtime_ns),
    }
    base_config_path = MANIFEST_DIR / "base_manifest_config.json"
    _base_signature_matches = False
    if base_manifest.exists() and base_config_path.exists():
        try:
            _base_signature_matches = _manifest_config_matches(
                json.loads(base_config_path.read_text(encoding="utf-8")), base_manifest_config)
        except Exception:
            _base_signature_matches = False
    if base_manifest.exists() and _base_signature_matches and not REBUILD_MANIFESTS:
        print("Reusing base manifest; source/schema fingerprint matches.")
    else:
        # Build lag expressions on the complete sample index. Lags are ordered by
        # actual date and numeric sample_id; the latter only breaks same-date ties.
        lag_expr = []
        lag_order = "CAST(sample_date AS DATE), TRY_CAST(sample_id AS BIGINT), CAST(sample_id AS VARCHAR)"
        for lag in range(1, HISTORY_STEPS):
            lag_expr.append(
                f"LAG(CAST(sample_date AS DATE), {lag}) OVER "
                f"(PARTITION BY station_key ORDER BY {lag_order}) AS lag_date_{lag}"
            )
            for feat in TEMP_FEATURES:
                lag_expr.append(
                    f'LAG(TRY_CAST("{feat}" AS DOUBLE), {lag}) OVER '
                    f'(PARTITION BY station_key ORDER BY {lag_order}) AS "{feat}_lag{lag}"'
                )
        lag_expr_sql = ",\n        ".join(lag_expr)

        # Build all candidate-date context first, then attach one daily label per sample_id.
        base_query = f"""
        WITH context AS (
            SELECT
                CAST(sample_id AS VARCHAR) AS sample_id,
                TRY_CAST(sample_id AS BIGINT) AS sample_order,
                CAST(station_id AS VARCHAR) AS station_id,
                CAST(network AS VARCHAR) AS network,
                CAST(network AS VARCHAR) || '::' || CAST(station_id AS VARCHAR) AS station_key,
                CAST(sample_date AS DATE) AS sample_date,
                CAST(year AS INTEGER) AS year,
                TRY_CAST(latitude AS DOUBLE) AS latitude,
                TRY_CAST(longitude AS DOUBLE) AS longitude,
                (COS(RADIANS(TRY_CAST(latitude AS DOUBLE))) * COS(RADIANS(TRY_CAST(longitude AS DOUBLE)))) AS location_x,
                (COS(RADIANS(TRY_CAST(latitude AS DOUBLE))) * SIN(RADIANS(TRY_CAST(longitude AS DOUBLE)))) AS location_y,
                SIN(RADIANS(TRY_CAST(latitude AS DOUBLE))) AS location_z,
                TRY_CAST(modis_row_index AS BIGINT) AS modis_row_index,
                TRY_CAST(s1_row_index AS BIGINT) AS s1_row_index,
                TRY_CAST(SMAP AS DOUBLE) AS SMAP,
                TRY_CAST(SMAP_n_obs AS DOUBLE) AS SMAP_n_obs,
                CAST(SMAP_cell_conflict AS DOUBLE) AS SMAP_cell_conflict_num,
                CASE WHEN SMAP IS NOT NULL AND isfinite(TRY_CAST(SMAP AS DOUBLE)) THEN 1.0 ELSE 0.0 END AS has_smap_value_num,
                {', '.join(f'TRY_CAST("{f}" AS DOUBLE) AS "{f}"' for f in TEMP_FEATURES)},
                TRY_CAST(clay_pct AS DOUBLE) AS clay_pct,
                TRY_CAST(sand_pct AS DOUBLE) AS sand_pct,
                TRY_CAST(silt_pct AS DOUBLE) AS silt_pct,
                TRY_CAST(terrain_elevation_m AS DOUBLE) AS terrain_elevation_m,
                TRY_CAST(slope_deg AS DOUBLE) AS slope_deg,
                SIN(RADIANS(TRY_CAST(aspect_deg AS DOUBLE))) AS aspect_sin,
                COS(RADIANS(TRY_CAST(aspect_deg AS DOUBLE))) AS aspect_cos,
                SIN(2 * PI() * DAYOFYEAR(CAST(sample_date AS DATE)) / 365.25) AS day_sin,
                COS(2 * PI() * DAYOFYEAR(CAST(sample_date AS DATE)) / 365.25) AS day_cos
            FROM {parquet_scan(SAMPLE_INDEX)}
        ), histories AS (
            SELECT *,
                {lag_expr_sql}
            FROM context
        ), labelled AS (
            SELECT
                h.* EXCLUDE (sample_order),
                TRY_CAST(t.{TARGET_COL} AS DOUBLE) AS target_mean_m3_m3,
                TRY_CAST(t.target_min_m3_m3 AS DOUBLE) AS target_min_m3_m3,
                TRY_CAST(t.target_max_m3_m3 AS DOUBLE) AS target_max_m3_m3,
                TRY_CAST(t.target_range_m3_m3 AS DOUBLE) AS target_range_m3_m3,
                TRY_CAST(t.target_std_m3_m3 AS DOUBLE) AS target_std_m3_m3,
                TRY_CAST(t.target_valid_observation_count AS BIGINT) AS target_valid_observation_count,
                TRY_CAST(t.n_depth_classes AS BIGINT) AS n_depth_classes,
                TRY_CAST(t.n_sensors AS BIGINT) AS n_sensors,
                TRY_CAST(t.min_depth_from_m AS DOUBLE) AS min_depth_from_m,
                TRY_CAST(t.max_depth_to_m AS DOUBLE) AS max_depth_to_m,
                CAST(t.sensors AS VARCHAR) AS target_sensors,
                CAST(t.depth_classes AS VARCHAR) AS target_depth_classes
            FROM histories h
            INNER JOIN {parquet_scan(DAILY_TARGETS)} t
              ON CAST(t.sample_id AS VARCHAR) = h.sample_id
            WHERE t.{TARGET_COL} IS NOT NULL
              AND isfinite(TRY_CAST(t.{TARGET_COL} AS DOUBLE))
              AND h.modis_row_index IS NOT NULL
              AND h.s1_row_index IS NOT NULL
        )
        SELECT * FROM labelled
        """
        if base_manifest.exists(): base_manifest.unlink()
        con.execute(f"""
            COPY ({base_query}) TO '{sql_path(base_manifest)}'
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 8192)
        """)
        base_rows = con.execute(f"SELECT COUNT(*) FROM {parquet_scan(base_manifest)}").fetchone()[0]
        print(f"Wrote base manifest: {base_rows:,} rows")
        base_config_path.write_text(json.dumps(base_manifest_config, indent=2) + "\n", encoding="utf-8")

    base_sql = parquet_scan(base_manifest)
    for mode in ["environment_only", "location_history"]:
        for split in ["train", "test"]:
            out_path = manifest_paths[(mode, split)]
            if out_path.exists(): out_path.unlink()

            if mode == "environment_only":
                # HASH is deterministic within this DuckDB version; the realized split
                # files are persisted so a future rerun can reuse the exact assignment.
                split_expr = (
                    f"CASE WHEN HASH(station_key || ':{SEED}') % 100 < 80 THEN 'train' ELSE 'test' END"
                )
                known_filter = "TRUE"
            else:
                split_expr = (
                    f"CASE WHEN sample_date <= DATE '{TRAIN_END_DATE}' THEN 'train' ELSE 'test' END"
                )
                # Only evaluate future periods for stations with at least one labelled train-period row.
                known_filter = f"(sample_date <= DATE '{TRAIN_END_DATE}' OR station_key IN (SELECT DISTINCT station_key FROM {base_sql} WHERE sample_date <= DATE '{TRAIN_END_DATE}'))"

            # Training rows retain temporal/locality ordering. Test is hash-ordered so a
            # capped Optuna evaluation samples across the whole test set, not just early dates.
            if split == "train":
                order_expr = "year, sample_date, TRY_CAST(sample_id AS BIGINT), sample_id"
            else:
                order_expr = f"HASH(CAST(sample_id AS VARCHAR) || ':{SEED}:{split}'), sample_id"
            query = f"""
            SELECT * FROM (
                SELECT *, {split_expr} AS split
                FROM {base_sql}
                WHERE {known_filter}
            ) q
            WHERE split = '{split}'
            ORDER BY {order_expr}
            """
            con.execute(f"COPY ({query}) TO '{sql_path(out_path)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 8192)")
            n = con.execute(f"SELECT COUNT(*) FROM {parquet_scan(out_path)}").fetchone()[0]
            print(f"{mode:18s} {split:5s}: {n:,} rows")

    config_path.write_text(json.dumps(manifest_config, indent=2) + "\n", encoding="utf-8")

# Split integrity summary: station disjointness for environmental split; temporal ranges and
# station coverage for location/history split.
summary_rows = []
for mode in ["environment_only", "location_history"]:
    for split in ["train", "test"]:
        p = manifest_paths[(mode, split)]
        summary_rows.append({
            "mode": mode,
            "split": split,
            "rows": int(con.execute(f"SELECT COUNT(*) FROM {parquet_scan(p)}").fetchone()[0]),
            "stations": int(con.execute(f"SELECT COUNT(DISTINCT station_key) FROM {parquet_scan(p)}").fetchone()[0]),
            "min_date": con.execute(f"SELECT MIN(sample_date) FROM {parquet_scan(p)}").fetchone()[0],
            "max_date": con.execute(f"SELECT MAX(sample_date) FROM {parquet_scan(p)}").fetchone()[0],
        })
display(pd.DataFrame(summary_rows))

# Assert that all environment-only partitions are pairwise station-disjoint.
for _left, _right in (("train", "test"),):
    _overlap = con.execute(f"""
    WITH l AS (SELECT DISTINCT station_key FROM {parquet_scan(manifest_paths[('environment_only', _left)])}),
         r AS (SELECT DISTINCT station_key FROM {parquet_scan(manifest_paths[('environment_only', _right)])})
    SELECT COUNT(*) FROM l JOIN r USING(station_key)
    """).fetchone()[0]
    if _overlap:
        raise RuntimeError(f"Environment-only split leaks {_overlap} station overlaps: {_left}/{_right}")
print("Environment-only train/test partitions are station-disjoint.")

for _mode in ("environment_only", "location_history"):
    for _split in ("train", "test"):
        _n = int(con.execute(f"SELECT COUNT(*) FROM {parquet_scan(manifest_paths[(_mode, _split)])}").fetchone()[0])
        if _n == 0:
            raise RuntimeError(f"Empty manifest split: {_mode}/{_split}. Check filtered data and date boundaries.")

## 2. Fit scalers on training data only

Tabular means and standard deviations are fitted from each mode's training manifest only. Spatial raster scalers are fitted from a bounded, deterministic sample of training patches. Test data are never used to estimate preprocessing parameters.

**MODIS decoding is handled before normalization:** EVI raw integers are multiplied by `0.0001` after invalid/fill values are masked; LST raw integers are multiplied by `0.02` to Kelvin. Sentinel-1 stores are already float arrays, with non-finite pixels masked. The model gets a separate validity-mask channel for each spatial band.

In [ ]:
# =============================== TRAIN-ONLY TABULAR SCALERS ===============================
NUMERIC_FEATURES = TEMP_FEATURES + STATIC_FEATURES + ANCHOR_FEATURES

def fit_tabular_scaler(mode: str):
    train_path = manifest_paths[(mode, "train")]
    train_sql = parquet_scan(train_path)
    stats = {}
    for feature in NUMERIC_FEATURES + (LOCATION_FEATURES if mode == "location_history" else []):
        mu, sd, n = con.execute(f"""
            SELECT AVG(x) FILTER (WHERE x IS NOT NULL AND isfinite(x)),
                   STDDEV_SAMP(x) FILTER (WHERE x IS NOT NULL AND isfinite(x)),
                   COUNT(*) FILTER (WHERE x IS NOT NULL AND isfinite(x))
            FROM (SELECT TRY_CAST(\"{feature}\" AS DOUBLE) AS x FROM {train_sql})
        """).fetchone()
        mu = float(mu) if mu is not None and np.isfinite(mu) else 0.0
        sd = float(sd) if sd is not None and np.isfinite(sd) and sd > 1e-8 else 1.0
        stats[feature] = {"mean": mu, "std": sd, "n_valid": int(n or 0)}
    output = SCALER_DIR / f"{mode}_tabular.json"
    output.write_text(json.dumps(stats, indent=2) + "\n", encoding="utf-8")
    return stats

def open_group(path: Path):
    # Supports current local Zarr stores without loading arrays into RAM.
    return zarr.open_group(str(path), mode="r")

def decode_modis_patch(raw, kind):
    arr = np.asarray(raw)
    if kind == "evi":
        valid = np.isfinite(arr) & (arr != EVI_FILL_RAW) & (arr >= EVI_RAW_MIN) & (arr <= EVI_RAW_MAX)
        physical = arr.astype(np.float32) * EVI_SCALE
    elif kind == "lst":
        valid = np.isfinite(arr) & (arr != LST_FILL_RAW) & (arr >= LST_RAW_MIN) & (arr <= LST_RAW_MAX)
        physical = arr.astype(np.float32) * LST_SCALE
    else:
        raise ValueError(kind)
    physical = np.asarray(physical, dtype=np.float32)
    physical[~valid] = np.nan
    return physical, valid

def decode_s1_patch(raw):
    arr = np.asarray(raw, dtype=np.float32)
    valid = np.isfinite(arr)
    out = arr.copy()
    out[~valid] = np.nan
    return out, valid

def sample_training_raster_rows(mode, n_sample=RASTER_SCALER_SAMPLE_ROWS):
    # Stratified sample: at most ceil(n_sample / 12) rows per year, sampled
    # by a stable hash so the scaler sees more than only the earliest year.
    per_year = max(1, int(math.ceil(n_sample / 12)))
    pieces = []
    psql = parquet_scan(manifest_paths[(mode, "train")])
    for year in range(2015, 2027):
        df = con.execute(f"""
            SELECT sample_id, year, modis_row_index, s1_row_index
            FROM {psql}
            WHERE year = {year}
            ORDER BY HASH(sample_id || ':raster_scaler:{SEED}')
            LIMIT {per_year}
        """).df()
        if not df.empty:
            pieces.append(df)
    if not pieces:
        raise RuntimeError(f"No training rows found to fit raster scalers for {mode}")
    return pd.concat(pieces, ignore_index=True)

def fit_raster_scaler(mode: str):
    rows = sample_training_raster_rows(mode)
    feature_names = ["evi", "lst"] + S1_ARRAYS
    sums = {k: 0.0 for k in feature_names}
    sums2 = {k: 0.0 for k in feature_names}
    counts = {k: 0 for k in feature_names}
    modis_cache, s1_cache = {}, {}

    for r in tqdm(rows.itertuples(index=False), total=len(rows), desc=f"Raster scaler sample: {mode}"):
        year = int(r.year)
        if year not in modis_cache:
            modis_cache[year] = open_group(MODIS_ROOT / str(year) / "patches.zarr")
            s1_cache[year] = open_group(S1_ROOT / f"s1_nearest_valid_{year}.zarr")
        mg, sg = modis_cache[year], s1_cache[year]

        evi, evi_valid = decode_modis_patch(mg["evi"][int(r.modis_row_index)], "evi")
        lst, lst_valid = decode_modis_patch(mg["lst"][int(r.modis_row_index)], "lst")
        arrays = {"evi": (evi, evi_valid), "lst": (lst, lst_valid)}
        for name in S1_ARRAYS:
            arrays[name] = decode_s1_patch(sg[name][int(r.s1_row_index)])

        for name, (arr, valid) in arrays.items():
            vals = np.asarray(arr)[np.asarray(valid, dtype=bool)]
            if vals.size:
                vals64 = vals.astype(np.float64, copy=False)
                sums[name] += float(vals64.sum())
                sums2[name] += float(np.square(vals64).sum())
                counts[name] += int(vals64.size)

    result = {}
    for name in feature_names:
        n = counts[name]
        mu = sums[name] / n if n else 0.0
        variance = max(sums2[name] / n - mu * mu, 0.0) if n else 1.0
        std = math.sqrt(variance) if variance > 1e-12 else 1.0
        result[name] = {"mean": mu, "std": std, "n_valid_pixels": n}

    out = SCALER_DIR / f"{mode}_raster.json"
    out.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result

# --------------------- durable scaler reuse ---------------------
# The compact-cache fingerprint includes scaler values. Re-fitting raster scalers on every
# kernel restart performs another Zarr sampling pass and can introduce tiny float jitter,
# which creates a new compact-cache fingerprint. Persist and validate the scaler provenance.
SCALER_SIGNATURE_PATH = SCALER_DIR / "train_only_scalers_signature_v2.json"
_SCALER_MODES = ("environment_only", "location_history")
_SCALER_FEATURES = {
    mode: {
        "tabular": NUMERIC_FEATURES + (LOCATION_FEATURES if mode == "location_history" else []),
        "raster": ["evi", "lst"] + list(S1_ARRAYS),
    }
    for mode in _SCALER_MODES
}

def _scaler_manifest_metadata():
    out = {}
    for mode in _SCALER_MODES:
        path = Path(manifest_paths[(mode, "train")])
        pf = pq.ParquetFile(str(path))
        out[mode] = {
            "name": path.name,
            "size": int(path.stat().st_size),
            "num_rows": int(pf.metadata.num_rows),
            "num_row_groups": int(pf.metadata.num_row_groups),
            "schema": str(pf.schema),
        }
    return out


def _scaler_signature_payload():
    # mtime is intentionally excluded: copying the same manifest between local filesystems
    # may alter timestamp precision. Size, row count, row groups and schema remain checked.
    _mc = {k: v for k, v in globals().get("manifest_config", {}).items()
           if not str(k).endswith("_mtime_ns")}
    return {
        "signature_version": 2,
        "manifest_config": _mc,
        "train_manifests": _scaler_manifest_metadata(),
        "features": _SCALER_FEATURES,
        "seed": int(SEED),
        "raster_scaler_sample_rows": int(RASTER_SCALER_SAMPLE_ROWS),
        "history_steps": int(HISTORY_STEPS),
        "history_max_days": int(HISTORY_MAX_DAYS),
        "s1_arrays": list(S1_ARRAYS),
        "modis_decode": [EVI_FILL_RAW, EVI_RAW_MIN, EVI_RAW_MAX, EVI_SCALE,
                         LST_FILL_RAW, LST_RAW_MIN, LST_RAW_MAX, LST_SCALE],
    }


def _scaler_signature_hash(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def _scaler_files_are_usable():
    required = []
    for mode in _SCALER_MODES:
        required += [SCALER_DIR / f"{mode}_tabular.json", SCALER_DIR / f"{mode}_raster.json"]
    if not all(p.is_file() for p in required):
        return False
    try:
        for mode in _SCALER_MODES:
            tab = json.loads((SCALER_DIR / f"{mode}_tabular.json").read_text(encoding="utf-8"))
            ras = json.loads((SCALER_DIR / f"{mode}_raster.json").read_text(encoding="utf-8"))
            if not set(_SCALER_FEATURES[mode]["tabular"]).issubset(tab):
                return False
            if not set(_SCALER_FEATURES[mode]["raster"]).issubset(ras):
                return False
            for stats in list(tab.values()) + list(ras.values()):
                if not np.isfinite(float(stats["mean"])) or not np.isfinite(float(stats["std"])) or float(stats["std"]) <= 0:
                    return False
    except Exception:
        return False
    return True


SCALER_SIGNATURE_PAYLOAD = _scaler_signature_payload()
SCALER_SIGNATURE_HASH = _scaler_signature_hash(SCALER_SIGNATURE_PAYLOAD)
_reuse_scalers = False
_legacy_scaler_reuse = False
if _scaler_files_are_usable():
    if SCALER_SIGNATURE_PATH.is_file():
        try:
            _saved_scaler_sig = json.loads(SCALER_SIGNATURE_PATH.read_text(encoding="utf-8"))
            _reuse_scalers = (_saved_scaler_sig.get("fingerprint") == SCALER_SIGNATURE_HASH)
        except Exception:
            _reuse_scalers = False
    elif bool(globals().get("signature_matches", False)) and all(
        Path(manifest_paths[(mode, split)]).is_file()
        for mode in _SCALER_MODES for split in ("train", "test")
    ):
        # One-time migration for scalers created by earlier v19 notebooks before the
        # signature sidecar existed. The existing manifest configuration is unchanged.
        _reuse_scalers = True
        _legacy_scaler_reuse = True

if _reuse_scalers:
    tabular_scalers = {
        mode: json.loads((SCALER_DIR / f"{mode}_tabular.json").read_text(encoding="utf-8"))
        for mode in _SCALER_MODES
    }
    raster_scalers = {
        mode: json.loads((SCALER_DIR / f"{mode}_raster.json").read_text(encoding="utf-8"))
        for mode in _SCALER_MODES
    }
    if _legacy_scaler_reuse:
        print("[SCALER CACHE] Reusing the existing train-only scaler JSON files; manifest signature is unchanged. "
              "Writing a provenance sidecar for subsequent restarts.", flush=True)
    else:
        print("[SCALER CACHE] REUSED train-only tabular + raster scalers; no raster scaler sampling pass.", flush=True)
else:
    print("[SCALER CACHE] No matching durable scaler signature; fitting train-only scalers once.", flush=True)
    tabular_scalers = {mode: fit_tabular_scaler(mode) for mode in _SCALER_MODES}
    raster_scalers = {mode: fit_raster_scaler(mode) for mode in _SCALER_MODES}

    _tmp_sig = SCALER_SIGNATURE_PATH.with_suffix(SCALER_SIGNATURE_PATH.suffix + ".tmp")
    with open(_tmp_sig, "w", encoding="utf-8") as _f:
        json.dump({"fingerprint": SCALER_SIGNATURE_HASH, "signature": SCALER_SIGNATURE_PAYLOAD}, _f, indent=2)
        _f.flush(); os.fsync(_f.fileno())
    os.replace(_tmp_sig, SCALER_SIGNATURE_PATH)
    print("[SCALER CACHE] Saved durable train-only scaler signature:", SCALER_SIGNATURE_PATH, flush=True)

# If a legacy scaler set was accepted, persist the sidecar only after all four files parsed.
if _legacy_scaler_reuse:
    _tmp_sig = SCALER_SIGNATURE_PATH.with_suffix(SCALER_SIGNATURE_PATH.suffix + ".tmp")
    with open(_tmp_sig, "w", encoding="utf-8") as _f:
        json.dump({"fingerprint": SCALER_SIGNATURE_HASH, "signature": SCALER_SIGNATURE_PAYLOAD}, _f, indent=2)
        _f.flush(); os.fsync(_f.fileno())
    os.replace(_tmp_sig, SCALER_SIGNATURE_PATH)
print("Train-only scalers ready under", SCALER_DIR)

## 3. Compact parallel cache, raster-local batches and adaptive resource tuning

The persistent compact cache contains per-sample temporal/static metadata, labels, anchors, validity and source raster indices. In v10, a separate bounded disk mirror stores the active **raw source arrays** once in native dtype. It does not expand data into duplicated per-sample raster patches or normalized tensors.

The builder creates fixed source-row slots in memory-mapped arrays and uses multiple CPU processes to preprocess separate Parquet row groups concurrently. Workers write directly to disjoint row ranges, so their arrays are not copied through multiprocessing queues. Completed groups are flushed and recorded in an atomic checkpoint; groups not in that checkpoint are safe to rebuild after interruption.

Training reads compact mmap arrays and gathers raster patches from read-only native-dtype raster mmaps when the bounded disk mirror is available; otherwise it reads the original Zarr stores. A byte-bounded worker-local LRU caches normalized source chunks (EVI=128 rows; LST/S1=256 rows). The locality-aware sampler shuffles blocks while sorting rows inside each block by year and raster indices.

Worker counts are bounded by visible CPUs, available RAM and data constraints. Data workers limit their own PyTorch/BLAS thread pools to avoid oversubscription. The cache builder and training loader show progress, throughput, CPU/RAM and cache-hit metrics so a low-utilization bottleneck is easier to locate.

**Resume and disk safety:** compatible complete caches are reused; compatible checkpoints skip completed row groups. Before a fresh build, the notebook checks estimated compact-cache size plus disk reserve. For a resumed build, the disk check considers the unfinished portion only. Incompatible staging directories are preserved for inspection rather than silently deleted.

In [ ]:
# =============================== ADAPTIVE RESOURCE HELPERS ===============================
_GIB = 1024 ** 3
S1_CACHE_ARRAYS = list(S1_ARRAYS)
RAM_ZARR_CACHE_ACTIVE = False


def available_ram_bytes():
    if psutil is not None:
        try:
            return int(psutil.virtual_memory().available)
        except Exception:
            pass
    try:
        return int(os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"))
    except Exception:
        return 0


def visible_cpu_count():
    try:
        return max(1, len(os.sched_getaffinity(0)))
    except Exception:
        if psutil is not None:
            try:
                return max(1, int(psutil.cpu_count(logical=True) or 1))
            except Exception:
                pass
        return max(1, int(os.cpu_count() or 1))


def resource_limited_worker_cap():
    cpu_cap = max(0, visible_cpu_count() - 1)
    cap = min(int(MAX_NUM_WORKERS), cpu_cap)
    avail_gib = available_ram_bytes() / _GIB
    # Train and test loaders may each retain their worker pool. Budget for both.
    memory_slots = max(0, int(max(0.0, avail_gib - HOST_RAM_RESERVE_GIB) /
                              max(2.0 * WORKER_RAM_BUDGET_GIB, 0.5)))
    cap = min(cap, memory_slots)
    return max(0, cap)


def _cache_array_path(modality, year, name):
    return RAM_CACHE_DIR / modality / str(year) / f"{name}.npy"


def _manifest_years():
    years = set()
    for path in set(manifest_paths.values()):
        if not Path(path).exists():
            continue
        try:
            query = f"SELECT DISTINCT TRY_CAST(year AS INTEGER) FROM {parquet_scan(Path(path))} WHERE year IS NOT NULL"
            years.update(int(r[0]) for r in con.execute(query).fetchall() if r[0] is not None)
        except Exception as exc:
            print(f"Could not inspect manifest years for RAM cache: {exc}")
            return set()
    return years


def _source_array_specs():
    specs = []
    years = sorted(_manifest_years())
    if not years:
        return None
    for year in years:
        paths = {
            "modis": MODIS_ROOT / str(year) / "patches.zarr",
            "s1": S1_ROOT / f"s1_nearest_valid_{year}.zarr",
        }
        if not paths["modis"].exists() or not paths["s1"].exists():
            return None
        for modality, names in (("modis", ["evi", "lst"]), ("s1", S1_CACHE_ARRAYS)):
            group = open_group(paths[modality])
            for name in names:
                arr = group[name]
                shape = tuple(int(d) for d in arr.shape)
                dtype = np.dtype(arr.dtype).str
                nbytes = int(np.prod(shape, dtype=np.int64)) * np.dtype(arr.dtype).itemsize
                cache_path = _cache_array_path(modality, year, name)
                specs.append({"year": year, "modality": modality, "name": name,
                              "source": str(paths[modality]), "shape": shape,
                              "dtype": dtype, "nbytes": nbytes, "cache_path": cache_path})
    return specs


def _memmap_valid(spec):
    path = spec["cache_path"]
    if not path.exists():
        return False
    try:
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        return tuple(arr.shape) == spec["shape"] and arr.dtype.str == spec["dtype"]
    except Exception:
        return False


def prepare_zarr_memmap_cache():
    """Build/reuse a bounded disk mirror of active source raster arrays.

    This is deliberately a RAW/native-dtype mirror, not normalized patches or batches.
    It is shared by both task modes, so task-specific training-only scalers remain correct.
    Only EVI, LST, VV, VH and VV/VH-ratio arrays are included; HH/HV and all auxiliary
    arrays are excluded. Writes are atomic per array and resumable after interruption.
    """
    cache_mode = str(globals().get("ZARR_ARRAY_CACHE_MODE", "off")).lower()
    if str(globals().get("MODEL_READY_CACHE_MODE", "off")).lower() != "off" and cache_mode != "disk":
        print("Skipping duplicate raw-raster mirror: compact model cache is active and disk mirror mode is not explicitly enabled.")
        return False
    if cache_mode == "off":
        print("Raster disk mirror disabled; using direct Zarr reads.")
        return False
    if cache_mode != "disk":
        print(f"Unknown ZARR_ARRAY_CACHE_MODE={cache_mode!r}; using direct Zarr reads.")
        return False

    specs = _source_array_specs()
    if not specs:
        print("Could not determine active raster source arrays; using direct Zarr reads.")
        return False

    total_bytes = int(sum(int(spec["nbytes"]) for spec in specs))
    max_bytes = int(float(globals().get("MAX_ZARR_DISK_CACHE_GIB", 16.0)) * _GIB)
    reserve_bytes = int(float(globals().get("MIN_DISK_FREE_RESERVE_GIB", 10.0)) * _GIB)
    if total_bytes > max_bytes:
        print(f"Active raster mirror would require {total_bytes/_GIB:.2f} GiB, exceeding the hard "
              f"{max_bytes/_GIB:.2f} GiB cache cap. No mirror files will be built; using Zarr directly.")
        return False

    missing_specs = [spec for spec in specs if not _memmap_valid(spec)]
    missing_bytes = int(sum(int(spec["nbytes"]) for spec in missing_specs))
    try:
        disk = shutil.disk_usage(RUN_DIR)
        disk_free = int(disk.free)
        disk_total = int(disk.total)
    except Exception as exc:
        print(f"Could not inspect free disk space ({exc}); refusing to build raster mirror.")
        return False

    projected_free = disk_free - missing_bytes
    print("Active raw-raster mmap mirror estimate:")
    print(f"  arrays: {len(specs)} (MODIS EVI/LST + S1 {', '.join(S1_CACHE_ARRAYS)})")
    print(f"  total cache size: {total_bytes/_GIB:.2f} GiB / hard cap {max_bytes/_GIB:.2f} GiB")
    print(f"  remaining to build: {missing_bytes/_GIB:.2f} GiB; disk currently free: {disk_free/_GIB:.2f} GiB")
    print(f"  projected free after build: {projected_free/_GIB:.2f} GiB; required reserve: {reserve_bytes/_GIB:.2f} GiB")
    if missing_bytes > 0 and projected_free < reserve_bytes:
        print("Disk reserve check failed. No new arrays will be copied; existing complete cache arrays are retained and direct Zarr reads will be used for this run.")
        return False

    (RAM_CACHE_DIR / "modis").mkdir(parents=True, exist_ok=True)
    (RAM_CACHE_DIR / "s1").mkdir(parents=True, exist_ok=True)
    built = 0
    started = time.perf_counter()
    for spec_i, spec in enumerate(specs, start=1):
        path = Path(spec["cache_path"])
        if _memmap_valid(spec):
            continue
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp_path = path.with_suffix(path.suffix + ".building")
        try:
            group_path = Path(spec["source"])
            src_array = open_group(group_path)[spec["name"]]
            progress_path = tmp_path.with_suffix(tmp_path.suffix + ".progress.json")
            signature = {"source": str(group_path), "name": str(spec["name"]),
                         "shape": list(spec["shape"]), "dtype": str(spec["dtype"]),
                         "nbytes": int(spec["nbytes"])}
            start_row = 0
            # Reuse a valid partial .npy instead of truncating it on every restart.
            partial_valid = False
            if tmp_path.exists():
                try:
                    partial = np.load(tmp_path, mmap_mode="r", allow_pickle=False)
                    partial_valid = (tuple(partial.shape) == tuple(spec["shape"])
                                     and partial.dtype.str == np.dtype(spec["dtype"]).str)
                    del partial
                except Exception:
                    partial_valid = False
            saved_progress = None
            if partial_valid and progress_path.exists():
                try:
                    saved_progress = json.loads(progress_path.read_text(encoding="utf-8"))
                except Exception:
                    saved_progress = None
            if saved_progress and saved_progress.get("signature") == signature:
                start_row = max(0, min(int(saved_progress.get("completed_rows", 0)),
                                       int(spec["shape"][0]) if spec["shape"] else 0))
                print(f"  [RAW-RASTER-MIRROR] resuming {spec['name']} at row {start_row:,}/"
                      f"{int(spec['shape'][0]) if spec['shape'] else 1:,}")
            elif tmp_path.exists() and not partial_valid:
                # A corrupt/incompatible partial file cannot be resumed safely.
                tmp_path.unlink()
            mode = "r+" if partial_valid else "w+"
            dst = np.lib.format.open_memmap(tmp_path, mode=mode, dtype=np.dtype(spec["dtype"]),
                                            shape=tuple(spec["shape"]))
            def save_progress(completed_rows):
                record = {"signature": signature, "completed_rows": int(completed_rows),
                          "updated_unix": time.time()}
                progress_tmp = progress_path.with_name(progress_path.name + ".tmp")
                progress_tmp.write_text(json.dumps(record, sort_keys=True), encoding="utf-8")
                os.replace(progress_tmp, progress_path)
            if len(spec["shape"]) == 0:
                dst[...] = src_array[()]
                dst.flush()
                save_progress(1)
            elif int(spec["shape"][0]) > 0:
                source_chunks = getattr(src_array, "chunks", None)
                source_chunk_rows = int(source_chunks[0]) if source_chunks and source_chunks[0] else int(MEMMAP_COPY_ROWS)
                copy_rows = max(1, min(max(int(MEMMAP_COPY_ROWS), source_chunk_rows), 4096))
                n_rows = int(spec["shape"][0])
                # Progress is durable at each source-chunk-aligned block. An interruption loses
                # at most the current block, and a restart continues at the last committed row.
                for row_start in range(start_row, n_rows, copy_rows):
                    row_end = min(n_rows, row_start + copy_rows)
                    dst[row_start:row_end] = src_array[row_start:row_end]
                    dst.flush()
                    save_progress(row_end)
                    if row_end == n_rows or row_end % max(copy_rows * 8, 1) == 0:
                        print(f"  [RAW-RASTER-MIRROR] {spec['modality']}/{spec['year']}/{spec['name']}: "
                              f"{row_end:,}/{n_rows:,} rows ({100.0*row_end/max(1,n_rows):.1f}%)")
            dst.flush()
            del dst
            # Flush file contents before publishing the completed array atomically.
            try:
                with open(tmp_path, "rb") as f:
                    os.fsync(f.fileno())
            except OSError:
                pass
            os.replace(tmp_path, path)
            try:
                progress_path.unlink(missing_ok=True)
            except Exception:
                pass
            built += 1
            elapsed = time.perf_counter() - started
            print(f"  [{spec_i}/{len(specs)}] cached {spec['modality']}/{spec['year']}/{spec['name']} "
                  f"({int(spec['nbytes'])/_GIB:.2f} GiB; {elapsed/60:.1f} min elapsed)")
        except Exception as exc:
            # Keep the .building file and progress sidecar. The next run can resume safely.
            print(f"[RAW-RASTER-MIRROR] Copy paused for {path.name}: {type(exc).__name__}: {exc}. "
                  f"Partial data retained at {tmp_path}; next run will resume from its checkpoint. "
                  "Direct Zarr reads will be used for this run.")
            return False

    # Write a small audit manifest after all arrays validate. The array files themselves
    # are reusable independently on a restart because each is published atomically.
    cache_manifest = {
        "schema_version": int(globals().get("ZARR_DISK_CACHE_SCHEMA_VERSION", 1)),
        "storage": "raw_native_dtype_npy_memmap",
        "active_modis_arrays": ["evi", "lst"],
        "active_s1_arrays": list(S1_CACHE_ARRAYS),
        "total_bytes": total_bytes,
        "max_cache_bytes": max_bytes,
        "arrays": [{"year": int(x["year"]), "modality": x["modality"], "name": x["name"],
                    "shape": list(x["shape"]), "dtype": x["dtype"], "nbytes": int(x["nbytes"])}
                   for x in specs],
    }
    manifest_path = RAM_CACHE_DIR / "cache_manifest.json"
    tmp_manifest = manifest_path.with_suffix(".json.tmp")
    tmp_manifest.write_text(json.dumps(cache_manifest, indent=2) + "\n", encoding="utf-8")
    os.replace(tmp_manifest, manifest_path)

    invalid = [x for x in specs if not _memmap_valid(x)]
    if invalid:
        print(f"Raster mirror incomplete ({len(invalid)} arrays not validated); using direct Zarr reads.")
        return False
    used_bytes = sum(Path(x["cache_path"]).stat().st_size for x in specs)
    print(f"Disk-backed active raster mirror ready: {len(specs)} arrays, {used_bytes/_GIB:.2f} GiB "
          f"({built} newly built). Native dtypes preserved; no normalized patch files written.")
    print(f"Free disk after mirror: {shutil.disk_usage(RUN_DIR).free/_GIB:.2f} GiB. "
          f"Hard cache cap: {max_bytes/_GIB:.2f} GiB; free-space reserve: {reserve_bytes/_GIB:.2f} GiB.")
    return True


def _worker_init_fn(_worker_id):
    # Keep each worker from spawning nested BLAS/OpenMP pools.
    try:
        torch.set_num_threads(1)
    except Exception:
        pass
    try:
        if psutil is not None:
            import threadpoolctl
            threadpoolctl.threadpool_limits(limits=1)
    except Exception:
        pass


def _candidate_worker_counts():
    cap = resource_limited_worker_cap()
    vals = [0]
    for n in (1, 2, 4, 6, 8, 12, 16):
        if 1 <= n <= cap and n not in vals:
            vals.append(n)
    if cap and cap not in vals:
        vals.append(cap)
    return sorted(set(vals))


# =============================== LAZY DATASET ===============================
class GlobalSoilDataset(IterableDataset):
    def __init__(self, manifest_path, task_mode, target_mode, tabular_scaler, raster_scaler,
                 max_rows=None, shuffle=False, seed=SEED, start_row=0, skip_raster=False):
        super().__init__()
        self.manifest_path = Path(manifest_path)
        self.task_mode = task_mode
        self.target_mode = target_mode
        self.tabular_scaler = tabular_scaler
        self.raster_scaler = raster_scaler
        self.max_rows = max_rows
        self.shuffle = shuffle
        self.seed = seed
        self.start_row = max(0, int(start_row))
        self.skip_raster = bool(skip_raster)
        if self.start_row and self.shuffle:
            raise ValueError("start_row/resume requires shuffle=False for deterministic source ordering")
        # Read lightweight Parquet metadata only; each worker opens its own reader.
        _metadata_reader = pq.ParquetFile(str(self.manifest_path))
        self.n_row_groups = _metadata_reader.num_row_groups
        self.n_rows = _metadata_reader.metadata.num_rows
        self.row_group_offsets = []
        offset = 0
        for rg in range(self.n_row_groups):
            self.row_group_offsets.append(offset)
            offset += int(_metadata_reader.metadata.row_group(rg).num_rows)
        close_reader = getattr(_metadata_reader, "close", None)
        if callable(close_reader):
            close_reader()
        del _metadata_reader
        self.modis_cache = {}
        self.s1_cache = {}
        # Mmaps are opened per worker and share OS page cache.
        self.use_mmap_cache = bool(RAM_ZARR_CACHE_ACTIVE)

    def __len__(self):
        return min(self.n_rows, self.max_rows) if self.max_rows is not None else self.n_rows

    def _groups(self, year):
        year = int(year)
        if year not in self.modis_cache:
            if self.use_mmap_cache:
                self.modis_cache[year] = {
                    name: np.load(_cache_array_path("modis", year, name), mmap_mode="r", allow_pickle=False)
                    for name in ("evi", "lst")
                }
                self.s1_cache[year] = {
                    name: np.load(_cache_array_path("s1", year, name), mmap_mode="r", allow_pickle=False)
                    for name in S1_CACHE_ARRAYS
                }
            else:
                self.modis_cache[year] = open_group(MODIS_ROOT / str(year) / "patches.zarr")
                self.s1_cache[year] = open_group(S1_ROOT / f"s1_nearest_valid_{year}.zarr")
        return self.modis_cache[year], self.s1_cache[year]

    def _number(self, feature, value):
        st = self.tabular_scaler.get(feature, {"mean": 0.0, "std": 1.0})
        try:
            x = float(value)
        except (TypeError, ValueError):
            x = float("nan")
        if not np.isfinite(x):
            x = st["mean"]
        return np.float32((x - st["mean"]) / st["std"])

    def _normalize_raster(self, name, arr, valid):
        st = self.raster_scaler[name]
        arr = np.asarray(arr, dtype=np.float32)
        valid = np.asarray(valid, dtype=bool) & np.isfinite(arr)
        filled = np.where(valid, arr, st["mean"]).astype(np.float32, copy=False)
        normalized = (filled - st["mean"]) / st["std"]
        # Clip pathological standardized outliers so rare extreme ratios cannot dominate.
        normalized = np.clip(normalized, -10.0, 10.0).astype(np.float32, copy=False)
        return normalized, valid.astype(np.float32, copy=False)

    def _make(self, row):
        try:
            target = float(row[TARGET_COL])
        except (TypeError, ValueError):
            return None
        if not np.isfinite(target):
            return None
        try:
            smap = float(row["SMAP"])
        except (TypeError, ValueError):
            smap = float("nan")
        if self.target_mode == "residual" and not np.isfinite(smap):
            return None

        year = int(row["year"])
        mi, si = int(row["modis_row_index"]), int(row["s1_row_index"])

        if self.skip_raster:
            # Compact-cache construction stores only raster keys, not expanded patches.
            # Avoid opening/decoding Zarr data during this pass.
            evi_tensor = np.empty((0,), dtype=np.float32)
            lst_tensor = np.empty((0,), dtype=np.float32)
            s1_tensor = np.empty((0,), dtype=np.float32)
        else:
            modis, s1 = self._groups(year)
            evi_raw = modis["evi"][mi]
            lst_raw = modis["lst"][mi]
            evi_phys, evi_valid = decode_modis_patch(evi_raw, "evi")
            lst_phys, lst_valid = decode_modis_patch(lst_raw, "lst")
            evi_norm, evi_mask = self._normalize_raster("evi", evi_phys, evi_valid)
            lst_norm, lst_mask = self._normalize_raster("lst", lst_phys, lst_valid)
            # A validity mask is provided as an additional CNN channel for each modality.
            evi_tensor = np.stack([evi_norm, evi_mask], axis=0)
            lst_tensor = np.stack([lst_norm, lst_mask], axis=0)

            s1_values, s1_masks = [], []
            for name in S1_ARRAYS:
                raw = s1[name][si]
                physical, valid = decode_s1_patch(raw)
                norm, mask = self._normalize_raster(name, physical, valid)
                s1_values.append(norm)
                s1_masks.append(mask)
            s1_tensor = np.stack(s1_values + s1_masks, axis=0)

        # Sequence tokens oldest -> current. Token features contain standardized
        # weather variables, day-of-year sine/cosine, and age in days / max window.
        # Per-token weather values + calendar/age encodings + one missingness bit per weather feature.
        seq = np.zeros((HISTORY_STEPS, 2 * len(TEMP_FEATURES) + 3), dtype=np.float32)
        seq_mask = np.ones(HISTORY_STEPS, dtype=bool)  # True means padding/key is masked.
        current_date = pd.Timestamp(row["sample_date"]).normalize()

        for lag in range(HISTORY_STEPS):
            pos = HISTORY_STEPS - 1 - lag
            if lag == 0:
                token_date = current_date
            else:
                date_value = row.get(f"lag_date_{lag}")
                if date_value is None or pd.isna(date_value):
                    continue
                token_date = pd.Timestamp(date_value).normalize()
                age_days = int((current_date - token_date).days)
                if age_days < 0 or age_days > HISTORY_MAX_DAYS:
                    continue
            if lag == 0:
                age_days = 0
            weather_missing = []
            for j, feat in enumerate(TEMP_FEATURES):
                col = feat if lag == 0 else f"{feat}_lag{lag}"
                raw_value = row.get(col)
                try:
                    raw_float = float(raw_value)
                    missing = not np.isfinite(raw_float)
                except (TypeError, ValueError):
                    missing = True
                seq[pos, j] = self._number(feat, raw_value)
                weather_missing.append(1.0 if missing else 0.0)
            doy = token_date.dayofyear
            seq[pos, len(TEMP_FEATURES)] = np.sin(2 * np.pi * doy / 365.25)
            seq[pos, len(TEMP_FEATURES) + 1] = np.cos(2 * np.pi * doy / 365.25)
            seq[pos, len(TEMP_FEATURES) + 2] = age_days / max(HISTORY_MAX_DAYS, 1)
            seq[pos, len(TEMP_FEATURES) + 3:] = np.asarray(weather_missing, dtype=np.float32)
            seq_mask[pos] = False

        static_features = STATIC_FEATURES + ANCHOR_FEATURES
        if self.task_mode == "location_history":
            static_features += LOCATION_FEATURES
        static_values, static_missing = [], []
        for feat in static_features:
            raw_value = row.get(feat)
            try:
                raw_float = float(raw_value)
                missing = not np.isfinite(raw_float)
            except (TypeError, ValueError):
                missing = True
            static_values.append(self._number(feat, raw_value))
            static_missing.append(1.0 if missing else 0.0)
        static = np.asarray(static_values + static_missing, dtype=np.float32)

        y = target - smap if self.target_mode == "residual" else target
        return {
            "evi": torch.from_numpy(evi_tensor),
            "lst": torch.from_numpy(lst_tensor),
            "s1": torch.from_numpy(s1_tensor),
            "seq": torch.from_numpy(seq),
            "seq_mask": torch.from_numpy(seq_mask),
            "static": torch.from_numpy(static),
            "y": torch.tensor([y], dtype=torch.float32),
            "anchor": torch.tensor([smap if np.isfinite(smap) else 0.0], dtype=torch.float32),
            "anchor_valid": torch.tensor([np.isfinite(smap)], dtype=torch.bool),
            "sample_id": str(row["sample_id"]),
            # Stable source keys let the compact cache reuse spatial patches without duplicating
            # them once per sample/date.
            "_year": year,
            "_modis_row_index": mi,
            "_s1_row_index": si,
        }

    def __iter__(self):
        worker = get_worker_info()
        worker_id = worker.id if worker is not None else 0
        worker_count = worker.num_workers if worker is not None else 1
        parquet = pq.ParquetFile(str(self.manifest_path))
        group_ids = list(range(self.n_row_groups))
        if self.shuffle:
            random.Random(self.seed + 17).shuffle(group_ids)
        group_ids = group_ids[worker_id::worker_count]
        if self.max_rows is None:
            worker_limit = None
        else:
            worker_limit = max(0, self.max_rows // worker_count + (1 if worker_id < self.max_rows % worker_count else 0))
        emitted = 0
        try:
            if worker_limit == 0:
                return
            for rg in group_ids:
                group_start = int(self.row_group_offsets[rg])
                group_rows = int(parquet.metadata.row_group(rg).num_rows)
                group_end = group_start + group_rows
                if group_end <= self.start_row:
                    continue
                source_row = group_start
                for batch_i, record_batch in enumerate(parquet.iter_batches(
                    batch_size=ARROW_BATCH_ROWS, row_groups=[rg], use_threads=False
                )):
                    rows = record_batch.to_pylist()  # bounded by ARROW_BATCH_ROWS
                    indexed_rows = [(source_row + j, row) for j, row in enumerate(rows)]
                    source_row += len(rows)
                    if self.shuffle and len(indexed_rows) > 1:
                        shuffle_seed = self.seed + worker_id * 1000003 + rg * 1009 + batch_i + (int(time.time()) // 60)
                        random.Random(shuffle_seed).shuffle(indexed_rows)
                    for source_idx, row in indexed_rows:
                        if source_idx < self.start_row:
                            continue
                        item = self._make(row)
                        if item is None:
                            continue
                        # Internal cursor used only by the resumable cache builder; other
                        # consumers can safely ignore this extra metadata field.
                        item["_source_row_index"] = int(source_idx)
                        yield item
                        emitted += 1
                        if worker_limit is not None and emitted >= worker_limit:
                            return
        finally:
            close_reader = getattr(parquet, "close", None)
            if callable(close_reader):
                close_reader()

def collate_global(items):
    items = [x for x in items if x is not None]
    if not items:
        return None
    tensor_keys = ["evi", "lst", "s1", "seq", "seq_mask", "static", "y", "anchor"]
    if "anchor_valid" in items[0]:
        tensor_keys.append("anchor_valid")
    batch = {key: torch.stack([x[key] for x in items]) for key in tensor_keys}
    batch["sample_id"] = [x["sample_id"] for x in items]
    if "_source_row_index" in items[0]:
        batch["_source_row_index"] = [int(x["_source_row_index"]) for x in items]
    if "_year" in items[0]:
        batch["_year"] = [int(x["_year"]) for x in items]
        batch["_modis_row_index"] = [int(x["_modis_row_index"]) for x in items]
        batch["_s1_row_index"] = [int(x["_s1_row_index"]) for x in items]
    return batch

def build_global_loader(dataset, batch_size, *, pin_memory=PIN_MEMORY, num_workers=None,
                        persistent_workers=None, multiprocessing_context=None,
                        eval_max_batches=None):
    """Build raw-stream or cached loader with notebook-safe multiprocessing.

    The Dataset classes live in this notebook's __main__ namespace. On Python 3.14,
    the default forkserver context tries to import them in a fresh child and fails with
    "__main__ has no attribute GlobalSoilDataset". Prefer an explicit fork context on
    POSIX. Workers must remain CPU-only: do not create CUDA tensors or call CUDA here.
    On systems without fork, use num_workers=0 rather than starting workers that cannot
    reliably import notebook-defined classes.
    """
    worker_count = max(0, int(NUM_WORKERS if num_workers is None else num_workers))
    requested_context = multiprocessing_context or DATALOADER_MP_CONTEXT
    if requested_context is None and worker_count > 0:
        print("[DATALOADER-SAFETY] multiprocessing disabled to avoid fork-after-CUDA; "
              "using main-process collation with asynchronous device prefetch.", flush=True)
        worker_count = 0
    if worker_count > 0 and requested_context is None:
        # Spawn/forkserver cannot reliably import Dataset classes declared in notebook cells.
        print("DataLoader multiprocessing disabled: this platform lacks POSIX fork, which this "
              "notebook-defined Dataset requires. Using num_workers=0 for reliability.")
        worker_count = 0
    batch_size = max(1, int(batch_size))
    is_cached = isinstance(dataset, CachedGlobalDataset)
    kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "num_workers": worker_count,
        "pin_memory": bool(pin_memory),
        "collate_fn": CachedGlobalBatchCollator(dataset.cache_dir, dataset.metadata, dataset.target_mode)
                     if is_cached else collate_global,
        "worker_init_fn": _worker_init_fn if worker_count > 0 else None,
    }
    if worker_count > 0:
        kwargs["multiprocessing_context"] = requested_context
    # Training uses true batch-level locality: samples are globally raster-sorted, split
    # into broad 131k-sample windows, then WHOLE BATCHES are shuffled within each window.
    # Samples inside each emitted batch retain their raster order. Evaluation remains ordered.
    if is_cached:
        if bool(dataset.shuffle):
            _physical_batch_size = int(kwargs.pop("batch_size"))
            kwargs["batch_sampler"] = RasterLocalityBatchSampler(
                dataset, batch_size=_physical_batch_size,
                window_size=int(TRAIN_LOCALITY_WINDOW_SIZE), seed=int(dataset.seed))
        else:
            # Evaluation order does not affect aggregate metrics. Select a deterministic,
            # evenly spaced subset when requested, then sort it by raster keys.
            _max_eval_samples = (None if eval_max_batches is None
                                 else max(1, int(eval_max_batches) * batch_size))
            kwargs["sampler"] = RasterLocalityEvalSampler(
                dataset, max_samples=_max_eval_samples, seed=int(globals().get("OPTUNA_SEED", dataset.seed)))
        kwargs["generator"] = torch.Generator().manual_seed(int(dataset.seed))
    if worker_count > 0:
        prefetch_budget = max(1, int(globals().get("MAX_PREFETCHED_SAMPLES_PER_LOADER", 16384)))
        prefetch_cap = max(1, prefetch_budget // max(1, worker_count * batch_size))
        kwargs["prefetch_factor"] = max(1, min(int(PREFETCH_FACTOR), prefetch_cap))
        kwargs["timeout"] = int(globals().get("DATALOADER_TIMEOUT_SEC", 300))
        kwargs["persistent_workers"] = (True if persistent_workers is None else bool(persistent_workers))
    return DataLoader(**kwargs)

def make_loader(split, task_mode=TASK_MODE, max_rows=None, shuffle=False, batch_size=None, num_workers=None):
    dataset = get_global_dataset(split=split, task_mode=task_mode, target_mode=TARGET_MODE,
                                 max_rows=max_rows, shuffle=shuffle, seed=SEED)
    return build_global_loader(dataset, min(int(TRAIN_MICROBATCH_CAP), int(BATCH_SIZE)) if batch_size is None else batch_size,
                               num_workers=num_workers)


# =============================== MODEL-READY TENSOR CACHE ===============================
# The cache is independent of target_mode. Each sample stores direct y and anchor;
# residual mode is formed cheaply at batch collation time and rows without a valid anchor
# are filtered by CachedGlobalDataset. Therefore direct/residual Optuna trials share cache.
_MODEL_CACHE_FIELDS = {
    "evi": "evi", "lst": "lst", "s1": "s1", "seq": "seq",
    "seq_mask": "seq_mask", "static": "static", "y_direct": "y",
    "anchor": "anchor", "anchor_valid": "anchor_valid",
}
_MODEL_CACHE_KEYS = ("evi", "lst", "s1", "seq", "seq_mask", "static", "y_direct", "anchor", "anchor_valid")
MODEL_READY_CACHE_PATHS = {}


def _json_fingerprint(obj):
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def _model_ready_cache_signature(task_mode, split):
    """Semantic inputs signature. Excludes builder ID so harmless code-version changes don't break resume."""
    manifest = Path(manifest_paths[(task_mode, split)])
    st = manifest.stat()
    parquet_file = pq.ParquetFile(str(manifest))
    metadata = parquet_file.metadata
    return {
        "schema": MODEL_CACHE_SCHEMA_VERSION,
        "task_mode": task_mode,
        "split": split,
        # A basename is sufficient here because task_mode/split are separate keys;
        # absolute parent directories differ when the bundle moves to another machine.
        "manifest_path": manifest.name,
        "manifest_size": int(st.st_size),
        "manifest_num_rows": int(metadata.num_rows),
        "manifest_num_row_groups": int(metadata.num_row_groups),
        "manifest_schema": str(parquet_file.schema),
        "tabular_scaler": tabular_scalers[task_mode],
        "raster_scaler": raster_scalers[task_mode],
        "temp_features": TEMP_FEATURES,
        "static_features": STATIC_FEATURES,
        "location_features": LOCATION_FEATURES if task_mode == "location_history" else [],
        "anchor_features": ANCHOR_FEATURES,
        "s1_arrays": S1_ARRAYS,
        "history_steps": HISTORY_STEPS,
        "history_max_days": HISTORY_MAX_DAYS,
        "modis_decode": [EVI_FILL_RAW, EVI_RAW_MIN, EVI_RAW_MAX, EVI_SCALE,
                         LST_FILL_RAW, LST_RAW_MIN, LST_RAW_MAX, LST_SCALE],
    }


def _model_ready_cache_identity(task_mode, split):
    """Return a stable cache fingerprint/path that survives harmless file-mtime changes."""
    sig = dict(_model_ready_cache_signature(task_mode, split))
    sig["cache_builder_id"] = MODEL_CACHE_BUILDER_ID
    # Scaler means/stds can differ in their last few float digits after independent
    # reductions. Those differences do not change the normalized cache at meaningful
    # precision, and must not create a new multi-GiB cache directory on each restart.
    _canonical_sig = _canonical_cache_signature(sig, float_digits=7)
    fingerprint = _json_fingerprint(_canonical_sig)
    path = Path(MODEL_READY_CACHE_DIR) / f"{task_mode}_{split}_{fingerprint[:16]}"
    return fingerprint, path


def _legacy_v2_cache_identity(task_mode, split):
    """Reconstruct the v2 fingerprint so an existing partial v2 cache can be migrated."""
    manifest = Path(manifest_paths[(task_mode, split)])
    st = manifest.stat()
    sig = {
        "schema": 2,
        "cache_builder_id": "resumable_checkpoint_v2_2026_10_18",
        "task_mode": task_mode,
        "split": split,
        "manifest_path": str(manifest.resolve()),
        "manifest_size": int(st.st_size),
        "manifest_mtime_ns": int(st.st_mtime_ns),
        "tabular_scaler": tabular_scalers[task_mode],
        "raster_scaler": raster_scalers[task_mode],
        "temp_features": TEMP_FEATURES,
        "static_features": STATIC_FEATURES,
        "location_features": LOCATION_FEATURES if task_mode == "location_history" else [],
        "anchor_features": ANCHOR_FEATURES,
        "s1_arrays": S1_ARRAYS,
        "history_steps": HISTORY_STEPS,
        "history_max_days": HISTORY_MAX_DAYS,
        "modis_decode": [EVI_FILL_RAW, EVI_RAW_MIN, EVI_RAW_MAX, EVI_SCALE,
                         LST_FILL_RAW, LST_RAW_MIN, LST_RAW_MAX, LST_SCALE],
    }
    legacy_fingerprint = _json_fingerprint(sig)
    legacy_path = Path(MODEL_READY_CACHE_DIR) / f"{task_mode}_{split}_{legacy_fingerprint[:16]}"
    return legacy_fingerprint, legacy_path


def _atomic_write_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
        f.flush()
        if bool(globals().get("MODEL_CACHE_FSYNC_CHECKPOINTS", True)):
            os.fsync(f.fileno())
    os.replace(tmp, path)


def _validate_legacy_staging_candidate(candidate, task_mode, split, capacity_rows, source_rows,
                                       current_signature):
    """Validate a prior-version partial cache structurally; return checkpoint and identity strength.

    Older checkpoints did not record the full input signature. Such candidates can only be
    recovered when split, capacity, source cursor, row layout, and every backing-file size agree.
    This is a weaker compatibility check, so the caller reports it explicitly.
    """
    candidate = Path(candidate)
    cp_path = candidate / "checkpoint.json"
    if not cp_path.exists():
        return None
    try:
        cp = json.loads(cp_path.read_text(encoding="utf-8"))
        if cp.get("layout_mismatch"):
            return None
        if int(cp.get("resume_schema", -1)) != 1:
            return None
        if cp.get("task_mode") != task_mode or cp.get("split") != split:
            return None
        if int(cp.get("capacity_rows", -1)) != int(capacity_rows):
            return None
        committed = int(cp.get("committed_rows", -1))
        cursor = int(cp.get("source_row_cursor", -1))
        eof = bool(cp.get("source_eof", False))
        if not (0 <= committed <= capacity_rows and 0 <= cursor <= source_rows):
            return None
        if committed > 0 and cursor <= 0:
            return None
        arrays = cp.get("arrays")
        if not isinstance(arrays, dict) or set(arrays) != set(_MODEL_CACHE_KEYS):
            return None

        # If a prior checkpoint captured these source fields, require an exact match.
        manifest = Path(manifest_paths[(task_mode, split)])
        st = manifest.stat()
        pf = pq.ParquetFile(str(manifest))
        optional_expected = {
            "manifest_path": str(manifest.resolve()),
            "manifest_size": int(st.st_size),
            "manifest_num_rows": int(pf.metadata.num_rows),
            "manifest_num_row_groups": int(pf.metadata.num_row_groups),
            "manifest_schema": str(pf.schema),
        }
        for key, expected in optional_expected.items():
            if key not in cp:
                continue
            actual = cp[key]
            if key == "manifest_path":
                if Path(str(actual)).name != Path(str(expected)).name:
                    return None
            elif actual != expected:
                return None

        cp_signature = cp.get("input_signature")
        if cp_signature is not None:
            if not _cache_signatures_match(cp_signature, current_signature):
                return None
            identity_strength = 2  # semantic source/scaler/features match; absolute root may differ
        else:
            identity_strength = 1  # legacy: structural compatibility only

        seen_files = set()
        for key, spec in arrays.items():
            if not isinstance(spec, dict) or not {"file", "dtype", "shape"}.issubset(spec):
                return None
            filename = str(spec["file"])
            if Path(filename).name != filename or filename in seen_files:
                return None
            seen_files.add(filename)
            shape = tuple(int(x) for x in spec["shape"])
            if any(x < 0 for x in shape):
                return None
            dtype = np.dtype(spec["dtype"])
            row_bytes = int(np.prod(shape, dtype=np.int64)) * dtype.itemsize
            capacity_bytes = int(capacity_rows) * row_bytes
            committed_bytes = int(committed) * row_bytes
            array_path = candidate / filename
            if not array_path.exists():
                return None
            actual_bytes = array_path.stat().st_size
            allowed = {capacity_bytes}
            if eof:
                allowed.add(committed_bytes)
            if actual_bytes not in allowed:
                return None

        # Keep the returned object purely serializable and record the actual confidence level.
        cp["_legacy_identity_strength"] = identity_strength
        return cp
    except Exception:
        return None


def _migrate_legacy_v2_cache(task_mode, split, fingerprint, cache_dir, capacity_rows):
    """Recover a completed legacy cache or the most advanced structurally-compatible staging cache."""
    legacy_fp, legacy_dir = _legacy_v2_cache_identity(task_mode, split)
    cache_dir = Path(cache_dir)
    current_building = cache_dir.with_name(cache_dir.name + ".building")
    current_signature = _model_ready_cache_signature(task_mode, split)
    manifest = Path(manifest_paths[(task_mode, split)])
    source_rows = int(pq.ParquetFile(str(manifest)).metadata.num_rows)

    # First try exact legacy-v2 identity migration, which is stronger than structural recovery.
    if legacy_dir != cache_dir and not cache_dir.exists() and not current_building.exists():
        legacy_building = legacy_dir.with_name(legacy_dir.name + ".building")
        if legacy_building.exists():
            cp = _load_resumable_cache_checkpoint(legacy_building, legacy_fp, capacity_rows, source_rows)
            if cp is not None and cp.get("task_mode") == task_mode and cp.get("split") == split:
                cp["fingerprint"] = fingerprint
                cp["input_signature"] = current_signature
                cp["migrated_from_fingerprint"] = legacy_fp
                cp["identity_migration"] = "v2_mtime_sensitive_to_v3_stable"
                _write_cache_checkpoint(legacy_building / "checkpoint.json", cp)
                os.replace(legacy_building, current_building)
                print(f"[Model cache {task_mode}/{split}] RECOVERED exact legacy-v2 checkpoint: "
                      f"{cp['committed_rows']:,}/{capacity_rows:,} committed samples; no rebuild.")
                return True
        if legacy_dir.exists():
            meta = _validate_model_ready_cache(legacy_dir, legacy_fp)
            if meta is not None:
                meta["fingerprint"] = fingerprint
                meta["cache_builder_id"] = MODEL_CACHE_BUILDER_ID
                _atomic_write_json(legacy_dir / "metadata.json", meta)
                os.replace(legacy_dir, cache_dir)
                print(f"[Model cache {task_mode}/{split}] Migrated completed legacy-v2 cache; reused {meta['n_rows']:,} samples.")
                return True

    # General recovery for older runs whose fingerprint changed due to builder IDs, timestamps,
    # or previous identity bugs. Prefer a matching saved input_signature; for older checkpoints
    # without one, use split/capacity/array-file validation and warn that source settings must be unchanged.
    candidates = sorted(Path(MODEL_READY_CACHE_DIR).glob(f"{task_mode}_{split}_*.building"))
    compatible = []
    for candidate in candidates:
        cp = _validate_legacy_staging_candidate(candidate, task_mode, split, capacity_rows,
                                                source_rows, current_signature)
        if cp is not None:
            strength = int(cp.pop("_legacy_identity_strength", 1))
            compatible.append((strength, int(cp.get("committed_rows", 0)),
                               int(cp.get("source_row_cursor", 0)), candidate, cp))

    if compatible:
        # Prefer the checkpoint with the most durable work so a newly-created zero-row current
        # staging directory cannot mask an older, much more advanced partial cache. Candidates
        # with an explicit mismatching input signature were already rejected above.
        compatible.sort(key=lambda item: (item[1], item[0], item[2]), reverse=True)
        strength, committed, cursor, chosen, cp = compatible[0]
        chosen = Path(chosen)

        # Preserve, never delete, a stale destination/staging directory before adopting another.
        if current_building.exists() and current_building.resolve() != chosen.resolve():
            quarantine = current_building.with_name(current_building.name + f".superseded_{int(time.time())}")
            os.replace(current_building, quarantine)
            print(f"[Model cache {task_mode}/{split}] Preserved superseded staging cache at {quarantine}")
        if cache_dir.exists() and _validate_model_ready_cache(cache_dir, fingerprint) is None:
            quarantine = cache_dir.with_name(cache_dir.name + f".invalid_{int(time.time())}")
            os.replace(cache_dir, quarantine)
            print(f"[Model cache {task_mode}/{split}] Preserved invalid cache destination at {quarantine}")
        if chosen.resolve() != current_building.resolve():
            os.replace(chosen, current_building)

        cp["fingerprint"] = fingerprint
        cp["input_signature"] = current_signature
        cp["legacy_adopted_from"] = str(chosen)
        cp["legacy_identity_recovery"] = ("saved_input_signature_match" if strength == 2
                                           else "structural_only_assumes_manifest_and_scalers_unchanged")
        cp["legacy_identity_strength"] = strength
        _write_cache_checkpoint(current_building / "checkpoint.json", cp)
        if strength == 2:
            print(f"[Model cache {task_mode}/{split}] RECOVERED preserved checkpoint with matching input signature: "
                  f"{committed:,}/{capacity_rows:,} rows; resuming at source row {cursor:,} from {current_building}")
        else:
            print(f"[Model cache {task_mode}/{split}] RECOVERED LEGACY checkpoint by structural validation: "
                  f"{committed:,}/{capacity_rows:,} committed samples; resuming at source row {cursor:,} from {current_building}")
            print("WARNING: this older checkpoint did not save the input/scaler signature. Resume assumes the "
                  "manifest contents, feature definitions, raster decoding and scalers have not changed since it began.")
        if len(compatible) > 1:
            print(f"[Model cache {task_mode}/{split}] Other compatible candidates preserved: {len(compatible)-1}; "
                  f"selected the best-advanced compatible checkpoint.")
        return True

    # No compatible candidate: explain all preserved checkpoints, without removing anything.
    for candidate in candidates:
        cp_path = candidate / "checkpoint.json"
        if cp_path.exists():
            try:
                cp = json.loads(cp_path.read_text(encoding="utf-8"))
                print(f"[Model cache {task_mode}/{split}] Preserved checkpoint not resumed: "
                      f"{cp.get('committed_rows', '?'):,}/{cp.get('capacity_rows', '?'):,} rows at {candidate}. "
                      "It failed task/split/capacity/signature/array-file validation.")
            except Exception:
                print(f"[Model cache {task_mode}/{split}] Unreadable staging directory preserved at {candidate}.")
    return False


def _validate_model_ready_cache(cache_dir, expected_fingerprint=None):
    cache_dir = Path(cache_dir)
    meta_path = cache_dir / "metadata.json"
    if not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if int(meta.get("schema", -1)) != int(MODEL_CACHE_SCHEMA_VERSION):
            return None
        if expected_fingerprint is not None and meta.get("fingerprint") != expected_fingerprint:
            return None
        n = int(meta["n_rows"])
        for key, spec in meta["arrays"].items():
            path = cache_dir / spec["file"]
            dt = np.dtype(spec["dtype"])
            row_bytes = int(np.prod(spec["shape"], dtype=np.int64)) * dt.itemsize
            if not path.exists() or path.stat().st_size != n * row_bytes:
                return None
        return meta
    except Exception:
        return None


class CachedGlobalDataset(Dataset):
    """Map-style dataset that returns cache row indices; arrays are fetched in batch by collator."""
    def __init__(self, cache_dir, target_mode="residual", max_rows=None, shuffle=False, seed=SEED):
        self.cache_dir = Path(cache_dir)
        self.metadata = _validate_model_ready_cache(self.cache_dir)
        if self.metadata is None:
            raise RuntimeError(f"Model-ready cache is invalid or incomplete: {self.cache_dir}")
        self.target_mode = str(target_mode)
        if self.target_mode not in ("direct", "residual"):
            raise ValueError("target_mode must be 'direct' or 'residual'")
        self.shuffle = bool(shuffle)
        self.seed = int(seed)
        self._valid_rows = None
        n = int(self.metadata["n_rows"])
        if self.target_mode == "residual":
            spec = self.metadata["arrays"]["anchor_valid"]
            valid = np.memmap(self.cache_dir / spec["file"], dtype=np.dtype(spec["dtype"]),
                              mode="r", shape=(n, *spec["shape"]))
            self._valid_rows = np.flatnonzero(np.asarray(valid).reshape(n)).astype(np.int64, copy=False)
            del valid
            self._length = len(self._valid_rows)
        else:
            self._length = n
        if max_rows is not None:
            self._length = min(self._length, max(0, int(max_rows)))

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        index = int(index)
        if index < 0:
            index += self._length
        if index < 0 or index >= self._length:
            raise IndexError(index)
        return index if self._valid_rows is None else int(self._valid_rows[index])

    def __getitems__(self, indices):
        # DataLoader's batched-fetch path accepts an ndarray. Returning it directly avoids
        # boxing every row index into a Python int/list and converting it back in collate.
        positions = np.asarray(indices, dtype=np.int64)
        return positions if self._valid_rows is None else self._valid_rows[positions]


class CachedGlobalBatchCollator:
    """Open each memory-mapped tensor once per loader worker; gather whole batches in NumPy."""
    def __init__(self, cache_dir, metadata, target_mode):
        self.cache_dir = str(cache_dir)
        self.metadata = metadata
        self.target_mode = str(target_mode)
        self._arrays = None

    def _open(self):
        if self._arrays is not None:
            return
        n = int(self.metadata["n_rows"])
        arrays = {}
        for key in _MODEL_CACHE_KEYS:
            spec = self.metadata["arrays"][key]
            arrays[key] = np.memmap(Path(self.cache_dir) / spec["file"],
                                    dtype=np.dtype(spec["dtype"]), mode="r",
                                    shape=(n, *spec["shape"]))
        self._arrays = arrays

    def __call__(self, row_indices):
        self._open()
        rows = np.asarray(row_indices, dtype=np.int64)
        batch = {}
        for key in ("evi", "lst", "s1", "seq", "seq_mask", "static", "anchor"):
            # Advanced indexing gives one contiguous batch copy instead of per-sample Python stacks.
            data = np.asarray(self._arrays[key][rows])
            if not data.flags.c_contiguous:
                data = np.ascontiguousarray(data)
            batch[key] = torch.from_numpy(data)
        direct_y = np.asarray(self._arrays["y_direct"][rows]).copy()
        if self.target_mode == "residual":
            batch["y"] = torch.from_numpy(direct_y - batch["anchor"].numpy())
        else:
            batch["y"] = torch.from_numpy(direct_y)
        # IDs are not model inputs or metric keys; expose stable cache-row IDs for diagnostics.
        batch["sample_id"] = [f"cache_row_{int(i)}" for i in rows]
        return batch


def _write_cache_checkpoint(checkpoint_path, payload):
    """Atomically commit the cache's durable write position."""
    checkpoint_path = Path(checkpoint_path)
    tmp_path = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
    payload = dict(payload)
    payload["updated_unix"] = time.time()
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
        f.flush()
        if bool(globals().get("MODEL_CACHE_FSYNC_CHECKPOINTS", True)):
            os.fsync(f.fileno())
    os.replace(tmp_path, checkpoint_path)
    if bool(globals().get("MODEL_CACHE_FSYNC_CHECKPOINTS", True)):
        try:
            fd = os.open(str(checkpoint_path.parent), os.O_RDONLY)
            try: os.fsync(fd)
            finally: os.close(fd)
        except Exception:
            pass


def _load_resumable_cache_checkpoint(tmp_dir, fingerprint, capacity_rows, source_rows):
    """Return a compatible checkpoint or None. Never trusts uncommitted rows."""
    tmp_dir = Path(tmp_dir)
    cp_path = tmp_dir / "checkpoint.json"
    if not cp_path.exists():
        return None
    try:
        cp = json.loads(cp_path.read_text(encoding="utf-8"))
        if (int(cp.get("resume_schema", -1)) != 1
                or cp.get("layout_mismatch")
                or cp.get("fingerprint") != fingerprint
                or int(cp.get("capacity_rows", -1)) != int(capacity_rows)
                or cp.get("task_mode") is None or cp.get("split") is None):
            return None
        committed = int(cp["committed_rows"])
        cursor = int(cp["source_row_cursor"])
        if not (0 <= committed <= capacity_rows and 0 <= cursor <= source_rows):
            return None
        arrays = cp["arrays"]
        if set(arrays) != set(_MODEL_CACHE_KEYS):
            return None
        source_eof = bool(cp.get("source_eof", False))
        for key, spec in arrays.items():
            path = tmp_dir / spec["file"]
            row_bytes = int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
            capacity_bytes = int(capacity_rows) * row_bytes
            committed_bytes = committed * row_bytes
            if not path.exists():
                return None
            size = path.stat().st_size
            permitted = {capacity_bytes}
            if source_eof:
                permitted.add(committed_bytes)
            if size not in permitted:
                return None
        return cp
    except Exception:
        return None


def _flush_cache_arrays(arrays, fsync=False):
    for arr in arrays.values():
        arr.flush()
    if fsync:
        for arr in arrays.values():
            filename = getattr(arr, "filename", None)
            if filename:
                try:
                    with open(filename, "rb") as f:
                        os.fsync(f.fileno())
                except Exception:
                    pass


def build_model_ready_cache(task_mode, split, force=False):
    """Build a persistent model-ready cache and resume from committed batches after interruption.

    Cache-build iteration is intentionally single-process so every output sample has a stable
    Parquet source-row cursor. Training on the completed cache still uses adaptive DataLoader
    workers. On interruption, a checkpoint records only fully flushed batches; an uncommitted
    partial batch is overwritten when resumed.
    """
    fingerprint, cache_dir = _model_ready_cache_identity(task_mode, split)
    manifest = Path(manifest_paths[(task_mode, split)])
    capacity_rows = int(pq.ParquetFile(str(manifest)).metadata.num_rows)
    MODEL_READY_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if not force:
        cached_meta = _validate_model_ready_cache(cache_dir, fingerprint)
        if cached_meta is not None:
            print(f"Model-ready cache reused [{task_mode}/{split}]: {cached_meta['n_rows']:,} samples at {cache_dir}")
            return cache_dir
        _migrate_legacy_v2_cache(task_mode, split, fingerprint, cache_dir, capacity_rows)
        cached_meta = _validate_model_ready_cache(cache_dir, fingerprint)
        if cached_meta is not None:
            print(f"Model-ready cache reused after identity migration [{task_mode}/{split}]: "
                  f"{cached_meta['n_rows']:,} samples at {cache_dir}")
            return cache_dir

    if capacity_rows <= 0:
        print(f"Model-ready cache skipped [{task_mode}/{split}]: empty manifest.")
        return None

    raw_ds_meta = GlobalSoilDataset(manifest, task_mode, "direct", tabular_scalers[task_mode],
                                    raster_scalers[task_mode], max_rows=None, shuffle=False, seed=SEED)
    tmp_dir = cache_dir.with_name(cache_dir.name + ".building")
    checkpoint_path = tmp_dir / "checkpoint.json"
    resume_cp = None if force else _load_resumable_cache_checkpoint(
        tmp_dir, fingerprint, capacity_rows, capacity_rows)
    if tmp_dir.exists() and resume_cp is None:
        # Never silently destroy a prior partial build. Quarantine it and leave it available
        # for manual inspection/recovery rather than erasing potentially valuable progress.
        quarantine = tmp_dir.with_name(tmp_dir.name + f".incompatible_{int(time.time())}")
        try:
            os.replace(tmp_dir, quarantine)
            print(f"[Model cache {task_mode}/{split}] Checkpoint was missing/incompatible. "
                  f"Preserved old staging data at {quarantine}; starting a new build in {tmp_dir}.")
        except Exception as move_exc:
            raise RuntimeError(f"Refusing to delete incompatible cache staging directory {tmp_dir}: {move_exc}")
    elif resume_cp is not None:
        print(f"[Model cache {task_mode}/{split}] RESUMING at source row "
              f"{resume_cp['source_row_cursor']:,}/{capacity_rows:,}; "
              f"{resume_cp['committed_rows']:,} samples durably committed from "
              f"{resume_cp.get('committed_batches', 0):,} committed batches.")

    start_cursor = int(resume_cp["source_row_cursor"]) if resume_cp else 0
    committed_rows = int(resume_cp["committed_rows"]) if resume_cp else 0
    written = committed_rows
    committed_batches = int(resume_cp.get("committed_batches", 0)) if resume_cp else 0
    batches_since_checkpoint = 0
    last_source_cursor = start_cursor
    source_eof = bool(resume_cp.get("source_eof", False)) if resume_cp else False
    specs = dict(resume_cp["arrays"]) if resume_cp else {}
    mmap_arrays = {}
    loader = None
    iterator = None
    pbar = None
    error = None
    published = False
    checkpoint_every = max(1, int(globals().get("MODEL_CACHE_CHECKPOINT_EVERY_BATCHES", 4)))
    fsync_checkpoints = bool(globals().get("MODEL_CACHE_FSYNC_CHECKPOINTS", True))
    write_batch = max(32, int(MODEL_CACHE_WRITE_BATCH_ROWS))

    try:
        # Resume requires a deterministic source cursor; DataLoader workers are therefore
        # disabled for this one-time cache build, not for subsequent cached model training.
        raw_ds = GlobalSoilDataset(manifest, task_mode, "direct", tabular_scalers[task_mode],
                                   raster_scalers[task_mode], max_rows=None, shuffle=False,
                                   seed=SEED, start_row=start_cursor)
        loader = build_global_loader(raw_ds, write_batch, pin_memory=False, num_workers=0,
                                     persistent_workers=False)
        iterator = iter(loader)
        first_batch = next(iterator, None)

        if specs and first_batch is not None:
            # A legacy checkpoint is never appended to until its array schema matches the current
            # model-ready sample representation. If not, preserve and mark it unusable for auto-resume.
            schema_mismatches = []
            for stored_key, source_key in _MODEL_CACHE_FIELDS.items():
                tensor = first_batch[source_key]
                actual_shape = list(int(d) for d in tensor.shape[1:])
                actual_dtype = tensor.detach().cpu().numpy().dtype.str
                saved_spec = specs.get(stored_key, {})
                if list(saved_spec.get("shape", [])) != actual_shape or np.dtype(saved_spec.get("dtype", actual_dtype)).str != actual_dtype:
                    schema_mismatches.append(f"{stored_key}: saved={saved_spec.get('shape')}/{saved_spec.get('dtype')} current={actual_shape}/{actual_dtype}")
            if schema_mismatches:
                try:
                    cp_now = json.loads(checkpoint_path.read_text(encoding="utf-8"))
                    cp_now["layout_mismatch"] = True
                    cp_now["layout_mismatch_details"] = schema_mismatches[:20]
                    _write_cache_checkpoint(checkpoint_path, cp_now)
                except Exception:
                    pass
                raise RuntimeError("Legacy cache array layout mismatch; no cached data was appended. "
                                   + "; ".join(schema_mismatches[:6]))

        if not specs:
            if first_batch is None:
                print(f"Model-ready cache skipped [{task_mode}/{split}]: no valid target samples.")
                return None
            bytes_per_row = 0
            for stored_key, source_key in _MODEL_CACHE_FIELDS.items():
                tensor = first_batch[source_key]
                sample_shape = tuple(int(d) for d in tensor.shape[1:])
                dtype = tensor.detach().cpu().numpy().dtype
                specs[stored_key] = {"file": f"{stored_key}.dat", "dtype": dtype.str,
                                     "shape": list(sample_shape)}
                bytes_per_row += int(np.prod(sample_shape, dtype=np.int64)) * dtype.itemsize
            estimated_bytes = capacity_rows * bytes_per_row
            free_bytes = shutil.disk_usage(MODEL_READY_CACHE_DIR).free
            reserve_bytes = int(float(MODEL_CACHE_DISK_RESERVE_GIB) * _GIB)
            print(f"\n[Model cache {task_mode}/{split}] estimated upper bound: {estimated_bytes/_GIB:.2f} GiB; "
                  f"free disk: {free_bytes/_GIB:.2f} GiB; deterministic cache builder=1 worker; "
                  f"write batch={write_batch}; checkpoint every {checkpoint_every} batches")
            if free_bytes < estimated_bytes * 1.05 + reserve_bytes:
                print(f"[Model cache {task_mode}/{split}] not enough disk headroom; falling back to raw streaming for this split.")
                return None
            if tmp_dir.exists():
                shutil.rmtree(tmp_dir, ignore_errors=True)
            tmp_dir.mkdir(parents=True, exist_ok=True)
            for key, spec in specs.items():
                mmap_arrays[key] = np.memmap(tmp_dir / spec["file"], mode="w+",
                                             dtype=np.dtype(spec["dtype"]),
                                             shape=(capacity_rows, *spec["shape"]))
            resume_cp = {
                "resume_schema": 1, "fingerprint": fingerprint,
                "input_signature": _model_ready_cache_signature(task_mode, split),
                "task_mode": task_mode, "split": split, "capacity_rows": capacity_rows,
                "manifest_path": str(manifest.resolve()),
                "manifest_size": int(manifest.stat().st_size),
                "manifest_num_rows": int(capacity_rows),
                "manifest_num_row_groups": int(pq.ParquetFile(str(manifest)).metadata.num_row_groups),
                "manifest_schema": str(pq.ParquetFile(str(manifest)).schema),
                "committed_rows": 0, "source_row_cursor": 0, "committed_batches": 0,
                "source_eof": False, "arrays": specs,
            }
            _flush_cache_arrays(mmap_arrays, fsync=fsync_checkpoints)
            _write_cache_checkpoint(checkpoint_path, resume_cp)
            start_cursor = 0
            last_source_cursor = 0
        else:
            # Validate/repair staging files if the process was interrupted during final truncation.
            if tmp_dir.exists():
                for key, spec in specs.items():
                    path = tmp_dir / spec["file"]
                    row_bytes = int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
                    expected_capacity_bytes = capacity_rows * row_bytes
                    if path.stat().st_size != expected_capacity_bytes:
                        if source_eof and path.stat().st_size == committed_rows * row_bytes:
                            os.truncate(path, expected_capacity_bytes)
                        else:
                            raise RuntimeError(f"Partial-cache array size mismatch: {path.name}")
                    mmap_arrays[key] = np.memmap(path, mode="r+", dtype=np.dtype(spec["dtype"]),
                                                 shape=(capacity_rows, *spec["shape"]))

        pbar = tqdm(total=capacity_rows, initial=committed_rows,
                    desc=f"Precompute cache [{task_mode}/{split}]", unit="sample",
                    dynamic_ncols=True, mininterval=0.5)

        def save_checkpoint(source_cursor, eof=False):
            nonlocal committed_rows, committed_batches, batches_since_checkpoint
            _flush_cache_arrays(mmap_arrays, fsync=fsync_checkpoints)
            committed_rows = written
            committed_batches += batches_since_checkpoint
            batches_since_checkpoint = 0
            payload = {
                "resume_schema": 1, "fingerprint": fingerprint, "input_signature": _model_ready_cache_signature(task_mode, split),
                "task_mode": task_mode, "split": split, "capacity_rows": capacity_rows,
                "manifest_path": str(manifest.resolve()),
                "manifest_size": int(manifest.stat().st_size),
                "manifest_num_rows": int(capacity_rows),
                "manifest_num_row_groups": int(pq.ParquetFile(str(manifest)).metadata.num_row_groups),
                "manifest_schema": str(pq.ParquetFile(str(manifest)).schema),
                "committed_rows": committed_rows, "source_row_cursor": int(source_cursor),
                "committed_batches": committed_batches, "source_eof": bool(eof), "arrays": specs,
            }
            _write_cache_checkpoint(checkpoint_path, payload)

        def write_batch_to_cache(batch):
            nonlocal written, batches_since_checkpoint, last_source_cursor
            count = int(batch["y"].shape[0])
            if "_source_row_index" not in batch:
                raise RuntimeError("Cache batch is missing source-row indices; cannot checkpoint safely.")
            source_indices = batch["_source_row_index"]
            if not source_indices:
                raise RuntimeError("Empty source index list for nonempty cache batch.")
            next_source_cursor = int(max(source_indices)) + 1
            if written + count > capacity_rows:
                raise RuntimeError(f"Cache writer received more valid rows ({written+count}) than manifest rows ({capacity_rows}).")
            for stored_key, source_key in _MODEL_CACHE_FIELDS.items():
                values = batch[source_key].detach().cpu().numpy()
                mmap_arrays[stored_key][written:written+count] = values
            written += count
            last_source_cursor = next_source_cursor
            batches_since_checkpoint += 1
            pbar.update(count)
            pbar.set_postfix(written=f"{written:,}", committed=f"{committed_rows:,}",
                             source=f"{last_source_cursor:,}/{capacity_rows:,}", workers=0)
            if batches_since_checkpoint >= checkpoint_every:
                save_checkpoint(last_source_cursor, eof=False)
                pbar.set_postfix(written=f"{written:,}", committed=f"{committed_rows:,}",
                                 source=f"{last_source_cursor:,}/{capacity_rows:,}", checkpoint="saved", workers=0)

        # The first batch may be None on resume if only invalid rows remain; in that case
        # the EOF checkpoint below safely completes the cache from existing committed rows.
        if first_batch is not None:
            write_batch_to_cache(first_batch)
        for batch in iterator:
            if batch is None:
                continue
            write_batch_to_cache(batch)

        # Iteration exhausted: mark EOF and commit all remaining rows before finalization.
        save_checkpoint(capacity_rows, eof=True)
        pbar.total = max(written, 1)
        pbar.refresh()
        pbar.close(); pbar = None
        if written <= 0:
            raise RuntimeError("No valid samples were written into the cache.")

        _flush_cache_arrays(mmap_arrays, fsync=fsync_checkpoints)
        for arr in mmap_arrays.values():
            try: arr._mmap.close()
            except Exception: pass
        mmap_arrays.clear()

        # Truncate the preallocated arrays to the true number of valid rows. EOF checkpoint
        # allows recovery if interruption occurs partway through this truncation loop.
        for key, spec in specs.items():
            row_bytes = int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
            os.truncate(tmp_dir / spec["file"], written * row_bytes)
        meta = {"schema": MODEL_CACHE_SCHEMA_VERSION, "cache_builder_id": MODEL_CACHE_BUILDER_ID,
                "fingerprint": fingerprint, "task_mode": task_mode, "split": split, "n_rows": written,
                "created_unix": time.time(), "arrays": specs,
                "target_policy": "direct target + anchor + anchor_valid; residual formed at collation",
                "resume": {"source_rows_scanned": capacity_rows, "checkpointed": True}}
        metadata_tmp = tmp_dir / "metadata.json.tmp"
        with open(metadata_tmp, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)
            f.flush()
            if fsync_checkpoints: os.fsync(f.fileno())
        os.replace(metadata_tmp, tmp_dir / "metadata.json")
        if cache_dir.exists():
            shutil.rmtree(cache_dir, ignore_errors=True)
        os.replace(tmp_dir, cache_dir)
        published = True
        if _validate_model_ready_cache(cache_dir, fingerprint) is None:
            raise RuntimeError("Cache validation failed immediately after build.")
        print(f"[Model cache {task_mode}/{split}] READY: {written:,} samples, "
              f"{sum((cache_dir/spec['file']).stat().st_size for spec in specs.values())/_GIB:.2f} GiB")
        return cache_dir
    except Exception as exc:
        error = exc
        print(f"[Model cache {task_mode}/{split}] BUILD INTERRUPTED/FAILED ({type(exc).__name__}: {exc}).")
        print(f"[Model cache {task_mode}/{split}] Progress checkpoint is retained at: {checkpoint_path}")
        print(f"[Model cache {task_mode}/{split}] Rerun this cell/search to resume from the last committed batch.")
        return None
    finally:
        if pbar is not None:
            pbar.close()
        for arr in mmap_arrays.values():
            try:
                arr.flush()
                arr._mmap.close()
            except Exception:
                pass
        mmap_arrays.clear()
        if iterator is not None:
            del iterator
        if loader is not None:
            del loader
        gc.collect()
        # IMPORTANT: do not delete .building on failure/interruption. The checkpoint and
        # memory-mapped array files are the resume state. Successful publish moves the dir.


def get_global_dataset(split, task_mode=TASK_MODE, target_mode=TARGET_MODE, max_rows=None,
                       shuffle=False, seed=SEED):
    """Return cached map-style samples when available, otherwise use the raw Zarr/Parquet stream."""
    key = (str(task_mode), str(split))
    cache_dir = MODEL_READY_CACHE_PATHS.get(key, "__not_checked__")
    if cache_dir == "__not_checked__":
        if str(MODEL_READY_CACHE_MODE).lower() != "off":
            cache_dir = build_model_ready_cache(task_mode, split)
        else:
            cache_dir = None
        MODEL_READY_CACHE_PATHS[key] = cache_dir
    if cache_dir is not None:
        return CachedGlobalDataset(cache_dir, target_mode=target_mode, max_rows=max_rows,
                                   shuffle=shuffle, seed=seed)
    return GlobalSoilDataset(manifest_paths[(task_mode, split)], task_mode, target_mode,
                             tabular_scalers[task_mode], raster_scalers[task_mode],
                             max_rows=max_rows, shuffle=shuffle, seed=seed)


def prepare_model_ready_caches():
    """Build/reuse each split cache with progress; missing-space splits remain on raw streaming."""
    if str(MODEL_READY_CACHE_MODE).lower() == "off":
        print("MODEL_READY_CACHE_MODE='off'; using raw streaming datasets.")
        return False
    MODEL_READY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print("Preparing persistent model-ready tensor caches. This is a one-time pass; later trials/epochs read cached tensors.")
    successes = 0
    for mode in ("location_history", "environment_only"):
        for split in ("train", "test"):
            key = (mode, split)
            path = build_model_ready_cache(mode, split)
            MODEL_READY_CACHE_PATHS[key] = path
            successes += int(path is not None)
    print(f"Model-ready cache availability: {successes}/4 task/split sets.")
    return successes == 4


def tune_dataloader_workers(task_mode="environment_only"):
    """Benchmark enough real rows to choose worker count by sustained throughput."""
    global NUM_WORKERS
    cap = resource_limited_worker_cap()
    manifest = manifest_paths[(task_mode, "train")]
    using_ready_cache = MODEL_READY_CACHE_PATHS.get((task_mode, "train")) is not None
    if not using_ready_cache:
        try:
            metadata = pq.ParquetFile(str(manifest)).metadata
            cap = min(cap, max(1, int(metadata.num_row_groups)))
        except Exception:
            pass
    candidates = [w for w in _candidate_worker_counts() if w <= cap]
    if not candidates:
        candidates = [0]
    if len(candidates) == 1:
        NUM_WORKERS = candidates[0]
        print(f"Worker autotune: resource/data limits permit {NUM_WORKERS} worker(s); "
              f"visible CPUs={visible_cpu_count()}, available RAM={available_ram_bytes() / _GIB:.1f} GiB.")
        return NUM_WORKERS

    results = []
    print(f"Worker autotune: benchmarking {WORKER_BENCHMARK_SAMPLES} timed rows / "
          f"batch={WORKER_BENCHMARK_BATCH_SIZE}; candidates={candidates}; "
          f"visible CPUs={visible_cpu_count()}, available RAM={available_ram_bytes() / _GIB:.1f} GiB")
    for workers in candidates:
        warmup_rows = workers * int(PREFETCH_FACTOR) * int(WORKER_BENCHMARK_BATCH_SIZE) if workers else 0
        if using_ready_cache:
            ds = get_global_dataset("train", task_mode=task_mode, target_mode="direct",
                                    max_rows=WORKER_BENCHMARK_SAMPLES + warmup_rows,
                                    shuffle=True, seed=SEED)
        else:
            ds = GlobalSoilDataset(manifest, task_mode, "direct", tabular_scalers[task_mode],
                                   raster_scalers[task_mode], max_rows=WORKER_BENCHMARK_SAMPLES + warmup_rows,
                                   shuffle=False, seed=SEED)
        loader = None
        try:
            loader = build_global_loader(ds, WORKER_BENCHMARK_BATCH_SIZE, num_workers=workers,
                                         pin_memory=False, persistent_workers=False)
            iterator = iter(loader)
            warmed = 0
            while warmed < warmup_rows:
                batch = next(iterator, None)
                if batch is None:
                    break
                warmed += int(batch["y"].shape[0])
            timed_rows = 0
            started = time.perf_counter()
            while timed_rows < WORKER_BENCHMARK_SAMPLES:
                batch = next(iterator, None)
                if batch is None:
                    break
                timed_rows += int(batch["y"].shape[0])
            elapsed = max(1e-6, time.perf_counter() - started)
            rate = timed_rows / elapsed if timed_rows else 0.0
            results.append((workers, float(rate), int(timed_rows), float(elapsed)))
            print(f"  workers={workers}: {rate:,.1f} samples/s; timed_rows={timed_rows}; window={elapsed:.2f}s")
        except Exception as exc:
            print(f"  workers={workers}: FAILED {type(exc).__name__}: {exc}")
        finally:
            if iterator is not None:
                del iterator
            if loader is not None:
                del loader
            del ds
            gc.collect()
    if not results:
        NUM_WORKERS = 0
        print("Worker benchmark failed for every setting; using zero-worker fallback so the underlying data error is diagnosable.")
    else:
        best_rate = max(r[1] for r in results)
        near_best = [r for r in results if r[1] >= best_rate * 0.90]
        nonzero_near_best = [r for r in near_best if r[0] > 0]
        NUM_WORKERS = min(nonzero_near_best, key=lambda r: r[0])[0] if nonzero_near_best else max(results, key=lambda r: r[1])[0]
        print(f"Worker autotune selected NUM_WORKERS={NUM_WORKERS}; best measured rate={best_rate:,.1f} samples/s.")
    return NUM_WORKERS

def _power_of_two_batch_sizes(min_batch=MIN_GPU_BATCH_SIZE, max_batch=MAX_GPU_BATCH_SIZE):
    vals, n = [], 1
    while n < min_batch: n *= 2
    while n <= max_batch:
        vals.append(n); n *= 2
    return vals or [max(1, int(min_batch))]


def probe_gpu_batch_limit(model, dataset, max_batch=MAX_GPU_BATCH_SIZE, min_batch=MIN_GPU_BATCH_SIZE):
    """Find a safe batch size by measuring real-shape forward+backward activation memory."""
    if device.type != "cuda" or not AUTO_GPU_BATCH_SIZE:
        return min(int(max_batch), max(256, int(BATCH_SIZE)))
    probe_loader = build_global_loader(dataset, 1, num_workers=0, pin_memory=False,
                                       persistent_workers=False)
    try:
        sample = next(iter(probe_loader))
    except StopIteration:
        raise RuntimeError("Cannot tune GPU batch size: training dataset produced no valid sample.")

    # In GPU raster-cache mode, the collator intentionally returns only raster indices
    # (_gpu_raster_years/_gpu_raster_modis_idx/_gpu_raster_s1_idx), not evi/lst/s1 tensors.
    # Resolve that compact batch using the same main-process path as real training BEFORE
    # creating the model-probe tensors. Without this, every Optuna trial fails with KeyError('evi').
    batch_mover = globals().get("move_batch")
    if callable(batch_mover):
        sample = batch_mover(sample)
    elif "_gpu_raster_years" in sample:
        raise RuntimeError(
            "GPU raster cache batch reached the batch-size probe before move_batch was defined. "
            "Run notebook cells in order through the Optuna live-metrics cell, then start the search."
        )

    required_model_inputs = {"evi", "lst", "s1", "seq", "seq_mask", "static", "anchor"}
    missing_inputs = required_model_inputs.difference(sample)
    if missing_inputs:
        raise RuntimeError(
            f"GPU batch-size probe received an unresolved batch; missing model inputs: {sorted(missing_inputs)}. "
            f"Available keys: {sorted(sample.keys())}"
        )
    tensors = {k: v[:1].to(device, non_blocking=True) for k, v in sample.items() if torch.is_tensor(v)}
    # cuDNN LSTM backward requires the LSTM module to be in training mode.
    # Keep the model in train mode for the probe backward pass, but freeze BatchNorm
    # running-stat updates and Dropout randomness. Restore every submodule's exact
    # previous mode in finally, including mixed parent/child mode configurations.
    module_modes = [(module, bool(module.training)) for module in model.modules()]
    model.train(True)
    for module in model.modules():
        if isinstance(module, (nn.modules.batchnorm._BatchNorm,
                               nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d,
                               nn.AlphaDropout, nn.FeatureAlphaDropout)):
            module.train(False)
    try:
        torch.cuda.empty_cache()
        best = None
        best_peak_delta = 0
        for n in _power_of_two_batch_sizes(min_batch, max_batch):
            free_before, _ = torch.cuda.mem_get_info(device)
            base_allocated = torch.cuda.memory_allocated(device)
            try:
                probe = {k: v.expand((n,) + tuple(v.shape[1:])).contiguous()
                         for k, v in tensors.items()}
                # Raster convolution inputs benefit from channels-last memory format.
                for key in ("evi", "lst", "s1"):
                    if key in probe and probe[key].ndim == 4:
                        probe[key] = probe[key].contiguous(memory_format=torch.channels_last)
                model.zero_grad(set_to_none=True)
                torch.cuda.reset_peak_memory_stats(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                    pred = model(probe)
                    loss = pred.float().square().mean()
                loss.backward()
                torch.cuda.synchronize(device)
                delta = max(0, torch.cuda.max_memory_allocated(device) - base_allocated)
                allowed = int(free_before * (1.0 - GPU_MEMORY_HEADROOM_FRACTION))
                if delta > allowed:
                    print(f"GPU batch probe: {n} needs ~{delta / _GIB:.2f} GiB additional memory; "
                          f"keeping headroom, stop.")
                    del probe, pred, loss
                    model.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    break
                best = n
                best_peak_delta = int(delta)
                print(f"GPU batch probe: {n} fits (~{delta / _GIB:.2f} GiB extra peak)")
                del probe, pred, loss
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
            except torch.cuda.OutOfMemoryError:
                print(f"GPU batch probe: {n} exceeded available memory; stopping at {best}.")
                # Release partially constructed inputs/activations before clearing allocator cache.
                probe = None
                pred = None
                loss = None
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                break
        if best is None:
            best = int(min_batch)
        globals()["_GPU_PROBE_PEAK_DELTA_BYTES"] = int(best_peak_delta)
        globals()["_GPU_PROBE_SAFE_BATCH"] = int(best)
        return int(best)
    finally:
        model.zero_grad(set_to_none=True)
        for module, was_training in module_modes:
            module.training = was_training
        del tensors
        torch.cuda.empty_cache()


def batch_size_for_scale(safe_max, scale):
    target = max(MIN_GPU_BATCH_SIZE, int(safe_max * float(scale)))
    # Keep minibatch sizes powers of two for clean memory/performance behavior.
    power = 2 ** int(math.floor(math.log2(max(1, target))))
    return max(MIN_GPU_BATCH_SIZE, min(int(safe_max), power))


# Resource snapshot and optional pre-materialization are performed once per kernel run.
NUM_WORKERS = min(NUM_WORKERS, resource_limited_worker_cap())
RAM_ZARR_CACHE_ACTIVE = prepare_zarr_memmap_cache()
print("Adaptive data pipeline ready | disk-backed active raster mmap:", RAM_ZARR_CACHE_ACTIVE,
      "| initial workers:", NUM_WORKERS, "| resource cap:", resource_limited_worker_cap(),
      "| candidates:", _candidate_worker_counts(), "| prefetch:", PREFETCH_FACTOR, "| Arrow rows:", ARROW_BATCH_ROWS,
      "| loader microbatch cap:", TRAIN_MICROBATCH_CAP)

# ======================= PARALLEL COMPACT CACHE + BYTE-BOUNDED RASTER LRU =======================
# Compact cache v2 writes directly into disjoint source-row slots. It parallelizes CPU work
# by Parquet row group, checkpoints completed groups, and never copies spatial patches to disk.
from collections import OrderedDict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, wait, FIRST_COMPLETED
import queue as _queue

_COMPACT_CACHE_KIND = "compact_metadata_parallel_rowgroup_v2"
_COMPACT_CACHE_KEYS = ("seq", "seq_mask", "static", "y_direct", "anchor", "anchor_valid",
                       "year", "modis_idx", "s1_idx", "sample_valid")
_COMPACT_CACHE_DTYPES = {
    "seq": np.dtype(np.float16),
    "seq_mask": np.dtype(np.bool_),
    "static": np.dtype(np.float16),
    "y_direct": np.dtype(np.float32),
    "anchor": np.dtype(np.float32),
    "anchor_valid": np.dtype(np.bool_),
    "year": np.dtype(np.int32),
    "modis_idx": np.dtype(np.int64),
    "s1_idx": np.dtype(np.int64),
    "sample_valid": np.dtype(np.bool_),
}
MODEL_READY_CACHE_PATHS = globals().get("MODEL_READY_CACHE_PATHS", {})


def _compact_cache_specs_for_task(task_mode):
    static_feature_names = STATIC_FEATURES + ANCHOR_FEATURES
    if task_mode == "location_history":
        static_feature_names += LOCATION_FEATURES
    shapes = {
        "seq": (HISTORY_STEPS, 2 * len(TEMP_FEATURES) + 3),
        "seq_mask": (HISTORY_STEPS,),
        "static": (2 * len(static_feature_names),),
        "y_direct": (1,), "anchor": (1,), "anchor_valid": (1,),
        "year": (), "modis_idx": (), "s1_idx": (), "sample_valid": (),
    }
    return {k: {"file": f"{k}.dat", "dtype": _COMPACT_CACHE_DTYPES[k].str,
                "shape": list(shapes[k])} for k in _COMPACT_CACHE_KEYS}


def _compact_cache_specs_from_batch(batch):
    # Compatibility helper, useful for checking the actual shape against the schema.
    shapes = {"seq": tuple(batch["seq"].shape[1:]),
              "seq_mask": tuple(batch["seq_mask"].shape[1:]),
              "static": tuple(batch["static"].shape[1:]),
              "y_direct": tuple(batch["y"].shape[1:]),
              "anchor": tuple(batch["anchor"].shape[1:]),
              "anchor_valid": tuple(batch["anchor_valid"].shape[1:]),
              "year": (), "modis_idx": (), "s1_idx": (), "sample_valid": ()}
    return {k: {"file": f"{k}.dat", "dtype": _COMPACT_CACHE_DTYPES[k].str,
                "shape": list(shapes[k])} for k in _COMPACT_CACHE_KEYS}


def _compact_cache_validate(cache_dir, expected_fingerprint=None):
    cache_dir = Path(cache_dir)
    meta_path = cache_dir / "metadata.json"
    if not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if int(meta.get("schema", -1)) != 4 or meta.get("cache_kind") != _COMPACT_CACHE_KIND:
            return None
        if expected_fingerprint is not None and meta.get("fingerprint") != expected_fingerprint:
            return None
        n = int(meta["n_rows"])
        n_valid = int(meta.get("n_valid_rows", -1))
        if n <= 0 or not (0 <= n_valid <= n) or set(meta.get("arrays", {})) != set(_COMPACT_CACHE_KEYS):
            return None
        for key, spec in meta["arrays"].items():
            path = cache_dir / spec["file"]
            row_bytes = int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
            if not path.exists() or path.stat().st_size != n * row_bytes:
                return None
        return meta
    except Exception:
        return None


def _canonical_cache_signature(value, float_digits=7):
    """Canonicalize only tiny floating-point jitter in scaler statistics for cache identity checks."""
    if isinstance(value, dict):
        return {str(k): _canonical_cache_signature(v, float_digits) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_canonical_cache_signature(v, float_digits) for v in value]
    if isinstance(value, (float, np.floating)):
        v = float(value)
        return round(v, int(float_digits)) if np.isfinite(v) else str(v)
    if isinstance(value, (int, np.integer, bool, str)) or value is None:
        return value
    return str(value)


def _cache_signatures_match(left, right):
    """Compare cache inputs across machines without comparing absolute manifest paths.

    v19 originally embedded manifest.resolve() in cache fingerprints. When the transfer
    bundle moved from /home/mayur/... to another machine, semantically identical caches
    were rejected and rebuilt. Normalize only that path to its basename; still require
    matching manifest size/rows/row-groups/schema, scalers, features and preprocessing.
    """
    def portable(value):
        if isinstance(value, dict):
            out = {}
            for key, item in value.items():
                if key == "manifest_path" and isinstance(item, str):
                    out[key] = Path(item).name
                else:
                    out[key] = portable(item)
            return out
        if isinstance(value, list):
            return [portable(x) for x in value]
        if isinstance(value, tuple):
            return [portable(x) for x in value]
        return value
    return (_json_fingerprint(_canonical_cache_signature(portable(left))) ==
            _json_fingerprint(_canonical_cache_signature(portable(right))))


def _find_compatible_completed_compact_cache(task_mode, split, capacity_rows, specs, current_signature, current_dir):
    """Find a completed same-run cache whose full signature matches, or a safe legacy cache.

    Legacy v4 metadata did not store input_signature. It is reused only if the manifest-config
    signature is unchanged, builder ID/schema/layout/row count match, and all array files validate.
    This lets a restart survive insignificant floating-point scaler jitter without blindly
    trusting a cache from a different data split or cache schema.
    """
    root = Path(MODEL_READY_CACHE_DIR)
    manifest_config_unchanged = bool(globals().get("signature_matches", False)) and not bool(globals().get("REBUILD_MANIFESTS", False))
    for candidate in sorted(root.glob(f"{task_mode}_{split}_*")):
        if candidate == Path(current_dir) or not candidate.is_dir():
            continue
        name = candidate.name
        if name.endswith(".building") or ".old_" in name or ".incompatible_" in name:
            continue
        meta = _compact_cache_validate(candidate, expected_fingerprint=None)
        if meta is None:
            continue
        # Schema + exact array layout define the serialized cache format. When a completed
        # cache also carries a full input_signature, that semantic signature is stronger than
        # a code-builder label: a harmless builder-ID bump must not force a multi-GiB rebuild.
        if (meta.get("task_mode") != task_mode or meta.get("split") != split
                or int(meta.get("n_rows", -1)) != int(capacity_rows)
                or int(meta.get("schema", -1)) != int(MODEL_CACHE_SCHEMA_VERSION)
                or _json_fingerprint(meta.get("arrays", {})) != _json_fingerprint(specs)):
            continue
        saved_signature = meta.get("input_signature")
        if saved_signature is not None:
            if _cache_signatures_match(saved_signature, current_signature):
                return candidate, ("full semantic input signature + schema + array layout match; "
                                   "reusing cache despite builder-ID-only change")
            continue
        # Very old metadata omitted the input signature. Reuse it only under the stricter
        # legacy condition: same builder ID, same manifests/config, and full structural validation.
        if meta.get("builder_id") == MODEL_CACHE_BUILDER_ID and manifest_config_unchanged:
            return candidate, "legacy v4 cache structurally compatible; manifest config unchanged"
    return None, None


def _load_parallel_compact_checkpoint(tmp_dir, fingerprint, capacity_rows, task_mode, split, n_row_groups, specs):
    cp_path = Path(tmp_dir) / "checkpoint.json"
    if not cp_path.exists():
        return None
    try:
        cp = json.loads(cp_path.read_text(encoding="utf-8"))
        if (cp.get("cache_kind") != _COMPACT_CACHE_KIND
                or cp.get("task_mode") != task_mode or cp.get("split") != split
                or int(cp.get("capacity_rows", -1)) != int(capacity_rows)
                or int(cp.get("n_row_groups", -1)) != int(n_row_groups)
                or set(cp.get("arrays", {})) != set(_COMPACT_CACHE_KEYS)):
            return None
        if not _cache_signatures_match(cp.get("input_signature", {}), _model_ready_cache_signature(task_mode, split)):
            return None
        if _json_fingerprint(cp["arrays"]) != _json_fingerprint(specs):
            return None
        completed = [int(x) for x in cp.get("completed_row_groups", [])]
        if len(completed) != len(set(completed)) or any(x < 0 or x >= n_row_groups for x in completed):
            return None
        counts = {str(k): int(v) for k, v in cp.get("valid_rows_by_group", {}).items()}
        if set(counts) != {str(x) for x in completed} or any(v < 0 for v in counts.values()):
            return None
        for key, spec in specs.items():
            path = Path(tmp_dir) / spec["file"]
            expected = int(capacity_rows) * int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
            if not path.exists() or path.stat().st_size != expected:
                return None
        if int(cp.get("committed_rows", -1)) != sum(counts.values()):
            return None
        return cp
    except Exception:
        return None


def _compact_flush(arrays, fsync=True):
    for arr in arrays.values():
        arr.flush()
    if fsync:
        for arr in arrays.values():
            try:
                with open(str(arr.filename), "rb") as f:
                    os.fsync(f.fileno())
            except Exception:
                pass


# Worker globals are initialized inside each forked process; workers only write their own
# Parquet row-group's source-row range, so no shared-array write locks are needed.
_COMPACT_BUILD_WORKER_DS = None
_COMPACT_BUILD_WORKER_PARQUET = None
_COMPACT_BUILD_WORKER_ARRAYS = None
_COMPACT_BUILD_WORKER_PROGRESS = None
_COMPACT_BUILD_WORKER_CAPACITY = None
_COMPACT_BUILD_EXECUTOR_KIND = "process"
_COMPACT_BUILD_THREAD_LOCAL = threading.local()


def _compact_parallel_worker_init(manifest_path, task_mode, cache_dir, specs, capacity_rows, progress_queue):
    global _COMPACT_BUILD_WORKER_DS, _COMPACT_BUILD_WORKER_PARQUET
    global _COMPACT_BUILD_WORKER_ARRAYS, _COMPACT_BUILD_WORKER_PROGRESS, _COMPACT_BUILD_WORKER_CAPACITY
    try:
        torch.set_num_threads(1)
    except Exception:
        pass
    try:
        import threadpoolctl
        threadpoolctl.threadpool_limits(limits=1)
    except Exception:
        pass
    _COMPACT_BUILD_WORKER_DS = GlobalSoilDataset(
        Path(manifest_path), task_mode, "direct", tabular_scalers[task_mode],
        raster_scalers[task_mode], max_rows=None, shuffle=False, seed=SEED, skip_raster=True)
    _COMPACT_BUILD_WORKER_PARQUET = pq.ParquetFile(str(manifest_path))
    _COMPACT_BUILD_WORKER_ARRAYS = {
        key: np.memmap(Path(cache_dir) / spec["file"], mode="r+", dtype=np.dtype(spec["dtype"]),
                       shape=(int(capacity_rows), *spec["shape"]))
        for key, spec in specs.items()
    }
    _COMPACT_BUILD_WORKER_PROGRESS = progress_queue
    _COMPACT_BUILD_WORKER_CAPACITY = int(capacity_rows)


def _compact_thread_worker_init(manifest_path, task_mode):
    """Initialize one independent Parquet reader per thread; mmap output arrays stay shared."""
    local = _COMPACT_BUILD_THREAD_LOCAL
    local.ds = GlobalSoilDataset(
        Path(manifest_path), task_mode, "direct", tabular_scalers[task_mode],
        raster_scalers[task_mode], max_rows=None, shuffle=False, seed=SEED, skip_raster=True)
    local.parquet = pq.ParquetFile(str(manifest_path))


def _compact_parallel_process_row_group(task):
    """Read/preprocess one Parquet row group; write compact fields at original source row indices."""
    rg, group_start, expected_rows = map(int, task)
    if _COMPACT_BUILD_EXECUTOR_KIND == "thread":
        local = _COMPACT_BUILD_THREAD_LOCAL
        ds = local.ds
        parquet = local.parquet
        arrays = _COMPACT_BUILD_WORKER_ARRAYS  # one shared mapping, disjoint row-group writes
        q = _COMPACT_BUILD_WORKER_PROGRESS
        worker_capacity = _COMPACT_BUILD_WORKER_CAPACITY
    else:
        ds = _COMPACT_BUILD_WORKER_DS
        parquet = _COMPACT_BUILD_WORKER_PARQUET
        arrays = _COMPACT_BUILD_WORKER_ARRAYS
        q = _COMPACT_BUILD_WORKER_PROGRESS
        worker_capacity = _COMPACT_BUILD_WORKER_CAPACITY
    group_end = group_start + expected_rows
    if group_start < 0 or group_end > worker_capacity:
        raise RuntimeError(f"Invalid row-group bounds for group {rg}: {group_start}:{group_end}")
    # A partially written/uncommitted group is always reset before reprocessing after restart.
    arrays["sample_valid"][group_start:group_end] = False
    source_row = group_start
    valid_count = 0
    for record_batch in parquet.iter_batches(batch_size=int(ARROW_BATCH_ROWS), row_groups=[rg], use_threads=False):
        rows = record_batch.to_pylist()
        batch_valid = 0
        for row in rows:
            pos = source_row
            source_row += 1
            item = ds._make(row)
            if item is None:
                continue
            arrays["seq"][pos] = item["seq"].numpy().astype(np.float16, copy=False)
            arrays["seq_mask"][pos] = item["seq_mask"].numpy().astype(np.bool_, copy=False)
            arrays["static"][pos] = item["static"].numpy().astype(np.float16, copy=False)
            arrays["y_direct"][pos] = item["y"].numpy().astype(np.float32, copy=False)
            arrays["anchor"][pos] = item["anchor"].numpy().astype(np.float32, copy=False)
            arrays["anchor_valid"][pos] = item["anchor_valid"].numpy().astype(np.bool_, copy=False)
            arrays["year"][pos] = int(item["_year"])
            arrays["modis_idx"][pos] = int(item["_modis_row_index"])
            arrays["s1_idx"][pos] = int(item["_s1_row_index"])
            # Mark valid only after every field for this sample has been written.
            arrays["sample_valid"][pos] = True
            valid_count += 1
            batch_valid += 1
        if q is not None:
            try: q.put_nowait((rg, len(rows), batch_valid))
            except Exception: pass
    if source_row != group_end:
        raise RuntimeError(f"Row group {rg} row count mismatch: read {source_row-group_start}, expected {expected_rows}")
    return rg, valid_count, expected_rows


def _parallel_cache_worker_count(n_row_groups):
    """Choose cache-build parallelism with an explicit override for I/O-heavy work.

    By default, the cache builder is intentionally allowed to overlap blocked Zarr/Parquet reads
    even if Jupyter/WSL exposes a narrow CPU affinity mask. Memory and number of row groups always
    cap the result. Set MODEL_CACHE_BUILD_WORKERS_OVERRIDE=0 and
    MODEL_CACHE_BUILD_ALLOW_CPU_OVERSUBSCRIPTION=False for strict affinity-based auto sizing.
    """
    visible_cpus = max(1, int(visible_cpu_count()))
    system_cpus = max(1, int(os.cpu_count() or visible_cpus))
    configured_max = max(1, int(MODEL_CACHE_BUILD_MAX_WORKERS))
    requested = int(globals().get("MODEL_CACHE_BUILD_WORKERS_OVERRIDE", 0) or 0)
    allow_oversub = bool(globals().get("MODEL_CACHE_BUILD_ALLOW_CPU_OVERSUBSCRIPTION", False))

    if requested > 0:
        desired = min(requested, configured_max)
        if not allow_oversub:
            desired = min(desired, max(1, visible_cpus - 1))
    else:
        # Reserve one visible CPU for the notebook/UI if using strict automatic selection.
        desired = min(configured_max, max(1, visible_cpus - 1))

    avail_bytes = available_ram_bytes()
    reserve_gib = max(0.0, float(globals().get("MODEL_CACHE_BUILD_RAM_RESERVE_GIB", 1.0)))
    reserve_bytes = int(reserve_gib * _GIB)
    avail_after_reserve = max(0, avail_bytes - reserve_bytes)
    # Respect a lower per-worker estimate for this metadata-only builder; workers share the
    # mmap outputs and never load raster patches. Keep one worker as diagnostic fallback.
    per_worker_gib = max(0.25, float(MODEL_CACHE_BUILD_RAM_PER_WORKER_GIB))
    per_worker = per_worker_gib * _GIB
    ram_cap = max(1, int(avail_after_reserve / per_worker))
    row_group_cap = max(1, int(n_row_groups))
    cap_inputs = {
        "configured_max": configured_max,
        "requested_or_cpu": desired,
        "RAM": ram_cap,
        "unfinished_row_groups": row_group_cap,
    }
    cap = max(1, min(cap_inputs.values()))
    limiting = [name for name, value in cap_inputs.items() if value == cap]

    backend = "process/fork" if (os.name == "posix" and "fork" in mp.get_all_start_methods()) else "thread pool"
    print(f"Cache worker sizing: requested={requested or 'auto'}, selected={cap}, limiting={limiting}, backend={backend}, "
          f"visible/affinity CPUs={visible_cpus}, os.cpu_count={system_cpus}, "
          f"oversubscribe_affinity={allow_oversub}, RAM available={avail_bytes/_GIB:.2f} GiB, "
          f"builder RAM reserve={reserve_gib:.2f} GiB, budget/worker={per_worker_gib:.2f} GiB, "
          f"RAM cap={ram_cap}, unfinished row_groups={n_row_groups}")
    if cap < desired and "RAM" in limiting:
        print("Cache worker sizing note: reduced by current available RAM. This is a safety cap, not a CPU limit; "
              "free memory before increasing the cap further.")
    if allow_oversub and visible_cpus < cap:
        print("Cache worker sizing note: using more processes than visible CPU affinity suggests; "
              "this can improve I/O overlap, but compare source-row/s and RAM to ensure it helps.")
    return cap


def build_model_ready_cache(task_mode, split, force=False):
    """Parallel resumable compact cache; disjoint workers write only compact metadata/raster keys."""
    if force and not bool(globals().get("MODEL_READY_CACHE_FORCE_REBUILD", False)):
        raise RuntimeError("Refusing to force-rebuild a persistent model-ready cache. Set MODEL_READY_CACHE_FORCE_REBUILD=True deliberately only after changing source data/features.")
    global _COMPACT_BUILD_EXECUTOR_KIND, _COMPACT_BUILD_WORKER_ARRAYS
    global _COMPACT_BUILD_WORKER_PROGRESS, _COMPACT_BUILD_WORKER_CAPACITY
    fingerprint, cache_dir = _model_ready_cache_identity(task_mode, split)
    cache_dir = Path(cache_dir)
    print(f"[COMPACT-CACHE] checking reusable cache for {task_mode}/{split} at {cache_dir}", flush=True)
    print("[COMPACT-CACHE] A cache is reusable only after its final metadata.json is published and the directory is promoted; "
          "checkpointed_valid is flushed progress in a .building directory, not a completed cache. Only [READY] + metadata.json + atomic promotion means complete.", flush=True)
    cached = _compact_cache_validate(cache_dir, fingerprint)
    if cache_dir.exists() and cached is None:
        _meta_path = cache_dir / "metadata.json"
        if not _meta_path.is_file():
            print(f"[COMPACT-CACHE] expected path exists but is not READY (metadata.json missing): {cache_dir}", flush=True)
        else:
            try:
                _old_meta = json.loads(_meta_path.read_text(encoding="utf-8"))
                print("[COMPACT-CACHE] expected path is not reusable: "
                      f"saved_fingerprint={str(_old_meta.get('fingerprint',''))[:16]} "
                      f"expected={fingerprint[:16]} schema={_old_meta.get('schema')} "
                      f"builder={_old_meta.get('builder_id')}; checking compatible alternatives.", flush=True)
            except Exception as _meta_exc:
                print(f"[COMPACT-CACHE] expected metadata is unreadable ({_meta_exc}); checking alternatives.", flush=True)
    if cached is not None and not force:
        used = sum((cache_dir / spec["file"]).stat().st_size for spec in cached["arrays"].values()) / _GIB
        print(f"Compact parallel cache reused [{task_mode}/{split}]: {cached['n_valid_rows']:,} valid / "
              f"{cached['n_rows']:,} source rows; {used:.2f} GiB at {cache_dir}")
        return cache_dir

    manifest = Path(manifest_paths[(task_mode, split)])
    parquet_meta_file = pq.ParquetFile(str(manifest))
    parquet_meta = parquet_meta_file.metadata
    capacity_rows = int(parquet_meta.num_rows)
    n_row_groups = int(parquet_meta.num_row_groups)
    if capacity_rows <= 0 or n_row_groups <= 0:
        print(f"Compact cache skipped [{task_mode}/{split}]: empty manifest.")
        return None
    group_rows = [int(parquet_meta.row_group(i).num_rows) for i in range(n_row_groups)]
    group_offsets = []
    pos = 0
    for count in group_rows:
        group_offsets.append(pos); pos += count
    if pos != capacity_rows:
        raise RuntimeError(f"Parquet row-group row total {pos} does not match manifest rows {capacity_rows}.")
    specs = _compact_cache_specs_for_task(task_mode)
    current_signature = _model_ready_cache_signature(task_mode, split)
    if not force:
        print(f"[COMPACT-CACHE] expected fingerprint cache absent/incompatible for {task_mode}/{split}; "
              "searching completed caches and resumable checkpoints before any rebuild.", flush=True)
        compatible_dir, compatible_reason = _find_compatible_completed_compact_cache(
            task_mode, split, capacity_rows, specs, current_signature, cache_dir)
        if compatible_dir is not None:
            used = sum((compatible_dir / spec["file"]).stat().st_size for spec in specs.values()) / _GIB
            print(f"Compact parallel cache reused [{task_mode}/{split}]: {compatible_reason}; "
                  f"{_compact_cache_validate(compatible_dir, None)['n_valid_rows']:,} valid / "
                  f"{capacity_rows:,} source rows; {used:.2f} GiB at {compatible_dir}")
            return compatible_dir
        nearby = [p for p in MODEL_READY_CACHE_DIR.glob(f"{task_mode}_{split}_*")
                  if p.is_dir() and not p.name.endswith(".building") and ".old_" not in p.name
                  and ".incompatible_" not in p.name and p != cache_dir]
        if nearby:
            print(f"[Cache identity] {len(nearby)} other completed {task_mode}/{split} cache candidate(s) found, "
                  "but none passed the semantic/legacy compatibility checks. They are preserved; "
                  "a fresh cache will be built to avoid silently using stale normalized inputs.")
        elif not cache_dir.exists():
            print(f"[Cache identity] No completed cache exists at the expected fingerprint path for "
                  f"{task_mode}/{split}; building it once now. Expected path: {cache_dir}")
    # Discover and validate an existing checkpoint BEFORE applying the disk-space gate.
    # A resume needs space for unfinished row groups, not the entire cache a second time.
    tmp_dir = cache_dir.with_name(cache_dir.name + ".building")
    if force and tmp_dir.exists():
        raise RuntimeError(f"force=True will not delete resumable staging data: {tmp_dir}")
    # If an interrupted build used a neighboring fingerprint path, relocate it only after its
    # stored semantic signature, row count, row groups and array layout all validate.
    if not tmp_dir.exists() and not force:
        for alternate_tmp in sorted(MODEL_READY_CACHE_DIR.glob(f"{task_mode}_{split}_*.building")):
            alternate_cp = _load_parallel_compact_checkpoint(
                alternate_tmp, fingerprint, capacity_rows, task_mode, split, n_row_groups, specs)
            if alternate_cp is not None:
                os.replace(alternate_tmp, tmp_dir)
                alternate_cp["fingerprint"] = fingerprint
                _atomic_write_json(tmp_dir / "checkpoint.json", alternate_cp)
                print(f"[Cache resume] Adopted compatible interrupted checkpoint from {alternate_tmp.name}; "
                      "continuing only unfinished row groups.")
                break
    resume_cp = _load_parallel_compact_checkpoint(
        tmp_dir, fingerprint, capacity_rows, task_mode, split, n_row_groups, specs)
    if resume_cp is not None and resume_cp.get("fingerprint") != fingerprint:
        resume_cp["fingerprint"] = fingerprint
        _atomic_write_json(tmp_dir / "checkpoint.json", resume_cp)
    if tmp_dir.exists() and resume_cp is None:
        raise RuntimeError(f"Incompatible parallel compact-cache staging data was preserved at {tmp_dir}. "
                           "Refusing to start a duplicate build that could consume more disk.")

    estimated_bytes = capacity_rows * sum(
        int(np.prod(spec["shape"], dtype=np.int64)) * np.dtype(spec["dtype"]).itemsize
        for spec in specs.values())
    disk_parent = cache_dir.parent if cache_dir.parent.exists() else RUN_DIR
    free_bytes = shutil.disk_usage(disk_parent).free
    reserve_bytes = int(float(MODEL_CACHE_DISK_RESERVE_GIB) * _GIB)
    if resume_cp is None:
        bytes_needed = estimated_bytes
        reserve = reserve_bytes
    else:
        already_done_groups = {int(x) for x in resume_cp.get("completed_row_groups", [])}
        already_done_rows = sum(group_rows[g] for g in already_done_groups)
        bytes_needed = int(estimated_bytes * max(0, capacity_rows - already_done_rows) / max(1, capacity_rows))
        # A resumed cache should not be blocked by the full-cache reserve after most data
        # has already been written, but still preserve a modest safety margin.
        reserve = min(reserve_bytes, int(0.5 * _GIB))
    print(f"\\n[Parallel compact cache {task_mode}/{split}] estimate={estimated_bytes/_GIB:.2f} GiB; "
          f"remaining allocation estimate={bytes_needed/_GIB:.2f} GiB; "
          f"free disk={free_bytes/_GIB:.2f} GiB; source rows={capacity_rows:,}; row groups={n_row_groups:,}")
    if free_bytes < bytes_needed * 1.10 + reserve:
        if resume_cp is not None:
            print(f"[Parallel compact cache {task_mode}/{split}] insufficient disk headroom to resume; "
                  f"checkpoint remains preserved at {tmp_dir}.")
        else:
            print(f"[Parallel compact cache {task_mode}/{split}] insufficient disk headroom; using raw stream for this split.")
        return None

    tmp_dir.mkdir(parents=True, exist_ok=True)
    arrays = {}
    progress_queue = None
    pbar = None
    worker_context_started = False
    try:
        if resume_cp is None:
            for key, spec in specs.items():
                arrays[key] = np.memmap(tmp_dir / spec["file"], mode="w+", dtype=np.dtype(spec["dtype"]),
                                        shape=(capacity_rows, *spec["shape"]))
            arrays["sample_valid"][:] = False
            _compact_flush(arrays, fsync=bool(MODEL_CACHE_FSYNC_CHECKPOINTS))
            completed_counts = {}
            cp = {"resume_schema": 2, "cache_kind": _COMPACT_CACHE_KIND,
                  "fingerprint": fingerprint, "input_signature": _model_ready_cache_signature(task_mode, split),
                  "task_mode": task_mode, "split": split, "capacity_rows": capacity_rows,
                  "n_row_groups": n_row_groups, "completed_row_groups": [], "valid_rows_by_group": {},
                  "committed_rows": 0, "source_eof": False, "arrays": specs,
                  "created_unix": time.time(), "builder_id": MODEL_CACHE_BUILDER_ID}
            _write_cache_checkpoint(tmp_dir / "checkpoint.json", cp)
        else:
            completed_counts = {str(k): int(v) for k, v in resume_cp["valid_rows_by_group"].items()}
            for key, spec in specs.items():
                arrays[key] = np.memmap(tmp_dir / spec["file"], mode="r+", dtype=np.dtype(spec["dtype"]),
                                        shape=(capacity_rows, *spec["shape"]))
            print(f"[Parallel compact cache {task_mode}/{split}] RESUME: groups={len(completed_counts)}/{n_row_groups}; "
                  f"valid committed rows={sum(completed_counts.values()):,}; source committed rows="
                  f"{sum(group_rows[int(g)] for g in completed_counts):,}/{capacity_rows:,}")

        completed_groups = {int(k) for k in completed_counts}
        committed_source_rows = sum(group_rows[g] for g in completed_groups)
        committed_valid_rows = sum(completed_counts.values())
        pbar = tqdm(total=capacity_rows, initial=committed_source_rows,
                    desc=f"Parallel compact cache [{task_mode}/{split}]", unit="source-row",
                    dynamic_ncols=True, mininterval=0.5)
        group_progressed = {g: group_rows[g] for g in completed_groups}
        finalized_progress_groups = set(completed_groups)
        observed_valid_rows = committed_valid_rows
        newly_completed = {}
        groups_since_checkpoint = 0
        checkpoint_every = max(1, int(MODEL_CACHE_CHECKPOINT_EVERY_GROUPS))
        fsync = bool(MODEL_CACHE_FSYNC_CHECKPOINTS)
        workers = _parallel_cache_worker_count(n_row_groups)
        unfinished = [(rg, group_offsets[rg], group_rows[rg]) for rg in range(n_row_groups)
                      if rg not in completed_groups]
        print(f"[Parallel compact cache {task_mode}/{split}] CPU workers={workers}; "
              f"completed row groups={len(completed_groups):,}; checkpoint every {checkpoint_every} groups; "
              f"Arrow rows/batch={ARROW_BATCH_ROWS}.")

        _cache_cpu_sample = {"at": None, "cpu_seconds": None}
        def _cache_resource_postfix():
            parts = {}
            if psutil is not None:
                try:
                    parts["hostCPU"] = f"{psutil.cpu_percent(interval=None):.0f}%"
                    vm = psutil.virtual_memory()
                    parts["RAM"] = f"{vm.percent:.0f}%"
                    proc = psutil.Process(os.getpid())
                    cpu_seconds = sum(proc.cpu_times()[:2])
                    for child in proc.children(recursive=True):
                        try:
                            ct = child.cpu_times()
                            cpu_seconds += float(ct.user + ct.system)
                        except (psutil.NoSuchProcess, psutil.AccessDenied):
                            continue
                    now = time.perf_counter()
                    prev_at, prev_cpu = _cache_cpu_sample["at"], _cache_cpu_sample["cpu_seconds"]
                    if prev_at is not None and now > prev_at:
                        parts["procCPU"] = f"{max(0.0, (cpu_seconds-prev_cpu)/(now-prev_at)*100.0):.0f}%"
                    _cache_cpu_sample["at"], _cache_cpu_sample["cpu_seconds"] = now, cpu_seconds
                except Exception:
                    pass
            return parts

        def drain_progress_queue():
            nonlocal observed_valid_rows
            raw_delta = 0
            valid_delta = 0
            while True:
                try: rg, raw_n, valid_n = progress_queue.get_nowait()
                except _queue.Empty:
                    break
                rg = int(rg)
                if rg in finalized_progress_groups:
                    continue
                raw_n = int(raw_n); valid_n = int(valid_n)
                already = int(group_progressed.get(rg, 0))
                # Messages carry source-row work completed within the active row group.
                accepted = min(raw_n, max(0, group_rows[rg] - already))
                group_progressed[rg] = already + accepted
                raw_delta += accepted
                valid_delta += valid_n
            if raw_delta or valid_delta:
                pbar.update(raw_delta)
                observed_valid_rows += valid_delta
                pbar.set_postfix(work_valid=f"~{observed_valid_rows:,}", checkpointed_valid=f"{committed_valid_rows:,}",
                                 groups=f"{len(completed_groups)+len(newly_completed)}/{n_row_groups}", workers=workers,
                                 rows_s=f"{pbar.n/max(time.perf_counter()-build_started,1e-6):.0f}",
                                 **_cache_resource_postfix())

        def commit_checkpoint(source_eof=False):
            nonlocal committed_valid_rows, committed_source_rows, groups_since_checkpoint
            if newly_completed:
                # Finished worker groups have returned. Flush shared mappings BEFORE committing their IDs.
                _compact_flush(arrays, fsync=fsync)
                completed_counts.update({str(k): int(v) for k, v in newly_completed.items()})
                completed_groups.update(newly_completed.keys())
                newly_completed.clear()
            committed_valid_rows = sum(completed_counts.values())
            committed_source_rows = sum(group_rows[int(g)] for g in completed_counts)
            cp = {"resume_schema": 2, "cache_kind": _COMPACT_CACHE_KIND,
                  "fingerprint": fingerprint, "input_signature": _model_ready_cache_signature(task_mode, split),
                  "task_mode": task_mode, "split": split, "capacity_rows": capacity_rows,
                  "n_row_groups": n_row_groups, "completed_row_groups": sorted(int(x) for x in completed_counts),
                  "valid_rows_by_group": completed_counts, "committed_rows": committed_valid_rows,
                  "source_eof": bool(source_eof), "arrays": specs, "updated_unix": time.time(),
                  "builder_id": MODEL_CACHE_BUILDER_ID}
            _write_cache_checkpoint(tmp_dir / "checkpoint.json", cp)
            groups_since_checkpoint = 0
            pbar.set_postfix(work_valid=f"~{observed_valid_rows:,}", checkpointed_valid=f"{committed_valid_rows:,}",
                             checkpointed_groups=f"{len(completed_groups)}/{n_row_groups}", workers=workers,
                             rows_s=f"{pbar.n/max(time.perf_counter()-build_started,1e-6):.0f}", ckpt="saved",
                             **_cache_resource_postfix())

        build_started = time.perf_counter()
        if unfinished:
            if workers <= 0:
                # Defensive fallback only; worker sizing currently returns at least one.
                _COMPACT_BUILD_EXECUTOR_KIND = "process"
                progress_queue = _queue.Queue()
                _compact_parallel_worker_init(str(manifest), task_mode, str(tmp_dir), specs,
                                              capacity_rows, progress_queue)
                for task in unfinished:
                    rg, valid_count, rows_count = _compact_parallel_process_row_group(task)
                    drain_progress_queue()
                    if int(group_progressed.get(rg, 0)) < group_rows[rg]:
                        pbar.update(group_rows[rg] - int(group_progressed.get(rg, 0)))
                    group_progressed[rg] = group_rows[rg]
                    finalized_progress_groups.add(rg)
                    newly_completed[rg] = int(valid_count)
                    groups_since_checkpoint += 1
                    if groups_since_checkpoint >= checkpoint_every:
                        commit_checkpoint(source_eof=False)
                # Cleanup in-process maps before parent array mappings are closed.
                try:
                    for a in (_COMPACT_BUILD_WORKER_ARRAYS or {}).values():
                        try: a.flush(); a._mmap.close()
                        except Exception: pass
                except Exception:
                    pass
                _COMPACT_BUILD_WORKER_ARRAYS = None
                _COMPACT_BUILD_WORKER_DS = None
                _COMPACT_BUILD_WORKER_PARQUET = None
            elif os.name != "posix" or "fork" not in mp.get_all_start_methods():
                # Windows/Jupyter spawn cannot safely import notebook-defined worker functions.
                # Threads are safe here because workers read separate ParquetFile handles and write
                # only disjoint source-row slices of shared memory-mapped output arrays.
                progress_queue = _queue.Queue()
                _COMPACT_BUILD_EXECUTOR_KIND = "thread"
                _COMPACT_BUILD_WORKER_ARRAYS = arrays
                _COMPACT_BUILD_WORKER_PROGRESS = progress_queue
                _COMPACT_BUILD_WORKER_CAPACITY = capacity_rows
                print(f"[Parallel compact cache {task_mode}/{split}] using {workers} ThreadPoolExecutor workers "
                      "(Windows/Jupyter-safe; disjoint mmap row-group writes).")
                with ThreadPoolExecutor(max_workers=workers, initializer=_compact_thread_worker_init,
                                        initargs=(str(manifest), task_mode)) as executor:
                    futures = {executor.submit(_compact_parallel_process_row_group, task): task for task in unfinished}
                    while futures:
                        done, _ = wait(tuple(futures), timeout=0.25, return_when=FIRST_COMPLETED)
                        drain_progress_queue()
                        for future in done:
                            task = futures.pop(future)
                            rg, valid_count, rows_count = future.result()
                            rg = int(rg)
                            if int(group_progressed.get(rg, 0)) < group_rows[rg]:
                                pbar.update(group_rows[rg] - int(group_progressed.get(rg, 0)))
                            group_progressed[rg] = group_rows[rg]
                            finalized_progress_groups.add(rg)
                            newly_completed[rg] = int(valid_count)
                            groups_since_checkpoint += 1
                            pbar.set_postfix(work_valid=f"~{observed_valid_rows:,}", checkpointed_valid=f"{committed_valid_rows:,}",
                                             groups=f"{len(completed_groups)+len(newly_completed)}/{n_row_groups}",
                                             workers=workers, rows_s=f"{pbar.n/max(time.perf_counter()-build_started,1e-6):.0f}")
                            if groups_since_checkpoint >= checkpoint_every:
                                commit_checkpoint(source_eof=False)
                    drain_progress_queue()
                _COMPACT_BUILD_EXECUTOR_KIND = "process"
                _COMPACT_BUILD_WORKER_ARRAYS = None
                _COMPACT_BUILD_WORKER_PROGRESS = None
                _COMPACT_BUILD_WORKER_CAPACITY = None
            else:
                _COMPACT_BUILD_EXECUTOR_KIND = "process"
                ctx = mp.get_context("fork")
                progress_queue = ctx.Queue(maxsize=max(2048, workers * 256))
                initargs = (str(manifest), task_mode, str(tmp_dir), specs, capacity_rows, progress_queue)
                with ProcessPoolExecutor(max_workers=workers, mp_context=ctx,
                                         initializer=_compact_parallel_worker_init, initargs=initargs) as executor:
                    futures = {executor.submit(_compact_parallel_process_row_group, task): task for task in unfinished}
                    while futures:
                        done, _ = wait(tuple(futures), timeout=0.25, return_when=FIRST_COMPLETED)
                        drain_progress_queue()
                        for future in done:
                            task = futures.pop(future)
                            rg, valid_count, rows_count = future.result()
                            rg = int(rg)
                            if int(group_progressed.get(rg, 0)) < group_rows[rg]:
                                pbar.update(group_rows[rg] - int(group_progressed.get(rg, 0)))
                            group_progressed[rg] = group_rows[rg]
                            finalized_progress_groups.add(rg)
                            newly_completed[rg] = int(valid_count)
                            groups_since_checkpoint += 1
                            pbar.set_postfix(work_valid=f"~{observed_valid_rows:,}", checkpointed_valid=f"{committed_valid_rows:,}",
                                             groups=f"{len(completed_groups)+len(newly_completed)}/{n_row_groups}",
                                             workers=workers, rows_s=f"{pbar.n/max(time.perf_counter()-build_started,1e-6):.0f}")
                            if groups_since_checkpoint >= checkpoint_every:
                                commit_checkpoint(source_eof=False)
                    drain_progress_queue()
                progress_queue.close()
                progress_queue.join_thread()

        # Commit any tail groups; the EOF bit is committed only when every row group is durable.
        if newly_completed:
            commit_checkpoint(source_eof=(len(completed_groups) + len(newly_completed) == n_row_groups))
        elif len(completed_groups) == n_row_groups:
            commit_checkpoint(source_eof=True)
        if len(completed_groups) != n_row_groups:
            missing = sorted(set(range(n_row_groups)) - completed_groups)
            raise RuntimeError(f"Parallel cache build incomplete; missing row groups: {missing[:12]}")
        _compact_flush(arrays, fsync=fsync)
        n_valid = int(np.asarray(arrays["sample_valid"], dtype=np.bool_).sum())
        if n_valid != committed_valid_rows:
            raise RuntimeError(f"Validity-map count {n_valid:,} differs from checkpoint valid count {committed_valid_rows:,}.")
        metadata = {"schema": 4, "cache_kind": _COMPACT_CACHE_KIND,
                    "fingerprint": fingerprint, "task_mode": task_mode, "split": split,
                    "n_rows": capacity_rows, "n_valid_rows": n_valid, "arrays": specs,
                    "input_signature": _model_ready_cache_signature(task_mode, split),
                    "created_unix": time.time(), "builder_id": MODEL_CACHE_BUILDER_ID,
                    "storage_note": "source-row slots; compact sequence/static metadata only; raster patches stay in source Zarr"}
        for arr in arrays.values():
            try: arr.flush(); arr._mmap.close()
            except Exception: pass
        arrays.clear()
        _atomic_write_json(tmp_dir / "metadata.json", metadata)
        try: (tmp_dir / "checkpoint.json").unlink()
        except Exception: pass
        if cache_dir.exists():
            backup = cache_dir.with_name(cache_dir.name + f".old_{int(time.time())}")
            os.replace(cache_dir, backup)
            print(f"Existing cache preserved at {backup}")
        os.replace(tmp_dir, cache_dir)
        validated = _compact_cache_validate(cache_dir, fingerprint)
        if validated is None:
            raise RuntimeError("Final parallel compact cache validation failed.")
        used = sum((cache_dir / spec["file"]).stat().st_size for spec in specs.values()) / _GIB
        print(f"[Parallel compact cache {task_mode}/{split}] READY: {n_valid:,} valid rows / "
              f"{capacity_rows:,} source rows; {used:.2f} GiB; builder workers={workers}.")
        return cache_dir
    except Exception as exc:
        print(f"[Parallel compact cache {task_mode}/{split}] BUILD FAILED/INTERRUPTED: {type(exc).__name__}: {exc}")
        print(f"Durable checkpoints/staging preserved at: {tmp_dir}")
        return None
    finally:
        if pbar is not None:
            pbar.close()
        # If the one-worker fallback raised inside a row group, its worker globals live in
        # this same process. Close those maps too so a retry does not leak file descriptors.
        _worker_arrays_cleanup = globals().get("_COMPACT_BUILD_WORKER_ARRAYS")
        if _worker_arrays_cleanup is not None:
            for _worker_arr in _worker_arrays_cleanup.values():
                try:
                    _worker_arr.flush()
                    _worker_arr._mmap.close()
                except Exception:
                    pass
            globals()["_COMPACT_BUILD_WORKER_ARRAYS"] = None
            globals()["_COMPACT_BUILD_WORKER_DS"] = None
            globals()["_COMPACT_BUILD_WORKER_PARQUET"] = None
            globals()["_COMPACT_BUILD_WORKER_PROGRESS"] = None
            globals()["_COMPACT_BUILD_WORKER_CAPACITY"] = None
        for arr in arrays.values():
            try: arr.flush(); arr._mmap.close()
            except Exception: pass
        arrays.clear()
        if progress_queue is not None and hasattr(progress_queue, "close"):
            try: progress_queue.close()
            except Exception: pass
        gc.collect()


class CachedGlobalDataset(Dataset):
    """Map-style view over valid source-row slots in the parallel compact cache."""
    def __init__(self, cache_dir, target_mode="residual", max_rows=None, shuffle=False, seed=SEED):
        self.cache_dir = Path(cache_dir)
        self.metadata = _compact_cache_validate(self.cache_dir)
        if self.metadata is None:
            raise RuntimeError(f"Parallel compact cache is invalid/incomplete: {self.cache_dir}")
        self.target_mode = str(target_mode)
        self.task_mode = str(self.metadata["task_mode"])
        if self.target_mode not in ("direct", "residual"):
            raise ValueError("target_mode must be 'direct' or 'residual'")
        self.shuffle = bool(shuffle)
        self.seed = int(seed)
        n = int(self.metadata["n_rows"])
        v = np.memmap(self.cache_dir / self.metadata["arrays"]["sample_valid"]["file"],
                      dtype=np.dtype(self.metadata["arrays"]["sample_valid"]["dtype"]), mode="r", shape=(n,))
        valid_rows = np.flatnonzero(np.asarray(v, dtype=bool)).astype(np.int64, copy=False)
        del v
        if self.target_mode == "residual":
            avspec = self.metadata["arrays"]["anchor_valid"]
            av = np.memmap(self.cache_dir / avspec["file"], dtype=np.dtype(avspec["dtype"]), mode="r", shape=(n,))
            valid_rows = valid_rows[np.asarray(av[valid_rows], dtype=bool)]
            del av
        if max_rows is not None:
            valid_rows = valid_rows[:max(0, int(max_rows))]
        self._valid_rows = valid_rows
        self._length = len(valid_rows)
    def __len__(self): return self._length
    def __getitem__(self, index):
        index = int(index)
        if index < 0: index += self._length
        if index < 0 or index >= self._length: raise IndexError(index)
        return int(self._valid_rows[index])
    def __getitems__(self, indices):
        # PyTorch calls this once per minibatch. Return the mapped source-row IDs in one
        # NumPy gather instead of executing one Python __getitem__ call per sample.
        positions = np.asarray(indices, dtype=np.int64)
        return self._valid_rows[positions]


class CachedGlobalBatchCollator:
    """Batch metadata plus byte-bounded raster patch cache; worker-local and CPU-only."""
    def __init__(self, cache_dir, metadata, target_mode):
        self.cache_dir = Path(cache_dir)
        self.metadata = metadata
        self.target_mode = str(target_mode)
        self.task_mode = str(metadata["task_mode"])
        self._arrays = None
        self._groups_by_year = {}
        self._lru = OrderedDict()
        self._lru_limit = max(0, int(globals().get("PATCH_LRU_MAX_ENTRIES", 64)))
        self._lru_budget_bytes = max(0, int(globals().get("PATCH_LRU_MAX_BYTES_PER_WORKER_MIB", 64))) * 1024**2
        self._lru_bytes = 0
        self._hits = 0
        self._misses = 0
        self._scaler = raster_scalers[self.task_mode]

    def _open_meta(self):
        if self._arrays is not None: return
        n = int(self.metadata["n_rows"])
        self._arrays = {key: np.memmap(self.cache_dir / spec["file"], dtype=np.dtype(spec["dtype"]),
                                       mode="r", shape=(n, *spec["shape"]))
                        for key, spec in self.metadata["arrays"].items()}

    def _groups(self, year):
        year = int(year)
        if year not in self._groups_by_year:
            if bool(RAM_ZARR_CACHE_ACTIVE):
                modis = {name: np.load(_cache_array_path("modis", year, name), mmap_mode="r", allow_pickle=False)
                         for name in ("evi", "lst")}
                s1 = {name: np.load(_cache_array_path("s1", year, name), mmap_mode="r", allow_pickle=False)
                      for name in S1_CACHE_ARRAYS}
            else:
                modis = open_group(MODIS_ROOT / str(year) / "patches.zarr")
                s1 = open_group(S1_ROOT / f"s1_nearest_valid_{year}.zarr")
            self._groups_by_year[year] = (modis, s1)
        return self._groups_by_year[year]

    def _normalize(self, name, values, valid):
        st = self._scaler[name]
        values = np.asarray(values, dtype=np.float32)
        valid = np.asarray(valid, dtype=bool) & np.isfinite(values)
        filled = np.where(valid, values, st["mean"]).astype(np.float32, copy=False)
        norm = np.clip((filled - st["mean"]) / st["std"], -10.0, 10.0).astype(np.float32, copy=False)
        return norm, valid.astype(np.float32, copy=False)

    @staticmethod
    def _nbytes(value):
        if isinstance(value, (tuple, list)):
            return sum(CachedGlobalBatchCollator._nbytes(x) for x in value)
        return int(getattr(value, "nbytes", 0))

    def _lru_put(self, key, value):
        size = self._nbytes(value)
        if self._lru_limit <= 0 or self._lru_budget_bytes <= 0 or size > self._lru_budget_bytes:
            return
        if key in self._lru:
            old = self._lru.pop(key)
            self._lru_bytes -= self._nbytes(old)
        while self._lru and (len(self._lru) >= self._lru_limit or self._lru_bytes + size > self._lru_budget_bytes):
            _, old = self._lru.popitem(last=False)
            self._lru_bytes -= self._nbytes(old)
        self._lru[key] = value
        self._lru_bytes += size

    def _raster_for(self, year, modis_idx, s1_idx):
        modis_key = ("modis", int(year), int(modis_idx))
        s1_key = ("s1", int(year), int(s1_idx))
        if modis_key in self._lru:
            self._hits += 1; self._lru.move_to_end(modis_key)
            evi_tensor, lst_tensor = self._lru[modis_key]
        else:
            self._misses += 1
            modis, _ = self._groups(year)
            evi_phys, evi_valid = decode_modis_patch(modis["evi"][int(modis_idx)], "evi")
            lst_phys, lst_valid = decode_modis_patch(modis["lst"][int(modis_idx)], "lst")
            evi_norm, evi_mask = self._normalize("evi", evi_phys, evi_valid)
            lst_norm, lst_mask = self._normalize("lst", lst_phys, lst_valid)
            evi_tensor = np.stack([evi_norm, evi_mask], axis=0)
            lst_tensor = np.stack([lst_norm, lst_mask], axis=0)
            self._lru_put(modis_key, (evi_tensor, lst_tensor))
        if s1_key in self._lru:
            self._hits += 1; self._lru.move_to_end(s1_key)
            s1_tensor = self._lru[s1_key]
        else:
            self._misses += 1
            _, s1 = self._groups(year)
            vals, masks = [], []
            for name in S1_ARRAYS:
                physical, valid = decode_s1_patch(s1[name][int(s1_idx)])
                norm, mask = self._normalize(name, physical, valid)
                vals.append(norm); masks.append(mask)
            s1_tensor = np.stack(vals + masks, axis=0)
            self._lru_put(s1_key, s1_tensor)
        return evi_tensor, lst_tensor, s1_tensor

    def __call__(self, row_indices):
        self._open_meta()
        hits_before, misses_before = self._hits, self._misses
        rows = np.asarray(row_indices, dtype=np.int64)
        batch = {}
        batch["seq"] = torch.from_numpy(np.asarray(self._arrays["seq"][rows], dtype=np.float32).copy())
        batch["seq_mask"] = torch.from_numpy(np.asarray(self._arrays["seq_mask"][rows], dtype=np.bool_).copy())
        batch["static"] = torch.from_numpy(np.asarray(self._arrays["static"][rows], dtype=np.float32).copy())
        batch["anchor"] = torch.from_numpy(np.asarray(self._arrays["anchor"][rows], dtype=np.float32).copy())
        direct_y = torch.from_numpy(np.asarray(self._arrays["y_direct"][rows], dtype=np.float32).copy())
        batch["y"] = direct_y - batch["anchor"] if self.target_mode == "residual" else direct_y
        years = np.asarray(self._arrays["year"][rows], dtype=np.int32)
        midx = np.asarray(self._arrays["modis_idx"][rows], dtype=np.int64)
        sidx = np.asarray(self._arrays["s1_idx"][rows], dtype=np.int64)
        raster_rows = [self._raster_for(y, mi, si) for y, mi, si in zip(years, midx, sidx)]
        batch["evi"] = torch.from_numpy(np.stack([x[0] for x in raster_rows]).astype(np.float32, copy=False))
        batch["lst"] = torch.from_numpy(np.stack([x[1] for x in raster_rows]).astype(np.float32, copy=False))
        batch["s1"] = torch.from_numpy(np.stack([x[2] for x in raster_rows]).astype(np.float32, copy=False))
        batch["anchor_valid"] = torch.from_numpy(np.asarray(self._arrays["anchor_valid"][rows], dtype=np.bool_).copy()).reshape(-1, 1)
        batch["sample_id"] = [f"compact_row_{int(i)}" for i in rows]
        batch["_patch_lru_batch_hits"] = int(self._hits - hits_before)
        batch["_patch_lru_batch_misses"] = int(self._misses - misses_before)
        batch["_patch_lru_bytes"] = int(self._lru_bytes)
        return batch


def get_global_dataset(split, task_mode=TASK_MODE, target_mode=TARGET_MODE, max_rows=None,
                       shuffle=False, seed=SEED):
    key = (str(task_mode), str(split))
    cache_dir = MODEL_READY_CACHE_PATHS.get(key, "__not_checked__")
    if cache_dir == "__not_checked__":
        if str(MODEL_READY_CACHE_MODE).lower() != "off":
            cache_dir = build_model_ready_cache(task_mode, split)
        else:
            cache_dir = None
        MODEL_READY_CACHE_PATHS[key] = cache_dir
    if cache_dir is not None:
        return CachedGlobalDataset(cache_dir, target_mode=target_mode, max_rows=max_rows,
                                   shuffle=shuffle, seed=seed)
    return GlobalSoilDataset(manifest_paths[(task_mode, split)], task_mode, target_mode,
                             tabular_scalers[task_mode], raster_scalers[task_mode],
                             max_rows=max_rows, shuffle=shuffle, seed=seed)


def prepare_model_ready_caches():
    """Build/reuse compact per-sample metadata caches in parallel; no raster patches are duplicated."""
    if str(MODEL_READY_CACHE_MODE).lower() == "off":
        print("MODEL_READY_CACHE_MODE='off'; using raw streaming datasets.")
        return False
    MODEL_READY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print("Preparing parallel compact metadata caches. Raster patches stay in source Zarr stores; "
          f"each training worker's normalized patch LRU is capped at {PATCH_LRU_MAX_BYTES_PER_WORKER_MIB} MiB.")
    successes = 0
    for mode in ("location_history", "environment_only"):
        for split in ("train", "test"):
            path = build_model_ready_cache(mode, split)
            MODEL_READY_CACHE_PATHS[(mode, split)] = path
            successes += int(path is not None)
    print(f"Parallel compact cache availability: {successes}/6 task/split sets.")
    return successes == 4


class RasterLocalityBatchSampler(Sampler):
    """Batch-shuffle within broad windows of a global native-raster ordering.

    The global ordering groups by year, MODIS EVI chunk, Sentinel chunk, then exact
    source row indices. A window contains approximately 131,072 samples. We split each
    window into ordinary physical batches, shuffle those batches (not the samples inside
    them), and then shuffle window order. This preserves batch-level locality while
    changing optimization order each epoch. No raster pixels are copied or cached here.
    """
    def __init__(self, dataset, batch_size, window_size=131072, seed=SEED):
        self.dataset = dataset
        self.batch_size = max(1, int(batch_size))
        self.window_size = max(self.batch_size, int(window_size))
        self.seed = int(seed)
        self.epoch = 0
        n_meta = int(self.dataset.metadata["n_rows"])
        specs = self.dataset.metadata["arrays"]
        self._year = np.memmap(self.dataset.cache_dir / specs["year"]["file"],
                               dtype=np.dtype(specs["year"]["dtype"]), mode="r", shape=(n_meta,))
        self._modis = np.memmap(self.dataset.cache_dir / specs["modis_idx"]["file"],
                                dtype=np.dtype(specs["modis_idx"]["dtype"]), mode="r", shape=(n_meta,))
        self._s1 = np.memmap(self.dataset.cache_dir / specs["s1_idx"]["file"],
                             dtype=np.dtype(specs["s1_idx"]["dtype"]), mode="r", shape=(n_meta,))
        self._n = int(len(self.dataset))
        self._order_path = (self.dataset.cache_dir /
            f"raster_order_{self.dataset.task_mode}_{self.dataset.target_mode}_"
            f"{globals().get('RASTER_LOCALITY_ORDER_VERSION', 'global_locality_v14')}.npy")
        self._sorted_positions = self._load_or_build_order()
        # Shuffle small groups of adjacent physical batches rather than all batches in a
        # 131k window. This keeps a worker near the same mmap pages for several batches.
        # With batch=2048 and group=4, optimization order changes every ~8k samples,
        # while the broad window remains sequential to avoid cold, distant mmap jumps.
        self.group_batches = max(1, int(globals().get("LOCALITY_SHUFFLE_GROUP_BATCHES", 4)))
        self.batches_per_window = max(1, self.window_size // self.batch_size)
        self.window_span_samples = self.batches_per_window * self.batch_size
        self.window_count = int(math.ceil(self._n / max(1, self.window_span_samples)))

    def _load_or_build_order(self):
        n = self._n
        if self._order_path.exists():
            try:
                cached = np.load(self._order_path, mmap_mode="r", allow_pickle=False)
                if cached.dtype == np.int64 and cached.shape == (n,):
                    print(f"Raster locality order reused: {self._order_path.name} ({n:,} rows)")
                    return cached
            except Exception as exc:
                print(f"Ignoring invalid raster-order cache ({type(exc).__name__}: {exc}); rebuilding.")

        started = time.perf_counter()
        print(f"[LOCALITY-ORDER] building raster-local order for {n:,} samples; "
              "this is preprocessing, not model training. A completed order is reused on restart.", flush=True)
        valid_rows = np.asarray(self.dataset._valid_rows, dtype=np.int64)
        years = np.asarray(self._year[valid_rows], dtype=np.int32)
        modis_idx = np.asarray(self._modis[valid_rows], dtype=np.int64)
        s1_idx = np.asarray(self._s1[valid_rows], dtype=np.int64)
        if n == 0:
            order = np.empty(0, dtype=np.int64)
        else:
            evi_rows = max(1, int(globals().get("RASTER_SORT_EVI_CHUNK_ROWS", 128)))
            s1_rows = max(1, int(globals().get("RASTER_SORT_S1_CHUNK_ROWS", 256)))
            if np.any(modis_idx < 0) or np.any(s1_idx < 0):
                raise RuntimeError("Raster locality sampler found negative raster indices in valid rows.")
            modis_chunk = modis_idx // evi_rows
            s1_chunk = s1_idx // s1_rows
            year_min = int(years.min())
            modis_base = int(modis_chunk.max()) + 1
            s1_base = int(s1_chunk.max()) + 1
            key = (((years.astype(np.int64) - year_min) * modis_base + modis_chunk) * s1_base
                   + s1_chunk)
            print(f"[LOCALITY-ORDER] sorting {n:,} rows by year/raster-chunk locality; "
                  f"key arrays ready after {time.perf_counter()-started:.1f}s.", flush=True)
            # Primary order is (year, EVI chunk, S1 chunk); exact row indices break ties.
            # This makes mmap access more sequential inside each batch while retaining chunk locality.
            order = np.lexsort((s1_idx, modis_idx, key)).astype(np.int64, copy=False)
            del key, years, modis_idx, s1_idx, modis_chunk, s1_chunk

        self._order_path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self._order_path.with_name(self._order_path.name + f".{os.getpid()}.tmp")
        try:
            with open(tmp, "wb") as f:
                np.save(f, order, allow_pickle=False)
                f.flush()
                os.fsync(f.fileno())
            os.replace(tmp, self._order_path)
        finally:
            try:
                if tmp.exists(): tmp.unlink()
            except OSError:
                pass
        elapsed = time.perf_counter() - started
        print(f"Built v14 raster locality order: {n:,} rows in {elapsed:.1f}s; "
              f"cached at {self._order_path.name} ({self._order_path.stat().st_size/1024**2:.1f} MiB).")
        return np.load(self._order_path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        return int(math.ceil(self._n / max(1, self.batch_size)))

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        self.epoch += 1
        # Traverse broad windows in sorted backing-file order. Do not shuffle broad windows:
        # distant mmap jumps caused major-fault/read_bytes spikes in v14. Within each window,
        # shuffle groups of adjacent batches, then keep batches sequential inside each group.
        for window_id in range(self.window_count):
            start = window_id * self.window_span_samples
            stop = min(self._n, start + self.window_span_samples)
            ordered_window = self._sorted_positions[start:stop]
            batch_starts = list(range(0, len(ordered_window), self.batch_size))
            group_width = max(1, self.group_batches)
            batch_groups = [batch_starts[i:i + group_width]
                            for i in range(0, len(batch_starts), group_width)]
            rng.shuffle(batch_groups)
            for batch_group in batch_groups:
                for offset in batch_group:
                    batch_positions = ordered_window[offset:min(len(ordered_window), offset + self.batch_size)]
                    # ndarray.tolist() converts the index vector in C rather than one Python
                    # int conversion at a time in a comprehension.
                    yield batch_positions


class RasterLocalityEvalSampler(Sampler):
    """Deterministic raster-local evaluation order, optionally over a split-wide subset.

    Aggregate metrics are order-independent. A full key sort makes each batch reuse nearby
    raster tiles; evenly spaced source positions provide broad coverage for fast trial scoring.
    """
    def __init__(self, dataset, max_samples=None, seed=SEED):
        self.dataset = dataset
        self.seed = int(seed)
        n = int(len(dataset))
        if max_samples is not None and int(max_samples) < n:
            # Systematic sample over the entire split (not just its chronological prefix).
            positions = np.linspace(0, n - 1, num=max(1, int(max_samples)), dtype=np.int64)
            positions = np.unique(positions)
        else:
            positions = np.arange(n, dtype=np.int64)
        valid_rows = dataset._valid_rows
        source_rows = positions if valid_rows is None else valid_rows[positions]
        meta_n = int(dataset.metadata["n_rows"])
        specs = dataset.metadata["arrays"]
        self._year = np.memmap(dataset.cache_dir / specs["year"]["file"],
            dtype=np.dtype(specs["year"]["dtype"]), mode="r", shape=(meta_n,))
        self._modis = np.memmap(dataset.cache_dir / specs["modis_idx"]["file"],
            dtype=np.dtype(specs["modis_idx"]["dtype"]), mode="r", shape=(meta_n,))
        self._s1 = np.memmap(dataset.cache_dir / specs["s1_idx"]["file"],
            dtype=np.dtype(specs["s1_idx"]["dtype"]), mode="r", shape=(meta_n,))
        order = np.lexsort((self._s1[source_rows], self._modis[source_rows], self._year[source_rows]))
        self.positions = positions[order]
    def __len__(self):
        return int(len(self.positions))
    def __iter__(self):
        return (int(p) for p in self.positions)

print(f"Parallel compact cache mode: {MODEL_READY_CACHE_MODE}; cache directory: {MODEL_READY_CACHE_DIR}; "
      f"cache-builder workers cap={MODEL_CACHE_BUILD_MAX_WORKERS}; training workers cap={MAX_NUM_WORKERS}.")
print("Persistent cache policy: completed compact caches and raw Zarr mmap arrays are reused across restarts;")
print("  absolute machine-specific manifest paths are ignored for cache identity, but row counts/schema/scalers/features must match.")

## 4. Model v19: multi-scale spatial CNNs, temporal attention pooling and gated fusion

Each raster modality uses a residual multi-scale encoder with local and dilated spatial filters. Three spatial summaries (mean, maximum and learned attention pooling) preserve more information than global-average pooling alone. The temporal encoder can use the selected Transformer/LSTM/TCN backbone, then learns an attention-weighted summary over all valid history steps and a gate between that summary and the latest valid step. A learned modality gate adaptively weights EVI, LST, Sentinel-1, temporal history and static context. The explicit center-pixel bypass and quadratic anchor/residual calibration are retained.

The new architecture is trained from scratch; v15/v17 model weights are not shape-compatible. Optuna tunes its spatial width, temporal hidden dimension, fusion-head width and the existing training hyperparameters. The train/test split and test-R² selection protocol are deliberately unchanged.

In [ ]:
# =============================== MODEL V18 ===============================
# v19 changes representation learning only: input schema, targets, calibration and
# training/evaluation contracts remain the same as v17.

class MultiScaleResidualBlock(nn.Module):
    """Two receptive-field scales with a residual path; keeps spatial maps intact."""
    def __init__(self, channels: int):
        super().__init__()
        groups = 8 if channels % 8 == 0 else 4 if channels % 4 == 0 else 1
        self.local = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False),
            nn.GroupNorm(groups, channels), nn.GELU(),
        )
        self.wide = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2,
                      groups=channels, bias=False),
            nn.GroupNorm(groups, channels), nn.GELU(),
        )
        self.mix = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.GroupNorm(groups, channels),
        )
        self.activation = nn.GELU()

    def forward(self, x):
        return self.activation(x + self.mix(torch.cat([self.local(x), self.wide(x)], dim=1)))


class MultiScaleSpatialStem(nn.Module):
    """Extract local and neighbourhood context, then pool with mean/max/learned attention."""
    def __init__(self, in_channels: int, width: int = 32, out_dim: int = 32):
        super().__init__()
        groups = 8 if width % 8 == 0 else 4 if width % 4 == 0 else 1
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, width, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, width), nn.GELU(),
        )
        self.block1 = MultiScaleResidualBlock(width)
        self.block2 = MultiScaleResidualBlock(width)
        self.spatial_score = nn.Conv2d(width, 1, kernel_size=1)
        self.project = nn.Sequential(
            nn.Linear(width * 3, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
        )

    def forward(self, x):
        feature_map = self.block2(self.block1(self.stem(x)))
        mean_pool = feature_map.mean(dim=(-2, -1))
        max_pool = feature_map.amax(dim=(-2, -1))
        attention = self.spatial_score(feature_map).flatten(2).softmax(dim=-1)
        attention_pool = torch.bmm(attention, feature_map.flatten(2).transpose(1, 2)).squeeze(1)
        return self.project(torch.cat([mean_pool, max_pool, attention_pool], dim=1))


class TemporalEncoder(nn.Module):
    """Encode history and learn how much to trust the latest step vs full-history context."""
    def __init__(self, input_dim: int, max_steps: int, backbone: str = "transformer",
                 out_dim: int = 32, hidden: int = 48):
        super().__init__()
        self.backbone = backbone.lower()
        self.hidden = int(hidden)
        self.input_proj = nn.Sequential(nn.Linear(input_dim, self.hidden), nn.LayerNorm(self.hidden))
        self.position = nn.Parameter(torch.zeros(1, max_steps, self.hidden))
        nn.init.normal_(self.position, mean=0.0, std=0.02)
        if self.backbone == "transformer":
            layer = nn.TransformerEncoderLayer(
                d_model=self.hidden, nhead=4, dim_feedforward=2 * self.hidden,
                dropout=0.1, batch_first=True, norm_first=True, activation="gelu",
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        elif self.backbone == "lstm":
            self.encoder = nn.LSTM(self.hidden, self.hidden, num_layers=1, batch_first=True)
        elif self.backbone == "tcn":
            self.encoder = nn.Sequential(
                nn.Conv1d(self.hidden, self.hidden, kernel_size=3, padding=1), nn.GELU(),
                nn.Conv1d(self.hidden, self.hidden, kernel_size=3, padding=2, dilation=2), nn.GELU(),
            )
        else:
            raise ValueError("TEMPORAL_BACKBONE must be 'transformer', 'lstm', or 'tcn'")
        self.history_score = nn.Sequential(nn.LayerNorm(self.hidden), nn.Linear(self.hidden, 1))
        self.last_history_gate = nn.Linear(self.hidden * 2, self.hidden)
        self.projection = nn.Sequential(
            nn.LayerNorm(self.hidden), nn.Linear(self.hidden, out_dim), nn.GELU(),
        )

    def forward(self, seq, seq_mask):
        # seq_mask=True denotes padding. Dataset histories are right-padded; lengths below
        # therefore identify the last valid observation without assuming fixed history length.
        mask = seq_mask.to(dtype=torch.bool)
        lengths = (~mask).sum(dim=1).clamp(min=1)
        x = self.input_proj(seq) + self.position[:, :seq.shape[1], :]
        if self.backbone == "transformer":
            x = self.encoder(x, src_key_padding_mask=mask)
        elif self.backbone == "lstm":
            # Histories are left-padded and can contain missing-date gaps. Do not use
            # pack_padded_sequence: it assumes valid tokens occupy the first `length` slots.
            x = x.masked_fill(mask.unsqueeze(-1), 0.0)
            x, _ = self.encoder(x)
        else:
            x = x.masked_fill(mask.unsqueeze(-1), 0.0).transpose(1, 2)
            x = self.encoder(x).transpose(1, 2)
        x = x.masked_fill(mask.unsqueeze(-1), 0.0)
        # Find the final actual valid position (the current-date token is expected here),
        # rather than assuming valid tokens are contiguous or right-padded.
        positions = torch.arange(seq.shape[1], device=x.device).unsqueeze(0).expand_as(mask)
        last_index = positions.masked_fill(mask, -1).amax(dim=1).clamp(min=0)
        last = x[torch.arange(x.shape[0], device=x.device), last_index]
        scores = self.history_score(x).squeeze(-1).masked_fill(mask, -1e4)
        weights = torch.softmax(scores, dim=1)
        pooled_history = torch.sum(x * weights.unsqueeze(-1), dim=1)
        gate = torch.sigmoid(self.last_history_gate(torch.cat([last, pooled_history], dim=1)))
        temporal = gate * last + (1.0 - gate) * pooled_history
        return self.projection(temporal)


class QuadraticCalibrationLayer(nn.Module):
    """Identity-initialized differentiable anchor/residual calibration."""
    def __init__(self):
        super().__init__()
        self.coefficients = nn.Parameter(torch.tensor([0.0, 1.0, 1.0, 0.0, 0.0, 0.0]))

    def forward(self, anchor, residual):
        a = anchor.reshape(-1, 1).float()
        r = residual.reshape(-1, 1).float()
        c = self.coefficients
        return c[0] + c[1] * a + c[2] * r + c[3] * a * r + c[4] * a.square() + c[5] * r.square()


class GlobalSoilNet(nn.Module):
    def __init__(self, static_dim: int, backbone: str, target_mode: str = "residual",
                 spatial_width: int = 32, temporal_hidden: int = 48, fusion_width: int = 192):
        super().__init__()
        self.target_mode = target_mode
        self.spatial_width = int(spatial_width)
        self.temporal_hidden = int(temporal_hidden)
        self.fusion_width = int(fusion_width)
        self.evi_cnn = MultiScaleSpatialStem(in_channels=2, width=self.spatial_width, out_dim=32)
        self.lst_cnn = MultiScaleSpatialStem(in_channels=2, width=self.spatial_width, out_dim=32)
        self.s1_cnn = MultiScaleSpatialStem(in_channels=2 * len(S1_ARRAYS), width=self.spatial_width, out_dim=32)
        self.temporal = TemporalEncoder(
            input_dim=2 * len(TEMP_FEATURES) + 3,
            max_steps=HISTORY_STEPS,
            backbone=backbone,
            out_dim=32,
            hidden=self.temporal_hidden,
        )
        self.static_mlp = nn.Sequential(
            nn.Linear(static_dim, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.GELU(),
        )
        # Softmax gates let the network adaptively prioritize EVI, LST, S1, history and static context.
        self.modality_gate = nn.Sequential(
            nn.Linear(32 * 5, 64), nn.GELU(), nn.Linear(64, 5),
        )
        self.center_dim = 2 + 2 + 2 * len(S1_ARRAYS)
        fusion_dim = 32 * 5 + 32 + self.center_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_dim, self.fusion_width), nn.LayerNorm(self.fusion_width),
            nn.GELU(), nn.Dropout(0.15),
            nn.Linear(self.fusion_width, self.fusion_width // 2), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(self.fusion_width // 2, 1),
        )
        self.calibrator = QuadraticCalibrationLayer()

    def forward(self, batch, return_raw=False):
        z_evi = self.evi_cnn(batch["evi"])
        z_lst = self.lst_cnn(batch["lst"])
        z_s1 = self.s1_cnn(batch["s1"])
        z_time = self.temporal(batch["seq"], batch["seq_mask"])
        z_static = self.static_mlp(batch["static"])
        tokens = torch.stack([z_evi, z_lst, z_s1, z_time, z_static], dim=1)
        modality_weights = torch.softmax(self.modality_gate(tokens.flatten(1)), dim=1)
        gated_tokens = tokens * (5.0 * modality_weights.unsqueeze(-1))
        weighted_context = torch.sum(tokens * modality_weights.unsqueeze(-1), dim=1)

        center_evi = batch["evi"][:, :, batch["evi"].shape[-2] // 2, batch["evi"].shape[-1] // 2]
        center_lst = batch["lst"][:, :, batch["lst"].shape[-2] // 2, batch["lst"].shape[-1] // 2]
        center_s1 = batch["s1"][:, :, batch["s1"].shape[-2] // 2, batch["s1"].shape[-1] // 2]
        center_rasters = torch.cat([center_evi, center_lst, center_s1], dim=1)
        fused = torch.cat([gated_tokens.flatten(1), weighted_context, center_rasters], dim=1)
        raw_output = self.fusion_head(fused)
        anchor = batch["anchor"].reshape(-1, 1)
        residual_coordinate = raw_output if self.target_mode == "residual" else raw_output - anchor
        calibrated_output = self.calibrator(anchor, residual_coordinate)
        if return_raw:
            raw_physical = anchor + raw_output if self.target_mode == "residual" else raw_output
            return calibrated_output, raw_physical
        return calibrated_output


def build_model(task_mode=TASK_MODE, backbone=TEMPORAL_BACKBONE, target_mode=TARGET_MODE,
                spatial_width=None, temporal_hidden=None, fusion_width=None):
    static_features = STATIC_FEATURES + ANCHOR_FEATURES
    if task_mode == "location_history":
        static_features += LOCATION_FEATURES
    model = GlobalSoilNet(
        static_dim=2 * len(static_features), backbone=backbone, target_mode=target_mode,
        spatial_width=int(MODEL_SPATIAL_WIDTH if spatial_width is None else spatial_width),
        temporal_hidden=int(MODEL_TEMPORAL_HIDDEN if temporal_hidden is None else temporal_hidden),
        fusion_width=int(MODEL_FUSION_WIDTH if fusion_width is None else fusion_width),
    ).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    return model


def physical_target(batch, target_mode):
    """Map each dataset target back to physical m³/m³ units."""
    return batch["y"] + batch["anchor"] if target_mode == "residual" else batch["y"]


def calibration_identity_penalty(model):
    identity = model.calibrator.coefficients.new_tensor([0.0, 1.0, 1.0, 0.0, 0.0, 0.0])
    return torch.mean((model.calibrator.coefficients - identity) ** 2)

CALIBRATION_IDENTITY_PENALTY = 1e-4
MODEL_SPATIAL_WIDTH = 32
MODEL_TEMPORAL_HIDDEN = 48
MODEL_FUSION_WIDTH = 192
model = build_model()
num_parameters = sum(p.numel() for p in model.parameters())
print("Model v19 parameters:", f"{num_parameters:,}", "| device:", device)
print("Model dimensions: spatial_width=", MODEL_SPATIAL_WIDTH,
      "temporal_hidden=", MODEL_TEMPORAL_HIDDEN, "fusion_width=", MODEL_FUSION_WIDTH)
print("Calibration coefficients (identity init):", model.calibrator.coefficients.detach().cpu().tolist())

## 5. Optional smoke test: real batches, patch shapes and backward pass

Disabled by default (`RUN_SMOKE_TEST=False`). Enable it only when you want the multi-batch forward/backward check. The separate single-sample preflight in the training cell remains enabled by default.

This reads only a few actual rows from the configured training split and does a forward/backward pass. It does not train an epoch. If it passes, you can enable the short training benchmark, then decide whether to start full training.

In [ ]:
# =============================== SMOKE TEST ===============================
if RUN_SMOKE_TEST:
    smoke_loader = make_loader(
        split="train", task_mode=TASK_MODE,
        max_rows=BATCH_SIZE * SMOKE_TEST_BATCHES,
        shuffle=False,
    )
    model.train()
    smoke_results = []
    for batch_i, batch in enumerate(smoke_loader):
        if batch is None:
            continue
        batch_device = move_batch(batch) if "move_batch" in globals() else {
            k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()
        }
        prediction = model(batch_device)
        target_physical = physical_target(batch_device, TARGET_MODE).reshape_as(prediction)
        loss = nn.functional.huber_loss(prediction, target_physical) + CALIBRATION_IDENTITY_PENALTY * calibration_identity_penalty(model)
        model.zero_grad(set_to_none=True)
        loss.backward()
        smoke_results.append({
            "batch": batch_i,
            "batch_size": int(batch_device["y"].shape[0]),
            "loss": float(loss.detach().cpu()),
            "evi_shape": tuple(batch_device["evi"].shape),
            "lst_shape": tuple(batch_device["lst"].shape),
            "s1_shape": tuple(batch_device["s1"].shape),
            "sequence_shape": tuple(batch_device["seq"].shape),
            "static_shape": tuple(batch_device["static"].shape),
            "prediction_shape": tuple(prediction.shape),
            "calibration_coefficients": model.calibrator.coefficients.detach().float().cpu().tolist(),
        })
        if batch_i + 1 >= SMOKE_TEST_BATCHES:
            break
    if not smoke_results:
        raise RuntimeError("No smoke-test batches were produced. Check target/anchor availability and split counts.")
    display(pd.DataFrame(smoke_results))
    print("Smoke test passed. Optuna search can now be started in the training cell.")

## 6. Training, adaptive microbatches, EMA evaluation, and checkpointing

Training optimizes Huber loss in physical soil-moisture units. The effective optimizer batch is assembled from prefetched loader microbatches with gradient accumulation. The model contains a trainable quadratic calibration layer initialized to `anchor + residual`; the calibration coefficients are regularized mildly toward identity. Direct-target mode is converted to residual coordinates only inside this layer, preserving the direct prediction at identity initialization. Metrics are calculated online so predictions for the entire split are not accumulated in RAM.

Optuna is deliberately a **bounded screening pass**: 3 epochs × the configured per-trial sample budget and 16 evaluation batches per trial. Those trial scores are proxies, not full-data results. `RUN_FINAL_RETRAIN=True` then trains each selected winner on the **complete training split** (`MAX_ROWS_PER_SPLIT=None`, `FINAL_RETRAIN_MAX_TRAIN_BATCHES=None`) for up to 52 epochs with early stopping. A coverage guard compares the number of trained batches against `len(train_loader)` every retraining epoch, and the notebook prints processed batches and samples. Set `RUN_FINAL_RETRAIN=False` only if you want screening without a full-data fit.

### Optional legacy VRAM-resident rolling-shard mode (disabled by default)

This older path is retained only for controlled comparisons. It is disabled in v10 because its startup fill creates idle time before every epoch. When explicitly enabled, it may stage one batch initially and then refill consumed slots asynchronously. The default path instead starts training on the first ready batch and stages up to three future batches on a background CUDA stream.


**Startup behaviour in v10:** streaming prefetch is the default and training starts as soon as the first batch is ready; there is no large synchronous VRAM fill. The legacy resident-shard mode can be enabled manually for comparison, but its initial fill is capped at one batch. MLflow launch is attempted inline; if the frontend does not render the extension, use the MLflow URL printed by the extension and the log directory printed at startup.

## Final training progress reporting

Each Optuna trial reports:
- epoch progress bar
- train/test loss and R²
- RMSE
- samples/sec
- data wait time
- GPU utilisation
- VRAM usage
- effective batch, microbatch and accumulation steps

The epoch bars are retained during the full search so long runs remain observable.


**Laptop throughput update:** the training collator fetches distinct raster indices in vectorized chunks rather than issuing individual Zarr reads in a per-sample Python loop. The Optuna stage is intentionally limited to 3 epochs × the configured per-trial sample budget × 16 evaluation batches per trial, across up to 12 trials per task. The selected configuration is then retrained on the complete training split by default (`RUN_FINAL_RETRAIN=True`, `FINAL_RETRAIN_MAX_TRAIN_BATCHES=None`, `MAX_ROWS_PER_SPLIT=None`); full-test metrics are run once at the end. This separates inexpensive model screening from the actual full-data fit.

### v10 input pipeline

Training uses streaming prefetch by default. The first ready batch is enough to start; subsequent batches are prepared in the background on a separate CUDA stream. There is no synchronous large resident-shard fill before each epoch. The optional resident-shard path is retained for comparison and is capped at one initial batch.

In [ ]:
# =============================== FINAL ADAPTIVE INPUT PIPELINE SETTINGS ===============================
# IMPORTANT: do not redefine build_global_loader here. The canonical implementation above
# accepts num_workers/pin_memory/persistent_workers and installs collate_global + worker_init_fn.
# This cell changes only tuning policy and candidate worker counts.

MAX_NUM_WORKERS = min(3, max(1, int(MAX_NUM_WORKERS)))
PREFETCH_FACTOR = 2  # controlled two-batch worker prefetch; bounded by sample budget below
# Reserve memory for the OS and interactive session. Worker RAM is budgeted across concurrent
# train/test pools; final counts are still bounded by visible CPUs, available RAM, and Parquet groups.
HOST_RAM_RESERVE_GIB = 1.5
WORKER_RAM_BUDGET_GIB = 0.50
MAX_PREFETCHED_SAMPLES_PER_LOADER = 12288  # at most two batches per worker at 3 workers x 2048 samples
DATALOADER_TIMEOUT_SEC = 300


def _candidate_worker_counts():
    cap = resource_limited_worker_cap()
    candidates = [0]
    for count in (1, 2, 4, 6, 8, 12, 16, 24, 32):
        if 1 <= count <= cap:
            candidates.append(count)
    if cap > 0 and cap not in candidates:
        candidates.append(cap)
    return sorted(set(candidates))

print("Adaptive loader policy active (canonical loader preserved)")
print("CPU logical/affinity count:", visible_cpu_count())
print("Worker search ceiling:", MAX_NUM_WORKERS)
print("Resource-limited worker cap:", resource_limited_worker_cap())
print("Prefetch factor ceiling:", PREFETCH_FACTOR)
print("Prefetch sample budget per loader:", MAX_PREFETCHED_SAMPLES_PER_LOADER)
print("RAM reserve GiB:", HOST_RAM_RESERVE_GIB, "| worker budget GiB/worker/pool:", WORKER_RAM_BUDGET_GIB)

In [ ]:
# =============================== VECTORIZED CACHED RASTER BATCH READER ===============================
# Chunk-aware raster reader (v10).
# The v8 per-patch LRU had almost no hits because individual sample raster indices are mostly
# unique. Zarr is chunked along axis 0 (EVI=128, LST/S1=256): cache normalized source CHUNKS,
# not individual patches. Each source chunk is read/decompressed once, normalized once, then
# gathered for all selected samples. Cache is strictly byte-bounded worker RAM; no raster cache
# files or expanded patch arrays are ever written to disk.
from collections import defaultdict, OrderedDict

RASTER_BATCH_READ_ROWS = 256  # compatibility setting; reads now follow each array's native Zarr chunks
PATCH_LRU_MAX_ENTRIES = max(16, int(PATCH_LRU_MAX_ENTRIES))
# One normalized chunk cache per DataLoader worker; hard RAM ceiling comes from cell 2.

class CachedGlobalBatchCollator:
    """Low-allocation vectorized raster gather from native Zarr or the raw mmap mirror.

    The previous implementation called np.unique(..., return_inverse=True) for each
    raster modality in every batch. The measured train stream had a unique/requested
    ratio of 1.000, so sorting and inverse-remapping every index was pure overhead.
    This implementation gathers rows in batch order (duplicates are allowed and remain
    correct), normalizes in-place in one float32 work buffer, and writes directly into
    FP16 channel-last backing arrays. DataLoader pinning and H2D can therefore preserve
    the channels-last layout instead of converting NCHW-contiguous tensors on the GPU.
    """
    def __init__(self, cache_dir, metadata, target_mode):
        self.cache_dir = Path(cache_dir)
        self.metadata = metadata
        self.target_mode = str(target_mode)
        self.task_mode = str(metadata["task_mode"])
        self._arrays = None
        self._groups_by_year = {}
        self._scaler = raster_scalers[self.task_mode]
        self._source_rows_read = 0
        self._source_rows_requested = 0
        self._source_chunks_touched = 0
        self._uses_mmap = bool(RAM_ZARR_CACHE_ACTIVE)

    def _open_meta(self):
        if self._arrays is not None:
            return
        n = int(self.metadata["n_rows"])
        self._arrays = {
            key: np.memmap(self.cache_dir / spec["file"], dtype=np.dtype(spec["dtype"]),
                           mode="r", shape=(n, *spec["shape"]))
            for key, spec in self.metadata["arrays"].items()
        }

    @staticmethod
    def _apply_mmap_advice(array):
        # Linux/WSL only. This is a per-mapping performance hint, never a data transformation.
        advice_name = str(globals().get("RASTER_MMAP_ADVICE", "normal")).lower()
        if advice_name in {"normal", "off", "none"} or os.name != "posix":
            return
        try:
            import mmap as _mmap
            advice = {"random": _mmap.MADV_RANDOM,
                      "sequential": _mmap.MADV_SEQUENTIAL}.get(advice_name)
            mapped = getattr(array, "_mmap", None)
            if advice is not None and mapped is not None and hasattr(mapped, "madvise"):
                mapped.madvise(advice)
        except (AttributeError, OSError, ValueError):
            # Unsupported kernel/filesystem: retain normal kernel behaviour.
            return

    def _groups(self, year):
        year = int(year)
        if year not in self._groups_by_year:
            if self._uses_mmap:
                modis = {name: np.load(_cache_array_path("modis", year, name),
                                       mmap_mode="r", allow_pickle=False)
                         for name in ("evi", "lst")}
                s1 = {name: np.load(_cache_array_path("s1", year, name),
                                    mmap_mode="r", allow_pickle=False)
                      for name in S1_CACHE_ARRAYS}
                for _array in (*modis.values(), *s1.values()):
                    self._apply_mmap_advice(_array)
            else:
                modis = open_group(MODIS_ROOT / str(year) / "patches.zarr")
                s1 = open_group(S1_ROOT / f"s1_nearest_valid_{year}.zarr")
            self._groups_by_year[year] = (modis, s1)
        return self._groups_by_year[year]

    @staticmethod
    def _read_rows(array, indices):
        """Gather requested rows in their existing order; do not sort/deduplicate them."""
        indices = np.asarray(indices, dtype=np.int64)
        if indices.size == 0:
            return np.empty((0, *tuple(int(x) for x in array.shape[1:])), dtype=np.dtype(array.dtype))
        if isinstance(array, np.ndarray):  # includes np.memmap
            return np.asarray(array[indices])
        oindex = getattr(array, "oindex", None)
        if oindex is not None:
            try:
                return np.asarray(oindex[indices])
            except Exception:
                pass
        return np.asarray(array[indices])

    @staticmethod
    def _read_rows_float32_into(array, indices, destination):
        """Gather/cast directly into the processing buffer, avoiding a second full raster copy."""
        indices = np.asarray(indices, dtype=np.int64)
        if isinstance(array, np.ndarray):  # includes np.memmap
            if np.dtype(array.dtype) == np.dtype(np.float32):
                # Common float32 source: gather straight into the final work buffer.
                np.take(array, indices, axis=0, out=destination)
            else:
                # NumPy take requires a same-dtype output buffer for integer/other sources.
                # Preallocate that gather explicitly, then cast once into the work buffer.
                raw = np.empty((len(indices), *tuple(int(x) for x in array.shape[1:])),
                               dtype=np.dtype(array.dtype))
                np.take(array, indices, axis=0, out=raw)
                np.copyto(destination, raw, casting="unsafe")
        else:
            # Zarr fallback: one temporary gather, then cast/copy into the same work buffer.
            raw = CachedGlobalBatchCollator._read_rows(array, indices)
            np.copyto(destination, raw, casting="unsafe")
        return destination

    @staticmethod
    def _allocate_channels_last(n, channels, height, width, dtype):
        # A transposed view of NHWC storage has logical NCHW shape and channels-last strides.
        backing = np.empty((n, height, width, channels), dtype=dtype)
        return backing.transpose(0, 3, 1, 2)

    def _write_modis(self, destination, positions, array, indices, kind):
        # np.take(..., out=...) casts the raw mmap rows directly to float32. The v16 path
        # first materialized a raw-dtype gather and then copied it into this float32 buffer.
        work = np.empty((len(indices), *tuple(int(x) for x in array.shape[1:])), dtype=np.float32)
        self._read_rows_float32_into(array, indices, work)
        valid = np.empty(work.shape, dtype=np.bool_)
        scratch = np.empty(work.shape, dtype=np.bool_)
        if kind == "evi":
            lo, hi, fill, scale = float(EVI_RAW_MIN), float(EVI_RAW_MAX), float(EVI_FILL_RAW), float(EVI_SCALE)
        elif kind == "lst":
            lo, hi, fill, scale = float(LST_RAW_MIN), float(LST_RAW_MAX), float(LST_FILL_RAW), float(LST_SCALE)
        else:
            raise ValueError(kind)
        # Range comparisons also reject NaN and +/-inf; use output masks to avoid chained
        # boolean-expression temporaries. Keep the explicit fill-value test for exact semantics.
        np.greater_equal(work, lo, out=valid)
        np.less_equal(work, hi, out=scratch)
        np.logical_and(valid, scratch, out=valid)
        np.not_equal(work, fill, out=scratch)
        np.logical_and(valid, scratch, out=valid)
        np.multiply(work, scale, out=work)
        stats = self._scaler[kind]
        np.subtract(work, float(stats["mean"]), out=work)
        np.divide(work, max(float(stats["std"]), 1e-8), out=work)
        np.clip(work, -10.0, 10.0, out=work)
        work[~valid] = 0.0
        destination[positions, 0] = work
        destination[positions, 1] = valid
        position_count = int(positions.stop - positions.start) if isinstance(positions, slice) else int(len(positions))
        self._source_rows_read += int(len(indices))
        self._source_rows_requested += position_count
        if not self._uses_mmap:
            chunks = self._chunk_rows(array, 128 if kind == "evi" else 256)
            self._source_chunks_touched += int(np.unique(np.asarray(indices) // chunks).size)

    def _write_s1(self, destination, positions, s1_group, indices):
        channel_count = len(S1_ARRAYS)
        for channel, name in enumerate(S1_ARRAYS):
            # Direct float32 gather replaces raw gather + dtype conversion copy.
            source = s1_group[name]
            work = np.empty((len(indices), *tuple(int(x) for x in source.shape[1:])), dtype=np.float32)
            self._read_rows_float32_into(source, indices, work)
            valid = np.empty(work.shape, dtype=np.bool_)
            np.isfinite(work, out=valid)
            stats = self._scaler[name]
            np.subtract(work, float(stats["mean"]), out=work)
            np.divide(work, max(float(stats["std"]), 1e-8), out=work)
            np.clip(work, -10.0, 10.0, out=work)
            work[~valid] = 0.0
            destination[positions, channel] = work
            destination[positions, channel_count + channel] = valid
        position_count = int(positions.stop - positions.start) if isinstance(positions, slice) else int(len(positions))
        self._source_rows_read += int(channel_count * len(indices))
        self._source_rows_requested += int(channel_count * position_count)
        if not self._uses_mmap:
            chunks = self._chunk_rows(s1_group[S1_ARRAYS[0]], 256)
            self._source_chunks_touched += int(np.unique(np.asarray(indices) // chunks).size)

    @staticmethod
    def _chunk_rows(array, fallback):
        chunks = getattr(array, "chunks", None)
        try:
            return max(1, int(chunks[0])) if chunks is not None else int(fallback)
        except Exception:
            return int(fallback)

    def _load_raster_batch(self, years, modis_indices, s1_indices):
        n = int(len(years))
        if n == 0:
            return (np.empty((0, 2, 0, 0), np.float32),
                    np.empty((0, 2, 0, 0), np.float32),
                    np.empty((0, 2 * len(S1_ARRAYS), 0, 0), np.float32))

        first_modis, first_s1 = self._groups(int(years[0]))
        eh, ew = tuple(int(x) for x in first_modis["evi"].shape[1:])
        lh, lw = tuple(int(x) for x in first_modis["lst"].shape[1:])
        sh, sw = tuple(int(x) for x in first_s1[S1_ARRAYS[0]].shape[1:])
        out_dtype = np.float16 if (device.type == "cuda" and USE_AMP) else np.float32
        out_evi = self._allocate_channels_last(n, 2, eh, ew, out_dtype)
        out_lst = self._allocate_channels_last(n, 2, lh, lw, out_dtype)
        out_s1 = self._allocate_channels_last(n, 2 * len(S1_ARRAYS), sh, sw, out_dtype)

        years = np.asarray(years, dtype=np.int32)
        modis_indices = np.asarray(modis_indices, dtype=np.int64)
        s1_indices = np.asarray(s1_indices, dtype=np.int64)

        # Fast path is common because the locality sampler sorts by year first.
        if np.all(years == years[0]):
            year_groups = ((int(years[0]), slice(0, n)),)
        else:
            year_groups = []
            for year in np.unique(years):
                year_groups.append((int(year), np.flatnonzero(years == year)))

        for year, positions in year_groups:
            modis, s1 = self._groups(year)
            # No np.unique(return_inverse=True): the measured stream had no repeated
            # source rows (unique/requested == 1.000), so sort+inverse only added work.
            # Direct gathers also remain correct if a future batch contains duplicates.
            midx = modis_indices[positions]
            sidx = s1_indices[positions]
            self._write_modis(out_evi, positions, modis["evi"], midx, "evi")
            self._write_modis(out_lst, positions, modis["lst"], midx, "lst")
            self._write_s1(out_s1, positions, s1, sidx)

        return out_evi, out_lst, out_s1

    def __call__(self, row_indices):
        collate_started = time.perf_counter()
        self._open_meta()
        rows = np.asarray(row_indices, dtype=np.int64)
        batch = {}
        meta_started = time.perf_counter()
        batch["seq"] = torch.from_numpy(np.asarray(self._arrays["seq"][rows], dtype=np.float32))
        batch["seq_mask"] = torch.from_numpy(np.asarray(self._arrays["seq_mask"][rows], dtype=np.bool_))
        batch["static"] = torch.from_numpy(np.asarray(self._arrays["static"][rows], dtype=np.float32))
        batch["anchor"] = torch.from_numpy(np.asarray(self._arrays["anchor"][rows], dtype=np.float32))
        direct_y = torch.from_numpy(np.asarray(self._arrays["y_direct"][rows], dtype=np.float32))
        batch["y"] = direct_y - batch["anchor"] if self.target_mode == "residual" else direct_y
        years = np.asarray(self._arrays["year"][rows], dtype=np.int32)  # fancy indexing already materializes one owned batch array
        batch["year"] = torch.from_numpy(years)
        midx = np.asarray(self._arrays["modis_idx"][rows], dtype=np.int64)
        sidx = np.asarray(self._arrays["s1_idx"][rows], dtype=np.int64)
        meta_elapsed = time.perf_counter() - meta_started

        raster_started = time.perf_counter()
        use_gpu_tiles = (device.type == "cuda" and
                         str(globals().get("GPU_RASTER_CACHE_MODE", "off")).lower() not in {"off", "false", "0", "none"})
        if use_gpu_tiles:
            batch["_gpu_raster_task_mode"] = self.task_mode
            batch["_gpu_raster_cache_dir"] = str(self.cache_dir)
            batch["_gpu_raster_years"] = torch.from_numpy(years)
            batch["_gpu_raster_modis_idx"] = torch.from_numpy(midx)
            batch["_gpu_raster_s1_idx"] = torch.from_numpy(sidx)
        else:
            before_read = self._source_rows_read
            before_requested = self._source_rows_requested
            before_chunks = self._source_chunks_touched
            evi, lst, s1 = self._load_raster_batch(years, midx, sidx)
            batch["evi"] = torch.from_numpy(evi)
            batch["lst"] = torch.from_numpy(lst)
            batch["s1"] = torch.from_numpy(s1)
            batch["_raster_rows_gathered"] = int(self._source_rows_read - before_read)
            batch["_raster_rows_requested"] = int(self._source_rows_requested - before_requested)
            batch["_raster_chunks_touched"] = int(self._source_chunks_touched - before_chunks)
            batch["_raster_source_mode"] = "raw_mmap_direct_rows" if self._uses_mmap else "zarr_orthogonal_rows"
        batch["anchor_valid"] = torch.from_numpy(np.asarray(self._arrays["anchor_valid"][rows], dtype=np.bool_)).reshape(-1, 1)
        raster_elapsed = time.perf_counter() - raster_started
        batch["_patch_lru_batch_hits"] = 0
        batch["_patch_lru_batch_misses"] = 0
        batch["_patch_lru_bytes"] = 0
        batch["_raster_source_rows_read"] = int(batch.get("_raster_rows_gathered", 0))
        batch["_raster_source_rows_requested"] = int(batch.get("_raster_rows_requested", 0))
        batch["_raster_source_chunks_read"] = int(batch.get("_raster_chunks_touched", 0))
        batch["_collate_meta_sec"] = float(meta_elapsed)
        batch["_collate_raster_sec"] = float(raster_elapsed)
        batch["_collate_total_sec"] = float(time.perf_counter() - collate_started)
        return batch

class GPURasterTileCache:
    """Bounded LRU of normalized raster tiles on dedicated CUDA memory.

    The cache uses fp16 normalized channels and 128-row tiles to make repeat reads cheap while
    preserving headroom for the model. It never assumes Windows shared GPU memory is fast VRAM.
    Cache misses are read from Zarr/mmap and normalized once for that tile; hits bypass Zarr and
    CPU normalization. If VRAM headroom is low, tiles are used for the current batch but not kept.
    """
    def __init__(self, cache_dir, metadata, task_mode):
        self.cache_dir = Path(cache_dir)
        self.metadata = metadata
        self.task_mode = str(task_mode)
        self.reader = CachedGlobalBatchCollator(self.cache_dir, metadata, "direct")
        self.tile_rows = max(16, int(globals().get("GPU_RASTER_TILE_ROWS", 128)))
        self.max_bytes = max(0, int(float(globals().get("GPU_RASTER_CACHE_MAX_GIB", 1.10)) * (1024**3)))
        self.reserve_bytes = max(0, int(float(globals().get("GPU_RASTER_CACHE_RESERVE_GIB", 0.60)) * (1024**3)))
        self.lru = OrderedDict()
        self.cache_bytes = 0
        self.hits = 0
        self.misses = 0
        self.batches = 0
        self._printed = False

    @staticmethod
    def _bytes(tile):
        if isinstance(tile, tuple):
            return sum(int(x.numel() * x.element_size()) for x in tile)
        return int(tile.numel() * tile.element_size())

    def _effective_capacity(self):
        if device.type != "cuda" or self.max_bytes <= 0:
            return 0
        try:
            free_bytes, _total_bytes = torch.cuda.mem_get_info(device)
            # free + existing cache approximates the headroom before this cache was allocated.
            after_reserve = max(0, int(free_bytes) + self.cache_bytes - self.reserve_bytes)
            return min(self.max_bytes, after_reserve)
        except Exception:
            return self.max_bytes

    def _get(self, key):
        if key not in self.lru:
            return None
        value = self.lru.pop(key)
        self.lru[key] = value
        self.hits += 1
        return value

    def _put(self, key, value):
        size = self._bytes(value)
        cap = self._effective_capacity()
        # If available headroom has shrunk since the last batch, promptly drop old tiles first.
        while self.lru and self.cache_bytes > cap:
            _old_key, old = self.lru.popitem(last=False)
            self.cache_bytes -= self._bytes(old)
        if size <= 0 or size > cap:
            return False
        while self.lru and self.cache_bytes + size > cap:
            _old_key, old = self.lru.popitem(last=False)
            self.cache_bytes -= self._bytes(old)
        # Never pin a tile if the model/desktop is currently using the reserved VRAM headroom.
        if self.cache_bytes + size > cap:
            return False
        self.lru[key] = value
        self.cache_bytes += size
        return True

    def _load_modis_tile(self, year, tile_id):
        modis, _s1 = self.reader._groups(year)
        start = int(tile_id) * self.tile_rows
        end = min(start + self.tile_rows, int(modis["evi"].shape[0]))
        if end <= start:
            raise IndexError(f"Empty MODIS raster tile: year={year}, tile={tile_id}")
        raw_evi = np.asarray(modis["evi"][start:end])
        raw_lst = np.asarray(modis["lst"][start:end])
        ev = self.reader._decode_modis_many(raw_evi, "evi").astype(np.float16, copy=False)
        ls = self.reader._decode_modis_many(raw_lst, "lst").astype(np.float16, copy=False)
        ev_t = self._pinned_fp16_to_device(ev)
        ls_t = self._pinned_fp16_to_device(ls)
        return (ev_t, ls_t)

    @staticmethod
    def _pinned_fp16_to_device(array):
        """Stage a contiguous FP16 tile in pinned RAM, then enqueue async H2D on current CUDA stream."""
        cpu = torch.from_numpy(np.ascontiguousarray(array, dtype=np.float16))
        try:
            cpu = cpu.pin_memory()
        except (RuntimeError, NotImplementedError):
            # Fallback remains correct; it simply may not overlap the copy with compute.
            return cpu.to(device=device, dtype=torch.float16)
        return cpu.to(device=device, dtype=torch.float16, non_blocking=True)

    def _load_s1_tile(self, year, tile_id):
        _modis, s1 = self.reader._groups(year)
        ref = s1[S1_ARRAYS[0]]
        start = int(tile_id) * self.tile_rows
        end = min(start + self.tile_rows, int(ref.shape[0]))
        if end <= start:
            raise IndexError(f"Empty Sentinel-1 raster tile: year={year}, tile={tile_id}")
        values, masks = [], []
        for name in S1_ARRAYS:
            physical = np.asarray(s1[name][start:end], dtype=np.float32).copy()
            valid = np.isfinite(physical)
            physical[~valid] = np.nan
            norm, mask = self.reader._normalize(name, physical, valid)
            values.append(norm)
            masks.append(mask)
        packed = np.concatenate((np.stack(values, axis=1), np.stack(masks, axis=1)), axis=1)
        packed = np.ascontiguousarray(packed.astype(np.float16, copy=False))
        return self._pinned_fp16_to_device(packed)

    def _obtain_tile(self, key):
        tile = self._get(key)
        if tile is not None:
            return tile
        self.misses += 1
        if key[0] == "m":
            tile = self._load_modis_tile(key[1], key[2])
        else:
            tile = self._load_s1_tile(key[1], key[2])
        self._put(key, tile)
        return tile

    def get_batch(self, years, modis_indices, s1_indices):
        years = np.asarray(years, dtype=np.int32).reshape(-1)
        modis_indices = np.asarray(modis_indices, dtype=np.int64).reshape(-1)
        s1_indices = np.asarray(s1_indices, dtype=np.int64).reshape(-1)
        n = len(years)
        if n == 0:
            return (torch.empty((0, 2, 27, 27), device=device),
                    torch.empty((0, 2, 13, 13), device=device),
                    torch.empty((0, 2 * len(S1_ARRAYS), 13, 13), device=device))

        first_modis, first_s1 = self.reader._groups(int(years[0]))
        eh, ew = tuple(int(x) for x in first_modis["evi"].shape[1:])
        lh, lw = tuple(int(x) for x in first_modis["lst"].shape[1:])
        sh, sw = tuple(int(x) for x in first_s1[S1_ARRAYS[0]].shape[1:])
        # Output tensors are small batch buffers; source tiles are streamed one-at-a-time to cap VRAM.
        out_evi = torch.empty((n, 2, eh, ew), dtype=torch.float16, device=device,
                              memory_format=torch.channels_last)
        out_lst = torch.empty((n, 2, lh, lw), dtype=torch.float16, device=device,
                              memory_format=torch.channels_last)
        out_s1 = torch.empty((n, 2 * len(S1_ARRAYS), sh, sw), dtype=torch.float16, device=device,
                             memory_format=torch.channels_last)
        modis_groups, s1_groups = defaultdict(list), defaultdict(list)
        for pos, (year, mi, si) in enumerate(zip(years, modis_indices, s1_indices)):
            modis_groups[(int(year), int(mi) // self.tile_rows)].append((pos, int(mi)))
            s1_groups[(int(year), int(si) // self.tile_rows)].append((pos, int(si)))

        for (year, tile_id), rows in modis_groups.items():
            ev_tile, ls_tile = self._obtain_tile(("m", int(year), int(tile_id)))
            pos = torch.as_tensor([r[0] for r in rows], dtype=torch.long, device=device)
            local = torch.as_tensor([r[1] - tile_id * self.tile_rows for r in rows], dtype=torch.long, device=device)
            out_evi.index_copy_(0, pos, ev_tile.index_select(0, local))
            out_lst.index_copy_(0, pos, ls_tile.index_select(0, local))

        for (year, tile_id), rows in s1_groups.items():
            s1_tile = self._obtain_tile(("s", int(year), int(tile_id)))
            pos = torch.as_tensor([r[0] for r in rows], dtype=torch.long, device=device)
            local = torch.as_tensor([r[1] - tile_id * self.tile_rows for r in rows], dtype=torch.long, device=device)
            out_s1.index_copy_(0, pos, s1_tile.index_select(0, local))

        self.batches += 1
        if not self._printed:
            print(f"GPU raster tile cache active [{self.task_mode}]: tile_rows={self.tile_rows}, "
                  f"max={self.max_bytes/(1024**3):.2f} GiB, reserved_headroom={self.reserve_bytes/(1024**3):.2f} GiB; "
                  "normalized tile values+validity channels stored as fp16 on dedicated VRAM.")
            self._printed = True
        log_every = max(1, int(globals().get("GPU_RASTER_CACHE_LOG_EVERY_BATCHES", 100)))
        if self.batches % log_every == 0:
            total = self.hits + self.misses
            print(f"GPU raster tile cache [{self.task_mode}]: batches={self.batches}, "
                  f"tile-hit-rate={100.0*self.hits/max(1,total):.1f}%, "
                  f"resident={self.cache_bytes/(1024**3):.2f} GiB, tiles={len(self.lru)}")
        # Under CUDA AMP the Conv2d branches accept FP16 directly. Keeping these output
        # buffers FP16 avoids an extra full-batch cast/copy and cuts input-buffer VRAM.
        if device.type == "cuda" and bool(globals().get("USE_AMP", True)):
            return (out_evi, out_lst, out_s1)
        return (out_evi.float().contiguous(memory_format=torch.channels_last),
                out_lst.float().contiguous(memory_format=torch.channels_last),
                out_s1.float().contiguous(memory_format=torch.channels_last))


_GPU_RASTER_TILE_PROVIDERS = {}

def _get_gpu_raster_tile_provider(cache_dir, task_mode):
    key = (str(task_mode), str(Path(cache_dir).resolve()))
    provider = _GPU_RASTER_TILE_PROVIDERS.get(key)
    if provider is None:
        cache_dir = Path(cache_dir)
        metadata = json.loads((cache_dir / "metadata.json").read_text(encoding="utf-8"))
        provider = GPURasterTileCache(cache_dir, metadata, task_mode)
        _GPU_RASTER_TILE_PROVIDERS[key] = provider
    return provider


print("Chunk-aware raster loader active: source arrays from", "disk-backed native-dtype mmap" if RAM_ZARR_CACHE_ACTIVE else "direct Zarr",
      "; normalized chunk LRU cap:", PATCH_LRU_MAX_BYTES_PER_WORKER_MIB, "MiB/worker;",
      "expanded normalized patch files: disabled;")

In [ ]:
import os
import psutil

print("Logical CPUs:", os.cpu_count())
print("CPU affinity:", len(os.sched_getaffinity(0)))
print(f"Total RAM visible to WSL: {psutil.virtual_memory().total / 2**30:.2f} GiB")
print(f"Available RAM: {psutil.virtual_memory().available / 2**30:.2f} GiB")
print(f"RAM used: {psutil.virtual_memory().percent:.1f}%")
print("Notebook worker cap:", resource_limited_worker_cap())

## v16: grouped locality windows and low-overhead diagnostics

The training sampler now emits batches directly. It first uses a global order based on year, native EVI/Sentinel chunk identifiers, and exact source row indices. It partitions that order into windows of approximately **131,072 samples**, splits each window into the configured physical microbatches, shuffles groups of adjacent batches while preserving sequential order inside each group, and then shuffles window order between epochs. Rows inside each batch remain raster-local. This is designed to smooth memory-mapped access while preserving batch-level stochasticity.

Training aggregates four locality windows before collecting process-tree minor/major page faults, physical disk bytes read, read characters, throughput, loader wait and collator time. This avoids a full process-tree scan and flushed console write at every window boundary. These are diagnostics only and are not logged to MLflow. Linux process-tree counters are best-effort: page faults/read bytes are summed across the notebook process and its DataLoader descendants.

`torch.compile` is enabled by default for full retraining. The notebook benchmarks a bounded real batch, tests repeated gradient-accumulation backward passes, and retains the compiled wrapper when it is not materially slower than eager.


The default group size is four adjacent physical batches (about 8,192 samples at a batch size of 2,048). The broad windows are still traversed sequentially. `RUN_RUNTIME_LOADER_BENCHMARK=True` enables a short data-only comparison of workers, grouping and window size before full training.

## v17 runtime baseline-only input benchmark

This optional benchmark compares real raster collation/DataLoader throughput without running the model. It tests the current 131,072-sample window with full-batch shuffle versus grouped-local batches, plus smaller/larger windows. It reports measured samples/s, DataLoader wait and collator time. It does not modify model weights, Optuna trials, checkpoints or production settings. Enable `RUN_RUNTIME_LOADER_BENCHMARK=True` in the configuration cell to run it after the persistent caches and pipeline preflight are ready.

The benchmark is deliberately short and should be treated as a screening result, not a definitive full-epoch measurement. Compare the first several training windows and full epoch wall-clock time as well.

In [ ]:
# =============================== RUNTIME-ONLY LOADER BENCHMARK ===============================
def run_runtime_loader_benchmark(task_mode="location_history"):
    """Compare loader/collator settings on real cached training rows, without model execution."""
    global TRAIN_LOCALITY_WINDOW_SIZE, LOCALITY_SHUFFLE_GROUP_BATCHES, NUM_WORKERS, RASTER_MMAP_ADVICE
    old_window = int(TRAIN_LOCALITY_WINDOW_SIZE)
    old_group = int(LOCALITY_SHUFFLE_GROUP_BATCHES)
    old_workers = int(NUM_WORKERS)
    old_advice = str(RASTER_MMAP_ADVICE)
    worker_cap = min(int(NUM_WORKERS), int(resource_limited_worker_cap()))
    worker_values = sorted(set([max(0, min(1, worker_cap)), max(0, min(2, worker_cap)), max(0, worker_cap)]))
    candidate_set = set()
    for workers in worker_values:
        for group_batches in (1, 2, 4, 8, 16):
            candidate_set.add((workers, group_batches, 131072, old_advice))
    max_workers = max(worker_values) if worker_values else 0
    candidate_set.add((max_workers, 4, 65536, old_advice))
    candidate_set.add((max_workers, 4, 262144, old_advice))
    candidate_set.add((max_workers, 4, 524288, old_advice))
    # Compare kernel mmap access hints at the existing production settings.
    if os.name == "posix":
        for advice in ("normal", "random", "sequential"):
            candidate_set.add((old_workers, old_group, old_window, advice))
    candidates = sorted(candidate_set)
    repeats = max(1, int(RUNTIME_BENCH_REPEATS))
    batch_size = min(int(TRAIN_MICROBATCH_CAP), int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE))
    warmup_batches = max(0, int(RUNTIME_BENCH_WARMUP_BATCHES))
    measure_batches = max(2, int(RUNTIME_BENCH_MEASURE_BATCHES))
    rows = []
    baseline_config = (old_workers, old_group, old_window, old_advice)
    print(f"Runtime loader benchmark [{task_mode}]: {len(candidates)} candidates x {repeats} repeats, "
          f"batch={batch_size}, warmup={warmup_batches}, timed={measure_batches} batches each.", flush=True)
    try:
        for repeat_id in range(repeats):
            repeat_candidates = list(candidates)
            random.Random(int(SEED) + 1601 + repeat_id).shuffle(repeat_candidates)
            for workers, group_batches, window_size, mmap_advice in repeat_candidates:
                TRAIN_LOCALITY_WINDOW_SIZE = int(window_size)
                LOCALITY_SHUFFLE_GROUP_BATCHES = int(group_batches)
                RASTER_MMAP_ADVICE = str(mmap_advice)
                ds = get_global_dataset("train", task_mode=task_mode, target_mode="residual",
                                        max_rows=None, shuffle=True, seed=SEED + 1601 + repeat_id)
                if not isinstance(ds, CachedGlobalDataset):
                    print("Skipping runtime loader benchmark: persistent cached dataset is unavailable; "
                          "this benchmark targets the v15 cached raster path.")
                    return pd.DataFrame(rows)
                loader = None
                iterator = None
                try:
                    loader = build_global_loader(ds, batch_size, pin_memory=False,
                                                 num_workers=int(workers), persistent_workers=False)
                    iterator = iter(loader)
                    for _ in range(warmup_batches):
                        batch = next(iterator, None)
                        if batch is None:
                            break
                    measured_samples = 0
                    collate_total = 0.0
                    collate_raster = 0.0
                    measured = 0
                    started = time.perf_counter()
                    for _ in range(measure_batches):
                        batch = next(iterator, None)
                        if batch is None:
                            break
                        measured += 1
                        measured_samples += int(batch["y"].shape[0])
                        collate_total += float(batch.get("_collate_total_sec", 0.0))
                        collate_raster += float(batch.get("_collate_raster_sec", 0.0))
                    elapsed = max(1e-6, time.perf_counter() - started)
                    rows.append({
                        "repeat": int(repeat_id + 1),
                        "workers": int(workers),
                        "group_batches": int(group_batches),
                        "window_samples": int(window_size),
                        "mmap_advice": str(mmap_advice),
                        "timed_batches": int(measured),
                        "samples": int(measured_samples),
                        "samples_per_sec": float(measured_samples / elapsed),
                        "loader_wait_ms_per_batch": float(1000.0 * elapsed / max(1, measured)),
                        "collate_ms_per_batch": float(1000.0 * collate_total / max(1, measured)),
                        "raster_collate_ms_per_batch": float(1000.0 * collate_raster / max(1, measured)),
                    })
                    print(f"[Runtime bench r{repeat_id + 1}] workers={workers} group={group_batches} "
                          f"window={window_size} advice={mmap_advice} rate={measured_samples / elapsed:,.0f} samples/s "
                          f"loader={1000.0 * elapsed / max(1, measured):.1f}ms/b "
                          f"collate={1000.0 * collate_total / max(1, measured):.1f}ms/b", flush=False)
                finally:
                    if iterator is not None:
                        shutdown = getattr(iterator, "_shutdown_workers", None)
                        if callable(shutdown):
                            try: shutdown()
                            except Exception: pass
                    if loader is not None:
                        del loader
                    del ds
                    gc.collect()
    finally:
        TRAIN_LOCALITY_WINDOW_SIZE = old_window
        LOCALITY_SHUFFLE_GROUP_BATCHES = old_group
        NUM_WORKERS = old_workers
        RASTER_MMAP_ADVICE = old_advice

    raw = pd.DataFrame(rows)
    if raw.empty:
        print("Runtime benchmark produced no measured batches; keeping existing settings.")
        return raw
    result = (raw.groupby(["workers", "group_batches", "window_samples", "mmap_advice"], as_index=False)
              .agg(samples_per_sec=("samples_per_sec", "median"),
                   min_samples_per_sec=("samples_per_sec", "min"),
                   repeats=("samples_per_sec", "count"),
                   loader_wait_ms_per_batch=("loader_wait_ms_per_batch", "median"),
                   collate_ms_per_batch=("collate_ms_per_batch", "median"),
                   raster_collate_ms_per_batch=("raster_collate_ms_per_batch", "median"))
              .sort_values("samples_per_sec", ascending=False).reset_index(drop=True))
    display(result)

    # Compare the winner against the exact production configuration before changing anything.
    baseline_mask = (
        (result["workers"] == baseline_config[0]) &
        (result["group_batches"] == baseline_config[1]) &
        (result["window_samples"] == baseline_config[2]) &
        (result["mmap_advice"] == baseline_config[3])
    )
    baseline_rate = (float(result.loc[baseline_mask, "samples_per_sec"].iloc[0])
                     if bool(baseline_mask.any()) else None)
    winner = result.iloc[0]
    winner_rate = float(winner["samples_per_sec"])
    min_gain = max(0.0, float(RUNTIME_BENCH_MIN_GAIN))
    print("Baseline configuration:", baseline_config, "median samples/s:", baseline_rate)
    print("Best measured configuration:", winner.to_dict())
    if (bool(RUNTIME_BENCH_AUTO_APPLY) and baseline_rate is not None and
            winner_rate >= baseline_rate * (1.0 + min_gain)):
        NUM_WORKERS = int(winner["workers"])
        LOCALITY_SHUFFLE_GROUP_BATCHES = int(winner["group_batches"])
        TRAIN_LOCALITY_WINDOW_SIZE = int(winner["window_samples"])
        RASTER_MMAP_ADVICE = str(winner["mmap_advice"])
        print(f"Applied runtime settings: workers={NUM_WORKERS}, "
              f"group_batches={LOCALITY_SHUFFLE_GROUP_BATCHES}, "
              f"window={TRAIN_LOCALITY_WINDOW_SIZE:,} samples, mmap_advice={RASTER_MMAP_ADVICE} "
              f"({100.0*(winner_rate/baseline_rate-1.0):.1f}% above baseline).")
    else:
        print("No candidate beat the current setting by the required margin; keeping current production settings.")
    return result

In [ ]:
# =============================== OPTUNA SEARCH / LIVE METRICS ===============================
_GPU_NVML_TRIED = False
_GPU_NVML_HANDLE = None
_GPU_NVML_MODULE = None
_LAST_PROCESS_TREE_CPU_SAMPLE = None
_PROCESS_TREE_CPU_LOCK = threading.Lock()

_GPU_UTIL_CACHE = {"at": 0.0, "value": None}
_GPU_UTIL_CACHE_LOCK = threading.Lock()


def _gpu_utilization_percent():
    """Rate-limited GPU utilization poll; avoids spawning nvidia-smi per status update."""
    global _GPU_NVML_TRIED, _GPU_NVML_HANDLE, _GPU_NVML_MODULE
    if device.type != "cuda":
        return None
    now = time.monotonic()
    poll_interval = max(1.0, float(globals().get("GPU_UTIL_POLL_INTERVAL_SEC", 3.0)))
    if now - float(_GPU_UTIL_CACHE["at"]) < poll_interval:
        return _GPU_UTIL_CACHE["value"]
    with _GPU_UTIL_CACHE_LOCK:
        now = time.monotonic()
        if now - float(_GPU_UTIL_CACHE["at"]) < poll_interval:
            return _GPU_UTIL_CACHE["value"]
        value = None
        if not _GPU_NVML_TRIED:
            _GPU_NVML_TRIED = True
            try:
                import pynvml as _pynvml
                _pynvml.nvmlInit()
                _GPU_NVML_MODULE = _pynvml
                _GPU_NVML_HANDLE = _pynvml.nvmlDeviceGetHandleByIndex(torch.cuda.current_device())
            except Exception:
                _GPU_NVML_MODULE = None
                _GPU_NVML_HANDLE = None
        if _GPU_NVML_MODULE is not None and _GPU_NVML_HANDLE is not None:
            try:
                value = float(_GPU_NVML_MODULE.nvmlDeviceGetUtilizationRates(_GPU_NVML_HANDLE).gpu)
            except Exception:
                value = None
        if value is None:
            try:
                result = subprocess.run(
                    ["nvidia-smi", "--id", str(torch.cuda.current_device()),
                     "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=2.0, check=True,
                )
                value = float(result.stdout.strip().splitlines()[0])
            except Exception:
                value = None
        _GPU_UTIL_CACHE["at"] = now
        _GPU_UTIL_CACHE["value"] = value
        return value

def _resource_snapshot():
    snap = {"gpu_util_pct": _gpu_utilization_percent(), "gpu_alloc_gib": None,
            "gpu_reserved_gib": None, "gpu_total_gib": None, "host_cpu_pct": None,
            "host_ram_pct": None, "process_rss_gib": None, "available_ram_gib": None,
            "process_tree_cpu_pct": None, "process_tree_rss_gib": None}
    if device.type == "cuda":
        try:
            props = torch.cuda.get_device_properties(device)
            snap["gpu_alloc_gib"] = torch.cuda.memory_allocated(device) / (1024 ** 3)
            snap["gpu_reserved_gib"] = torch.cuda.memory_reserved(device) / (1024 ** 3)
            snap["gpu_total_gib"] = props.total_memory / (1024 ** 3)
        except Exception:
            pass
    if psutil is not None:
        try:
            global _LAST_PROCESS_TREE_CPU_SAMPLE
            vm = psutil.virtual_memory()
            proc = psutil.Process(os.getpid())
            snap["host_cpu_pct"] = float(psutil.cpu_percent(interval=None))
            snap["host_ram_pct"] = float(vm.percent)
            snap["available_ram_gib"] = float(vm.available / (1024 ** 3))
            parent_mem = proc.memory_info()
            snap["process_rss_gib"] = float(parent_mem.rss / (1024 ** 3))
            tree_cpu_seconds = sum(proc.cpu_times()[:2])
            tree_rss_bytes = int(parent_mem.rss)
            for child in proc.children(recursive=True):
                try:
                    child_times = child.cpu_times()
                    tree_cpu_seconds += float(child_times.user + child_times.system)
                    tree_rss_bytes += int(child.memory_info().rss)
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    continue
            now = time.perf_counter()
            with _PROCESS_TREE_CPU_LOCK:
                previous = _LAST_PROCESS_TREE_CPU_SAMPLE
                if previous is not None and now > previous[0]:
                    snap["process_tree_cpu_pct"] = max(0.0, (tree_cpu_seconds - previous[1]) / (now - previous[0]) * 100.0)
                _LAST_PROCESS_TREE_CPU_SAMPLE = (now, tree_cpu_seconds)
            snap["process_tree_rss_gib"] = tree_rss_bytes / (1024 ** 3)
        except Exception:
            pass
    return snap

def _training_resource_summary(resource_samples):
    """Report in-loop averages/peaks, not the idle state after input queues are released."""
    final = _resource_snapshot()
    samples = list(resource_samples or [])
    if samples:
        for key in ("gpu_util_pct", "host_cpu_pct", "process_tree_cpu_pct"):
            vals = [float(x[key]) for x in samples if x.get(key) is not None and np.isfinite(float(x[key]))]
            if vals:
                final[key] = float(np.mean(vals))
        for key in ("host_ram_pct",):
            vals = [float(x[key]) for x in samples if x.get(key) is not None and np.isfinite(float(x[key]))]
            if vals:
                final[key] = float(np.max(vals))
        for key in ("process_rss_gib", "process_tree_rss_gib"):
            vals = [float(x[key]) for x in samples if x.get(key) is not None and np.isfinite(float(x[key]))]
            if vals:
                final[key] = float(np.max(vals))
        vals = [float(x["available_ram_gib"]) for x in samples
                if x.get("available_ram_gib") is not None and np.isfinite(float(x["available_ram_gib"]))]
        if vals:
            final["available_ram_gib"] = float(np.min(vals))
    if device.type == "cuda":
        try:
            # These are the peak active-epoch values and remain meaningful after tensors are freed.
            final["gpu_alloc_gib"] = float(torch.cuda.max_memory_allocated(device) / _GIB)
            final["gpu_reserved_gib"] = float(torch.cuda.max_memory_reserved(device) / _GIB)
        except Exception:
            pass
    return final


def _make_optimizer(name, parameters, lr, weight_decay):
    """Prefer fused CUDA Adam/AdamW; fall back safely on foreach or the standard implementation."""
    optimizer_cls = torch.optim.AdamW if str(name) == "AdamW" else torch.optim.Adam
    params = list(parameters)
    base = {"lr": float(lr), "weight_decay": float(weight_decay)}
    if device.type == "cuda":
        try:
            opt = optimizer_cls(params, **base, fused=True)
            opt._speed_impl = "fused"
            return opt
        except (TypeError, RuntimeError, ValueError):
            try:
                opt = optimizer_cls(params, **base, foreach=True)
                opt._speed_impl = "foreach"
                return opt
            except (TypeError, RuntimeError, ValueError):
                pass
    opt = optimizer_cls(params, **base)
    opt._speed_impl = "standard"
    return opt


def _cpu_state_dict(state_dict):
    return {key: (value.detach().cpu().clone() if torch.is_tensor(value) else value)
            for key, value in state_dict.items()}


def _atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temp_path)
    os.replace(temp_path, path)


def _update_ema_model(ema_model, source_model, decay):
    """EMA trainable weights; copy BatchNorm/non-parameter buffers from the current model."""
    decay = float(decay)
    with torch.no_grad():
        for ema_param, source_param in zip(ema_model.parameters(), source_model.parameters()):
            ema_param.mul_(decay).add_(source_param, alpha=1.0 - decay)
        for ema_buffer, source_buffer in zip(ema_model.buffers(), source_model.buffers()):
            ema_buffer.copy_(source_buffer)


def _format_live_status(samples, elapsed, data_wait_total, host_step_total, loss_value=None):
    snap = _resource_snapshot()
    out = {"samples": f"{samples:,}", "samp/s": f"{samples / max(elapsed, 1e-6):.1f}",
           "wait/batch": f"{data_wait_total / max(1, samples):.4f}s/sample",
           "step_host": f"{host_step_total / max(1, samples):.4f}s/sample"}
    if loss_value is not None and np.isfinite(loss_value):
        out["loss"] = f"{loss_value:.4f}"
    if snap["gpu_util_pct"] is not None:
        out["GPU"] = f"{snap['gpu_util_pct']:.0f}%"
    if snap["gpu_alloc_gib"] is not None:
        out["VRAM"] = f"{snap['gpu_alloc_gib']:.2f}/{snap['gpu_total_gib']:.1f}G"
    if snap["host_cpu_pct"] is not None:
        out["CPU"] = f"{snap['host_cpu_pct']:.0f}%"
    if snap["host_ram_pct"] is not None:
        out["RAM"] = f"{snap['host_ram_pct']:.0f}%"
    return out

def move_batch(batch):
    """Resolve GPU-cached raster indices, then transfer remaining CPU fields on the caller's stream."""
    if isinstance(batch, dict) and "_gpu_raster_years" in batch:
        task_mode = str(batch.pop("_gpu_raster_task_mode"))
        cache_dir = batch.pop("_gpu_raster_cache_dir")
        years = batch.pop("_gpu_raster_years").detach().cpu().numpy()
        modis_idx = batch.pop("_gpu_raster_modis_idx").detach().cpu().numpy()
        s1_idx = batch.pop("_gpu_raster_s1_idx").detach().cpu().numpy()
        provider = _get_gpu_raster_tile_provider(cache_dir, task_mode)
        evi, lst, s1 = provider.get_batch(years, modis_idx, s1_idx)
        batch["evi"], batch["lst"], batch["s1"] = evi, lst, s1
    out = {}
    for key, value in batch.items():
        if not torch.is_tensor(value):
            out[key] = value
        elif device.type == "cuda" and key in ("evi", "lst", "s1") and value.ndim == 4:
            out[key] = value.to(device, non_blocking=True, memory_format=torch.channels_last)
        else:
            out[key] = value.to(device, non_blocking=True)
    return out

class AsyncDeviceBatchPrefetcher:
    """Stream one ready batch at a time to the compute stream while preparing future batches.

    The producer starts immediately. It never waits for a large VRAM shard to fill before
    training can begin. Pinned CPU source batches are retained only until their H2D event ends.
    """
    _END = object()

    def __init__(self, loader, enabled_override=None):
        self.loader = loader
        self.source_iterator = iter(loader)  # create DataLoader workers on the calling thread
        prefetch_enabled = (globals().get("ASYNC_DEVICE_PREFETCH", True)
                            if enabled_override is None else bool(enabled_override))
        self.enabled = bool(device.type == "cuda" and prefetch_enabled)
        self.closed = False
        self._stats_lock = threading.Lock()
        self._source_refs = []
        self._transfer_event_pairs = []
        self._stats = {
            "loader_wait_sec": 0.0,
            "move_enqueue_sec": 0.0,
            "batches_prepared": 0,
            "collate_total_sec": 0.0,
            "collate_meta_sec": 0.0,
            "collate_raster_sec": 0.0,
            "pinned_tensor_fraction": None,
        }
        if self.enabled:
            self.queue = _queue.Queue(maxsize=max(1, int(globals().get("DEVICE_PREFETCH_QUEUE_SIZE", 2))))
            self.stop_event = threading.Event()
            self.stream = torch.cuda.Stream(device=device)
            self.thread = threading.Thread(target=self._producer, name="cuda-batch-prefetch", daemon=True)
            self.thread.start()
        else:
            self.queue = None
            self.stop_event = None
            self.stream = None
            self.thread = None

    def __iter__(self):
        return self

    def _put(self, item):
        while not self.stop_event.is_set():
            try:
                self.queue.put(item, timeout=0.1)
                return True
            except _queue.Full:
                continue
        return False

    def _prune_source_refs(self):
        # Keep pinned CPU buffers alive until DMA has finished, but do not retain the epoch.
        with self._stats_lock:
            retained = []
            for event, source in self._source_refs:
                try:
                    complete = bool(event.query())
                except Exception:
                    complete = False
                if not complete:
                    retained.append((event, source))
            self._source_refs = retained

    def _producer(self):
        try:
            while not self.stop_event.is_set():
                source_started = time.perf_counter()
                with self._stats_lock:
                    is_first_batch = int(self._stats["batches_prepared"]) == 0
                if is_first_batch:
                    print("[FIRST-BATCH] requesting first production-loader batch; "
                          "if this stalls, the model has not started training yet.", flush=True)
                try:
                    raw = next(self.source_iterator)
                except StopIteration:
                    break
                loader_wait = time.perf_counter() - source_started
                if is_first_batch:
                    print(f"[FIRST-BATCH] loader returned its first batch in {loader_wait:.1f}s; "
                          "continuing to device transfer/model step.", flush=True)
                if raw is None:
                    with self._stats_lock:
                        self._stats["loader_wait_sec"] += loader_wait
                    if not self._put(("batch", (None, None, None))):
                        break
                    continue

                with self._stats_lock:
                    batch_no = int(self._stats["batches_prepared"])
                sample_every = max(1, int(globals().get("PIPELINE_DIAGNOSTICS_SAMPLE_EVERY_BATCHES", 64)))
                sample_transfer = (batch_no % sample_every == 0)
                tensor_values = [v for v in raw.values() if torch.is_tensor(v)]
                pinned_fraction = (sum(bool(v.is_pinned()) for v in tensor_values) / max(1, len(tensor_values)))
                with torch.cuda.stream(self.stream):
                    start_event = torch.cuda.Event(enable_timing=True) if sample_transfer else None
                    if start_event is not None:
                        start_event.record(self.stream)
                    move_started = time.perf_counter()
                    prepared = move_batch(raw)
                    move_elapsed = time.perf_counter() - move_started
                    ready_event = torch.cuda.Event(enable_timing=sample_transfer, blocking=False)
                    ready_event.record(self.stream)

                with self._stats_lock:
                    self._stats["loader_wait_sec"] += loader_wait
                    self._stats["move_enqueue_sec"] += move_elapsed
                    self._stats["batches_prepared"] += 1
                    self._stats["collate_total_sec"] += float(raw.get("_collate_total_sec", 0.0))
                    self._stats["collate_meta_sec"] += float(raw.get("_collate_meta_sec", 0.0))
                    self._stats["collate_raster_sec"] += float(raw.get("_collate_raster_sec", 0.0))
                    if self._stats["pinned_tensor_fraction"] is None:
                        self._stats["pinned_tensor_fraction"] = float(pinned_fraction)
                    if start_event is not None:
                        self._transfer_event_pairs.append((start_event, ready_event))
                    self._source_refs.append((ready_event, raw))

                self._prune_source_refs()
                # Queue only device tensors; `_source_refs` keeps the pinned CPU sources alive
                # until the corresponding ready_event reports completion.
                if not self._put(("batch", (prepared, ready_event, None))):
                    break
                del raw, prepared
        except BaseException as exc:
            self._put(("error", exc))
        finally:
            self._put(("end", self._END))

    def __next__(self):
        if self.closed:
            raise StopIteration
        if not self.enabled:
            raw = next(self.source_iterator)
            return None if raw is None else move_batch(raw)
        tag, payload = self.queue.get()
        if tag == "end":
            self.close()
            raise StopIteration
        if tag == "error":
            self.close()
            raise RuntimeError(f"Asynchronous device prefetch failed: {type(payload).__name__}: {payload}") from payload
        batch, ready_event, _unused_source = payload
        if batch is not None and ready_event is not None:
            consumer_stream = torch.cuda.current_stream(device)
            consumer_stream.wait_event(ready_event)
            for value in batch.values():
                if torch.is_tensor(value) and value.device.type == "cuda":
                    value.record_stream(consumer_stream)
        self._prune_source_refs()
        return batch

    def diagnostics(self):
        with self._stats_lock:
            result = dict(self._stats)
            result["transfer_event_pairs"] = list(self._transfer_event_pairs)
            result["source_refs_pending"] = len(self._source_refs)
        return result

    def close(self):
        if self.closed:
            return
        self.closed = True
        if self.enabled:
            self.stop_event.set()
            if self.thread is not None and self.thread.is_alive():
                self.thread.join(timeout=8.0)
            try:
                # Wait only for the copy stream, not for unrelated device work.
                self.stream.synchronize()
            except Exception:
                pass
            with self._stats_lock:
                self._source_refs.clear()

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass


def _cuda_event_ms_summary(event_pairs):
    """Summarize sampled CUDA event pairs after their streams have completed."""
    values = []
    for start_event, end_event in event_pairs or []:
        try:
            values.append(float(start_event.elapsed_time(end_event)))
        except Exception:
            continue
    if not values:
        return {"n": 0, "mean": float("nan"), "p50": float("nan"), "p95": float("nan")}
    a = np.asarray(values, dtype=np.float64)
    return {"n": int(a.size), "mean": float(a.mean()),
            "p50": float(np.percentile(a, 50)), "p95": float(np.percentile(a, 95))}


# Metric helper used by evaluation, calibration diagnostics, and per-year reports.
def metric_pack(y_true, y_pred, robust_multiplier=ROBUST_MAD_MULTIPLIER):
    y = np.asarray(y_true, dtype=np.float64).reshape(-1)
    p = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    ok = np.isfinite(y) & np.isfinite(p)
    y, p = y[ok], p[ok]
    if len(y) == 0:
        return {"n": 0, "rmse": np.nan, "nrmse": np.nan, "mae": np.nan, "r2": np.nan,
                "robust_n": 0, "robust_rmse": np.nan, "robust_nrmse": np.nan, "robust_mae": np.nan, "robust_r2": np.nan,
                "robust_error_cutoff": np.nan}
    err = p - y
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = float(1.0 - np.sum(err ** 2) / sst) if sst > 0 else float("nan")
    normalization_scale_m3_m3 = 1.0
    abs_err = np.abs(err)
    med = float(np.median(abs_err))
    mad = float(np.median(np.abs(abs_err - med)))
    robust_scale = max(1.4826 * mad, MIN_MAD_SCALE)
    cutoff = med + robust_multiplier * robust_scale
    keep = abs_err <= cutoff
    yr, pr = y[keep], p[keep]
    er = pr - yr
    if len(yr):
        robust_rmse = float(np.sqrt(np.mean(er ** 2)))
        robust_mae = float(np.mean(np.abs(er)))
        robust_sst = float(np.sum((yr - np.mean(yr)) ** 2))
        robust_r2 = float(1.0 - np.sum(er ** 2) / robust_sst) if robust_sst > 0 else float("nan")
    else:
        robust_rmse = robust_mae = robust_r2 = float("nan")
    return {"n": int(len(y)), "rmse": rmse, "nrmse": rmse / normalization_scale_m3_m3,
            "mae": mae, "r2": r2, "robust_n": int(keep.sum()),
            "robust_rmse": robust_rmse, "robust_nrmse": robust_rmse / normalization_scale_m3_m3,
            "robust_mae": robust_mae, "robust_r2": robust_r2, "robust_error_cutoff": float(cutoff)}



def evaluate_predictions(model, loader, target_mode, max_batches=None, desc=None):
    model.eval()
    ys, preds, raw_preds, years_seen = [], [], [], []
    n_samples = 0
    started = time.perf_counter()
    wait_total = 0.0
    host_step_total = 0.0
    try:
        estimated_total = len(loader)
    except Exception:
        estimated_total = None
    if max_batches is not None:
        estimated_total = min(estimated_total, max_batches) if estimated_total is not None else max_batches
    iterator = AsyncDeviceBatchPrefetcher(
        loader, enabled_override=bool(globals().get("EVAL_ASYNC_DEVICE_PREFETCH", False))
    )
    pbar = tqdm(total=estimated_total, desc=desc or "Evaluation", leave=False, dynamic_ncols=True, mininterval=0.25, position=1)
    wait_state = {"start": None, "samples": 0, "started": started, "wait_total": 0.0, "step_total": 0.0}
    stop_heartbeat = threading.Event()
    def _heartbeat():
        while not stop_heartbeat.wait(LIVE_PROGRESS_INTERVAL_SEC):
            ws = wait_state.get("start")
            if ws is not None:
                waited = time.perf_counter() - ws
                if waited >= LIVE_PROGRESS_INTERVAL_SEC:
                    snap = _resource_snapshot()
                    info = {"status": "WAIT_DATA", "wait_s": f"{waited:.1f}",
                            "samples": f"{wait_state['samples']:,}",
                            "samp/s": f"{wait_state['samples'] / max(1e-6, time.perf_counter()-wait_state['started']):.1f}"}
                    if snap["gpu_util_pct"] is not None: info["GPU"] = f"{snap['gpu_util_pct']:.0f}%"
                    if snap["gpu_alloc_gib"] is not None: info["VRAM"] = f"{snap['gpu_alloc_gib']:.2f}/{snap['gpu_total_gib']:.1f}G"
                    if snap["host_cpu_pct"] is not None: info["CPU"] = f"{snap['host_cpu_pct']:.0f}%"
                    if snap["process_tree_cpu_pct"] is not None: info["procCPU"] = f"{snap['process_tree_cpu_pct']:.0f}%"
                    if snap["host_ram_pct"] is not None: info["RAM"] = f"{snap['host_ram_pct']:.0f}%"
                    pbar.set_postfix(info, refresh=True)
    heartbeat = threading.Thread(target=_heartbeat, name="eval-data-wait-heartbeat", daemon=True)
    heartbeat.start()
    batch_i = 0
    try:
        with torch.no_grad():
            while True:
                wait_start = time.perf_counter(); wait_state["start"] = wait_start
                try:
                    batch = next(iterator)
                except StopIteration:
                    wait_state["start"] = None
                    break
                data_wait = time.perf_counter() - wait_start
                wait_total += data_wait; wait_state["wait_total"] = wait_total; wait_state["start"] = None
                if batch is None:
                    batch_i += 1; pbar.update(1)
                    if max_batches is not None and batch_i >= max_batches: break
                    continue
                step_start = time.perf_counter()
                b = batch  # already resolved/transferred by AsyncDeviceBatchPrefetcher
                with torch.autocast(device_type="cuda", dtype=torch.float16,
                                    enabled=(USE_AMP and device.type == "cuda")):
                    pred, raw_pred = model(b, return_raw=True)
                y_physical = physical_target(b, target_mode)
                ys.append(y_physical.detach().float().reshape(-1))
                preds.append(pred.detach().float().reshape(-1))
                raw_preds.append(raw_pred.detach().float().reshape(-1))
                if "year" in b and torch.is_tensor(b["year"]):
                    years_seen.append(b["year"].detach().long().reshape(-1))
                n_samples += int(y_physical.shape[0]); wait_state["samples"] = n_samples
                host_step_total += time.perf_counter() - step_start
                batch_i += 1; pbar.update(1)
                if batch_i == 1 or batch_i % max(1, int(EVAL_RESOURCE_SNAPSHOT_EVERY_BATCHES)) == 0 or data_wait > LIVE_PROGRESS_INTERVAL_SEC:
                    snap = _resource_snapshot()
                    info = {"samples": f"{n_samples:,}", "samp/s": f"{n_samples/max(time.perf_counter()-started,1e-6):.1f}",
                            "data_wait": f"{data_wait:.2f}s", "batches": batch_i}
                    if snap["gpu_util_pct"] is not None: info["GPU"] = f"{snap['gpu_util_pct']:.0f}%"
                    if snap["gpu_alloc_gib"] is not None: info["VRAM"] = f"{snap['gpu_alloc_gib']:.2f}/{snap['gpu_total_gib']:.1f}G"
                    if snap["process_tree_cpu_pct"] is not None: info["procCPU"] = f"{snap['process_tree_cpu_pct']:.0f}%"
                    if snap["host_ram_pct"] is not None: info["RAM"] = f"{snap['host_ram_pct']:.0f}%"
                    pbar.set_postfix(info, refresh=False)
                if max_batches is not None and batch_i >= max_batches: break
    finally:
        if hasattr(iterator, "close"):
            iterator.close()
        stop_heartbeat.set(); heartbeat.join(timeout=1.0); pbar.close()
    if not ys:
        empty = metric_pack([], [])
        empty.update({"raw_r2": np.nan, "raw_rmse": np.nan, "raw_mae": np.nan,
                      "calibrated_r2": np.nan, "calibration_r2_delta": np.nan})
        return empty
    # One device-to-host synchronization for all evaluation outputs, not one per batch.
    y_all = torch.cat(ys).cpu().numpy()
    p_all = torch.cat(preds).cpu().numpy()
    raw_all = torch.cat(raw_preds).cpu().numpy()
    calibrated_metrics = metric_pack(y_all, p_all)
    raw_metrics = metric_pack(y_all, raw_all)
    calibrated_metrics["calibrated_r2"] = calibrated_metrics["r2"]
    calibrated_metrics["raw_r2"] = raw_metrics["r2"]
    calibrated_metrics["raw_rmse"] = raw_metrics["rmse"]
    calibrated_metrics["raw_mae"] = raw_metrics["mae"]
    calibrated_metrics["calibration_r2_delta"] = (
        calibrated_metrics["r2"] - raw_metrics["r2"]
        if np.isfinite(calibrated_metrics["r2"]) and np.isfinite(raw_metrics["r2"]) else np.nan
    )
    # If the active cached collator provides year IDs, add a per-year view. This is a diagnostic
    # report, not an optimization target; small years are explicitly identifiable via n.
    if years_seen and sum(len(x) for x in years_seen) == len(y_all):
        yr_all = torch.cat(years_seen).cpu().numpy()
        for year in sorted(np.unique(yr_all).tolist()):
            mask = yr_all == year
            year_metrics = metric_pack(y_all[mask], p_all[mask])
            calibrated_metrics[f"year_{int(year)}_n"] = year_metrics["n"]
            calibrated_metrics[f"year_{int(year)}_r2"] = year_metrics["r2"]
            calibrated_metrics[f"year_{int(year)}_rmse"] = year_metrics["rmse"]
    return calibrated_metrics

def _locality_process_io_snapshot():
    """Best-effort Linux process-tree page-fault and physical-read counters.

    DataLoader workers perform mmap reads in child processes, so inspecting only the
    notebook process would miss the relevant activity. Returns aggregate counters for
    the notebook process and its descendants; unavailable fields are reported as None.
    """
    if not sys.platform.startswith("linux"):
        return {"minor_faults": None, "major_faults": None,
                "read_bytes": None, "rchar": None, "processes": 0}
    pids = {os.getpid()}
    if psutil is not None:
        try:
            pids.update(int(p.pid) for p in psutil.Process(os.getpid()).children(recursive=True))
        except Exception:
            pass
    totals = {"minor_faults": 0, "major_faults": 0, "read_bytes": 0, "rchar": 0}
    seen = 0
    for pid in pids:
        try:
            stat_text = Path(f"/proc/{pid}/stat").read_text()
            # comm is parenthesized and may itself contain spaces/parentheses.
            fields = stat_text[stat_text.rfind(")") + 2:].split()
            totals["minor_faults"] += int(fields[7])  # proc stat field 10
            totals["major_faults"] += int(fields[9])  # proc stat field 12
            io_values = {}
            for line in Path(f"/proc/{pid}/io").read_text().splitlines():
                if ":" in line:
                    key, value = line.split(":", 1)
                    if key.strip() in ("read_bytes", "rchar"):
                        io_values[key.strip()] = int(value.strip())
            totals["read_bytes"] += int(io_values.get("read_bytes", 0))
            totals["rchar"] += int(io_values.get("rchar", 0))
            seen += 1
        except (FileNotFoundError, ProcessLookupError, PermissionError, ValueError, IndexError):
            continue
    totals["processes"] = seen
    return totals


def _locality_counter_delta(after, before, key):
    a, b = after.get(key), before.get(key)
    return None if a is None or b is None else max(0, int(a) - int(b))


def _mark_cudagraph_step_begin():
    """Tell CUDAGraph Trees that the previous microbatch backward is complete."""
    if not bool(globals().get("COMPILE_MARK_STEP_BEGIN", True)):
        return
    try:
        marker = getattr(getattr(torch, "compiler", None), "cudagraph_mark_step_begin", None)
        if callable(marker):
            marker()
    except Exception:
        # Safe no-op on older PyTorch builds or the no-CUDA-graphs fallback mode.
        pass


def _ensure_stable_parameter_grad_buffers(model):
    """Preallocate .grad storage before CUDA-graph-backed compiled backward begins.

    With torch.compile(mode="reduce-overhead"), a gradient that is first allocated during
    graph capture can alias a graph-owned buffer. Replaying another microbatch can then
    overwrite it before the accumulated gradient is consumed. Keep stable parameter-owned
    buffers for the entire compiled training run.
    """
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.grad is None:
            parameter.grad = torch.zeros_like(parameter, memory_format=torch.preserve_format)
        elif parameter.requires_grad:
            parameter.grad.zero_()
    model.zero_grad(set_to_none=False)


def benchmark_torch_compile_training_step(model, batch, target_mode, *, max_samples=1024, steps=5):
    """Benchmark compiled forward/backward and retain it for full training when faster.

    reduce-overhead uses CUDA graphs. Parameter .grad tensors are therefore allocated
    before the first compiled backward and preserved at the same addresses across calls.
    The helper also performs a multi-backward accumulation smoke test before returning
    the compiled wrapper. If CUDA-graph accumulation is unsupported by this runtime, it
    retries max-autotune-no-cudagraphs instead of immediately abandoning torch.compile.
    """
    if device.type != "cuda" or not hasattr(torch, "compile"):
        print("[COMPILE] skipped: CUDA or torch.compile is unavailable.", flush=True)
        return None
    import sysconfig
    _python_include = sysconfig.get_paths().get("include")
    _python_header = (Path(_python_include) / "Python.h") if _python_include else None
    if _python_header is None or not _python_header.is_file():
        print(f"[COMPILE] matching Python.h is missing under {_python_include!r}; cannot compile on this environment.", flush=True)
        return None

    n = min(int(max_samples), int(batch["y"].shape[0]))
    if n < 8:
        print(f"[COMPILE] skipped: only {n} samples are available.", flush=True)
        return None
    model_input_keys = ("evi", "lst", "s1", "seq", "seq_mask", "static", "anchor")
    bench_batch = {}
    for key in model_input_keys:
        value = batch[key]
        if value.ndim == 4:
            bench_batch[key] = value[:n].contiguous(memory_format=torch.channels_last)
        else:
            bench_batch[key] = value[:n].contiguous() if value.ndim > 0 else value
    bench_target = batch["y"][:n].contiguous()
    model.train(True)
    _ensure_stable_parameter_grad_buffers(model)

    _probe_grad_scaler = torch.amp.GradScaler("cuda", enabled=bool(USE_AMP))

    def _loss_and_backward(forward_model, *, clear_first=False):
        if clear_first:
            # Zero in-place; never release parameter-owned .grad buffers in this function.
            model.zero_grad(set_to_none=False)
        if forward_model is not model:
            _mark_cudagraph_step_begin()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=bool(USE_AMP)):
            pred = forward_model(bench_batch).reshape(-1)
            if target_mode == "residual":
                residual_target = bench_target.reshape(-1)
                anchor = bench_batch["anchor"].reshape(-1)
                error = (pred - anchor) - residual_target
                abs_error = error.abs()
                delta = float(RESIDUAL_HUBER_DELTA)
                if LOSS_FUNCTION == "mse":
                    per_item = error.square()
                else:
                    per_item = torch.where(abs_error <= delta, 0.5 * error.square(),
                                           delta * (abs_error - 0.5 * delta))
                weights = (1.0 / (1.0 + residual_target.abs())) if RESIDUAL_WEIGHTING else 1.0
                loss = (per_item * weights).mean()
            else:
                target = bench_target.reshape(-1)
                loss = (nn.functional.mse_loss(pred, target) if LOSS_FUNCTION == "mse" else
                        nn.functional.huber_loss(pred, target, delta=float(RESIDUAL_HUBER_DELTA)))
            loss = loss + CALIBRATION_IDENTITY_PENALTY * calibration_identity_penalty(model)
        # Use the same scaled-backward pattern as the real FP16/GradScaler training loop.
        _probe_grad_scaler.scale(loss).backward()
        return loss

    def _timed_compiled(mode_name):
        compiled_model = torch.compile(model, mode=mode_name, fullgraph=False, dynamic=False)
        # Trigger forward + AOT-autograd compilation with stable parameter-grad addresses.
        for _ in range(3):
            _loss_and_backward(compiled_model, clear_first=True)
        torch.cuda.synchronize(device)
        # Explicitly test true accumulation across multiple CUDAGraph replays; this is the
        # pattern used by full-data microbatch accumulation.
        model.zero_grad(set_to_none=False)
        for _ in range(3):
            _loss_and_backward(compiled_model, clear_first=False)
        torch.cuda.synchronize(device)
        model.zero_grad(set_to_none=False)
        timings = []
        for _ in range(max(3, int(steps))):
            start_t = time.perf_counter()
            _loss_and_backward(compiled_model, clear_first=True)
            torch.cuda.synchronize(device)
            timings.append(time.perf_counter() - start_t)
        return compiled_model, 1000.0 * float(np.median(timings))

    compiled = None
    try:
        # Eager baseline uses identical input shape, AMP and preallocated gradients.
        for _ in range(2):
            _loss_and_backward(model, clear_first=True)
        torch.cuda.synchronize(device)
        eager_times = []
        for _ in range(max(3, int(steps))):
            start_t = time.perf_counter()
            _loss_and_backward(model, clear_first=True)
            torch.cuda.synchronize(device)
            eager_times.append(time.perf_counter() - start_t)
        eager_ms = 1000.0 * float(np.median(eager_times))

        primary_mode = str(globals().get("COMPILE_BENCHMARK_MODE", "reduce-overhead"))
        fallback_mode = str(globals().get("COMPILE_FALLBACK_MODE", "max-autotune-no-cudagraphs"))
        print(f"[COMPILE] compiling on {n:,} real samples (mode={primary_mode}); "
              "first-time compilation is excluded from the steady-state timing.", flush=True)
        try:
            compiled, compiled_ms = _timed_compiled(primary_mode)
            used_mode = primary_mode
        except Exception as primary_exc:
            if primary_mode == fallback_mode:
                raise
            print(f"[COMPILE] {primary_mode} accumulation smoke test failed: "
                  f"{type(primary_exc).__name__}: {primary_exc}", flush=True)
            print(f"[COMPILE] retrying torch.compile(mode={fallback_mode!r}) without CUDA graphs.", flush=True)
            if compiled is not None:
                del compiled
                compiled = None
            model.zero_grad(set_to_none=False)
            try:
                torch._dynamo.reset()
            except Exception:
                pass
            torch.cuda.empty_cache()
            _ensure_stable_parameter_grad_buffers(model)
            compiled, compiled_ms = _timed_compiled(fallback_mode)
            used_mode = fallback_mode

        speedup = eager_ms / max(compiled_ms, 1e-9)
        globals()["_TORCH_COMPILE_ACTIVE_MODE"] = used_mode
        print(f"[COMPILE benchmark] eager={eager_ms:.2f}ms; compiled={compiled_ms:.2f}ms; "
              f"speedup={speedup:.2f}x; batch={n:,}; mode={used_mode}.", flush=True)
        model.zero_grad(set_to_none=False)
        threshold = float(globals().get("COMPILE_TRAINING_MIN_SPEEDUP", 1.00))
        if speedup >= threshold:
            print(f"[COMPILE] enabled for full-data training: {speedup:.2f}x >= {threshold:.2f}x threshold.", flush=True)
            return {"compiled_model": compiled, "eager_ms": eager_ms,
                    "compiled_ms": compiled_ms, "speedup": speedup, "samples": n,
                    "mode": used_mode}
        print(f"[COMPILE] benchmark measured {speedup:.2f}x (<{threshold:.2f}x); keeping eager for safety.", flush=True)
        del compiled
        compiled = None
        try:
            torch._dynamo.reset()
        except Exception:
            pass
        model.zero_grad(set_to_none=True)
        return {"compiled_model": None, "eager_ms": eager_ms,
                "compiled_ms": compiled_ms, "speedup": speedup, "samples": n,
                "mode": used_mode}
    except Exception as exc:
        print(f"[COMPILE] compiled paths failed ({type(exc).__name__}: {exc}); eager fallback.", flush=True)
        model.zero_grad(set_to_none=True)
        try:
            if compiled is not None:
                del compiled
            torch._dynamo.reset()
        except Exception:
            pass
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return None
def _train_epoch_streaming(model, loader, optimizer, scaler, target_mode, effective_batch_size,
                             microbatch_size, clip_norm, desc, max_batches=None, forward_model=None):
    """Train with small prefetched loader batches and gradient accumulation to an effective batch."""
    model.train()
    _compiled_active = forward_model is not None and bool(globals().get("TORCH_COMPILE_ENABLED", True))
    if _compiled_active and bool(globals().get("COMPILE_STABLE_GRAD_BUFFERS", True)):
        _ensure_stable_parameter_grad_buffers(model)
        optimizer.zero_grad(set_to_none=False)
    else:
        optimizer.zero_grad(set_to_none=True)
    accum_steps = max(1, int(math.ceil(effective_batch_size / max(1, microbatch_size))))
    try:
        total_batches = len(loader)
    except Exception:
        total_batches = None
    if max_batches is not None:
        total_batches = min(total_batches, max_batches) if total_batches is not None else max_batches
    pbar = tqdm(total=total_batches, desc=desc, leave=False, dynamic_ncols=True, mininterval=0.25, position=1)
    epoch_start = time.perf_counter()
    resource_samples = []
    last_resource_snapshot = epoch_start
    if device.type == "cuda":
        try: torch.cuda.reset_peak_memory_stats(device)
        except Exception: pass
    wait_total = 0.0; host_step_total = 0.0; samples = 0; batches = 0
    collate_total_sec = 0.0; collate_meta_sec = 0.0; collate_raster_sec = 0.0
    compute_event_pairs = []
    pipeline_event_every = max(1, int(globals().get("PIPELINE_DIAGNOSTICS_SAMPLE_EVERY_BATCHES", 64)))
    patch_lru_hits = 0; patch_lru_misses = 0
    patch_lru_bytes_sum = 0; patch_lru_bytes_samples = 0
    raster_source_rows_read = 0; raster_source_rows_requested = 0; raster_source_chunks_read = 0
    loss_total = torch.zeros((), device=device)
    pending_microbatches = 0
    iterator = AsyncDeviceBatchPrefetcher(loader)
    _batch_sampler = getattr(loader, "batch_sampler", None)
    _locality_window_batches_target = getattr(_batch_sampler, "batches_per_window", None)
    if _locality_window_batches_target is not None:
        _locality_window_batches_target = max(1, int(_locality_window_batches_target))
    _locality_window_id = 0
    _locality_window_batches_done = 0
    _locality_window_samples = 0
    _locality_window_wait_sec = 0.0
    _locality_window_collate_sec = 0.0
    _locality_window_started = time.perf_counter()
    _locality_window_io_start = (_locality_process_io_snapshot()
                                 if _locality_window_batches_target is not None else None)
    _locality_report_windows = 0
    _locality_report_samples = 0
    _locality_report_batches = 0
    _locality_report_wait_sec = 0.0
    _locality_report_collate_sec = 0.0
    _locality_report_started = time.perf_counter()
    _locality_report_start_window = 1
    def _emit_locality_window(partial=False):
        nonlocal _locality_window_id, _locality_window_batches_done
        nonlocal _locality_window_samples, _locality_window_wait_sec, _locality_window_collate_sec
        nonlocal _locality_window_started, _locality_window_io_start
        nonlocal _locality_report_windows, _locality_report_samples, _locality_report_batches
        nonlocal _locality_report_wait_sec, _locality_report_collate_sec
        nonlocal _locality_report_started, _locality_report_start_window
        if _locality_window_batches_done <= 0:
            return

        # Accumulate four windows and take the expensive /proc process-tree snapshot only
        # once per report group. The old implementation scanned every descendant process
        # and flushed notebook output at every single window boundary.
        _locality_report_windows += 1
        _locality_report_samples += int(_locality_window_samples)
        _locality_report_batches += int(_locality_window_batches_done)
        _locality_report_wait_sec += float(_locality_window_wait_sec)
        _locality_report_collate_sec += float(_locality_window_collate_sec)
        _report_every = max(1, int(globals().get("LOCALITY_DIAGNOSTICS_EVERY_WINDOWS", 4)))
        _should_report = (_locality_report_windows >= _report_every) or bool(partial)

        _locality_window_id += 1
        _locality_window_batches_done = 0
        _locality_window_samples = 0
        _locality_window_wait_sec = 0.0
        _locality_window_collate_sec = 0.0
        _locality_window_started = time.perf_counter()

        if not _should_report:
            return

        _now = time.perf_counter()
        _elapsed = max(1e-6, _now - _locality_report_started)
        _io_end = _locality_process_io_snapshot() if _locality_window_io_start is not None else {}
        _io_start = _locality_window_io_start or {}
        _minor = _locality_counter_delta(_io_end, _io_start, "minor_faults")
        _major = _locality_counter_delta(_io_end, _io_start, "major_faults")
        _disk = _locality_counter_delta(_io_end, _io_start, "read_bytes")
        _rchar = _locality_counter_delta(_io_end, _io_start, "rchar")
        _fault_text = (f"minor/major_faults={_minor if _minor is not None else 'n/a'}/"
                       f"{_major if _major is not None else 'n/a'}")
        _disk_text = (f"disk_read={_disk/(1024**2):.1f}MiB" if _disk is not None else "disk_read=n/a")
        _rchar_text = (f"rchar={_rchar/(1024**2):.1f}MiB" if _rchar is not None else "rchar=n/a")
        _suffix = " partial" if partial else ""
        print(f"[Locality windows {_locality_report_start_window}-{_locality_window_id}] "
              f"windows={_locality_report_windows} batches={_locality_report_batches}{_suffix} "
              f"samples={_locality_report_samples:,} throughput={_locality_report_samples/_elapsed:.0f} samples/s "
              f"loader_wait={1000.0*_locality_report_wait_sec/max(1,_locality_report_batches):.1f}ms/b "
              f"collate={1000.0*_locality_report_collate_sec/max(1,_locality_report_batches):.1f}ms/b "
              f"{_fault_text} {_disk_text} {_rchar_text} processes={_io_end.get('processes', 'n/a')}", flush=False)
        _locality_window_io_start = _io_end
        _locality_report_windows = 0
        _locality_report_samples = 0
        _locality_report_batches = 0
        _locality_report_wait_sec = 0.0
        _locality_report_collate_sec = 0.0
        _locality_report_started = time.perf_counter()
        _locality_report_start_window = _locality_window_id + 1
    wait_state = {"start": None, "samples": 0, "started": epoch_start}
    stop_heartbeat = threading.Event()
    def _heartbeat():
        while not stop_heartbeat.wait(LIVE_PROGRESS_INTERVAL_SEC):
            ws = wait_state.get("start")
            if ws is not None:
                waited = time.perf_counter() - ws
                if waited >= LIVE_PROGRESS_INTERVAL_SEC:
                    snap = _resource_snapshot()
                    info = {"status": "WAIT_DATA", "wait_s": f"{waited:.1f}",
                            "samples": f"{wait_state['samples']:,}",
                            "samp/s": f"{wait_state['samples']/max(1e-6,time.perf_counter()-wait_state['started']):.1f}",
                            "micro": microbatch_size, "accum": accum_steps}
                    if snap["gpu_util_pct"] is not None: info["GPU"] = f"{snap['gpu_util_pct']:.0f}%"
                    if snap["gpu_alloc_gib"] is not None: info["VRAM"] = f"{snap['gpu_alloc_gib']:.2f}/{snap['gpu_total_gib']:.1f}G"
                    if snap["host_cpu_pct"] is not None: info["CPU"] = f"{snap['host_cpu_pct']:.0f}%"
                    if snap["process_tree_cpu_pct"] is not None: info["procCPU"] = f"{snap['process_tree_cpu_pct']:.0f}%"
                    if snap["host_ram_pct"] is not None: info["RAM"] = f"{snap['host_ram_pct']:.0f}%"
                    pbar.set_postfix(info, refresh=True)
    heartbeat = threading.Thread(target=_heartbeat, name="train-data-wait-heartbeat", daemon=True)
    heartbeat.start()
    batch_i = 0
    def _optimizer_step():
        nonlocal pending_microbatches
        if pending_microbatches <= 0:
            return
        scaler.unscale_(optimizer)
        # Average gradients over the actual partial/full accumulation group.
        inv = 1.0 / float(pending_microbatches)
        for param in model.parameters():
            if param.grad is not None:
                param.grad.mul_(inv)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=not _compiled_active)
        pending_microbatches = 0
    try:
        while True:
            wait_start = time.perf_counter(); wait_state["start"] = wait_start
            try:
                batch = next(iterator)
            except StopIteration:
                wait_state["start"] = None
                break
            data_wait = time.perf_counter() - wait_start
            wait_total += data_wait; wait_state["start"] = None
            batch_i += 1
            if batch is None:
                pbar.update(1)
                if max_batches is not None and batch_i >= max_batches: break
                continue
            patch_lru_hits += int(batch.get("_patch_lru_batch_hits", 0))
            patch_lru_misses += int(batch.get("_patch_lru_batch_misses", 0))
            patch_lru_bytes_sum += int(batch.get("_patch_lru_bytes", 0))
            patch_lru_bytes_samples += 1
            raster_source_rows_read += int(batch.get("_raster_source_rows_read", 0))
            raster_source_rows_requested += int(batch.get("_raster_source_rows_requested", 0))
            raster_source_chunks_read += int(batch.get("_raster_source_chunks_read", 0))
            collate_total_sec += float(batch.get("_collate_total_sec", 0.0))
            collate_meta_sec += float(batch.get("_collate_meta_sec", 0.0))
            collate_raster_sec += float(batch.get("_collate_raster_sec", 0.0))
            if _locality_window_batches_target is not None:
                _locality_window_wait_sec += data_wait
                _locality_window_collate_sec += float(batch.get("_collate_total_sec", 0.0))
            step_start = time.perf_counter()
            b = batch  # already resolved/transferred by AsyncDeviceBatchPrefetcher
            sample_compute_event = (device.type == "cuda" and
                                    (batches % pipeline_event_every == 0))
            compute_start_event = torch.cuda.Event(enable_timing=True) if sample_compute_event else None
            compute_end_event = torch.cuda.Event(enable_timing=True) if sample_compute_event else None
            if compute_start_event is not None:
                compute_start_event.record(torch.cuda.current_stream(device))
            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")):
                if forward_model is None:
                    pred = model(b).reshape(-1)
                else:
                    # Each microbatch has completed its previous backward before this point.
                    # Marking the new graph iteration avoids CUDAGraph Trees retaining/reusing
                    # an output generation while gradient accumulation consumes prior gradients.
                    _mark_cudagraph_step_begin()
                    _forward_batch = {k: b[k] for k in ("evi", "lst", "s1", "seq", "seq_mask", "static", "anchor")}
                    pred = forward_model(_forward_batch).reshape(-1)
                if target_mode == "residual":
                    # model(b) returns physical target units (anchor + predicted residual),
                    # while b["y"] is target-minus-anchor. Subtract the anchor first.
                    # Flattening every vector prevents accidental [N,1] vs [N] -> [N,N] broadcasting.
                    residual_target = b["y"].reshape(-1)
                    anchor = b["anchor"].reshape(-1)
                    error = (pred - anchor) - residual_target
                    abs_error = error.abs()
                    delta = float(RESIDUAL_HUBER_DELTA)
                    if LOSS_FUNCTION == "mse":
                        per_item_loss = error.square()
                    else:
                        per_item_loss = torch.where(
                            abs_error <= delta,
                            0.5 * error.square(),
                            delta * (abs_error - 0.5 * delta),
                        )
                    weights = (1.0 / (1.0 + residual_target.abs())) if RESIDUAL_WEIGHTING else 1.0
                    loss = (per_item_loss * weights).mean()
                else:
                    target_physical = physical_target(b, target_mode).reshape(-1)
                    if LOSS_FUNCTION == "mse":
                        loss = nn.functional.mse_loss(pred, target_physical)
                    else:
                        loss = nn.functional.huber_loss(pred, target_physical, delta=float(RESIDUAL_HUBER_DELTA))
                loss = loss + CALIBRATION_IDENTITY_PENALTY * calibration_identity_penalty(model)
            scaler.scale(loss).backward()
            pending_microbatches += 1
            _batch_samples = int(b["y"].shape[0])
            samples += _batch_samples; wait_state["samples"] = samples
            if _locality_window_batches_target is not None:
                _locality_window_samples += _batch_samples
            loss_total += loss.detach()
            if pending_microbatches >= accum_steps:
                _optimizer_step()
            if compute_end_event is not None:
                compute_end_event.record(torch.cuda.current_stream(device))
                compute_event_pairs.append((compute_start_event, compute_end_event))
            host_step_total += time.perf_counter() - step_start
            pbar.update(1)
            elapsed = max(time.perf_counter() - epoch_start, 1e-6)
            now = time.perf_counter()
            report_due = (batch_i == 1 or batch_i % max(1, int(RESOURCE_SNAPSHOT_EVERY_BATCHES)) == 0
                          or now - last_resource_snapshot >= float(RESOURCE_SNAPSHOT_INTERVAL_SEC)
                          or data_wait >= LIVE_PROGRESS_INTERVAL_SEC)
            if report_due:
                try: current_loss = float(loss.detach().float().cpu())
                except Exception: current_loss = float("nan")
                snap = _resource_snapshot()
                resource_samples.append(snap)
                last_resource_snapshot = now
                info = {"samples": f"{samples:,}", "eff/micro": f"{effective_batch_size}/{microbatch_size}",
                        "accum": accum_steps, "samp/s": f"{samples/elapsed:.1f}",
                        "wait": f"{data_wait:.2f}s", "step_h": f"{host_step_total/max(1,batch_i):.3f}s/b",
                        "collate_ms": f"{1000.0*float(batch.get('_collate_total_sec',0.0)):.0f}",
                        "raster_ms": f"{1000.0*float(batch.get('_collate_raster_sec',0.0)):.0f}"}
                if np.isfinite(current_loss): info["loss"] = f"{current_loss:.4f}"
                if snap["gpu_util_pct"] is not None: info["GPU"] = f"{snap['gpu_util_pct']:.0f}%"
                if snap["gpu_alloc_gib"] is not None: info["VRAM"] = f"{snap['gpu_alloc_gib']:.2f}/{snap['gpu_total_gib']:.1f}G"
                if snap["host_cpu_pct"] is not None: info["CPU"] = f"{snap['host_cpu_pct']:.0f}%"
                if snap["host_ram_pct"] is not None: info["RAM"] = f"{snap['host_ram_pct']:.0f}%"
                if raster_source_rows_requested:
                    info["gather_ratio"] = f"{raster_source_rows_read/max(1,raster_source_rows_requested):.2f}x"
                pbar.set_postfix(info, refresh=False)
            batches += 1
            if _locality_window_batches_target is not None:
                _locality_window_batches_done += 1
                _is_window_boundary = (_locality_window_batches_done >= _locality_window_batches_target)
                _is_training_boundary = (max_batches is not None and batch_i >= max_batches)
                if _is_window_boundary or _is_training_boundary:
                    _emit_locality_window(partial=(_is_training_boundary and not _is_window_boundary))
            if max_batches is not None and batch_i >= max_batches:
                break
        if _locality_window_batches_done > 0:
            _emit_locality_window(partial=True)
        if pending_microbatches:
            _optimizer_step()
    finally:
        if hasattr(iterator, "close"):
            iterator.close()
        stop_heartbeat.set(); heartbeat.join(timeout=1.0); pbar.close()
    # The scalar read synchronizes the compute stream; the prefetcher close synchronizes
    # its transfer stream. CUDA event timings are therefore ready without per-batch syncs.
    try:
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    except Exception:
        pass
    elapsed = max(time.perf_counter() - epoch_start, 1e-6)
    mean_loss = float((loss_total / max(1, batches)).detach().cpu()) if batches else float("nan")
    snap = _training_resource_summary(resource_samples)
    prefetch_diag = iterator.diagnostics() if hasattr(iterator, "diagnostics") else {}
    h2d_ms = _cuda_event_ms_summary(prefetch_diag.get("transfer_event_pairs", []))
    compute_ms = _cuda_event_ms_summary(compute_event_pairs)
    return {"train_loss": mean_loss, "samples": int(samples), "train_batches": int(batches),
            "optimizer_steps": int(math.ceil(batches / accum_steps)) if batches else 0,
            "microbatch_size": int(microbatch_size), "effective_batch_size": int(effective_batch_size),
            "accumulation_steps": int(accum_steps), "train_elapsed_sec": float(elapsed),
            "samples_per_sec": float(samples / elapsed), "data_wait_sec": float(wait_total),
            "data_wait_per_batch_sec": float(wait_total / max(1,batches)),
            "host_step_sec_per_batch": float(host_step_total / max(1,batches)),
            "collate_total_ms_per_batch": float(1000.0 * collate_total_sec / max(1,batches)),
            "collate_meta_ms_per_batch": float(1000.0 * collate_meta_sec / max(1,batches)),
            "collate_raster_ms_per_batch": float(1000.0 * collate_raster_sec / max(1,batches)),
            "loader_source_wait_ms_per_batch": float(1000.0 * prefetch_diag.get("loader_wait_sec", 0.0) / max(1, prefetch_diag.get("batches_prepared", 0))),
            "h2d_enqueue_ms_per_batch": float(1000.0 * prefetch_diag.get("move_enqueue_sec", 0.0) / max(1, prefetch_diag.get("batches_prepared", 0))),
            "h2d_cuda_ms_mean": h2d_ms["mean"], "h2d_cuda_ms_p50": h2d_ms["p50"],
            "h2d_cuda_ms_p95": h2d_ms["p95"], "h2d_cuda_samples": h2d_ms["n"],
            "gpu_compute_ms_mean": compute_ms["mean"], "gpu_compute_ms_p50": compute_ms["p50"],
            "gpu_compute_ms_p95": compute_ms["p95"], "gpu_compute_samples": compute_ms["n"],
            "input_pinned_tensor_fraction": prefetch_diag.get("pinned_tensor_fraction"),
            "gpu_util_pct": snap["gpu_util_pct"], "gpu_alloc_gib": snap["gpu_alloc_gib"],
            "gpu_reserved_gib": snap["gpu_reserved_gib"], "gpu_total_gib": snap["gpu_total_gib"],
            "host_cpu_pct": snap["host_cpu_pct"], "host_ram_pct": snap["host_ram_pct"],
            "process_rss_gib": snap["process_rss_gib"], "available_ram_gib": snap["available_ram_gib"],
            "process_tree_cpu_pct": snap["process_tree_cpu_pct"],
            "process_tree_rss_gib": snap["process_tree_rss_gib"],
            "patch_lru_hits": int(patch_lru_hits), "patch_lru_misses": int(patch_lru_misses),
            "patch_lru_hit_pct": float(100.0*patch_lru_hits/max(1, patch_lru_hits+patch_lru_misses)),
            "patch_lru_bytes_mean": float(patch_lru_bytes_sum / max(1, patch_lru_bytes_samples)),
            "raster_source_rows_read": int(raster_source_rows_read),
            "raster_source_rows_requested": int(raster_source_rows_requested),
            "raster_source_chunks_read": int(raster_source_chunks_read),
            "raster_read_amplification": float(raster_source_rows_read / max(1, raster_source_rows_requested))}


def _gpu_shard_effective_free_bytes():
    """Use driver-reported free VRAM only; allocator-reserved blocks are not driver-free memory."""
    free_driver, _total = torch.cuda.mem_get_info(device)
    return int(free_driver)

_GPU_SHARD_REQUIRED_KEYS = ("evi", "lst", "s1", "seq", "seq_mask", "static", "anchor", "y")
_GPU_SHARD_RASTER_KEYS = {"evi", "lst", "s1"}

def _gpu_shard_estimated_batch_bytes(raw):
    total = 0
    for key in _GPU_SHARD_REQUIRED_KEYS:
        value = raw.get(key)
        if not torch.is_tensor(value):
            raise RuntimeError(f"VRAM shard loader expected tensor '{key}', got {type(value).__name__}; batch keys={list(raw)}")
        itemsize = 2 if (key in _GPU_SHARD_RASTER_KEYS and USE_AMP) else int(value.element_size())
        total += int(value.numel()) * itemsize
    return int(total)

def _gpu_shard_to_device(raw, *, non_blocking=False):
    """Transfer required model tensors; async refill uses a dedicated CUDA stream and pinned CPU batches."""
    # GPU-index mode is intentionally not the default for this path, but resolve it correctly if enabled.
    if isinstance(raw, dict) and "_gpu_raster_years" in raw:
        # move_batch uses non_blocking transfers on the currently active CUDA stream.
        return move_batch(raw)
    out = {}
    for key in _GPU_SHARD_REQUIRED_KEYS:
        value = raw[key]
        # Non-blocking H2D is truly overlapped when DataLoader has pinned the source tensor.
        async_copy = bool(non_blocking and value.device.type == "cpu" and value.is_pinned())
        if key in _GPU_SHARD_RASTER_KEYS and device.type == "cuda" and USE_AMP:
            out[key] = value.to(device=device, dtype=torch.float16, non_blocking=async_copy,
                                memory_format=torch.channels_last)
        elif key in _GPU_SHARD_RASTER_KEYS and device.type == "cuda":
            out[key] = value.to(device=device, dtype=torch.float32, non_blocking=async_copy,
                                memory_format=torch.channels_last)
        else:
            out[key] = value.to(device=device, non_blocking=async_copy)
    # Preserve collator diagnostics as small Python integers, not extra CUDA tensors.
    out["_patch_lru_batch_hits"] = int(raw.get("_patch_lru_batch_hits", 0))
    out["_patch_lru_batch_misses"] = int(raw.get("_patch_lru_batch_misses", 0))
    out["_patch_lru_bytes"] = int(raw.get("_patch_lru_bytes", 0))
    return out


class _GPUShardAsyncRefiller:
    """One-batch-ahead CPU+H2D refiller for the rolling VRAM-resident batch queue.

    A permit is released only after the consumer has freed a completed batch's inputs.
    Therefore at most one replacement batch is in flight, and it occupies memory released
    by already-consumed inputs instead of allocating an unbounded second GPU shard.
    """
    def __init__(self, source_iterator, *, stream=None):
        self.source_iterator = iter(source_iterator)
        self.queue = _queue.Queue(maxsize=1)
        self.permit = threading.Semaphore(1)  # start one refill immediately
        self.stop_event = threading.Event()
        self._release_event = None
        self.done = False
        self.stream = stream if stream is not None else torch.cuda.Stream(device=device)
        self.thread = threading.Thread(target=self._run, name="gpu-shard-async-refill", daemon=True)
        self.thread.start()

    def _run(self):
        while not self.stop_event.is_set():
            if not self.permit.acquire(timeout=0.1):
                continue
            if self.stop_event.is_set():
                return
            release_event = self._release_event
            self._release_event = None
            try:
                raw = next(self.source_iterator)
            except StopIteration:
                self.queue.put(("end", None))
                return
            except BaseException as exc:
                self.queue.put(("error", exc))
                return
            if raw is None:
                self.queue.put(("skip", None))
                continue
            try:
                with torch.cuda.stream(self.stream):
                    # Reuse memory from a consumed batch only after its backward pass has
                    # completed on the compute stream. The copy can then overlap the NEXT batch.
                    if release_event is not None:
                        self.stream.wait_event(release_event)
                    prepared = _gpu_shard_to_device(raw, non_blocking=True)
                    event = torch.cuda.Event(blocking=False)
                    event.record(self.stream)
                # Keep raw alive with the queued transfer until its CUDA event is consumed.
                self.queue.put(("batch", (prepared, event, raw)))
                del raw, prepared, event
            except BaseException as exc:
                self.queue.put(("error", exc))
                return

    def take(self, block=False):
        try:
            item = self.queue.get(block=block)
        except _queue.Empty:
            return None
        kind, payload = item
        if kind == "end":
            self.done = True
            return ("end", None)
        if kind == "error":
            self.done = True
            raise RuntimeError(f"GPU shard asynchronous refill failed: {type(payload).__name__}: {payload}") from payload
        if kind == "skip":
            # Caller grants the next permit with its latest compute-completion event, if available.
            return ("skip", None)
        return ("batch", payload)

    def allow_next(self, after_event=None):
        if not self.done and not self.stop_event.is_set():
            self._release_event = after_event
            self.permit.release()

    def close(self):
        self.stop_event.set()
        # Wake a thread waiting for the next permit; it will notice stop_event and exit.
        try:
            self.permit.release()
        except ValueError:
            pass
        if self.thread.is_alive():
            self.thread.join(timeout=5.0)


def _train_epoch_vram_resident_shards(model, loader, optimizer, scaler, target_mode,
                                      effective_batch_size, microbatch_size, clip_norm,
                                      desc, max_batches=None, forward_model=None):
    """Fill VRAM with resolved input batches, then asynchronously refill slots as batches finish."""
    if device.type != "cuda":
        return _train_epoch_streaming(model, loader, optimizer, scaler, target_mode,
                                      effective_batch_size, microbatch_size, clip_norm, desc, max_batches,
                                      forward_model=forward_model)

    from collections import deque
    model.train(True)
    _compiled_active = forward_model is not None and bool(globals().get("TORCH_COMPILE_ENABLED", True))
    if _compiled_active and bool(globals().get("COMPILE_STABLE_GRAD_BUFFERS", True)):
        _ensure_stable_parameter_grad_buffers(model)
        optimizer.zero_grad(set_to_none=False)
    else:
        optimizer.zero_grad(set_to_none=not _compiled_active)
    accum_steps = max(1, int(math.ceil(effective_batch_size / max(1, microbatch_size))))
    try:
        total_batches = len(loader)
    except Exception:
        total_batches = None
    if max_batches is not None:
        total_batches = min(total_batches, int(max_batches)) if total_batches is not None else int(max_batches)
    pbar = tqdm(total=total_batches, desc=desc, leave=False, dynamic_ncols=True, mininterval=0.25, position=1)
    epoch_start = time.perf_counter()
    resource_samples = []
    last_resource_snapshot = epoch_start
    if device.type == "cuda":
        try: torch.cuda.reset_peak_memory_stats(device)
        except Exception: pass
    data_wait_total = host_step_total = 0.0
    samples = batches = 0
    patch_lru_hits = patch_lru_misses = 0
    loss_total = torch.zeros((), device=device)
    pending_microbatches = 0
    gpu_batches_loaded = 0
    max_resident_batches = 0
    initial_resident_gib = 0.0
    pending_cpu_batch = None
    raw_batches_read = 0
    prefetcher = None
    source_exhausted = False
    latest_release_event = None

    probe_peak = int(globals().get("_GPU_SHARD_CURRENT_PROBE_PEAK_BYTES",
                                  globals().get("_GPU_PROBE_PEAK_DELTA_BYTES", 0)) or 0)
    reserve_bytes = max(
        int(float(globals().get("GPU_SHARD_RESERVE_GIB", 0.60)) * _GIB),
        int(probe_peak * float(globals().get("GPU_SHARD_ACTIVATION_MULTIPLIER", 1.20))
            + float(globals().get("GPU_SHARD_EXTRA_ACTIVATION_GIB", 0.20)) * _GIB),
    )
    reserve_gib = reserve_bytes / _GIB
    print(f"[VRAM rolling-shard mode] model + optimizer resident; activation/workspace reserve={reserve_gib:.2f} GiB. "
          "The queue will refill one freed batch slot asynchronously on a separate CUDA stream.")

    data_iter = iter(loader)  # construct DataLoader workers in the calling thread
    # Use one dedicated input/copy stream for initial shard allocations and later refills.
    # This lets the caching allocator safely reuse freed input blocks on that same stream;
    # compute-stream use is recorded below and guarded by an event before refill begins.
    refill_stream = torch.cuda.Stream(device=device)
    preload_ready_event = torch.cuda.Event(blocking=False)
    shard = deque()
    prefetch_slot_bytes = 0
    preload_bytes = 0
    preload_start = time.perf_counter()

    def _optimizer_step():
        nonlocal pending_microbatches
        if pending_microbatches <= 0:
            return
        scaler.unscale_(optimizer)
        inv = 1.0 / float(pending_microbatches)
        for param in model.parameters():
            if param.grad is not None:
                param.grad.mul_(inv)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=not _compiled_active)
        pending_microbatches = 0

    try:
        # Initial fill intentionally leaves one-batch worth of space in addition to the
        # measured activation reserve. The async refiller uses this slot for its first copy.
        _initial_fill_started = time.perf_counter()
        _initial_fill_max_seconds = float(globals().get("GPU_SHARD_INITIAL_FILL_MAX_SECONDS", 30.0))
        _initial_fill_max_batches = max(1, int(globals().get("GPU_SHARD_INITIAL_FILL_MAX_BATCHES", 16)))
        while len(shard) < int(globals().get("GPU_SHARD_MAX_BATCHES", 4096)):
            # Seed a useful resident working set, but never spend the whole startup waiting
            # for VRAM to fill. Remaining batches are fetched by the async refiller during training.
            if shard and (len(shard) >= _initial_fill_max_batches or
                          time.perf_counter() - _initial_fill_started >= _initial_fill_max_seconds):
                print(f"[VRAM fill] Starting training with {len(shard)} resident batches; "
                      f"initial fill capped at {_initial_fill_max_seconds:.0f}s/{_initial_fill_max_batches} batches. "
                      "Async refill continues during training.")
                break
            if max_batches is not None and raw_batches_read >= int(max_batches):
                source_exhausted = True
                break
            load_started = time.perf_counter()
            try:
                raw = next(data_iter)
            except StopIteration:
                source_exhausted = True
                break
            if raw is None:
                data_wait_total += time.perf_counter() - load_started
                continue
            raw_batches_read += 1  # count usable batches, matching the training loop's batch limit
            estimated = _gpu_shard_estimated_batch_bytes(raw)
            if prefetch_slot_bytes <= 0:
                prefetch_slot_bytes = estimated
            allocatable = _gpu_shard_effective_free_bytes()
            desired_free = reserve_bytes + prefetch_slot_bytes
            if allocatable - estimated < desired_free:
                if shard:
                    pending_cpu_batch = raw
                    break
                # Without room for one real batch plus a refill slot and activation reserve,
                # the overlap design cannot run safely. Fall back before any optimizer update.
                del raw, data_iter
                pbar.close()
                gc.collect()
                torch.cuda.empty_cache()
                print("[VRAM rolling-shard mode] Not enough room for a resident batch + async refill slot "
                      "and activation reserve; using streamed training for this epoch.")
                return _train_epoch_streaming(model, loader, optimizer, scaler, target_mode,
                    effective_batch_size, microbatch_size, clip_norm, desc, max_batches)
            try:
                with torch.cuda.stream(refill_stream):
                    resident = _gpu_shard_to_device(raw, non_blocking=False)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if shard:
                    # Keep the CPU batch and let the async refiller retry after a consumed batch frees space.
                    pending_cpu_batch = raw
                    break
                del raw, data_iter
                pbar.close()
                gc.collect()
                print("[VRAM rolling-shard mode] Initial allocation OOM; retrying streamed training.")
                return _train_epoch_streaming(model, loader, optimizer, scaler, target_mode,
                    effective_batch_size, microbatch_size, clip_norm, desc, max_batches)
            del raw
            data_wait_total += time.perf_counter() - load_started
            shard.append((resident, preload_ready_event, None))
            preload_bytes += estimated
            gpu_batches_loaded += 1
            pbar.set_postfix({"status": "GPU_FILL", "resident_batches": len(shard),
                              "resident_GiB": f"{preload_bytes/_GIB:.2f}",
                              "free_GiB": f"{_gpu_shard_effective_free_bytes()/_GIB:.2f}"}, refresh=True)
        if not shard and source_exhausted:
            raise RuntimeError("Training loader produced no usable batches.")
        # Initial inputs were allocated on refill_stream. Make all copies visible before training;
        # the same event can be attached to every initial batch.
        preload_ready_event.record(refill_stream)
        initial_resident_gib = preload_bytes / _GIB
        max_resident_batches = len(shard)
        pbar.set_postfix({"status": "GPU_RESIDENT", "initial_batches": len(shard),
                          "resident_GiB": f"{initial_resident_gib:.2f}",
                          "free_GiB": f"{_gpu_shard_effective_free_bytes()/_GIB:.2f}",
                          "reserve_GiB": f"{reserve_gib:.2f}"}, refresh=True)
        print(f"[VRAM rolling-shard mode] Initial resident set: {len(shard)} batches / {initial_resident_gib:.2f} GiB; "
              f"prefetch slot target={prefetch_slot_bytes/_GIB:.3f} GiB; raw batches read={raw_batches_read}.")

        def _remaining_source():
            # The held batch already came out of the loader and is counted in raw_batches_read.
            if pending_cpu_batch is not None:
                yield pending_cpu_batch
            if max_batches is None:
                yield from data_iter
            else:
                additional = max(0, int(max_batches) - raw_batches_read)
                while additional > 0:
                    try:
                        raw_next = next(data_iter)
                    except StopIteration:
                        return
                    if raw_next is not None:
                        additional -= 1
                    yield raw_next

        # Even if the initial fill reached the batch cap, this lightweight source exits immediately;
        # otherwise it supplies the held CPU batch then continues the same DataLoader iterator.
        prefetcher = _GPUShardAsyncRefiller(_remaining_source(), stream=refill_stream)

        while shard or not source_exhausted:
            if not shard:
                # Empty resident queue means the GPU has consumed its current set. Wait only here;
                # while resident batches remain, CPU read + H2D refill proceeds concurrently.
                wait_started = time.perf_counter()
                message = prefetcher.take(block=True) if prefetcher is not None else ("end", None)
                data_wait_total += time.perf_counter() - wait_started
                if message[0] == "end":
                    source_exhausted = True
                    break
                if message[0] == "skip":
                    prefetcher.allow_next(latest_release_event)
                    latest_release_event = None
                    continue
                if message[0] == "batch":
                    shard.append(message[1])
                    gpu_batches_loaded += 1
                    prefetcher.allow_next(latest_release_event)
                    latest_release_event = None
            if not shard:
                continue

            b, ready_event, cpu_source = shard.popleft()
            compute_stream = torch.cuda.current_stream(device)
            if ready_event is not None:
                compute_stream.wait_event(ready_event)
            # Inputs were allocated on refill_stream. Mark their compute-stream use so the
            # caching allocator cannot recycle their storage until this backward pass completes.
            for value in b.values():
                if torch.is_tensor(value) and value.device.type == "cuda":
                    value.record_stream(compute_stream)

            step_start = time.perf_counter()
            patch_lru_hits += int(b.get("_patch_lru_batch_hits", 0))
            patch_lru_misses += int(b.get("_patch_lru_batch_misses", 0))
            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")):
                if forward_model is None:
                    pred = model(b).reshape(-1)
                else:
                    _forward_batch = {k: b[k] for k in ("evi", "lst", "s1", "seq", "seq_mask", "static", "anchor")}
                    pred = forward_model(_forward_batch).reshape(-1)
                if target_mode == "residual":
                    # Model output is in physical units; b["y"] is target-minus-anchor.
                    # Align both representations and flatten before computing per-example loss.
                    residual_target = b["y"].reshape(-1)
                    anchor = b["anchor"].reshape(-1)
                    error = (pred - anchor) - residual_target
                    abs_error = error.abs()
                    delta = float(RESIDUAL_HUBER_DELTA)
                    if LOSS_FUNCTION == "mse":
                        per_item_loss = error.square()
                    else:
                        per_item_loss = torch.where(abs_error <= delta, 0.5 * error.square(),
                                                    delta * (abs_error - 0.5 * delta))
                    weights = (1.0 / (1.0 + residual_target.abs())) if RESIDUAL_WEIGHTING else 1.0
                    loss = (per_item_loss * weights).mean()
                else:
                    target_physical = physical_target(b, target_mode).reshape(-1)
                    if LOSS_FUNCTION == "mse":
                        loss = nn.functional.mse_loss(pred, target_physical)
                    else:
                        loss = nn.functional.huber_loss(pred, target_physical, delta=float(RESIDUAL_HUBER_DELTA))
                loss = loss + CALIBRATION_IDENTITY_PENALTY * calibration_identity_penalty(model)
            scaler.scale(loss).backward()
            pending_microbatches += 1
            batch_n = int(b["y"].shape[0])
            samples += batch_n
            loss_total += loss.detach()
            host_step_total += time.perf_counter() - step_start
            if pending_microbatches >= accum_steps:
                _optimizer_step()
            batches += 1
            pbar.update(1)
            del b, ready_event, cpu_source, pred, loss
            latest_release_event = torch.cuda.Event(blocking=False)
            latest_release_event.record(torch.cuda.current_stream(device))

            # If the background copy is ready, move it into the rolling queue and immediately
            # grant one refill permit. The newly freed batch allocation is then reused by that
            # next H2D transfer while the next resident microbatch computes on the default stream.
            if prefetcher is not None:
                message = prefetcher.take(block=False)
                if message is not None:
                    if message[0] == "batch":
                        shard.append(message[1])
                        gpu_batches_loaded += 1
                        max_resident_batches = max(max_resident_batches, len(shard))
                        prefetcher.allow_next(latest_release_event)
                        latest_release_event = None
                    elif message[0] == "skip":
                        prefetcher.allow_next(latest_release_event)
                        latest_release_event = None
                    elif message[0] == "end":
                        source_exhausted = True

            now = time.perf_counter()
            report_due = (batches == 1 or batches % max(1, int(RESOURCE_SNAPSHOT_EVERY_BATCHES)) == 0
                          or now - last_resource_snapshot >= float(RESOURCE_SNAPSHOT_INTERVAL_SEC))
            if report_due:
                elapsed = max(now - epoch_start, 1e-6)
                snap = _resource_snapshot()
                resource_samples.append(snap)
                last_resource_snapshot = now
                postfix = {"status": "ASYNC_REFILL", "resident_batches": len(shard),
                    "samples": f"{samples:,}", "samp/s": f"{samples/elapsed:.1f}",
                    "queue_wait/b": f"{data_wait_total/max(1,batches):.3f}s",
                    "launch/b": f"{host_step_total/max(1,batches):.3f}s",
                    "GPU": f"{snap['gpu_util_pct']:.0f}%" if snap["gpu_util_pct"] is not None else "n/a",
                    "VRAM_peak": f"{torch.cuda.max_memory_allocated(device)/_GIB:.2f}G" if device.type == "cuda" else "n/a"}
                pbar.set_postfix(postfix, refresh=False)

            if max_batches is not None and batches >= int(max_batches):
                source_exhausted = True
                break

        # Keep optimizer-step semantics identical at the end of the epoch, including partial accumulation.
        if pending_microbatches:
            _optimizer_step()
    finally:
        if prefetcher is not None:
            prefetcher.close()
        # A batch may have been queued for H2D but not consumed when a capped screening run ends.
        # Finish that bounded transfer before releasing the queue/source references.
        try:
            torch.cuda.synchronize(device)
        except Exception:
            pass
        try:
            if prefetcher is not None:
                while True:
                    prefetcher.queue.get_nowait()
        except _queue.Empty:
            pass
        try:
            del data_iter
        except Exception:
            pass
        pbar.close()

    elapsed = max(time.perf_counter() - epoch_start, 1e-6)
    mean_loss = float((loss_total / max(1, batches)).detach().cpu()) if batches else float("nan")
    snap = _training_resource_summary(resource_samples)
    return {"train_loss": mean_loss, "samples": int(samples), "train_batches": int(batches),
        "optimizer_steps": int(math.ceil(batches / accum_steps)) if batches else 0,
        "microbatch_size": int(microbatch_size), "effective_batch_size": int(effective_batch_size),
        "accumulation_steps": int(accum_steps), "train_elapsed_sec": float(elapsed),
        "samples_per_sec": float(samples / elapsed), "data_wait_sec": float(data_wait_total),
        "data_wait_per_batch_sec": float(data_wait_total / max(1, batches)),
        "host_step_sec_per_batch": float(host_step_total / max(1, batches)),
        "gpu_util_pct": snap["gpu_util_pct"], "gpu_alloc_gib": snap["gpu_alloc_gib"],
        "gpu_reserved_gib": snap["gpu_reserved_gib"], "gpu_total_gib": snap["gpu_total_gib"],
        "host_cpu_pct": snap["host_cpu_pct"], "host_ram_pct": snap["host_ram_pct"],
        "process_rss_gib": snap["process_rss_gib"], "available_ram_gib": snap["available_ram_gib"],
        "process_tree_cpu_pct": snap["process_tree_cpu_pct"], "process_tree_rss_gib": snap["process_tree_rss_gib"],
        "patch_lru_hits": int(patch_lru_hits), "patch_lru_misses": int(patch_lru_misses),
        "patch_lru_hit_pct": float(100.0 * patch_lru_hits / max(1, patch_lru_hits + patch_lru_misses)),
        "gpu_shards": 1 if batches else 0, "max_gpu_shard_batches": int(max_resident_batches),
        "avg_gpu_shard_gib": float(initial_resident_gib), "gpu_shard_reserve_gib": float(reserve_gib),
        "gpu_batches_loaded": int(gpu_batches_loaded), "full_loader_batches": int(len(loader)) if hasattr(loader, "__len__") else None,
        "max_batches_limit": int(max_batches) if max_batches is not None else None}

def _train_epoch_accumulated(model, loader, optimizer, scaler, target_mode, effective_batch_size,
                             microbatch_size, clip_norm, desc, max_batches=None, forward_model=None):
    """Use model-first VRAM-resident shards when enabled; otherwise use async streamed batches."""
    if device.type == "cuda" and bool(globals().get("GPU_RESIDENT_SHARD_TRAINING", True)):
        return _train_epoch_vram_resident_shards(model, loader, optimizer, scaler, target_mode,
            effective_batch_size, microbatch_size, clip_norm, desc, max_batches,
            forward_model=forward_model)
    return _train_epoch_streaming(model, loader, optimizer, scaler, target_mode,
        effective_batch_size, microbatch_size, clip_norm, desc, max_batches,
        forward_model=forward_model)

def train_one_trial(trial, task_mode):
    global TASK_MODE, TARGET_MODE, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP_NORM, NUM_WORKERS, RESIDUAL_WEIGHTING, RESIDUAL_HUBER_DELTA, LOSS_FUNCTION
    TASK_MODE = task_mode
    backbone = trial.suggest_categorical("temporal_backbone", ["transformer", "lstm", "tcn"])
    spatial_width = trial.suggest_categorical("spatial_width", [32, 48])
    temporal_hidden = trial.suggest_categorical("temporal_hidden", [32, 48, 64])
    fusion_width = trial.suggest_categorical("fusion_width", [128, 192, 256])
    target_mode = trial.suggest_categorical("target_mode", ["residual", "direct"])
    lr = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    wd = trial.suggest_float("weight_decay", 1e-7, 3e-3, log=True)
    batch_scale = trial.suggest_categorical("batch_scale", BATCH_SCALE_CHOICES)
    clip_norm = trial.suggest_categorical("grad_clip_norm", [0.5, 1.0, 2.0, 5.0])
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "Adam"])
    residual_weighting = trial.suggest_categorical("residual_weighting", [False, True])
    loss_function = trial.suggest_categorical("loss_function", ["mse", "huber"])
    huber_delta = trial.suggest_float("huber_delta", 0.05, 0.5, log=True)
    TARGET_MODE = target_mode
    RESIDUAL_WEIGHTING = bool(residual_weighting)
    LOSS_FUNCTION = str(loss_function)
    RESIDUAL_HUBER_DELTA = float(huber_delta)

    model = build_model(task_mode, backbone, target_mode,
                        spatial_width=spatial_width, temporal_hidden=temporal_hidden,
                        fusion_width=fusion_width)
    train_ds = get_global_dataset("train", task_mode=task_mode, target_mode=target_mode,
        max_rows=MAX_ROWS_PER_SPLIT, shuffle=True, seed=SEED + trial.number)
    test_selection_ds = get_global_dataset("test", task_mode=task_mode, target_mode=target_mode,
        max_rows=MAX_ROWS_PER_SPLIT, shuffle=False, seed=SEED)

    # Probe the physical microbatch that actually enters the GPU. The optimizer batch can
    # be larger through gradient accumulation and therefore need not consume matching VRAM.
    probe_key = (task_mode, backbone, int(spatial_width), int(temporal_hidden), int(fusion_width))
    if not hasattr(train_one_trial, "_safe_microbatch_cache"):
        train_one_trial._safe_microbatch_cache = {}
    if not hasattr(train_one_trial, "_probe_peak_cache"):
        train_one_trial._probe_peak_cache = {}
    if probe_key not in train_one_trial._safe_microbatch_cache:
        probed = probe_gpu_batch_limit(model, train_ds, max_batch=TRAIN_MICROBATCH_CAP,
                                       min_batch=MIN_GPU_BATCH_SIZE)
        train_one_trial._safe_microbatch_cache[probe_key] = int(probed)
        train_one_trial._probe_peak_cache[probe_key] = int(globals().get("_GPU_PROBE_PEAK_DELTA_BYTES", 0) or 0)
        print(f"GPU probe [{task_mode}/{backbone}/S{spatial_width}-T{temporal_hidden}-F{fusion_width}]: safe physical microbatch ceiling={probed}; "
              f"measured peak delta={train_one_trial._probe_peak_cache[probe_key]/_GIB:.2f} GiB")
    safe_microbatch = int(train_one_trial._safe_microbatch_cache[probe_key])
    probe_peak_bytes = int(train_one_trial._probe_peak_cache.get(probe_key, 0))
    globals()["_GPU_SHARD_CURRENT_PROBE_PEAK_BYTES"] = probe_peak_bytes
    effective_batch = batch_size_for_scale(max(EFFECTIVE_BATCH_CEILING, BATCH_SIZE), batch_scale)
    effective_batch = max(MIN_GPU_BATCH_SIZE, int(effective_batch))
    # Never choose a microbatch larger than this trial's effective batch; larger physical batches
    # are allowed up to 2048 only when the real forward/backward probe says they are safe.
    microbatch_size = max(MIN_GPU_BATCH_SIZE,
                          min(int(TRAIN_MICROBATCH_CAP), int(safe_microbatch), effective_batch))
    accumulation_steps = max(1, int(math.ceil(effective_batch / microbatch_size)))
    trial_max_train_batches = min(int(TRIAL_MAX_TRAIN_BATCHES), max(1, int(math.ceil(
        float(TRIAL_TRAIN_SAMPLES_PER_EPOCH) / float(microbatch_size)))))
    BATCH_SIZE = effective_batch
    trial.set_user_attr("effective_batch_size", int(effective_batch))
    trial.set_user_attr("loader_microbatch_size", int(microbatch_size))
    trial.set_user_attr("accumulation_steps", int(accumulation_steps))
    trial.set_user_attr("trial_max_train_batches", int(trial_max_train_batches))
    trial.set_user_attr("trial_train_samples_per_epoch", int(TRIAL_TRAIN_SAMPLES_PER_EPOCH))
    trial.set_user_attr("safe_microbatch_ceiling", int(safe_microbatch))
    trial.set_user_attr("architecture_version", "multiscale_spatial_temporal_attention_gated_fusion_v19")
    trial.set_user_attr("gpu_probe_peak_bytes", int(probe_peak_bytes))
    trial.set_user_attr("num_workers", int(NUM_WORKERS))
    trial.set_user_attr("prefetch_factor_ceiling", int(PREFETCH_FACTOR))
    trial.set_user_attr("prefetch_sample_budget", int(MAX_PREFETCHED_SAMPLES_PER_LOADER))
    if mlflow is not None and mlflow.active_run() is not None:
        _trial_params = {**trial.params, "task_mode": task_mode, "effective_batch_size": effective_batch,
                         "residual_weighting": RESIDUAL_WEIGHTING,
                         "loss_function": LOSS_FUNCTION, "residual_huber_delta": RESIDUAL_HUBER_DELTA,
                         "trial_train_samples_per_epoch": int(TRIAL_TRAIN_SAMPLES_PER_EPOCH)}
        mlflow.log_params({str(k): (str(v) if isinstance(v, (list, tuple, dict)) else v)
                           for k, v in _trial_params.items()})

    optimizer = _make_optimizer(optimizer_name, model.parameters(), lr=lr, weight_decay=wd)
    train_loader = build_global_loader(train_ds, microbatch_size)
    test_selection_loader = build_global_loader(test_selection_ds, min(int(EVAL_BATCH_SIZE), int(microbatch_size)),
        pin_memory=False, num_workers=0, persistent_workers=False,
        eval_max_batches=TRIAL_MAX_TEST_BATCHES)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
    best_score = -float("inf")
    best_epoch_metrics = None
    trial_start = time.perf_counter()

    epoch_bar = tqdm(range(1, TRIAL_EPOCHS + 1), desc=f"Trial {trial.number} [{task_mode}]",
                      leave=False, dynamic_ncols=True, mininterval=0.5, position=0)
    try:
        for epoch in epoch_bar:
            train_metrics = _train_epoch_accumulated(
                model, train_loader, optimizer, scaler, target_mode, effective_batch, microbatch_size,
                clip_norm, desc=f"T{trial.number} E{epoch} train", max_batches=trial_max_train_batches)
            metrics = evaluate_predictions(model, test_selection_loader, target_mode,
                                           max_batches=TRIAL_MAX_TEST_BATCHES,
                                           desc=f"T{trial.number} E{epoch} test-eval")
            score = metrics["r2"]
            combined = {**train_metrics, **{f"test_{k}": v for k, v in metrics.items()},
                        "epoch": int(epoch), "trial_elapsed_sec": float(time.perf_counter()-trial_start)}
            trial.set_user_attr(f"epoch_{epoch}_metrics", combined)
            trial.set_user_attr("last_epoch_metrics", combined)
            if mlflow is not None and mlflow.active_run() is not None:
                _mlflow_scalars = {
                    "test_r2": score,
                    "test_rmse": metrics.get("rmse", np.nan),
                }
                _mlflow_scalars = {k: float(v) for k, v in _mlflow_scalars.items()
                                   if v is not None and np.isfinite(float(v))}
                if _mlflow_scalars:
                    mlflow.log_metrics(_mlflow_scalars, step=epoch)
            trial.report(score if np.isfinite(score) else -1e6, epoch)
            if np.isfinite(score) and score > best_score:
                best_score, best_epoch_metrics = score, metrics
                trial.set_user_attr("best_epoch", epoch)
                trial.set_user_attr("best_test_metrics", metrics)
            epoch_bar.set_postfix(
                train_loss=f"{train_metrics['train_loss']:.4f}" if np.isfinite(train_metrics['train_loss']) else "N/A",
                test_r2=f"{score:.4f}" if np.isfinite(score) else "N/A",
                rmse=f"{metrics['rmse']:.4f}" if np.isfinite(metrics['rmse']) else "N/A",
                NMRSE=f"{metrics['nrmse']:.4f}" if np.isfinite(metrics['nrmse']) else "N/A",
                samp_s=f"{train_metrics['samples_per_sec']:.1f}",
                wait=f"{train_metrics['data_wait_per_batch_sec']:.2f}s",
                gpu=(f"{train_metrics['gpu_util_pct']:.0f}%" if train_metrics['gpu_util_pct'] is not None else "n/a"),
                vram=(f"{train_metrics['gpu_alloc_gib']:.2f}G" if train_metrics['gpu_alloc_gib'] is not None else "n/a"))
            if trial.should_prune():
                raise optuna.TrialPruned()
    finally:
        # Explicitly close persistent worker pools when a trial ends or is pruned.
        del train_loader, test_selection_loader
        gc.collect()

    if best_epoch_metrics is None:
        return -1e6
    trial.set_user_attr("target_mode", target_mode)
    trial.set_user_attr("temporal_backbone", backbone)
    trial.set_user_attr("optimizer", optimizer_name)
    trial.set_user_attr("trial_elapsed_sec", float(time.perf_counter()-trial_start))
    return float(best_score)

def _objective_with_diagnostics(trial, task_mode):
    """Run one Optuna trial in MLflow and persist diagnostics for failures."""
    import traceback
    if mlflow is None:
        raise ImportError("MLflow is required for experiment tracking. Run `%pip install mlflow` in this environment, then rerun the import/config cells.")
    run_name = f"optuna_{task_mode}_trial_{trial.number:04d}"
    with mlflow.start_run(run_name=run_name, tags={
        "phase": "optuna_trial", "task_mode": str(task_mode),
        "optuna_trial_number": str(trial.number), "study_suffix": str(OPTUNA_STUDY_SUFFIX),
    }):
        mlflow.log_params({"task_mode": str(task_mode), "trial_number": int(trial.number),
                           "trial_epochs": int(TRIAL_EPOCHS),
                           "trial_train_samples_per_epoch": int(TRIAL_TRAIN_SAMPLES_PER_EPOCH),
                           "trial_test_batches": int(TRIAL_MAX_TEST_BATCHES),
                           "seed": int(OPTUNA_SEED)})
        try:
            result = train_one_trial(trial, task_mode)
            if result is not None and np.isfinite(float(result)):
                mlflow.log_metric("best_test_r2", float(result))
            _best_test_metrics = trial.user_attrs.get("best_test_metrics", {}) or {}
            if _best_test_metrics.get("rmse") is not None and np.isfinite(float(_best_test_metrics["rmse"])):
                mlflow.log_metric("best_test_rmse", float(_best_test_metrics["rmse"]))
            mlflow.set_tag("optuna_state", "COMPLETE")
            mlflow.log_params({f"optuna_{k}": (str(v) if isinstance(v, (list, tuple, dict)) else v)
                               for k, v in trial.params.items()})
            return result
        except optuna.TrialPruned:
            mlflow.set_tag("optuna_state", "PRUNED")
            raise
        except Exception as exc:
            tb = traceback.format_exc()
            record = {"timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
                      "trial_number": int(trial.number), "task_mode": str(task_mode),
                      "params": dict(trial.params), "exception_type": type(exc).__name__,
                      "exception": str(exc), "traceback": tb}
            with open(RUN_DIR / "optuna_trial_errors_filtered_s1.jsonl", "a", encoding="utf-8") as f:
                f.write(json.dumps(record, default=str) + "\n")
            try:
                trial.set_user_attr("failure_type", type(exc).__name__)
                trial.set_user_attr("failure_message", str(exc)[:1500])
                trial.set_user_attr("failure_traceback_tail", tb[-5000:])
            except Exception:
                pass
            mlflow.set_tag("optuna_state", "FAILED")
            mlflow.set_tag("failure_type", type(exc).__name__)
            mlflow.set_tag("failure_message", str(exc)[:2000])
            print(f"\n[TRIAL {trial.number} FAILED — {task_mode}] {type(exc).__name__}: {exc}", flush=True)
            print(tb, flush=True)
            raise


def optuna_live_callback(study, frozen_trial):
    """Print one durable summary when each Optuna trial finishes."""
    state = getattr(frozen_trial.state, "name", str(frozen_trial.state))
    elapsed = frozen_trial.duration.total_seconds() if frozen_trial.duration is not None else float("nan")
    value = frozen_trial.value
    try:
        best = float(study.best_value)
    except Exception:
        best = float("nan")
    attrs = frozen_trial.user_attrs or {}
    last = attrs.get("last_epoch_metrics", {})
    print(f"[Optuna] trial={frozen_trial.number} state={state} "
          f"trial_score={value if value is not None else float('nan'):.4f} "
          f"best={best:.4f} duration={elapsed/60:.1f}min "
          f"backbone={frozen_trial.params.get('temporal_backbone','?')} "
          f"effective_batch={attrs.get('effective_batch_size','?')} "
          f"microbatch={attrs.get('loader_microbatch_size','?')} "
          f"samples/s={last.get('samples_per_sec',float('nan')):.1f} "
          f"data_wait/batch={last.get('data_wait_per_batch_sec',float('nan')):.2f}s "
          f"gather_ratio_ratio={last.get('raster_read_amplification',float('nan')):.3f} "
          f"GPU={last.get('gpu_util_pct','n/a')} VRAM={last.get('gpu_alloc_gib','n/a')} GiB "
          f"CPU(host)={last.get('host_cpu_pct','n/a')}% CPU(tree)={last.get('process_tree_cpu_pct','n/a')}% "
          f"RAM={last.get('host_ram_pct','n/a')}%")

def plot_optuna_progress(study, mode, *, show=True):
    """Persist trial and per-epoch test R² plots for deciding search budgets."""
    import matplotlib.pyplot as plt
    r2_target = R2_TARGETS[mode]
    completed = [t for t in study.trials if t.value is not None and np.isfinite(t.value)]
    epoch_rows = []
    for trial in completed:
        for key, metrics in (trial.user_attrs or {}).items():
            if key.startswith("epoch_") and isinstance(metrics, dict):
                score = metrics.get("test_r2")
                if score is not None and np.isfinite(score):
                    epoch_rows.append({"trial": trial.number, "epoch": int(key.split("_")[1]), "r2": float(score)})
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    if completed:
        axes[0].plot([t.number for t in completed], [float(t.value) for t in completed], "o-")
        axes[0].axhline(r2_target, color="tab:red", linestyle="--", label=f"target {r2_target:.2f}")
        axes[0].set(xlabel="Trial number", ylabel="Best test R²", title=f"{mode}: R² vs trial")
        axes[0].legend()
    if epoch_rows:
        frame = pd.DataFrame(epoch_rows)
        for trial_number, group in frame.groupby("trial"):
            axes[1].plot(group["epoch"], group["r2"], marker="o", alpha=0.45, label=f"T{trial_number}")
        axes[1].axhline(r2_target, color="tab:red", linestyle="--")
        axes[1].set(xlabel="Epoch", ylabel="Test R²", title=f"{mode}: R² vs epoch")
        if len(frame["trial"].unique()) <= 10:
            axes[1].legend(fontsize=8)
    fig.tight_layout()
    plot_path = RUN_DIR / f"{mode}_test_selection_r2_progress_mlflow.png"
    fig.savefig(plot_path, dpi=140)
    if show:
        display(fig)
    plt.close(fig)
    return plot_path


if RUN_OPTUNA_SEARCH:
    try:
        import optuna
    except ImportError as exc:
        raise ImportError("Optuna is required for hyperparameter search. Run `%pip install optuna`, then rerun this cell.") from exc

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # Configure local MLflow tracking. The tracking store is SQLite; artifacts stay local.
    if mlflow is None:
        raise ImportError("MLflow is required. Run `%pip install mlflow` in this Python environment, restart the kernel, and run the imports again.")
    MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
    MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.resolve()}"
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    if mlflow.active_run() is not None:
        print("Closing stale MLflow run left active by an interrupted notebook cell.")
        mlflow.end_run(status="FAILED")
    MLFLOW_EXPERIMENT_NAME = "global_soil_moisture_model_v19_multiscale_attnpool_gatedfusion"
    _existing_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    if _existing_experiment is None:
        mlflow.create_experiment(MLFLOW_EXPERIMENT_NAME,
                                 artifact_location=MLFLOW_ARTIFACT_DIR.resolve().as_uri())
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    # Launch the dashboard from this notebook against the SAME database used for logging.
    # This replaces the separate shell launcher and avoids tracking-DB / dashboard mismatch.
    MLFLOW_UI_URL = start_mlflow_ui(
        backend_store_uri=MLFLOW_TRACKING_URI,
        artifact_root=MLFLOW_ARTIFACT_DIR,
        log_dir=MLFLOW_DIR,
        host=MLFLOW_UI_HOST,
        preferred_port=MLFLOW_UI_PREFERRED_PORT,
        port_search_limit=MLFLOW_UI_PORT_SEARCH_LIMIT,
        startup_timeout=MLFLOW_UI_STARTUP_TIMEOUT_SEC,
    )
    print("MLflow tracking URI:", MLFLOW_TRACKING_URI)
    print("MLflow artifact directory:", MLFLOW_ARTIFACT_DIR.resolve())
    print("MLflow dashboard:", MLFLOW_UI_URL)
    try:
        from IPython.display import HTML as _HTML
        display(_HTML(f'<a href="{MLFLOW_UI_URL}" target="_blank" rel="noopener">Open MLflow dashboard</a>'))
    except Exception:
        pass

    # Precompute model-ready tensors once (rather than decoding/scaling Zarr patches every epoch).
    # The cache uses memory-mapped, uncompressed arrays so hot pages are shared by all loader workers.
    if str(MODEL_READY_CACHE_MODE).lower() != "off":
        _cache_all_available = prepare_model_ready_caches()
        print("Compact metadata cache active for all splits:", _cache_all_available)

    # Lightweight inference-only preflight after the final GPU-index collator is defined.
    # Independent of RUN_SMOKE_TEST so we can skip the multi-batch backward test by default
    # while still catching missing input keys before any Optuna trials begin.
    if RUN_PIPELINE_PREFLIGHT:
        print("Checking one real raster-cache -> model batch before starting Optuna...")
        _preflight_loader = make_loader(
            split="train", task_mode="environment_only", max_rows=1, shuffle=False,
            batch_size=1, num_workers=0,
        )
        try:
            _preflight_raw = next(iter(_preflight_loader))
        except StopIteration as _exc:
            raise RuntimeError("Final pipeline preflight got no training rows.") from _exc
        _preflight_batch = move_batch(_preflight_raw)
        _required = {"evi", "lst", "s1", "seq", "seq_mask", "static", "anchor"}
        _missing = _required.difference(_preflight_batch)
        if _missing:
            raise RuntimeError(
                f"Final pipeline preflight is missing model inputs {sorted(_missing)}; "
                f"available keys={sorted(_preflight_batch)}"
            )
        _was_training = model.training
        _preflight_prediction = None
        model.eval()
        try:
            with torch.no_grad(), torch.autocast(
                device_type="cuda", dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda"),
            ):
                _preflight_prediction = model(_preflight_batch)
            if not torch.isfinite(_preflight_prediction).all():
                raise RuntimeError("Final pipeline preflight produced non-finite predictions.")
            print("Final raster-cache -> model preflight PASSED.")
        finally:
            model.train(_was_training)
            del _preflight_prediction, _preflight_batch, _preflight_raw, _preflight_loader
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()

    if AUTO_TUNE_RESOURCES and AUTO_TUNE_WORKERS:
        NUM_WORKERS = tune_dataloader_workers("environment_only")
    if RUN_RUNTIME_LOADER_BENCHMARK:
        runtime_benchmark_results = run_runtime_loader_benchmark("location_history")
        print("Runtime loader benchmark completed. Production settings remain unchanged; use the table to choose the next measured configuration.")
    search_summaries = []
    best_artifacts = {}
    for mode in ["location_history", "environment_only"]:
        TASK_MODE = mode
        r2_target = R2_TARGETS[mode]
        print(f"\n========== OPTUNA SEARCH: {mode} (target test R²={r2_target:.2f}) ==========")
        study_path = RUN_DIR / f"optuna_filtered_s1_{mode}{OPTUNA_STUDY_SUFFIX}.sqlite3"
        study = optuna.create_study(
            study_name=f"global_soil_{mode}_filtered_s1{OPTUNA_STUDY_SUFFIX}", direction="maximize",
            storage=f"sqlite:///{study_path}", load_if_exists=True,
            sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=2),
        )
        remaining_trials = max(0, int(OPTUNA_MAX_TOTAL_TRIALS_PER_MODE) - len(study.trials))
        study.optimize(lambda trial, _mode=mode: _objective_with_diagnostics(trial, _mode),
                       n_trials=min(OPTUNA_TRIALS_PER_MODE, remaining_trials), gc_after_trial=True, show_progress_bar=True,
                       callbacks=[optuna_live_callback], catch=(Exception,))
        if PLOT_PROGRESS_EVERY_TRIAL and study.trials:
            _plot_path = plot_optuna_progress(study, mode)
            with mlflow.start_run(run_name=f"optuna_summary_{mode}", tags={"phase": "optuna_summary", "task_mode": mode}):
                mlflow.log_params({"study_name": study.study_name, "study_suffix": OPTUNA_STUDY_SUFFIX,
                                   "completed_trials": sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)})
                try:
                    _plot_best_value = float(study.best_value)
                except ValueError:
                    _plot_best_value = None
                if _plot_best_value is not None and np.isfinite(_plot_best_value):
                    mlflow.log_metric("best_test_r2", _plot_best_value)
                    _best_rmse = (study.best_trial.user_attrs.get("best_test_metrics", {}) or {}).get("rmse")
                    if _best_rmse is not None and np.isfinite(float(_best_rmse)):
                        mlflow.log_metric("best_test_rmse", float(_best_rmse))
                if Path(_plot_path).exists():
                    mlflow.log_artifact(str(_plot_path), artifact_path="plots")
        try:
            best = study.best_trial
        except ValueError:
            print("No completed Optuna trial for", mode, "— see optuna_trial_errors_filtered_s1.jsonl")
            continue
        params = dict(best.params)
        params["batch_size"] = int(best.user_attrs.get("effective_batch_size", BATCH_SIZE))
        params["safe_microbatch_ceiling"] = int(best.user_attrs.get("safe_microbatch_ceiling", params["batch_size"]))
        params["loader_microbatch_size"] = int(best.user_attrs.get("loader_microbatch_size", min(TRAIN_MICROBATCH_CAP, params["batch_size"])))
        params["accumulation_steps"] = int(best.user_attrs.get("accumulation_steps", 1))

        print("Best fast-search candidate:", {"trial": best.number, "proxy_test_r2": best.value,
              "params": params})
        if not bool(globals().get("RUN_FINAL_RETRAIN", False)):
            # Keep the fast local pass fast: do not immediately launch a full-data, multi-epoch fit.
            # To fit the selected configuration, set RUN_FINAL_RETRAIN=True and rerun this cell;
            # the persistent Optuna DB will be reused and already-completed trials will not repeat.
            search_summaries.append({
                "task_mode": mode, "best_trial": best.number, "optuna_best_proxy_test_r2": best.value,
                "trial_epochs": int(TRIAL_EPOCHS), "trial_max_train_batches": TRIAL_MAX_TRAIN_BATCHES,
                "trial_max_test_batches": TRIAL_MAX_TEST_BATCHES,
                "loader_workers": int(NUM_WORKERS), **{f"param_{k}": v for k, v in params.items()},
                "final_retrain_skipped": True, "checkpoint": "not created in fast-search-only mode"
            })
            print(f"[FAST LOCAL MODE] Skipping full-data retraining for {mode}. "
                  "Set RUN_FINAL_RETRAIN=True and rerun this cell when ready for the longer fit.")
            continue

        TASK_MODE = mode
        TARGET_MODE = params["target_mode"]
        RESIDUAL_WEIGHTING = bool(params.get("residual_weighting", False))
        LOSS_FUNCTION = str(params.get("loss_function", "huber"))
        RESIDUAL_HUBER_DELTA = float(params.get("huber_delta", RESIDUAL_HUBER_DELTA))
        GRAD_CLIP_NORM = float(params["grad_clip_norm"])
        backbone = params["temporal_backbone"]

        # Start with the full-retrain target, NOT the trial's physical microbatch.
        # The trial's `safe_microbatch_ceiling` was measured during short proxy training
        # and can be overly conservative (e.g. 512 -> 4,330 loader batches/epoch).
        # A fresh probe of the exact winning architecture is run below after train_ds exists.
        requested_microbatch = min(int(TRAIN_MICROBATCH_CAP),
                                    int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE))
        requested_microbatch = max(int(MIN_GPU_BATCH_SIZE), requested_microbatch)
        microbatch_size = 2 ** int(math.floor(math.log2(requested_microbatch)))
        microbatch_size = max(int(MIN_GPU_BATCH_SIZE), min(microbatch_size, requested_microbatch))
        effective_batch = max(int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE), microbatch_size)
        accum_steps = max(1, int(math.ceil(effective_batch / microbatch_size)))
        BATCH_SIZE = effective_batch

        # Full retraining makes thousands of updates per epoch, unlike the tiny proxy trial.
        # A conservative LR reduction plus EMA reduces large epoch-to-epoch test-R² swings.
        trial_effective_batch = int(best.user_attrs.get("effective_batch_size", params.get("batch_size", effective_batch)))
        final_lr = max(1e-6, float(params["learning_rate"]) * float(FINAL_RETRAIN_LR_MULTIPLIER))
        final_weight_decay = max(float(params["weight_decay"]), float(FINAL_RETRAIN_MIN_WEIGHT_DECAY))
        LEARNING_RATE = final_lr
        WEIGHT_DECAY = final_weight_decay
        best_model = build_model(
            mode, backbone, TARGET_MODE,
            spatial_width=int(params["spatial_width"]),
            temporal_hidden=int(params["temporal_hidden"]),
            fusion_width=int(params["fusion_width"]),
        )

        # Resume from the best selected-R² checkpoint first, not the latest epoch.
        # A later epoch can have worse test R²; using its live weights would discard
        # the best model after Ctrl-C. The best file is written atomically on improvement.
        warm_payload = None
        _resume_latest_path = CHECKPOINT_DIR / f"{mode}_latest_model_v19.pt"
        _resume_best_path = CHECKPOINT_DIR / f"{mode}_best_model_v19.pt"
        _expected_arch = {"spatial_width": int(params["spatial_width"]),
                          "temporal_hidden": int(params["temporal_hidden"]),
                          "fusion_width": int(params["fusion_width"])}
        _resume_param_keys = ("optimizer", "loss_function", "residual_weighting", "huber_delta",
                              "grad_clip_norm", "learning_rate", "weight_decay")

        def _resume_candidate_is_compatible(_candidate):
            _candidate_arch = _candidate.get("architecture_params", {}) or {}
            _candidate_params = _candidate.get("best_trial_params", {}) or {}
            _same_arch = (not _candidate_arch) or all(
                int(_candidate_arch.get(k, -1)) == v for k, v in _expected_arch.items()
            )
            _same_training_params = (not _candidate_params) or all(
                k in params and _candidate_params.get(k) == params[k] for k in _resume_param_keys
            )
            return bool(
                _candidate.get("training_version") == "model_v19" and
                _candidate.get("task_mode") == mode and
                _candidate.get("target_mode") == TARGET_MODE and
                _candidate.get("temporal_backbone") == backbone and
                _same_arch and _same_training_params and
                isinstance(_candidate.get("model_state_dict"), dict)
            )

        if bool(AUTO_RESUME_FINAL_RETRAIN):
            # Best checkpoint has priority. Latest is only a compatibility fallback.
            for _resume_path in (_resume_best_path, _resume_latest_path):
                if not _resume_path.is_file():
                    continue
                try:
                    _candidate = torch.load(_resume_path, map_location="cpu", weights_only=False)
                    if not _resume_candidate_is_compatible(_candidate):
                        print(f"[RESUME {mode}] skipping incompatible checkpoint: {_resume_path.name}", flush=True)
                        continue
                    if _resume_path == _resume_latest_path:
                        _embedded_best = _candidate.get("best_model_state_dict")
                        if _embedded_best is not None:
                            _candidate["model_state_dict"] = _embedded_best
                            _candidate["epoch"] = int(_candidate.get("best_epoch", _candidate.get("epoch", 0)))
                            _candidate["optimizer_state_dict"] = None
                            _candidate["scaler_state_dict"] = None
                            print(f"[RESUME {mode}] using embedded best weights from latest checkpoint.", flush=True)
                        else:
                            print(f"[RESUME {mode}] WARNING: best checkpoint is unavailable; falling back to latest "
                                  "weights for this legacy run.", flush=True)
                    best_model.load_state_dict(_candidate["model_state_dict"], strict=True)
                    warm_payload = _candidate
                    print(f"[RESUME {mode}] loaded selected BEST checkpoint: "
                          f"epoch={int(warm_payload.get('best_epoch', warm_payload.get('epoch', 0)))}, "
                          f"best_test_R2={float(warm_payload.get('best_test_r2', float('nan'))):.4f}; "
                          f"source={_resume_path.name}. Resuming from the epoch after this best checkpoint.",
                          flush=True)
                    break
                except Exception as _resume_exc:
                    print(f"[RESUME {mode}] could not load {_resume_path.name} "
                          f"({type(_resume_exc).__name__}: {_resume_exc}); trying fallback.", flush=True)
        if warm_payload is None:
            print(f"Fresh v19 initialization [{mode}]: spatial={params['spatial_width']}, "
                  f"temporal={params['temporal_hidden']}, fusion={params['fusion_width']}; "
                  "no compatible best checkpoint was found.", flush=True)

        optimizer = _make_optimizer(params["optimizer"], best_model.parameters(),
                                    lr=final_lr, weight_decay=final_weight_decay)
        lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=float(FINAL_SCHEDULER_FACTOR),
            patience=int(FINAL_SCHEDULER_PATIENCE), threshold=2e-4,
            threshold_mode="abs", min_lr=float(FINAL_SCHEDULER_MIN_LR)
        )
        ema_model = copy.deepcopy(best_model).to(device).eval()
        ema_model.requires_grad_(False)
        ema_updates = int(warm_payload.get("ema_updates", 1)) if warm_payload is not None else 0
        if warm_payload is not None:
            ema_model.load_state_dict(best_model.state_dict(), strict=True)

        train_ds = get_global_dataset("train", task_mode=mode, target_mode=TARGET_MODE,
            max_rows=MAX_ROWS_PER_SPLIT, shuffle=True, seed=SEED)
        test_selection_ds = get_global_dataset("test", task_mode=mode, target_mode=TARGET_MODE,
            max_rows=MAX_ROWS_PER_SPLIT, shuffle=False, seed=SEED)
        # IMPORTANT v19 fix: re-probe the selected architecture for the full-data run.
        # Do not trust the proxy trial's cached/serialized batch ceiling; it can cause a
        # 4x increase in batches per epoch and a severe throughput collapse.
        if device.type == "cuda" and AUTO_GPU_BATCH_SIZE:
            final_safe_microbatch = int(probe_gpu_batch_limit(
                best_model, train_ds, max_batch=min(int(TRAIN_MICROBATCH_CAP),
                int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE)), min_batch=int(MIN_GPU_BATCH_SIZE)))
            final_probe_peak = int(globals().get("_GPU_PROBE_PEAK_DELTA_BYTES", 0) or 0)
            requested_microbatch = min(int(TRAIN_MICROBATCH_CAP),
                int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE), final_safe_microbatch)
            requested_microbatch = max(int(MIN_GPU_BATCH_SIZE), requested_microbatch)
            microbatch_size = 2 ** int(math.floor(math.log2(requested_microbatch)))
            microbatch_size = max(int(MIN_GPU_BATCH_SIZE), min(microbatch_size, requested_microbatch))
            globals()["_GPU_SHARD_CURRENT_PROBE_PEAK_BYTES"] = final_probe_peak
            print(f"[v19 final-batch probe {mode}] fresh_safe_microbatch={final_safe_microbatch}; "
                  f"selected_microbatch={microbatch_size}; effective_batch={effective_batch}; "
                  f"accumulation={max(1, int(math.ceil(effective_batch / microbatch_size)))}; "
                  f"torch_cuda_allocated={torch.cuda.memory_allocated(device)/_GIB:.3f}GiB; "
                  f"torch_cuda_reserved={torch.cuda.memory_reserved(device)/_GIB:.3f}GiB")
        else:
            microbatch_size = min(int(TRAIN_MICROBATCH_CAP), int(FINAL_RETRAIN_EFFECTIVE_BATCH_SIZE))
            globals()["_GPU_SHARD_CURRENT_PROBE_PEAK_BYTES"] = 0
        accum_steps = max(1, int(math.ceil(effective_batch / microbatch_size)))
        BATCH_SIZE = effective_batch
        if next(best_model.parameters()).device != device:
            raise RuntimeError(f"Model/device mismatch: model={next(best_model.parameters()).device}, expected={device}")
        print(f"[v19 device check {mode}] device={device}; model_device={next(best_model.parameters()).device}; "
              f"cuda_available={torch.cuda.is_available()}; microbatch={microbatch_size}; "
              f"loader_batches/epoch≈{math.ceil(len(train_ds)/microbatch_size):,}")
        train_loader = build_global_loader(train_ds, microbatch_size)
        eval_batch_size = min(int(EVAL_BATCH_SIZE), int(microbatch_size))
        # Critical split guard: epoch selection MUST use the held-out test dataset.
        # v14 accidentally passed train_ds here, producing in-sample scores.
        if test_selection_ds is train_ds or bool(test_selection_ds.shuffle):
            raise RuntimeError("Test-selection dataset is not a distinct, ordered test split.")
        test_selection_loader = build_global_loader(test_selection_ds, eval_batch_size,
            pin_memory=PIN_MEMORY, num_workers=EVAL_NUM_WORKERS, persistent_workers=False,
            eval_max_batches=FINAL_TEST_SELECTION_MAX_BATCHES)
        scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
        _optimizer_state_restored = False
        _scheduler_state_restored = False
        _scaler_state_restored = False
        if warm_payload is not None:
            # Resume weights are the selected EMA snapshot. Optimizer moments from a
            # later live/latest model do not correspond exactly, so reset those moments.
            _saved_scheduler_state = warm_payload.get("scheduler_state_dict")
            if _saved_scheduler_state is not None:
                try:
                    lr_scheduler.load_state_dict(_saved_scheduler_state)
                    _scheduler_state_restored = True
                except Exception as _sched_resume_exc:
                    print(f"[RESUME {mode}] scheduler state could not be restored: "
                          f"{type(_sched_resume_exc).__name__}: {_sched_resume_exc}", flush=True)
            _saved_scaler_state = warm_payload.get("scaler_state_dict")
            if _saved_scaler_state:
                try:
                    scaler.load_state_dict(_saved_scaler_state)
                    _scaler_state_restored = True
                except Exception as _scaler_resume_exc:
                    print(f"[RESUME {mode}] GradScaler state could not be restored: "
                          f"{type(_scaler_resume_exc).__name__}: {_scaler_resume_exc}", flush=True)
            _saved_resume_lr = warm_payload.get("resume_learning_rate")
            if _saved_resume_lr is not None and np.isfinite(float(_saved_resume_lr)) and float(_saved_resume_lr) > 0:
                for _group in optimizer.param_groups:
                    _group["lr"] = float(_saved_resume_lr)
                print(f"[RESUME {mode}] restored best-checkpoint learning rate={float(_saved_resume_lr):.3g}.", flush=True)

        # Adaptive compilation benchmark on the selected full-retrain architecture and
        # actual physical microbatch. The compiled wrapper shares `best_model` parameters.
        _final_forward_model = None
        _compile_done_modes = globals().setdefault("_TORCH_COMPILE_BENCHMARK_DONE_MODES", set())
        if bool(TORCH_COMPILE_ENABLED) and bool(RUN_TORCH_COMPILE_BENCHMARK) and mode not in _compile_done_modes:
            import sysconfig
            _compile_include = sysconfig.get_paths().get("include")
            _compile_header = (Path(_compile_include) / "Python.h") if _compile_include else None
            if _compile_header is None or not _compile_header.is_file():
                print("[COMPILE] matching Python.h not found; keeping eager training.", flush=True)
                _compile_done_modes.add(mode)
            else:
                _compile_iter = iter(train_loader)
                try:
                    _compile_raw_batch = next(_compile_iter)
                    _compile_device_batch = move_batch(_compile_raw_batch)
                    _compile_result = benchmark_torch_compile_training_step(
                        best_model, _compile_device_batch, TARGET_MODE,
                        max_samples=min(int(COMPILE_BENCHMARK_MAX_SAMPLES), int(microbatch_size)),
                        steps=COMPILE_BENCHMARK_STEPS)
                    if _compile_result and _compile_result.get("compiled_model") is not None:
                        _final_forward_model = _compile_result["compiled_model"]
                    _compile_done_modes.add(mode)
                except StopIteration:
                    print("[COMPILE] skipped: training loader was empty; eager training will be used.", flush=True)
                    _compile_done_modes.add(mode)
                except Exception as _compile_exc:
                    print(f"[COMPILE] preflight failed ({type(_compile_exc).__name__}: {_compile_exc}); "
                          "eager training will be used.", flush=True)
                    _compile_done_modes.add(mode)
                finally:
                    try: del _compile_iter, _compile_raw_batch, _compile_device_batch
                    except Exception: pass
                    gc.collect()
                    if device.type == "cuda" and _final_forward_model is None:
                        torch.cuda.empty_cache()
        if _final_forward_model is not None:
            if bool(COMPILE_STABLE_GRAD_BUFFERS):
                _ensure_stable_parameter_grad_buffers(best_model)
                optimizer.zero_grad(set_to_none=False)
            print(f"[TRAINING COMPUTE] torch.compile(mode='{globals().get('_TORCH_COMPILE_ACTIVE_MODE', COMPILE_BENCHMARK_MODE)}') is ACTIVE; "
                  "stable parameter-gradient buffers are pinned and CUDAGraph iteration markers are enabled for accumulation.", flush=True)
        else:
            print("[TRAINING COMPUTE] eager forward/backward path is ACTIVE for this task mode.", flush=True)
        if warm_payload is not None:
            # Same-v19 resume: keep metrics from the prior epoch and continue at epoch+1.
            # A fresh optimizer is created because the earlier checkpoint predates optimizer-state saves.
            best_test_score = float(warm_payload.get("best_test_r2", -float("inf")))
            best_test_metrics = (warm_payload.get("best_test_metrics") or warm_payload.get("test_selection_metrics") or warm_payload.get("last_epoch_test_metrics"))
            best_epoch = int(warm_payload.get("best_epoch", warm_payload.get("epoch", 0)))
            start_epoch = max(1, int(best_epoch) + 1)
            _saved_smooth_score = warm_payload.get("smooth_scheduler_score")
            smooth_scheduler_score = (
                float(_saved_smooth_score) if _saved_smooth_score is not None and np.isfinite(float(_saved_smooth_score))
                else (best_test_score if np.isfinite(best_test_score) else None)
            )
            stale_epochs = int(warm_payload.get("stale_epochs", 0))
            regression_streak = int(warm_payload.get("regression_streak", 0))
            best_state = warm_payload.get("model_state_dict")
            if best_state is None:
                best_state = _cpu_state_dict(ema_model.state_dict())
            # Ensure both live model and EMA start from the selected best R² weights.
            best_model.load_state_dict(best_state, strict=True)
            ema_model.load_state_dict(best_state, strict=True)
            print(f"[RESUME {mode}] restored BEST epoch={best_epoch}, best_test_R2={best_test_score:.4f}; "
                  f"next_epoch={start_epoch}; optimizer_moments_reset=True; "
                  f"scheduler_state_restored={_scheduler_state_restored}; "
                  f"scaler_state_restored={_scaler_state_restored}.", flush=True)
        else:
            best_test_score, best_test_metrics, best_state = -float("inf"), None, None
            best_epoch, start_epoch, smooth_scheduler_score = 0, 1, None
            stale_epochs = 0
            regression_streak = 0

        best_checkpoint_path = CHECKPOINT_DIR / f"{mode}_best_model_v19.pt"
        latest_checkpoint_path = CHECKPOINT_DIR / f"{mode}_latest_model_v19.pt"
        # Preserve the authoritative best checkpoint on startup. Only create a fallback
        # file when recovering a legacy run whose best checkpoint is missing.
        if best_state is not None and not best_checkpoint_path.is_file():
            _atomic_torch_save({
                "model_state_dict": best_state, "task_mode": mode, "target_mode": TARGET_MODE,
                "temporal_backbone": backbone, "best_test_r2": best_test_score,
                "best_test_metrics": best_test_metrics, "best_epoch": best_epoch,
                "epoch": best_epoch, "best_trial_params": params,
                "architecture_params": _expected_arch,
                "training_version": "model_v19", "test_used_for_selection": True,
            }, best_checkpoint_path)

        mlflow.start_run(run_name=f"final_retrain_{mode}", tags={"phase": "final_retrain", "task_mode": mode,
                                                                  "study_suffix": OPTUNA_STUDY_SUFFIX,
                                                                  "training_version": "model_v19"})
        _mlflow_final_params = {k: v for k, v in params.items()
                                if k not in {"safe_microbatch_ceiling", "loader_microbatch_size", "accumulation_steps"}}
        mlflow.log_params({**{str(k): (str(v) if isinstance(v, (list, tuple, dict)) else v) for k, v in _mlflow_final_params.items()},
                           "selected_optuna_trial": int(best.number), "target_mode": str(TARGET_MODE),
                           "residual_weighting": bool(RESIDUAL_WEIGHTING), "loss_function": str(LOSS_FUNCTION),
                           "residual_huber_delta": float(RESIDUAL_HUBER_DELTA),
                           "final_retrain_epochs_max": int(FINAL_RETRAIN_EPOCHS),
                           "final_effective_batch_size": int(effective_batch),
                           "final_microbatch_size": int(microbatch_size),
                           "final_accumulation_steps": int(accum_steps),
                           "final_learning_rate": float(final_lr),
                           "warm_start_lr_multiplier": float(FINAL_RETRAIN_WARM_START_LR_MULTIPLIER),
                           "rollback_r2_drop": float(FINAL_ROLLBACK_R2_DROP),
                           "rollback_consecutive_epochs": int(FINAL_ROLLBACK_CONSECUTIVE_EPOCHS),
                           "final_weight_decay": float(final_weight_decay),
                           "optimizer_implementation": str(getattr(optimizer, "_speed_impl", "standard")),
                           "ema_decay": float(FINAL_EMA_DECAY),
                           "evaluation_batch_size": int(eval_batch_size)})

        print(f"\n--- Speed/stability retraining [{mode}] ---")
        print(f"Optuna effective batch={trial_effective_batch}; final effective batch={effective_batch}; "
              f"physical microbatch={microbatch_size}; accumulation={accum_steps}; "
              f"workers={NUM_WORKERS}; optimizer={getattr(optimizer, '_speed_impl', 'standard')}")
        print(f"Final LR={final_lr:.3g} (Optuna LR={float(params['learning_rate']):.3g}; "
              f"final-retrain multiplier={FINAL_RETRAIN_LR_MULTIPLIER}); "
              f"weight decay={final_weight_decay:.3g}; EMA decay={FINAL_EMA_DECAY:.2f}")
        print("FULL TRAIN SPLIT ENABLED:", len(train_loader), "loader batches/epoch; max_batches=", FINAL_RETRAIN_MAX_TRAIN_BATCHES)
        retrain_epoch_bar = tqdm(range(start_epoch, FINAL_RETRAIN_EPOCHS + 1),
                                 desc=f"Retrain [{mode}]", dynamic_ncols=True, mininterval=0.5,
                                 position=0, leave=True)
        for epoch in retrain_epoch_bar:
            _epoch_wall_started = time.perf_counter()
            retrain_epoch_bar.set_postfix(status=f"training E{epoch}", best_R2=f"{best_test_score:.4f}")
            try:
                train_metrics = _train_epoch_accumulated(
                    best_model, train_loader, optimizer, scaler, TARGET_MODE, effective_batch, microbatch_size,
                    GRAD_CLIP_NORM, desc=f"Retrain {mode} E{epoch} batches",
                    max_batches=FINAL_RETRAIN_MAX_TRAIN_BATCHES, forward_model=_final_forward_model)
            except KeyboardInterrupt:
                print(f"\n[INTERRUPTED {mode}] stopped during epoch {epoch}; the partial epoch was discarded. "
                      f"Next launch will restore best selected-R² checkpoint E{best_epoch} "
                      f"(R²={best_test_score:.4f}) from {best_checkpoint_path}.", flush=True)
                raise
            expected_train_batches = len(train_loader)
            if FINAL_RETRAIN_MAX_TRAIN_BATCHES is None and int(train_metrics.get("train_batches", -1)) != expected_train_batches:
                raise RuntimeError(
                    f"Full-data coverage check failed for {mode} epoch {epoch}: trained on "
                    f"{train_metrics.get('train_batches')} batches, expected {expected_train_batches}."
                )
            print(f"[Training coverage {mode} E{epoch}] processed {train_metrics.get('train_batches')} / "
                  f"{expected_train_batches} training batches ({train_metrics.get('samples'):,} samples).")

            # EMA starts from the trained weights on a fresh run; warm-start continuation
            # applies EMA immediately so the previous best checkpoint is not discarded.
            if ema_updates == 0:
                ema_model.load_state_dict(best_model.state_dict())
            else:
                _update_ema_model(ema_model, best_model, FINAL_EMA_DECAY)
            ema_updates += 1
            ema_model.eval()
            epoch_test_metrics = evaluate_predictions(ema_model, test_selection_loader, TARGET_MODE,
                max_batches=FINAL_TEST_SELECTION_MAX_BATCHES,
                desc=f"{mode} epoch {epoch} EMA test subset")
            score = float(epoch_test_metrics.get("r2", np.nan))
            _test_scalars = {"test_r2": score, "test_rmse": epoch_test_metrics.get("rmse", np.nan)}
            _test_scalars = {k: float(v) for k, v in _test_scalars.items()
                             if v is not None and np.isfinite(float(v))}
            if _test_scalars:
                mlflow.log_metrics(_test_scalars, step=epoch)

            if np.isfinite(score):
                smooth_scheduler_score = score if smooth_scheduler_score is None else (
                    0.5 * float(score) + 0.5 * float(smooth_scheduler_score))
                lr_scheduler.step(float(smooth_scheduler_score))

            improved = np.isfinite(score) and score > best_test_score + FINAL_MIN_DELTA
            if improved:
                best_test_score = score
                best_test_metrics = epoch_test_metrics
                best_epoch = epoch
                best_state = _cpu_state_dict(ema_model.state_dict())
                stale_epochs = 0
                best_payload = {
                    "model_state_dict": best_state, "task_mode": mode, "target_mode": TARGET_MODE,
                    "temporal_backbone": backbone, "loss_function": LOSS_FUNCTION,
                    "architecture_version": "multiscale_residual_spatial_temporal_attention_gated_fusion_v19",
                    "best_trial_number": int(best.number), "best_trial_params": params,
                    "architecture_params": {"spatial_width": int(params["spatial_width"]),
                                             "temporal_hidden": int(params["temporal_hidden"]),
                                             "fusion_width": int(params["fusion_width"])},
                    "best_test_r2": float(best_test_score), "best_test_metrics": best_test_metrics,
                    "best_epoch": int(best_epoch), "epoch": int(epoch),
                    "final_learning_rate": float(final_lr), "final_weight_decay": float(final_weight_decay),
                    "final_effective_batch_size": int(effective_batch),
                    "loader_microbatch_size": int(microbatch_size), "accumulation_steps": int(accum_steps),
                    "ema_decay": float(FINAL_EMA_DECAY), "training_version": "model_v19",
                    "ema_updates": int(ema_updates),
                    "scheduler_state_dict": lr_scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "smooth_scheduler_score": smooth_scheduler_score,
                    "resume_learning_rate": float(optimizer.param_groups[0]["lr"]),
                    "test_used_for_selection": True, "manifest_config": manifest_config,
                }
                _atomic_torch_save(best_payload, best_checkpoint_path)
            else:
                stale_epochs += 1

            # The previous loop retained the best checkpoint but continued optimizing
            # from deteriorated live weights. Roll back after two consecutive large
            # regressions on the same deterministic test subset, then restart Adam
            # moments and lower the current LR before continuing.
            severe_regression = (
                np.isfinite(score) and best_state is not None and
                score <= float(best_test_score) - float(FINAL_ROLLBACK_R2_DROP)
            )
            regression_streak = regression_streak + 1 if severe_regression else 0
            if regression_streak >= int(FINAL_ROLLBACK_CONSECUTIVE_EPOCHS):
                best_model.load_state_dict(best_state, strict=True)
                ema_model.load_state_dict(best_state, strict=True)
                ema_model.eval()
                optimizer.state.clear()  # reset stale Adam/AdamW moments after rollback
                optimizer.zero_grad(set_to_none=(_final_forward_model is None))
                for _group in optimizer.param_groups:
                    _group["lr"] = max(
                        float(_group["lr"]) * float(FINAL_ROLLBACK_LR_FACTOR),
                        float(FINAL_SCHEDULER_MIN_LR),
                    )
                smooth_scheduler_score = float(best_test_score)
                regression_streak = 0
                print(
                    f"[ROLLBACK {mode} E{epoch}] restored best checkpoint from E{best_epoch}; "
                    f"R²={best_test_score:.4f}; reduced LR to "
                    f"{optimizer.param_groups[0]['lr']:.3g}; reset optimizer moments."
                )

            if SAVE_LATEST_EPOCH_CHECKPOINT:
                _atomic_torch_save({
                    "model_state_dict": _cpu_state_dict(best_model.state_dict()),
                    "ema_model_state_dict": _cpu_state_dict(ema_model.state_dict()),
                    "task_mode": mode, "target_mode": TARGET_MODE, "temporal_backbone": backbone,
                    "epoch": int(epoch), "best_epoch": int(best_epoch),
                    "best_test_r2": float(best_test_score), "best_test_metrics": best_test_metrics,
                    "last_epoch_test_metrics": epoch_test_metrics,
                    "best_trial_params": params,
                    "architecture_params": {"spatial_width": int(params["spatial_width"]),
                                             "temporal_hidden": int(params["temporal_hidden"]),
                                             "fusion_width": int(params["fusion_width"])},
                    # Preserve training dynamics on future restarts. v19.1 checkpoints lack
                    # these fields, but v19.2 writes them after every completed epoch.
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": lr_scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "smooth_scheduler_score": smooth_scheduler_score,
                    "stale_epochs": int(stale_epochs),
                    "regression_streak": int(regression_streak),
                    "ema_updates": int(ema_updates),
                    "training_version": "model_v19",
                }, latest_checkpoint_path)

            retrain_epoch_bar.set_postfix(
                train_loss=f"{train_metrics['train_loss']:.4f}",
                samples_s=f"{train_metrics['samples_per_sec']:.1f}",
                best_test_r2=f"{best_test_score:.4f}",
                test_r2=f"{score:.4f}" if np.isfinite(score) else "N/A",
                lr=f"{optimizer.param_groups[0]['lr']:.2g}",
                gpu=(f"{train_metrics['gpu_util_pct']:.0f}%" if train_metrics['gpu_util_pct'] is not None else "n/a"),
                vram_peak=(f"{train_metrics['gpu_alloc_gib']:.2f}G" if train_metrics['gpu_alloc_gib'] is not None else "n/a"))
            _epoch_wall_elapsed_sec = max(0.0, time.perf_counter() - _epoch_wall_started)
            print(f"[Retrain {mode} epoch {epoch}] wall_epoch={_epoch_wall_elapsed_sec/60.0:.2f}min "
                  f"(train={train_metrics['train_elapsed_sec']/60.0:.2f}min) "
                  f"train_loss={train_metrics['train_loss']:.5f} "
                  f"samples/s={train_metrics['samples_per_sec']:.1f} queue_wait/b={train_metrics['data_wait_per_batch_sec']:.3f}s "
                  f"test_R2={score:.4f} raw_test_R2={epoch_test_metrics.get('raw_r2', float('nan')):.4f} "
                  f"RMSE={epoch_test_metrics['rmse']:.4f} MAE={epoch_test_metrics['mae']:.4f} "
                  f"LR={optimizer.param_groups[0]['lr']:.3g} GPU_avg={train_metrics['gpu_util_pct']}% "
                  f"VRAM_peak={train_metrics['gpu_alloc_gib']}GiB RAM_peak={train_metrics['host_ram_pct']}%")
            print(f"[Pipeline {mode} E{epoch}] collate={train_metrics['collate_total_ms_per_batch']:.1f}ms/b "
                  f"(metadata={train_metrics['collate_meta_ms_per_batch']:.1f}, raster={train_metrics['collate_raster_ms_per_batch']:.1f}); "
                  f"DataLoader next={train_metrics['loader_source_wait_ms_per_batch']:.1f}ms/b; "
                  f"H2D enqueue={train_metrics['h2d_enqueue_ms_per_batch']:.2f}ms/b "
                  f"CUDA-copy p50/p95={train_metrics['h2d_cuda_ms_p50']:.2f}/{train_metrics['h2d_cuda_ms_p95']:.2f}ms; "
                  f"GPU fwd+bwd p50/p95={train_metrics['gpu_compute_ms_p50']:.2f}/{train_metrics['gpu_compute_ms_p95']:.2f}ms; "
                  f"CPU launch={1000.0*train_metrics['host_step_sec_per_batch']:.1f}ms/b; "
                  f"pinned tensors={train_metrics['input_pinned_tensor_fraction']}")
            _raster_rows = int(train_metrics.get("raster_source_rows_read", 0))
            _raster_requested = int(train_metrics.get("raster_source_rows_requested", 0))
            if RAM_ZARR_CACHE_ACTIVE:
                print(f"[Raster gather {mode} E{epoch}] direct_mmap_rows={_raster_rows:,}/"
                      f"{_raster_requested:,}; gathered/requested="
                      f"{train_metrics.get('raster_read_amplification', float('nan')):.3f}; "
                      f"storage=raw_numpy_mmap (Zarr chunk count not applicable)")
            else:
                print(f"[Raster gather {mode} E{epoch}] Zarr_rows={_raster_rows:,}/"
                      f"{_raster_requested:,}; estimated_native_chunks="
                      f"{train_metrics.get('raster_source_chunks_read', 0):,}; "
                      f"workers={NUM_WORKERS}")

            if FINAL_EARLY_STOPPING_PATIENCE and stale_epochs >= FINAL_EARLY_STOPPING_PATIENCE:
                print(f"Early stopping after {stale_epochs} test epochs without improvement. Best epoch={best_epoch}.")
                break

        if best_state is None:
            raise RuntimeError(f"No finite test R² was produced for {mode}; no best checkpoint is available.")
        best_model.load_state_dict(best_state)
        del ema_model
        print(f"\n[{mode}] Running one full held-out TEST metric pass AFTER test-based model selection...")
        # The test partition is used directly for trial scoring and epoch/checkpoint selection, per the requested protocol.
        del test_selection_loader
        del test_selection_ds
        gc.collect()
        test_ds = get_global_dataset("test", task_mode=mode, target_mode=TARGET_MODE,
            max_rows=MAX_ROWS_PER_SPLIT, shuffle=False, seed=SEED)
        full_test_loader = build_global_loader(test_ds, min(int(EVAL_BATCH_SIZE), int(microbatch_size)),
            pin_memory=PIN_MEMORY, num_workers=EVAL_NUM_WORKERS, persistent_workers=False, eval_max_batches=None)
        best_full_metrics = evaluate_predictions(best_model, full_test_loader, TARGET_MODE,
            max_batches=FINAL_TEST_MAX_BATCHES, desc=f"{mode} final full test metrics")
        del train_loader, full_test_loader, test_ds
        gc.collect()
        checkpoint_path = CHECKPOINT_DIR / f"{mode}_best_model_v19.pt"
        final_checkpoint = {
            "model_state_dict": _cpu_state_dict(best_model.state_dict()), "task_mode": mode,
            "target_mode": TARGET_MODE, "temporal_backbone": backbone,
            "loss_function": LOSS_FUNCTION,
            "architecture_version": "multiscale_residual_spatial_temporal_attention_gated_fusion_v19",
            "training_version": "model_v19",
            "calibration_coefficients": best_model.calibrator.coefficients.detach().float().cpu().tolist(),
            "best_trial_number": best.number, "best_trial_params": params,
            "architecture_params": {"spatial_width": int(params["spatial_width"]),
                                     "temporal_hidden": int(params["temporal_hidden"]),
                                     "fusion_width": int(params["fusion_width"])},
            "optuna_study_suffix": OPTUNA_STUDY_SUFFIX,
            "best_epoch": int(best_epoch), "best_test_r2": float(best_test_score),
            "final_learning_rate": float(final_lr), "final_weight_decay": float(final_weight_decay),
            "final_effective_batch_size": int(effective_batch), "loader_microbatch_size": int(microbatch_size),
            "accumulation_steps": int(accum_steps), "ema_decay": float(FINAL_EMA_DECAY),
            "test_selection_metrics": best_test_metrics, "test_metrics": best_full_metrics,
            "test_used_for_selection": True, "manifest_config": manifest_config,
        }
        _atomic_torch_save(final_checkpoint, checkpoint_path)
        row = {"task_mode": mode, "best_trial": best.number, "optuna_best_test_r2": best.value,
                 "best_epoch_test_r2": best_test_score,
                 "r2_target": r2_target,
                 "test_r2_target_met": bool(np.isfinite(best_test_score) and best_test_score >= r2_target),
                 "final_test_r2_target_met": bool(np.isfinite(best_full_metrics.get("r2", np.nan)) and best_full_metrics.get("r2", np.nan) >= r2_target),
                "loader_workers": NUM_WORKERS, "effective_batch_size": effective_batch,
                "loader_microbatch_size": microbatch_size, "accumulation_steps": accum_steps,
                "final_learning_rate": final_lr, "final_weight_decay": final_weight_decay,
                "ema_decay": FINAL_EMA_DECAY, "best_epoch": best_epoch,
                "last_epoch_samples_per_sec": train_metrics["samples_per_sec"],
                "last_epoch_train_elapsed_sec": train_metrics["train_elapsed_sec"],
                "last_epoch_wall_elapsed_sec": _epoch_wall_elapsed_sec,
                "last_epoch_data_wait_per_batch_sec": train_metrics["data_wait_per_batch_sec"],
                "last_epoch_gpu_util_pct": train_metrics["gpu_util_pct"],
                "last_epoch_gpu_alloc_gib": train_metrics["gpu_alloc_gib"],
                 "last_epoch_patch_lru_hit_pct": train_metrics.get("patch_lru_hit_pct", np.nan),
                 "last_epoch_patch_lru_hits": train_metrics.get("patch_lru_hits", 0),
                 "last_epoch_patch_lru_misses": train_metrics.get("patch_lru_misses", 0),
                **{f"param_{k}": v for k, v in params.items()},
                 **{f"test_selection_{k}": v for k, v in (best_test_metrics or {}).items()},
                 **{f"test_{k}": v for k, v in (best_full_metrics or {}).items()},
                "checkpoint": str(checkpoint_path)}
        search_summaries.append(row); best_artifacts[mode] = row
        print("Best parameters:", params)
        print("Test-subset metrics used for epoch selection (raw vs calibrated):", best_test_metrics)
        print("Final held-out test metrics (raw vs calibrated):", best_full_metrics)
        print("Calibration R2 change on test:", best_full_metrics.get("calibration_r2_delta", np.nan))
        print(f"Test R² target >= {r2_target:.2f}: {'MET' if best_test_score >= r2_target else 'NOT MET'}")
        print(f"Final held-out test R² >= {r2_target:.2f}: {'MET' if best_full_metrics.get('r2', float('nan')) >= r2_target else 'NOT MET'}")
        print("Saved checkpoint:", checkpoint_path)
        _final_mlflow_metrics = {}
        if np.isfinite(best_test_score):
            _final_mlflow_metrics["best_test_r2"] = float(best_test_score)
        if best_test_metrics and best_test_metrics.get("rmse") is not None and np.isfinite(float(best_test_metrics["rmse"])):
            _final_mlflow_metrics["best_test_rmse"] = float(best_test_metrics["rmse"])
        if best_full_metrics.get("r2") is not None and np.isfinite(float(best_full_metrics["r2"])):
            _final_mlflow_metrics["final_test_r2"] = float(best_full_metrics["r2"])
        if best_full_metrics.get("rmse") is not None and np.isfinite(float(best_full_metrics["rmse"])):
            _final_mlflow_metrics["final_test_rmse"] = float(best_full_metrics["rmse"])
        if _final_mlflow_metrics:
            mlflow.log_metrics(_final_mlflow_metrics)
        if checkpoint_path.exists():
            mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
        mlflow.set_tag("test_used_for_selection", "true")
        mlflow.end_run(status="FINISHED")
    summary_path = RUN_DIR / f"optuna_filtered_s1_test_summary{OPTUNA_STUDY_SUFFIX}.csv"
    pd.DataFrame(search_summaries).to_csv(summary_path, index=False)
    print("\nSearch summary saved:", summary_path)
    if summary_path.exists():
        with mlflow.start_run(run_name="search_summary", tags={"phase": "summary"}):
            mlflow.log_params({"study_suffix": OPTUNA_STUDY_SUFFIX, "run_name": RUN_NAME})
            mlflow.log_artifact(str(summary_path), artifact_path="summaries")
        print("MLflow summary artifact logged:", summary_path)
else:
    print("RUN_OPTUNA_SEARCH=False; Optuna training not started.")

In [ ]:
# =============================== RELOAD / FINAL TEST METRICS ===============================
# The training cell already runs the full held-out test pass once and saves metrics in each checkpoint.
# Do not run another expensive full-test pass unless explicitly requested.
final_rows = []
if RUN_FINAL_TEST_RECHECK:
    print("RUN_FINAL_TEST_RECHECK=True: re-running the full test set (can be expensive).")
    for mode in ["location_history", "environment_only"]:
        checkpoint_path = CHECKPOINT_DIR / f"{mode}_best_model_v19.pt"
        if not checkpoint_path.exists():
            print("No selected checkpoint for", mode, "— run the Optuna cell first.")
            continue
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        task_mode, target_mode, backbone = ckpt["task_mode"], ckpt["target_mode"], ckpt["temporal_backbone"]
        arch = ckpt.get("architecture_params", ckpt.get("best_trial_params", {}))
        test_model = build_model(
            task_mode, backbone, target_mode,
            spatial_width=int(arch.get("spatial_width", MODEL_SPATIAL_WIDTH)),
            temporal_hidden=int(arch.get("temporal_hidden", MODEL_TEMPORAL_HIDDEN)),
            fusion_width=int(arch.get("fusion_width", MODEL_FUSION_WIDTH)),
        )
        test_model.load_state_dict(ckpt["model_state_dict"])
        ds = get_global_dataset("test", task_mode=task_mode, target_mode=target_mode,
            max_rows=MAX_ROWS_PER_SPLIT, shuffle=False, seed=SEED)
        batch_size = int(ckpt["best_trial_params"].get("loader_microbatch_size", min(TRAIN_MICROBATCH_CAP, int(ckpt["best_trial_params"]["batch_size"]))))
        loader = build_global_loader(ds, batch_size,
            num_workers=min(int(globals().get("EVAL_NUM_WORKERS", 2)), int(NUM_WORKERS)),
            persistent_workers=False, eval_max_batches=None)
        metrics = evaluate_predictions(test_model, loader, target_mode, max_batches=None,
            desc=f"Final re-evaluation {mode}")
        final_rows.append({"task_mode": task_mode, "target_mode": target_mode, "backbone": backbone,
                           "best_trial": ckpt.get("best_trial_number"), **metrics,
                           "NMRSE_scale_1": metrics.get("nrmse"),
                           "calibration_coefficients": ckpt.get("calibration_coefficients")})
        print(f"\n{mode} — test (raw vs calibrated):", metrics)
        del loader, ds, test_model
        gc.collect()
else:
    print("Using saved final-test metrics from the training checkpoint; set RUN_FINAL_TEST_RECHECK=True to re-run the full split.")
    for mode in ["location_history", "environment_only"]:
        checkpoint_path = CHECKPOINT_DIR / f"{mode}_best_model_v19.pt"
        if not checkpoint_path.exists():
            print("No selected checkpoint for", mode, "— run the Optuna cell first.")
            continue
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if ckpt.get("optuna_study_suffix") != OPTUNA_STUDY_SUFFIX:
            print("Ignoring checkpoint from an older study/config for", mode,
                  "— current fast-search-only run has not produced a final model yet.")
            continue
        metrics = ckpt.get("test_metrics")
        if not isinstance(metrics, dict):
            print("Checkpoint has no saved full-test metrics for", mode, "; set RUN_FINAL_TEST_RECHECK=True to compute them.")
            continue
        final_rows.append({"task_mode": ckpt["task_mode"], "target_mode": ckpt["target_mode"],
                           "backbone": ckpt["temporal_backbone"], "best_trial": ckpt.get("best_trial_number"),
                           **metrics, "NMRSE_scale_1": metrics.get("nrmse"),
                           "calibration_coefficients": ckpt.get("calibration_coefficients")})
final_metrics_df = pd.DataFrame(final_rows)
display(final_metrics_df)
final_metrics_path = RUN_DIR / "final_test_metrics_model_v19.csv"
final_metrics_df.to_csv(final_metrics_path, index=False)
print("Saved final test metrics:", final_metrics_path)

## Interpretation, stability, and caveats

### Cache persistence across restarts
- The compact cache is a disk-backed set of memory-mapped `.dat` arrays plus `metadata.json` under `RUN_DIR/model_ready_compact_cache_parallel_v2`; valid caches are reused automatically when the manifest/scaler/schema fingerprint matches.
- `REBUILD_MANIFESTS=False` avoids rerunning the DuckDB manifest creation step on a normal restart. If source Parquet files or split/history settings change, update/rebuild the manifests intentionally and expect the corresponding compact cache to need rebuilding if its identity changes.
- The v11 raw active-array mirror is persistent under `RUN_DIR/zarr_active_input_mmap_v11`; it preserves source dtypes, is shared by both task modes, and is capped at 16 GiB with a 10 GiB free-space reserve. It includes only EVI, LST, VV, VH and VV/VH ratio, not HH/HV or auxiliary arrays. The mirror is built sequentially and atomically per array, then read with NumPy memmaps. It does not contain normalized patch copies. The normalized source-chunk LRU remains in worker RAM (32 MiB per worker by default) and is discarded on kernel restart. If the disk budget check fails, direct Zarr reading remains available.


### Why metrics may have jumped around
- The earlier notebook evaluated only a small, fixed prefix of the test rows (`MAX_EVAL_BATCHES`) rather than a representative full pass. With spatial/temporal ordering in Parquet, that prefix can be distributionally unrepresentative and R² can swing sharply.
- Very few training batches/epochs make epoch-level scores noisy. A large learning rate, residual target scale, or missingness can also destabilize optimization. Optuna now explores learning rate, optimizer, clipping, batch size, temporal backbone, and target mode.
- This notebook reports metrics over the full test split for the selected model by default (`FINAL_TEST_MAX_BATCHES=None`). During trials, a deterministic subset of the test split is used as a fast proxy. Set `TRIAL_MAX_TEST_BATCHES=None` for exhaustive trial scoring at substantially higher cost. The final selected model gets one full-test pass unless `FINAL_TEST_MAX_BATCHES` is capped.

### Metrics
- `rmse`, `mae`, `r2`: raw, all finite target/prediction pairs in the evaluated portion.
- `nrmse`: RMSE divided by the explicitly requested soil-moisture scale of **1 m³/m³**. Numerically it equals RMSE; this scale is much larger than the usual observed soil-moisture range, so state the denominator whenever reporting NRMSE.
- `robust_*`: diagnostic metrics after excluding predictions whose absolute error exceeds `median(|error|) + 3.5 × 1.4826 × MAD(|error|)`. This is a post-prediction, data-adaptive filter; it may remove genuine difficult cases as well as artifacts. Always report `robust_n` alongside `n`, and never report filtered R² alone.

### Split meanings
- `environment_only`: 80% of stations train and 20% test; the partitions are station-disjoint. Coordinates and station identity are not model inputs.
- `location_history`: train through 2022-12-31 and test on 2023+ data; test stations are restricted to stations represented in the training period. Coordinates are inputs. Past in-situ target values are not inputs; the temporal context is meteorological/environmental history.

### Train/test protocol
There is no validation partition. Optuna trial objectives, learning-rate scheduling, early stopping, and checkpoint selection use test R². Trial and per-epoch selection use capped deterministic test subsets; the selected checkpoint is then scored on the full test split by default. This is intentional for this experiment, but the final test score is selection-biased and should not be interpreted as an unbiased generalization estimate.

### Data integrity and runtime
- Daily target remains Option A: mean of finite linked in-situ observations per `sample_id`.
- Raster patches are read lazily by row index from the active raw-array memmap mirror, or directly from Zarr if the mirror is unavailable. Tabular and raster scaling is fitted on training rows only.
- MODIS EVI/LST are decoded using documented fill/range/scale factors. Sentinel-1 validity currently treats finite values as valid; add product-specific handling if the Zarr uses finite sentinel fills.
- Throughput depends heavily on local storage and Zarr chunk layout. Start with `NUM_WORKERS=0` if multiprocessing causes stalls or excess RAM; try 2–4 workers if storage can keep up. Monitor GPU utilization before increasing batch size.


### Python 3.14 DataLoader multiprocessing
- `GlobalSoilDataset` opens its PyArrow reader inside each worker and streams `ARROW_BATCH_ROWS` records at a time; it no longer retains a live ParquetFile on the parent dataset or materializes whole row groups as pandas frames.
- Worker exit without a Python traceback often indicates the operating system killed a worker (commonly host-RAM pressure or a native-library crash). The configured starting point is up to 3 workers, then `resource_limited_worker_cap()` reduces it according to visible CPUs and free RAM. `PREFETCH_FACTOR=1` limits each worker to one queued batch. For diagnosis, set `NUM_WORKERS = 0` and rerun from the configuration cell; do not force `fork` as a blind workaround.


### Architecture and throughput updates in this version
- **Calibration:** trainable quadratic map over SMAP anchor and predicted residual, identity-initialized and fitted only through the training objective. Do not fit calibration coefficients against test labels.
- **Center bypass:** concatenates each modality's center-pixel values and masks alongside pooled CNN embeddings.
- **Input pipeline:** worker-local PyArrow readers stream bounded Arrow batches; each worker receives disjoint Parquet row groups. Worker count is capped at 3 and DataLoader prefetch is 1 batch per worker to reduce host-memory spikes; `max_rows` is apportioned across workers.
- **GPU use:** AMP/TF32 plus a single-batch CUDA-stream prefetcher. The streaming path uses a bounded three-batch CUDA-stream prefetch queue, retaining pinned source buffers until their H2D event completes. This overlaps next-batch loading/transfer with the current forward/backward pass without a large synchronous VRAM fill.


### Filtered-data and R² protocol
- The input Parquet files come from `s1_valid_global_v1`; all-empty VV/VH patches are removed, while partially empty patches and shared patch references remain. HH/HV are excluded because the producer metadata states their remote reads are disabled.
- Optuna trial objectives and best-epoch selection use test R² directly, as requested.
- Every evaluation records `raw_r2`, `calibrated_r2`, and `calibration_r2_delta`, plus per-year R² when the active collator provides year IDs. A calibration delta below zero means the calibration layer is harming R² on that split.
- R² ≥0.80 is a goal, not guaranteed. If test R² remains low, inspect regional/year-level errors, target distribution, missingness and prediction calibration; test R² is the explicit optimization target in this experiment.


### Test R² targets
There are only two partitions. `environment_only` uses station-disjoint 80/20 train/test partitions. `location_history` trains through `TRAIN_END_DATE` and tests on all later dates for stations observed in the training period. The final retraining pass processes the whole training split each epoch and uses a deterministic subset of test R² for epoch selection, then reports metrics across the full test set.

The notebook tracks raw and calibrated R² side by side. A negative `calibration_r2_delta` means the quadratic calibration reduced R² on that evaluation split. The target test R² is 0.80 for both location_history and environment_only; these are optimization goals, not guarantees.

### GPU cache and Windows shared GPU memory
- `GPU_RASTER_CACHE_MODE = "off"` by default; it can be set to `"auto"` to cache hot normalized raster tiles in dedicated CUDA memory, using fp16 cache storage and a bounded LRU (`GPU_RASTER_CACHE_MAX_GIB`). It does not attempt a full-dataset or whole-year preload on a 4 GiB GPU. Increase the cap only if measured free VRAM and the smoke test leave safe room for model activations.
- The cache keeps `GPU_RASTER_CACHE_RESERVE_GIB` free for the model, activation tensors, optimizer, and display workloads; tiles that cannot safely stay resident are still used for the current batch but not cached. Logs report tile hit rate every `GPU_RASTER_CACHE_LOG_EVERY_BATCHES` batches. A low hit rate means GPU caching is unlikely to help much; set `GPU_RASTER_CACHE_MODE = "off"` to compare.
- Windows Task Manager's “Shared GPU memory” is system RAM the graphics driver may borrow, not extra high-bandwidth dedicated VRAM. PyTorch CUDA allocations do not automatically spill into that pool as a transparent extension of VRAM, so this notebook only caches into dedicated CUDA memory and deliberately preserves headroom.
- On Windows, the compact metadata cache build uses `ThreadPoolExecutor` and one independent Parquet reader per thread, writing disjoint row-group spans in the shared mmap arrays. The default requests up to five workers, while a 1 GiB host-memory reserve and per-worker budget can reduce that number when free RAM is low. The log now prints each cap and the cap that actually limits workers. Threads overlap I/O, but Python-heavy preprocessing can still be limited by the GIL; compare source rows/s and process CPU before increasing the cap. On POSIX with `fork`, the existing process pool is retained.

- The high-throughput collator omits per-row `sample_id` strings because no training/evaluation path consumes them; this avoids creating and carrying thousands of Python strings per batch.


### Default smoke-test and live-metric behaviour
- The multi-batch smoke test is off by default (`RUN_SMOKE_TEST=False`). Turn it on explicitly only when debugging batch shapes or gradients.
- A one-row inference preflight remains enabled by default (`RUN_PIPELINE_PREFLIGHT=True`) because it verifies the final GPU-cache-to-model path without running a backward pass. Set it to `False` only if you intentionally want to bypass even that check.
- MLflow logs only model-quality metric series: `test_r2` and `test_rmse`, plus best/final R² and RMSE summary values. It does not log GPU/CPU/VRAM/throughput diagnostics as metrics. Optuna trial details, plots, summaries, and selected checkpoints may still be stored as run parameters or artifacts. The notebook starts the local MLflow UI automatically when the tracking experiment is configured.


### Reading the R² curve in v5

The per-epoch test subset is deterministic and fixed across epochs. Therefore, a sharp rise/fall is not caused by a newly sampled test set each epoch. In the uploaded v4 run, the winning proxy trial used a learning rate near 9.4e-4 and effective batch 512 after only about 96 optimizer updates across its three short epochs; the full-data run then made roughly 4,330 optimizer updates per epoch. That mismatch can make the selected learning rate too aggressive for the long run. v5 lowers the final learning rate, raises the final effective batch to 2,048 (subject to the actual GPU probe), and evaluates EMA weights. Model selection still uses the best observed test R², as requested.

The logged `raw_test_r2` is the fusion head before the learned quadratic calibration layer; `test_r2` is the calibrated model output used for checkpoint selection. They can differ substantially, so interpret the two as separate diagnostics rather than interchangeable scores.


### Reading v9 pipeline diagnostics
- `queue_wait/b` is time the training loop was blocked because no prepared GPU batch was ready. High values point to input preparation/refill starvation.
- `collate` and `raster` are worker-side CPU times per batch. High raster time points to raster/mmap access, decoding or normalization. These are summed worker times, so they can overlap with GPU computation.
- `DataLoader next` is how long the producer spent waiting for a DataLoader batch. High values indicate workers cannot prepare batches quickly enough.
- `H2D enqueue` is CPU time spent issuing copies; sampled CUDA-copy event times estimate actual transfer time.
- `GPU fwd+bwd` is sampled CUDA event time for model forward/backward and any optimizer step in that iteration. It is device time, not Python enqueue time.
- `CPU launch` is host-side time spent in the model/loss/backward/optimizer Python path. High host time with low CUDA event time indicates launch/Python overhead.
- If queue wait is low, DataLoader next is low, and GPU compute dominates, the GPU/model is the bottleneck. Only then should we focus on architecture/compile/CUDA-graph changes.

All these timings are console diagnostics only. MLflow still logs only R² and RMSE.


### Raster-cache disk safety (v11)
- The raw active-array mirror is stored in `RUN_DIR/zarr_active_input_mmap_v11` and capped at 16 GiB. The builder also requires at least 10 GiB of free disk space to remain after a new build.
- It mirrors only MODIS EVI/LST and Sentinel-1 VV, VH and VV/VH ratio, in the original source dtype. HH/HV and auxiliary arrays are not copied.
- The raw mirror is shared between `location_history` and `environment_only`; normalization still uses each task mode's own training-only raster scalers.
- Each completed `.npy` array is atomically published and reused after restart. If the source size exceeds the cap or the free-space reserve would be violated, the notebook falls back to direct Zarr reads.
- The normalized source-chunk LRU is still RAM-only, capped at 32 MiB per worker. No expanded normalized raster-patch or batch cache is written to disk.


### Reading v11 read-amplification diagnostics
- `source_chunks` counts normalized source chunks that had to be read from the raw-array mirror (or Zarr fallback) during the epoch. Cache hits do not count as new source reads.
- `rows_read/requested` counts source array rows materialized versus source rows requested; S1 is weighted for its three active arrays. `read_amplification` near 1 means few unused chunk rows were read; high values indicate scatter and chunk over-fetch.
- The inter-batch chunk-LRU hit rate can remain low when the sorted traversal consumes each chunk once. Read amplification and raster collation time are the more useful performance indicators.

### v19.4 cache and compiled-training notes
- The four model-ready compact caches are expected to show `REUSED` on a normal restart. Train-only scaler JSONs are also persisted; they are not re-fitted (and the 256-patch raster scaler sample is not reread) unless the signature changes.
- `checkpointed_valid` describes flushed row groups in a `.building` directory. Only `READY` means a completed cache was validated and atomically promoted; the next launch should print `Compact parallel cache reused`.
- `torch.compile(mode="reduce-overhead")` is enabled for full-data retraining. Its gradient buffers are allocated before compiled backward and zeroed in-place between optimizer steps, allowing the physical microbatches to accumulate without CUDAGraph gradient-buffer aliasing. If the CUDA-graph path still fails on a backend/version combination, `max-autotune-no-cudagraphs` is attempted before eager fallback.
- cuDNN benchmark/autotuning is enabled because the image shapes are fixed. Channels-last, FP16 AMP and fused Adam/AdamW remain active.
